<a href="https://colab.research.google.com/github/globalenglish01/-/blob/main/pipeline_JapaneseAnalysisTxt2Anki2Html.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1jb7fP72YQuGmEahG0lDzI6IMkWbXOkjT


In [ ]:
from google.colab import drive


In [ ]:
drive.mount("/content/drive", force_remount=False)


ValueError: mount failed

In [ ]:
!pip install edge-tts beautifulsoup4 pandas > /dev/null


In [ ]:
!pip install pydub nest_asyncio > /dev/null


In [ ]:
!pip install genanki


In [ ]:
!pip install fasttext langid "numpy<2"
import requests
import fasttext
import langid
import time
from typing import List, Tuple
import re
from bs4 import BeautifulSoup
from datetime import datetime
import random
import os
import json
import csv
import html
import asyncio
import base64
import pandas as pd
from io import StringIO
import edge_tts
import nest_asyncio
from pydub import AudioSegment
from pydub.exceptions import CouldntDecodeError

nest_asyncio.apply()


In [ ]:
# ============================================================
# 📂 根目录和配置文件


In [ ]:
# ============================================================

ANKI_ROOT = "/content/drive/MyDrive/ChatGPTAnkiKaraOK"

# 配置
MODEL_NAME = "deepseek-chat"
DEEPSEEK_API_KEY = "sk-55021361ba944cbaaa5ef004e8fe9467"
DEEPSEEK_API_URL = "https://api.deepseek.com/v1/chat/completions"


In [ ]:
# ============================================================
# 🎯 只需要修改这里！输入文件名


In [ ]:
# ============================================================

# 在这里修改为您要处理的文件名
# INPUT_FILENAME = "Consultant02.txt"  # 👈 只需要修改这里！

#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import re
import time
import random
import uuid
import sqlite3
import csv
import html
import asyncio
import requests
from bs4 import BeautifulSoup
from typing import List


In [ ]:
# ============================================================
# 📁 配置变量（请根据实际情况修改）


In [ ]:
# ============================================================

INPUT_FILENAME = "Consultant02.txt"          # 输入文件名


In [ ]:
# ============================================================
# 📂 自动生成相关文件名


In [ ]:
# ============================================================
sentences_file = os.path.join(ANKI_ROOT, INPUT_FILENAME)
base_name = os.path.splitext(INPUT_FILENAME)[0]

output_file = os.path.join(ANKI_ROOT, f"{base_name}_解析结果.txt")
checkpoint_file = os.path.join(ANKI_ROOT, f"{base_name}_检查点.txt")
log_file = os.path.join(ANKI_ROOT, f"{base_name}_日志.txt")
anki_export_file = os.path.join(ANKI_ROOT, f"{base_name}_anki导出.txt")

print("=" * 80)
print("🚀 配置信息")
print("=" * 80)
print(f"📁 输入文件: {sentences_file}")
print(f"📄 解析结果文件: {output_file}")
print(f"📍 检查点文件: {checkpoint_file}")
print(f"📝 日志文件: {log_file}")
print(f"🃏 Anki导出文件: {anki_export_file}")
print("=" * 80)


In [ ]:
# ============================================================
# 🗣 Edge TTS 配置（保留，但未使用）


In [ ]:
# ============================================================

CHINESE_VOICES = [
    "zh-CN-XiaoxiaoNeural",
    "zh-CN-YunxiNeural",
    "zh-CN-YunjianNeural",
    "zh-CN-XiaoyiNeural",
    "zh-CN-YunyangNeural",
]

JAPANESE_VOICES = [
    "ja-JP-NanamiNeural",
    "ja-JP-KeitaNeural",
]

ENGLISH_VOICES = [
    "en-US-JennyNeural",
    "en-US-GuyNeural",
    "en-GB-SoniaNeural",
    "en-AU-NatashaNeural",
]


In [ ]:
# ============================================================
# 📝 系统提示（核心教学模式）


In [ ]:
# ============================================================

CORE_SENTENCE_PROMPT = """
【核心角色设定】
顶级日语解析教练
精通：
日语初学者教学路径
日语语法体系
日语口语、省略、缩略、惯用语
片假名外来语来源与日式语义变化
同时你也是：
语言数据工程师
TTS 数据清洗工程师
对话分析工程师
学习系统调度工程师
考试与面试出题官
输出协议校验器

【批量处理模式指令】
MODE = 教学
BATCH_MODE = ON
INTERRUPT = OFF
NO_PROMPT = ON

【指令说明】
1. 用户每次提供一个日语句子进行解析
2. 只解析当前提供的句子
3. 输出完整的A到M模块，严格按照语言纯度规则
4. 绝对不要在输出末尾添加"本句解析完成"、"请回复下一句"等交互提示语
5. 输出完成后直接结束，等待下一个请求

【我的当前水平】
只学完日语五十音
不熟悉汉字
不熟悉语法
不熟悉口语省略
不熟悉外来语
不熟悉惯用表达

【模式切换指令（强制）】
MODE = 教学 | 考试 | 面试 | 实战对话
未给 MODE：
默认：教学

【一次性列出所有 MODE（强制）】
输出最前部必须列出：
当前模式 教学
可用模式 教学
可用模式 考试
可用模式 面试
可用模式 实战对话

通用硬性规则（语言纯度 V2.4）

R1 严格分语言分行
每一行只能包含一种语言
行首必须标注语言类型
[jp] 只允许日文
[ch] 只允许中文
[en] 只允许英文
禁止一行中出现多语言字符
禁止用括号或箭头做跨语言说明

R2 标签行 / 内容行结构锁死
凡涉及以下元素：
原型
当前形
假名
句型
助词
词性
外来语
活用形
口语形
缩略形
必须使用二行结构：
[ch] 标签
[目标语言] 内容

R3 词性语言锁死规则
凡是：
词性
语法类别
功能标签
变化说明
含义说明
对应内容行：
必须使用 [ch]
严禁在 [jp] 行输出：
动词
形容词
助词
句型
进行体
省略
口语
缩略
原型
当前形

R4 助词 / 句型强制结构化规则
凡涉及：
助词
句型
活用形
原型
口语变化
必须使用 4 行结构：
[ch] 标签
[jp] 语言实体
[ch] 含义
[ch] 中文说明

R5 变化路径禁止混行规则
禁止输出：
[ch] ている 省略为 てる
必须拆为：
[ch] 原始形
[jp] ている
[ch] 当前形
[jp] てる
[ch] 变化说明
[ch] 进行体的口语缩略

R6 解释行严禁日文规则
凡是标注为：
[ch] 含义
[ch] 说明
[ch] 用法
[ch] 功能
[ch] 变化说明
对应内容行：
严禁出现任何日文字符

R7 外来语语义变化强制规则
凡是：
[ch] 是否发生日式语义变化
[ch] 是
必须追加：
[ch] 英文原词语义
[en] 英文原始含义中文解释

R8 自检问题语言纯度规则
自检问题必须拆为：
[jp] 具体日文词
[ch] 中文问题内容
严禁中日混行

R9 引导与指令语言标注规则
凡是：
引导语
操作提示
系统性说明
回合控制语
推进提示语
一律：
不加任何语言类型标注

R10 Edge TTS 兼容优先
[jp] 行不得包含任何中文或英文
禁止一行内含括号或箭头
禁止混排注音

违规修复引擎 V2.4（强制）

V1 违规扫描
逐行扫描：
MIX_CH_JP
MIX_CH_EN
MIX_JP_EN
LABEL_LANG_MISMATCH
TAG_LINE_HAS_FOREIGN
CONTENT_LINE_HAS_CHINESE
PARTICLE_LINE_HAS_CHINESE
FORM_LINE_HAS_CHINESE
SELFTEST_MIXED_LANG
EN_SEMANTIC_MISSING
INSTRUCTION_HAS_LANG_TAG

V2 违规分类
为每条违规记录类型

V3 自动回滚（强制）
一旦发现任意违规：
丢弃当前输出
重新生成当前句 A → M
不允许精简
不允许合并行
不允许压缩结构

V4 修复确认
重写完成后输出末尾：
违规修复引擎已执行
未检测到语言混杂
当前输出已通过纯度校验

V5 自检失败保护
同一回合连续两次违规：
强制进入最小可用输出模式
仅输出：
当前句
A 模块
B 模块
并标记：
纯度降级输出模式已触发
请回复 重来 以恢复完整解析

MODE = 教学

0）自动拆句 + 角色 + 语气

1）逐句回合制
一次只解析一句

2）逐句完整解析模块（A → M）

【A 原句与翻译三层】
[jp] 原始日语句子
[ch] 逐词直译
[ch] 自然中文
[ch] 可脱口而出的中文口语表达

【B 完整读音标注】
[ch] 整句假名
[jp]

[ch] 汉字对应假名
[jp]

[ch] 分词假名
[jp]

【C 逐词拆解】
[ch] 假名
[jp]

[ch] 词性
[ch]

[ch] 原型
[jp]

[ch] 当前活用形
[jp]

[ch] 中文含义
[ch]

【D 语法点讲解】
所有助词
所有句型
所有难点
使用 R2 / R4 结构化输出

【E 口语、省略、缩略变化路径】
使用 R5 四层结构

【F 片假名外来语】
[ch] 日语词
[jp]

[ch] 中文含义
[ch]

[ch] 英文原词
[en]

[ch] 是否发生日式语义变化
[ch]

若为 是 必须追加：
[ch] 英文原词中文语义
[ch]

【G 缩略语、年轻人口语、惯用语】
[ch] 完整原型
[jp]

[ch] 缩略路径
[ch]

[ch] 使用语境
[ch]

[ch] 中文自然说法
[ch]

【H 语言纯度规则】
一行只允许一种语言
禁止跨语言混排

【I 同句型跟读训练 五句】
【I1 日语句子（每句真实输出三遍）】
[jp]
[jp]
[jp]

[ch] 中文对应

【J 自检问题（必须具体实例化）】
[jp] 具体日文词
[ch] 原型是什么

[jp] 具体口语变化
[ch] 来自哪里

[jp] 具体句型
[ch] 在这里表达什么语气

[jp] 具体名词
[ch] 换成这个主语该怎么说

【K Shadowing 分段朗读稿（每块三遍）】
[jp]
[jp]
[jp]

【L 跟读节奏版（每块三遍）】
[jp]
[jp]
[jp]

【M 跟读提示】
先听完整句
再逐块跟读
最后整句连读
只看日文

【批处理模式特别规则】
1. 不输出任何交互提示语（如"本句解析完成"等）
2. 每个句子的解析之间自动添加分隔线（"---"）
3. 严格遵守所有语言纯度规则
4. 确保输出可直接用于TTS系统
"""

TEACHING_PROMPT = CORE_SENTENCE_PROMPT

EXAM_PROMPT = """
你是JLPT N1出题官。

任务：基于输入句子生成N1难度题目。

【必须生成】
1. 词汇题（4选1）
2. 语法题（4选1）
3. 改错题（指出错误并修改）

【输出格式要求】
严格按照以下格式输出，每行必须使用语言标识符：
- [ch] 用于中文说明、标签、题目题干、选项文字（如果是中文）、解析等。
- [jp] 用于日文例句、选项（如果是日文）、错误句子等。
- 禁止使用Markdown语法（如##、**），如需强调请使用HTML标签如<b>...</b>。
- 每行只能包含一种语言，严禁中日混排。
- 每个题目之间用空行分隔。

【具体格式示例】
[ch] 1. 词汇题（4选1）
[ch] 题干：以下哪个选项最适合填入句子「＿＿＿」中？
[jp] 彼は＿＿＿が上手だ。
[jp] A. 話す
[jp] B. 話し
[jp] C. 話して
[jp] D. 話そう
[ch] 正确答案：B
[ch] 解析：「話す」的连用形「話し」可以作名词，表示“说话的方式”。

[ch] 2. 语法题（4选1）
[ch] 题干：...
[jp] ...
[jp] A. ...
[jp] B. ...
[jp] C. ...
[jp] D. ...
[ch] 正确答案：...
[ch] 解析：...

[ch] 3. 改错题
[jp] 错误句子：＿＿＿
[ch] 错误：＿＿＿
[ch] 修改：＿＿＿
[ch] 解析：＿＿＿

输入句子：
{sentence}
"""
INTERVIEW_PROMPT = """
你是日语面试表达专家。

任务：针对输入句子，生成不同礼貌程度的表达。

【必须生成】
1. 普通表达
2. 礼貌表达
3. 高级表达
4. 使用场景说明

【输出格式要求】
严格按照以下格式输出，每行使用语言标识符：
- [ch] 用于中文说明、标签、场景说明等。
- [jp] 用于日文表达句子。
- 禁止使用Markdown语法，如需强调请用HTML标签<b>...</b>。
- 每行只能包含一种语言。

【具体格式示例】
[ch] 1. 普通表达
[jp] 句子内容
[ch] 2. 礼貌表达
[jp] 句子内容
[ch] 3. 高级表达
[jp] 句子内容
[ch] 4. 使用场景说明
[ch] 说明文字

输入句子：
{sentence}
"""

DIALOG_PROMPT = """
生成2轮自然日语对话。

要求：
- 口语
- 自然
- 每句附中文

【输出格式要求】
严格按照以下格式输出，每行使用语言标识符：
- [ch] 用于中文翻译、说明。
- [jp] 用于日文对话内容。
- 对话轮次按顺序编号，用[ch]标注。
- 禁止使用Markdown语法，如需强调请用HTML标签<b>...</b>。
- 每行只能包含一种语言。

【具体格式示例】
[ch] 第1轮
[jp] 日文句子1
[ch] 中文翻译1
[jp] 日文句子2
[ch] 中文翻译2

[ch] 第2轮
[jp] 日文句子3
[ch] 中文翻译3
[jp] 日文句子4
[ch] 中文翻译4

输入句子：
{sentence}
"""


In [ ]:
# ============================================================
# 📦 工具函数定义


In [ ]:
# ============================================================

def analyze_one_sentence_with_retry(sentence, max_retries=5, base_delay=2, max_delay=60):
    """
    带有重试机制的API调用函数（用于教学模式）
    """
    print(f"🔍 调试信息:")
    print(f"   句子内容: {sentence[:50]}...")
    print(f"   句子长度: {len(sentence)} 字符")
    print(f"   系统提示长度: {len(CORE_SENTENCE_PROMPT)} 字符")

    if not CORE_SENTENCE_PROMPT or len(CORE_SENTENCE_PROMPT.strip()) == 0:
        print("❌ 错误: CORE_SENTENCE_PROMPT 未定义或为空！")
        raise Exception("CORE_SENTENCE_PROMPT 未定义")

    payload = {
        "model": MODEL_NAME,
        "messages": [
            {"role": "system", "content": CORE_SENTENCE_PROMPT},
            {"role": "user", "content": sentence}
        ],
        "temperature": 0,
        "max_tokens": 8192
    }

    headers = {
        "Authorization": f"Bearer {DEEPSEEK_API_KEY}",
        "Content-Type": "application/json"
    }

    last_exception = None
    last_status_code = None

    for attempt in range(max_retries):
        try:
            if attempt > 0:
                delay = min(base_delay * (2 ** (attempt - 1)) + random.uniform(0, 1), max_delay)
                print(f"🔄 第{attempt}次重试，等待{delay:.1f}秒后重试...")
                time.sleep(delay)

            print(f"📡 尝试请求API (尝试 {attempt + 1}/{max_retries})...")
            print(f"   请求payload大小: {len(str(payload))} 字符")

            start_time = time.time()
            response = requests.post(DEEPSEEK_API_URL, json=payload, headers=headers, timeout=60)
            request_time = time.time() - start_time

            last_status_code = response.status_code

            print(f"   响应时间: {request_time:.2f}秒")
            print(f"   状态码: {response.status_code}")

            if response.status_code == 200:
                try:
                    response_json = response.json()
                    print(f"   响应JSON键: {list(response_json.keys())}")

                    if "choices" in response_json and len(response_json["choices"]) > 0:
                        text = response_json["choices"][0]["message"]["content"].strip()
                        print(f"✅ 第{attempt + 1}次尝试成功")
                        print(f"   返回文本长度: {len(text)} 字符")
                        print(f"   返回文本前100字符: {text[:100]}...")
                        return text
                    else:
                        print(f"❌ 响应中没有choices字段: {response_json}")
                        raise Exception("响应中没有choices字段")

                except Exception as parse_error:
                    print(f"❌ 解析响应JSON失败: {str(parse_error)}")
                    print(f"   原始响应前500字符: {response.text[:500]}")
                    raise parse_error

            elif response.status_code == 400:
                print(f"❌ 400错误 - 请求格式可能有误")
                if response.text:
                    print(f"   错误详情: {response.text[:500]}")
                raise Exception("400 Bad Request - 检查请求格式")

            elif response.status_code == 429:
                print(f"⚠️ 请求过多429错误 (尝试 {attempt + 1}/{max_retries})")
                if attempt < max_retries - 1:
                    wait_time = min(30 * (attempt + 1), 120)
                    print(f"⏳ 速率限制，等待{wait_time}秒...")
                    time.sleep(wait_time)
                    continue
                else:
                    raise Exception("达到最大重试次数，仍然被限速")

            elif response.status_code == 503:
                print(f"⚠️ 服务器503错误 (尝试 {attempt + 1}/{max_retries})")
                if attempt < max_retries - 1:
                    continue
                else:
                    raise Exception("达到最大重试次数，服务器仍然503")

            else:
                print(f"❌ 收到 {response.status_code} 状态码")
                if response.text:
                    print(f"   错误响应: {response.text[:500]}")
                response.raise_for_status()

        except requests.exceptions.Timeout:
            print(f"⏱️ 请求超时 (尝试 {attempt + 1}/{max_retries})")
            last_exception = "请求超时"
            if attempt < max_retries - 1:
                continue

        except requests.exceptions.ConnectionError:
            print(f"🔌 连接错误 (尝试 {attempt + 1}/{max_retries})")
            last_exception = "连接错误"
            if attempt < max_retries - 1:
                time.sleep(5)
                continue

        except Exception as e:
            print(f"❌ 其他错误: {str(e)} (尝试 {attempt + 1}/{max_retries})")
            last_exception = str(e)
            if attempt < max_retries - 1:
                continue

    error_msg = f"❌ 所有{max_retries}次尝试都失败"
    if last_status_code:
        error_msg += f"，最后状态码: {last_status_code}"
    if last_exception:
        error_msg += f"，最后错误: {last_exception}"
    raise Exception(error_msg)

def call_llm_safe(prompt_text, system_prompt=None, max_retries=3):
    for attempt in range(max_retries):
        try:
            messages = []
            if system_prompt:
                messages.append({"role": "system", "content": system_prompt})
            messages.append({"role": "user", "content": prompt_text})

            payload = {
                "model": MODEL_NAME,
                "messages": messages,
                "temperature": 0.7,
                "max_tokens": 4096
            }
            headers = {
                "Authorization": f"Bearer {DEEPSEEK_API_KEY}",
                "Content-Type": "application/json"
            }
            response = requests.post(DEEPSEEK_API_URL, json=payload, headers=headers, timeout=60)
            response.raise_for_status()
            text = response.json()["choices"][0]["message"]["content"].strip()

            # 打印返回内容的前200字符，帮助调试
            print(f"📄 返回内容预览: {text[:200]}...")

            if validate_output(text):
                return text
            print(f"⚠️ 输出验证失败（长度不足或含省略），重试 {attempt+1}")
        except Exception as e:
            print(f"❌ API调用失败: {e}")
            if attempt == max_retries-1:
                raise
            time.sleep(2 ** attempt)
    raise Exception("多次重试后仍然失败")

def validate_output(text):
    """更宽松的验证函数"""
    if not text or len(text.strip()) == 0:
        return False
    # 只拦截明显无效的内容（完全由省略符组成）
    if text.strip() in ["略", "同上", "省略", "暂无"]:
        return False
    # 避免过短但可能有效的内容被拒绝（例如简短的对话或考试答案）
    # 移除长度硬性检查，只保留基本的非空判断
    return True

def merge_multi_mode(teaching, exam, interview, dialog):
    """合并四种模式输出"""
    return f"""
########## MULTI MODE OUTPUT ##########

===== 教学模式 =====
{teaching}

===== 考试模式 =====
{exam}

===== 面试模式 =====
{interview}

===== 实战对话 =====
{dialog}

######################################
"""

def analyze_multi_mode(sentence):
    """生成四种模式的内容并合并"""
    # 教学模式：使用系统提示 + 用户消息
    teaching = call_llm_safe(
        prompt_text=sentence,
        system_prompt=CORE_SENTENCE_PROMPT,
        max_retries=3
    )

    # 考试模式
    exam_prompt = EXAM_PROMPT.format(sentence=sentence)
    exam = call_llm_safe(exam_prompt, max_retries=3)

    # 面试模式
    interview_prompt = INTERVIEW_PROMPT.format(sentence=sentence)
    interview = call_llm_safe(interview_prompt, max_retries=3)

    # 实战对话模式
    dialog_prompt = DIALOG_PROMPT.format(sentence=sentence)
    dialog = call_llm_safe(dialog_prompt, max_retries=3)

    return merge_multi_mode(teaching, exam, interview, dialog)

def split_japanese_sentences(text):
    """将日语文本按句子拆分"""
    pattern = r'[^。！？]*[。！？]'
    sentences = re.findall(pattern, text)
    sentences = [s.strip() for s in sentences if s.strip()]
    return sentences

def load_sentences_from_file(file_path: str) -> List[str]:
    """从文件加载句子列表"""
    encodings = [
        "utf-8",
        "utf-16",
        "utf-16-le",
        "utf-16-be",
        "cp932",
        "shift_jis",
        "euc-jp",
        "latin1"
    ]

    raw = None
    used_encoding = None

    for enc in encodings:
        try:
            with open(file_path, "r", encoding=enc) as f:
                raw = f.read()
            if enc == "latin1" and "\x00" in raw:
                continue
            used_encoding = enc
            print(f"✅ 文件编码识别成功: {enc}")
            break
        except UnicodeDecodeError:
            continue

    if raw is None:
        raise ValueError("❌ 无法识别文件编码，请确认文件格式")

    soup = BeautifulSoup(raw, "lxml")
    for tag in soup(["script", "style"]):
        tag.decompose()

    text = soup.get_text(separator="\n")
    sentences = split_japanese_sentences(text)

    print(f"📖 共提取句子数: {len(sentences)}")
    return sentences

def load_checkpoint(checkpoint_file: str) -> int:
    """加载检查点，返回下一个要处理的句子编号（1-based）"""
    try:
        with open(checkpoint_file, 'r', encoding='utf-8') as f:
            checkpoint = f.read().strip()
            if checkpoint.isdigit():
                return int(checkpoint)
    except FileNotFoundError:
        pass
    return 1

def check_parsing_progress(checkpoint_file: str, total_sentences: int) -> tuple:
    """检查解析进度"""
    if not os.path.exists(checkpoint_file):
        return False, 0, 0

    try:
        with open(checkpoint_file, 'r', encoding='utf-8') as f:
            checkpoint = f.read().strip()
            if checkpoint.isdigit():
                next_to_process = int(checkpoint)
                processed = next_to_process - 1
                if processed >= total_sentences:
                    return True, total_sentences, total_sentences
                else:
                    return False, processed, processed
    except:
        pass
    return False, 0, 0

def count_parsed_sentences_in_file(output_file: str) -> int:
    """统计解析结果文件中已解析的句子数量"""
    if not os.path.exists(output_file):
        return 0

    try:
        with open(output_file, 'r', encoding='utf-8') as f:
            content = f.read()
            count = len(re.findall(r'当前模式 教学', content))
            return count
    except:
        return 0

def resume_processing_sync(
    sentences: List[str],
    start_index: int,
    output_file: str,
    checkpoint_file: str,
    log_file: str = None,
    max_retries_per_sentence: int = 5
):
    """同步版本的处理函数，使用多模式生成"""
    total = len(sentences)
    print(f"📊 总共 {total} 个句子，从第 {start_index + 1} 句开始")
    print(f"📁 输出到: {output_file}")
    print(f"📍 检查点: {checkpoint_file}")
    print(f"🔄 每句最大重试次数: {max_retries_per_sentence}")

    output_handle = open(output_file, 'a', encoding='utf-8')
    log_handle = open(log_file, 'a', encoding='utf-8') if log_file else None

    success_count = 0

    try:
        for i in range(start_index, total):
            sentence = sentences[i]
            current_num = i + 1

            print(f"\n{'='*60}")
            print(f"🔹 句 {current_num} ｜ {sentence[:50]}...")
            print(f"进度: {current_num}/{total} ({current_num/total*100:.1f}%)")

            if log_handle:
                log_handle.write(f"🔹 句 {current_num} ｜ {sentence}\n")
                log_handle.flush()

            start_time = time.time()

            try:
                # 调用多模式生成
                result = analyze_multi_mode(sentence)
                elapsed_time = time.time() - start_time

                success_count += 1
                print(f"✅ 成功！耗时: {elapsed_time:.1f}秒，成功数: {success_count}")

                output_handle.write(result + "\n\n" + "="*80 + "\n\n")
                output_handle.flush()

                if log_handle:
                    log_handle.write(f"{result}\n\n")
                    log_handle.flush()

            except Exception as e:
                print(f"❌ 处理句子失败: {str(e)}")
                error_result = f"[处理失败: {str(e)}]"
                output_handle.write(error_result + "\n\n" + "="*80 + "\n\n")
                output_handle.flush()

                if log_handle:
                    log_handle.write(f"{error_result}\n\n")
                    log_handle.flush()

                print("⚠️ 跳过此句，继续处理下一句...")

            # 保存检查点（下一个要处理的句子编号）
            with open(checkpoint_file, 'w', encoding='utf-8') as cp:
                cp.write(str(current_num + 1))

            # 避免请求过快
            time.sleep(1 + random.uniform(0, 0.5))

    except KeyboardInterrupt:
        print("\n\n⚠️ 用户中断处理")
        current_position = i if 'i' in locals() else start_index
        print(f"下次可以从第 {current_position + 1} 句继续")
        if 'i' in locals():
            with open(checkpoint_file, 'w', encoding='utf-8') as cp:
                cp.write(str(current_position + 1))
    finally:
        output_handle.close()
        if log_handle:
            log_handle.close()

        print(f"\n🎉 处理完成！")
        print(f"成功处理: {success_count}/{total - start_index} 句")
        print(f"结果保存在: {output_file}")
        print(f"检查点文件: {checkpoint_file}")

def parse_by_complete_structure(input_file_path: str) -> List[tuple]:
    """基于完整解析结构拆分句子块"""
    print(f"🔍 基于完整结构解析文件: {input_file_path}")

    with open(input_file_path, 'r', encoding='utf-8') as f:
        content = f.read()

    modules = [
        "【A 原句与翻译三层】",
        "【B 完整读音标注】",
        "【C 逐词拆解】",
        "【D 语法点讲解】",
        "【E 口语、省略、缩略变化路径】",
        "【F 片假名外来语】",
        "【G 缩略语、年轻人口语、惯用语】",
        "【H 语言纯度规则】",
        "【I 同句型跟读训练 五句】",
        "【J 自检问题（必须具体实例化）】",
        "【K Shadowing 分段朗读稿（每块三遍）】",
        "【L 跟读节奏版（每块三遍）】",
        "【M 跟读提示】"
    ]

    sentence_starts = [m.start() for m in re.finditer(r'当前模式 教学', content)]
    print(f"📊 找到 {len(sentence_starts)} 个句子开始位置")

    raw_blocks = []

    for i, start_pos in enumerate(sentence_starts):
        end_pos = sentence_starts[i + 1] if i + 1 < len(sentence_starts) else len(content)
        block_content = content[start_pos:end_pos].strip()

        module_count = sum(1 for m in modules if m in block_content)
        if module_count >= 3:
            raw_blocks.append(block_content)

    print(f"✅ 结构完整的解析块数量: {len(raw_blocks)}")

    # 去重
    unique_blocks = []
    seen_signatures = set()
    for block in raw_blocks:
        signature = block[:300]
        if signature not in seen_signatures:
            seen_signatures.add(signature)
            unique_blocks.append(block)

    print(f"📊 去重后唯一句子数: {len(unique_blocks)}")

    renumbered_blocks = [(i + 1, block) for i, block in enumerate(unique_blocks)]
    return renumbered_blocks

import re

def convert_to_anki_html_format(content: str) -> str:
    """将解析内容转换为HTML格式，空行替换为<hr>"""
    escaped_content = html.escape(content)
    # 先将连续2个及以上换行替换为 <hr>（保留一个<br>用于占位）
    # 注意：需要保留换行符标记，避免直接删除
    # 方案：将 \n\n+ 替换为 <hr>，再将单个 \n 替换为 <br>
    escaped_content = re.sub(r'\n\s*\n', '<hr>', escaped_content)   # 空行 → <hr>
    html_content = escaped_content.replace('\n', '<br>')
    html_content = html_content.replace('  ', ' &nbsp;')
    # 避免出现 <hr><br> 这种多余标记，清理一下
    html_content = html_content.replace('<hr><br>', '<hr>')
    return html_content

import genanki

def save_as_apkg(anki_notes: List[List[str]], output_path: str, deck_name: str = "日语解析"):
    """将Anki笔记列表保存为.apkg文件"""
    # 定义模型：正面为解析内容，反面可以为空或重复
    model_id = random.randrange(1 << 30, 1 << 31)
    model = genanki.Model(
        model_id,
        '日语解析模型',
        fields=[
            {'name': 'Content'},      # 解析内容
        ],
        templates=[
            {
                'name': 'Card 1',
                'qfmt': '{{Content}}',   # 正面显示解析内容
                'afmt': '{{FrontSide}}<hr id="answer">',  # 反面默认（可自定义）
            },
        ]
    )

    deck_id = random.randrange(1 << 30, 1 << 31)
    deck = genanki.Deck(deck_id, deck_name)

    for note_fields in anki_notes:
        # note_fields 现在是 [html_content] （单字段）
        # genanki.Note 需要字段列表，即使只有一个字段也要放在列表中
        note = genanki.Note(model=model, fields=[note_fields[0]])
        deck.add_note(note)

    # 导出为.apkg文件
    genanki.Package(deck).write_to_file(output_path)
    print(f"✅ 已生成 .apkg 文件: {output_path}")

def convert_to_anki_note_format(blocks: List[tuple]) -> List[List[str]]:
    """将解析结果转换为Anki Note导出的TXT格式"""
    anki_notes = []
    for num, content in blocks:
        html_content = convert_to_anki_html_format(content)
        note_fields = [html_content, "日语解析"]
        anki_notes.append(note_fields)
    return anki_notes

def save_as_anki_format_without_tags(anki_notes: List[List[str]], output_file: str):
    """保存为Anki格式（只有内容，没有标签）"""
    with open(output_file, 'w', encoding='utf-8-sig', newline='\r\n') as f:
        writer = csv.writer(f, delimiter='\t', quoting=csv.QUOTE_MINIMAL, lineterminator='\r\n')
        for note_fields in anki_notes:
            writer.writerow([note_fields[0]])
    print(f"✅ 已保存为Anki格式（无标签）: {output_file}")

def verify_anki_format(output_file: str):
    """验证文件格式"""
    try:
        with open(output_file, 'r', encoding='utf-8-sig') as f:
            first_line = f.readline().rstrip('\r\n')
            if '\t' in first_line:
                fields = first_line.split('\t')
                print(f"\n🔍 Anki格式验证:")
                print(f"   ✅ 文件编码: UTF-8 with BOM")
                print(f"   ✅ 分隔符: 制表符 (Tab)")
                print(f"   ✅ 字段数: {len(fields)}")
                if '<br>' in fields[0]:
                    print(f"   ✅ 内容格式: HTML (包含<br>标签)")
                else:
                    print(f"   ⚠️ 内容可能不是HTML格式")
                return True
            else:
                print(f"\n❌ 格式验证失败: 未找到制表符分隔符")
                return False
    except Exception as e:
        print(f"\n❌ 验证文件时出错: {e}")
        return False


In [ ]:
# ============================================================
# 🚀 主函数


In [ ]:
# ============================================================

async def main():
    print("=" * 80)
    print("🔧 日语解析处理系统（多模式）")
    print("=" * 80)
    print(f"📁 正在处理文件: {INPUT_FILENAME}")
    print("=" * 80)

    if not os.path.exists(sentences_file):
        print(f"❌ 错误：文件不存在！")
        print(f"   请检查路径: {sentences_file}")
        return

    print("\n📖 加载句子列表...")
    sentences = load_sentences_from_file(sentences_file)
    total_sentences = len(sentences)
    print(f"✅ 总句子数: {total_sentences}")

    print("\n" + "=" * 60)
    print("📊 检查解析进度")
    print("=" * 60)

    is_complete, next_index, processed_count = check_parsing_progress(checkpoint_file, total_sentences)
    actual_parsed = count_parsed_sentences_in_file(output_file)

    print(f"   检查点记录: 已处理 {processed_count}/{total_sentences} 句")
    print(f"   结果文件中: 实际有 {actual_parsed}/{total_sentences} 句")

    actual_processed = max(processed_count, actual_parsed)

    if actual_processed >= total_sentences:
        print(f"\n✅ 所有 {total_sentences} 句都已解析完成！")
        print(f"   跳过解析步骤，直接进行下一步...")
    else:
        print(f"\n📊 需要继续解析: 已完成 {actual_processed}/{total_sentences} 句")
        print(f"   还有 {total_sentences - actual_processed} 句待处理")

        start_index = actual_processed

        if start_index < total_sentences:
            next_sentence = sentences[start_index]
            print(f"\n📌 下一句预览: {next_sentence[:80]}...")

            if actual_processed > 0:
                confirm = input(f"从第 {actual_processed + 1} 句继续处理？(y/n，直接回车继续): ")
                if confirm.lower() == 'n':
                    start_index = 0
                    print("从第1句开始重新处理...")

        print("\n🔍 运行API测试...")
        test_sentence = "これはテスト文です。"
        try:
            result = analyze_one_sentence_with_retry(test_sentence, max_retries=1)
            print(f"✅ API测试成功！")
        except Exception as e:
            print(f"⚠️ API测试失败: {e}")
            confirm = input("API测试失败，是否继续处理？(y/n): ")
            if confirm.lower() != 'y':
                print("❌ 处理终止")
                return

        print("\n" + "=" * 60)
        print(f"🔄 开始处理剩余 {total_sentences - start_index} 句...")
        print("=" * 60)

        try:
            resume_processing_sync(
                sentences=sentences,
                start_index=start_index,
                output_file=output_file,
                checkpoint_file=checkpoint_file,
                log_file=log_file,
                max_retries_per_sentence=5
            )
            print("\n" + "=" * 60)
            print("✅ 解析处理完成！")
            print("=" * 60)
            print(f"📄 解析结果: {output_file}")
            print(f"📍 检查点文件: {checkpoint_file}")
            print(f"📝 日志文件: {log_file}")
        except Exception as e:
            print(f"\n❌ 处理过程中出错: {e}")
            import traceback
            traceback.print_exc()
            return

    if not os.path.exists(output_file):
        print(f"\n❌ 解析结果文件不存在，请检查: {output_file}")
        return

    final_parsed = count_parsed_sentences_in_file(output_file)
    if final_parsed < total_sentences:
        print(f"\n⚠️ 警告：解析结果文件只有 {final_parsed}/{total_sentences} 句")
        print(f"   请检查处理过程是否完整")
        confirm = input("是否继续生成Anki导出文件？(y/n，直接回车继续): ")
        if confirm.lower() == 'n':
            return

    print("\n" + "=" * 60)
    print("🃏 正在生成Anki导出文件...")
    print("=" * 60)

    if os.path.exists(anki_export_file):
        print(f"✅ Anki导出文件已存在: {anki_export_file}")
        regenerate = input("是否重新生成？(y/n，直接回车跳过): ")
        if regenerate.lower() == 'y':
            blocks = parse_by_complete_structure(output_file)
            anki_notes = convert_to_anki_note_format(blocks)
            # 原来：
            # save_as_anki_format_without_tags(anki_notes, anki_export_file)

            # 改为：
            apkg_file = anki_export_file.replace('.txt', '.apkg')  # 或直接指定路径
            save_as_apkg(anki_notes, apkg_file, deck_name="日语解析")
            verify_anki_format(anki_export_file)
            print(f"\n✅ Anki导出文件已重新生成: {anki_export_file}")
        else:
            print("⏭ 跳过生成，使用现有文件")
    else:
        print(f"❌ Anki导出文件不存在，开始生成...")
        blocks = parse_by_complete_structure(output_file)
        anki_notes = convert_to_anki_note_format(blocks)
        # 原来：
        # save_as_anki_format_without_tags(anki_notes, anki_export_file)

        # 改为：
        apkg_file = anki_export_file.replace('.txt', '.apkg')  # 或直接指定路径
        save_as_apkg(anki_notes, apkg_file, deck_name="日语解析")

        verify_anki_format(anki_export_file)
        print(f"\n✅ Anki导出文件已生成: {anki_export_file}")

    print("\n" + "=" * 80)
    print("🎉 所有处理完成！")
    print("=" * 80)
    print(f"📁 输入文件: {sentences_file}")
    print(f"📄 解析结果: {output_file} ({final_parsed}/{total_sentences} 句)")
    print(f"🃏 Anki导出: {anki_export_file}")
    print("=" * 80)

    if os.path.exists(anki_export_file):
        print("\n📝 Anki导出文件预览（前3行）:")
        print("-" * 60)
        with open(anki_export_file, 'r', encoding='utf-8-sig') as f:
            for i, line in enumerate(f):
                if i >= 3:
                    break
                preview = line[:100] + "..." if len(line) > 100 else line
                print(f"行 {i+1}: {preview}")
        print("-" * 60)

    print("\n✅ 处理完成！")

# 【批量处理模式指令】

# 【指令说明】

# 【我的当前水平】
# 只学完日语五十音
# 不熟悉汉字
# 不熟悉语法
# 不熟悉口语省略
# 不熟悉外来语
# 不熟悉惯用表达

# 【模式切换指令（强制）】

# 【一次性列出所有 MODE（强制）】
# 输出最前部必须列出：
# 当前模式 教学
# 可用模式 教学
# 可用模式 考试
# 可用模式 面试
# 可用模式 实战对话

# 通用硬性规则（语言纯度 V2.4）

# 违规修复引擎 V2.4（强制）

# V2 违规分类
# 为每条违规记录类型

# MODE = 教学

# 0）自动拆句 + 角色 + 语气

# 1）逐句回合制
# 一次只解析一句

# 2）逐句完整解析模块（A → M）

# 【A 原句与翻译三层】

# 【B 完整读音标注】
# [ch] 整句假名
# [jp]

# [ch] 汉字对应假名
# [jp]

# [ch] 分词假名
# [jp]

# 【C 逐词拆解】
# [ch] 假名
# [jp]

# [ch] 词性
# [ch]

# [ch] 原型
# [jp]

# [ch] 当前活用形
# [jp]

# [ch] 中文含义
# [ch]

# 【D 语法点讲解】
# 所有助词
# 所有句型
# 所有难点
# 使用 R2 / R4 结构化输出

# 【E 口语、省略、缩略变化路径】
# 使用 R5 四层结构

# 【F 片假名外来语】
# [ch] 日语词
# [jp]

# [ch] 中文含义
# [ch]

# [ch] 英文原词
# [en]

# [ch] 是否发生日式语义变化
# [ch]

# 若为 是 必须追加：
# [ch] 英文原词中文语义
# [ch]

# 【G 缩略语、年轻人口语、惯用语】
# [ch] 完整原型
# [jp]

# [ch] 缩略路径
# [ch]

# [ch] 使用语境
# [ch]

# [ch] 中文自然说法
# [ch]

# 【H 语言纯度规则】
# 一行只允许一种语言
# 禁止跨语言混排

# 【I 同句型跟读训练 五句】
# 【I1 日语句子（每句真实输出三遍）】

# [ch] 中文对应

# 【J 自检问题（必须具体实例化）】
# [jp] 具体日文词
# [ch] 原型是什么

# [jp] 具体口语变化
# [ch] 来自哪里

# [jp] 具体句型
# [ch] 在这里表达什么语气

# [jp] 具体名词
# [ch] 换成这个主语该怎么说

# 【K Shadowing 分段朗读稿（每块三遍）】

# 【L 跟读节奏版（每块三遍）】

# 【M 跟读提示】
# 先听完整句
# 再逐块跟读
# 最后整句连读
# 只看日文

# 【批处理模式特别规则】

# 生成：

# 要求：

# 句子：
# {sentence}
# """

# 生成：

# 句子：
# {sentence}
# """

# 要求：

# 句子：
# {sentence}
# """

# 任务：
# 基于输入句子生成N1难度题目

# 【必须生成】

# 【要求】

# 【干扰项生成规则】

# 输入句子：
# {sentence}
# """

#             print(f"📡 尝试请求API (尝试 {attempt + 1}/{max_retries})...")

#             last_status_code = response.status_code

#     raise Exception(error_msg)

#     日语句子分割规则：

#     return sentences

#         except UnicodeDecodeError:
#             continue

#     text = soup.get_text(separator="\n")

#     print(f"📖 共提取句子数: {len(sentences)}")

#     return sentences

# DB = "jlpt_n1.db"

#     raw = call_llm(prompt)

#     parsed = parse_questions(raw)

#     save_questions(sentence, parsed)

#     return parsed

#     return True

#             current_num = i + 1

#     raw_blocks = []

#         module_count = sum(1 for m in modules if m in block_content)

#     print(f"✅ 结构完整的解析块数量: {len(raw_blocks)}")

#     print(f"📊 去重后唯一句子数: {len(unique_blocks)}")

#     return renumbered_blocks

#     return html_content

#         anki_notes.append(note_fields)

#     return anki_notes

#     print(f"✅ 已保存为Anki格式（带标签）: {output_file}")

#     return False, 0, 0

#         print(f"⚠️ 输出异常，重试 {i+1}")

#     raise Exception("❌ 多次生成失败")


In [ ]:
# ===== 教学模式 =====
# {teaching}


In [ ]:
# ===== 考试模式 =====
# {exam}


In [ ]:
# ===== 面试模式 =====
# {interview}


In [ ]:
# ===== 实战对话 =====
# {dialog}

# ######################################
# """

#     result = merge_multi_mode(teaching, exam, interview, dialog)

#     # return merge_multi_mode(teaching, exam, interview, dialog)

#     # 从检查点获取进度
#     is_complete, next_index, processed_count = check_parsing_progress(checkpoint_file, total_sentences)

#日语解析文本专用

def build_reader_oppo(base_dir, chapters, html_name):
    """
    生成小说朗读器 HTML 文件
    """
    try:
        # 将章节数据转换为 JSON 字符串
        try:
            chapters_json = json.dumps(chapters, ensure_ascii=False)
        except Exception as e:
            print(f"    ❌ JSON序列化错误: {e}")
            chapters_json = "[]"

        # 确保 html_name 是完整的路径
        if not os.path.isabs(html_name):
            html_name = os.path.join(base_dir, html_name)

        # 确保输出目录存在
        os.makedirs(os.path.dirname(html_name), exist_ok=True)

        # 生成HTML内容
        html = f"""<!DOCTYPE html>
<html lang="zh">
<head>
<meta charset="UTF-8">
<title>{os.path.basename(html_name)}</title>
<meta name="viewport" content="width=device-width, initial-scale=1.0, maximum-scale=1.0, user-scalable=no, viewport-fit=cover">
<style>
:root {{
  --bg: #111;
  --text: #eee;        /* ✅ 新增：前景文字色 */
  --panel: #1b1b1b;
  --accent: #00eaff;
  --ui-scale: 0.85;   /* 👈 一键缩放工具栏 */
}}
* {{
  box-sizing: border-box;
  -webkit-tap-highlight-color: transparent;
}}
body {{
  margin: 0;
  display: flex;
  min-height: 100%;
  background: var(--bg);
  color: var(--text);   /* ✅ 用变量 */
  font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
}}

#chapters {{
  width: 220px;
  background: var(--panel);
  overflow-y: auto;
  flex-shrink: 0;
}}

#chapters div {{
  padding: 8px 10px;
  font-size: 13px;
  cursor: pointer;
}}
#chapters .active {{
  background: #333;
}}
#content {{
  flex: 1 1 auto;
  padding: 16px;
  padding-bottom: 160px;
  overflow-y: auto;
  font-size: 1.15em;
  line-height: 1.2;
}}
.line {{
  padding: 0px 38px;   /* 减小内边距 */
  margin: 2px 0;      /* 减小外边距 */
  border-radius: 6px;
  line-height: 1.2;   /* 更紧凑 */
}}

.line.active {{
  background: var(--accent);
  color: #000;
  font-weight: bold;
}}
.line.done {{
  opacity: .5;
}}
.line:empty {{
    display: none;  /* 防止意外空行显示 */
}}

.line[data-skip="1"] {{
    display: none;
}}

#ctrl {{
  position: fixed;
  left: 0;
  right: 0;
  bottom: 0;
  padding: 6px 10px calc(env(safe-area-inset-bottom) + 6px);
  background: #222;
  font-size: 12px;
  z-index: 50;
}}
#ctrl input[type=range] {{
  width: 100%;
  margin: 4px 0;
}}
#ctrl span {{
  font-size: 12px;
}}
button {{
  width: 100%;
  padding: 6px;
  margin-top: 4px;
  font-size: 14px;
  border-radius: 6px;
}}
button:active {{
  background: #666;
}}
audio {{
  display: none;
}}

@media (max-width: 768px) {{
  #chapters {{
    position: fixed;
    left: -220px;
    top: 0;
    bottom: 0;
    z-index: 10;
    transition: .25s;
  }}
  #chapters.show {{ left: 0; }}
  #toggle {{
    position: fixed;
    top: 12px;
    left: 12px;
    z-index: 20;
    background: #333;
    padding: 8px 10px;
    border-radius: 6px;
  }}
}}
@media (orientation: landscape) {{
  #content {{
    font-size: 1.25em;
    line-height: 1.9;
  }}
}}
#themeBar {{
  position: fixed;
  top: 52px;
  right: 8px;
  z-index: 55;
  display: none;          /* 👈 默认隐藏 */
  gap: 4px;
  background: var(--panel);
  padding: 6px;
  border-radius: 6px;
}}

/* 展开状态 */
#themeBar.show {{
  display: flex;
}}

#themeBar button {{
  font-size: 11px;
  padding: 4px 6px;
  border-radius: 5px;
  border: none;
  background: #444;
  color: var(--text);
}}

#themeBar button:active {{
  background: #666;
}}
/* ===== 一键全局缩放工具栏 ===== */

  transform: scale(var(--ui-scale));
}}

/* 播放控制小图标 */
#ctrlToggle {{
  position: fixed;
  right: 12px;
  bottom: 90px;
  z-index: 60;
  background: #333;
  color: #fff;
  padding: 8px 10px;
  border-radius: 10px;
  font-size: 16px;
}}

/* ctrl 默认隐藏（完全不占位） */
#ctrl {{
  transform: translateY(110%);
  transition: transform .25s ease;
}}

/* ctrl 展开 */
#ctrl.show {{
  transform: translateY(0);
}}
/* 主题小图标 */
#themeToggle {{
  position: fixed;
  top: 12px;
  right: 12px;
  z-index: 60;
  background: #333;
  color: #fff;
  padding: 8px 10px;
  border-radius: 10px;
  font-size: 16px;
}}

/* 主题面板默认隐藏 */
#colorPickerDiv {{
  position: fixed;
  top: 52px;
  right: 12px;
  z-index: 55;
  background: #333;
  padding: 8px 10px;
  border-radius: 6px;
  display: none;
}}

/* 展开 */
#colorPickerDiv.show {{
  display: flex;
}}

body.immersive .ui {{
  opacity: 0;
  visibility: hidden;
  pointer-events: none;
}}

/* 边缘召回区 */
#edge-left,
#edge-right {{
  position: fixed;
  top: 0;
  bottom: 0;
  width: 24px;
  z-index: 200;
}}
#edge-left {{ left: 0; }}
#edge-right {{ right: 0; }}
</style>
</head>
<body>
<div id="ui-layer">
  <div id="toggle" class="ui">📖</div>
  <div id="chapters" class="ui"></div>

  <div id="ctrlToggle" class="ui">🎛</div>
  <div id="ctrl" class="ui">
    速度 <span id="rateLabel">1.0×</span>
    <input id="rate" type="range" min="0.5" max="4" step="0.1" value="1">
    跟读停顿 <span id="pauseLabel">800 ms</span>
    <input id="pauseRange" type="range" min="0" max="3000" step="100" value="800">

    模式：
    <select id="repeatMode">
      <option value="line">每行重复</option>
      <option value="chapter">整章重复</option>
    </select>

    <!-- 重复朗读次数 -->
    次数 <span id="repeatLabel">1</span>
    <input id="repeatCount" type="number" min="1" max="10" value="1" style="width:60px">

    <!-- 每次速度变化 -->
    速度增量 <span id="speedIncLabel">0</span>
    <input id="speedIncrement" type="number" step="0.1" min="0" max="1" value="0" style="width:60px">

    <button id="btn">▶ 播放 / 暂停</button>
  </div>
  <!-- 颜色选择 -->
  <!-- 主题颜色折叠按钮 -->
  <div id="themeToggle" class="ui">🎨</div>
  <div id="themeBar" class="ui">
    <button data-theme="dark">🌙 夜间</button>
    <button data-theme="paper">📜 纸书</button>
    <button data-theme="green">🌿 护眼</button>
    <button data-theme="blue">🌌 蓝夜</button>
    <button data-theme="coffee">☕ 咖啡</button>
  </div>
</div>

<div id="content"></div>

<!-- 边缘召回区 -->
<div id="edge-left"></div>
<div id="edge-right"></div>

<audio id="player"></audio>
<script>
const chapters = {chapters_json};
const chaptersBox = document.getElementById("chapters");
const content = document.getElementById("content");
const player = document.getElementById("player");
const rate = document.getElementById("rate");
const rateLabel = document.getElementById("rateLabel");
const pauseRange = document.getElementById("pauseRange");
const pauseLabel = document.getElementById("pauseLabel");

const repeatCount = document.getElementById("repeatCount");
const repeatLabel = document.getElementById("repeatLabel");
repeatCount.oninput = () => repeatLabel.textContent = repeatCount.value;

const speedIncrement = document.getElementById("speedIncrement");
const speedIncLabel = document.getElementById("speedIncLabel");
speedIncrement.oninput = () => speedIncLabel.textContent = speedIncrement.value;

const repeatModeSelect = document.getElementById("repeatMode");
repeatModeSelect.onchange = () => {{
  chapterRepeatMode = repeatModeSelect.value;
}};

let chapterRepeatMode = "line"; // "line" = 每行重复, "chapter" = 整章重复
let currentRepeat = 0;          // 当前行/章的重复计数
let maxRepeat = 1;              // 从控制面板读取

const btn = document.getElementById("btn");
const toggle = document.getElementById("toggle");

// 点击正文 → 进入沉浸
content.addEventListener("click", () => {{
  document.body.classList.add("immersive");
}});
content.addEventListener("dblclick", () => {{
  document.body.classList.remove("immersive");
}});

// UI 内点击不触发正文
document.querySelectorAll(".ui").forEach(el => {{
  el.addEventListener("click", e => e.stopPropagation());
}});

// 点击边缘 → 召回 UI
document.querySelectorAll("#edge-left, #edge-right").forEach(edge => {{
  edge.addEventListener("click", () => {{
    document.body.classList.remove("immersive");
  }});
}});

// 颜色选择控件
const THEMES = {{
  dark: {{
    bg: "#111111",
    text: "#eaeaea",
    panel: "#1b1b1b",
    accent: "#00eaff",
  }},
  paper: {{
    bg: "#f4f1ea",
    text: "#2a2a2a",
    panel: "#e6e1d6",
    accent: "#ffd24d",
  }},
  green: {{
    bg: "#0f1f1a",
    text: "#d9f3ea",
    panel: "#162a23",
    accent: "#3ddc97",
  }},
  blue: {{
    bg: "#0d1b2a",
    text: "#e0e6ed",
    panel: "#13293d",
    accent: "#4cc9f0",
  }},
  coffee: {{
    bg: "#1a1410",
    text: "#f1e7dc",
    panel: "#261c16",
    accent: "#c9a66b",
  }}
}};

function applyTheme(name) {{
  const t = THEMES[name];
  if (!t) return;
  Object.entries(t).forEach(([k, v]) => {{
    document.documentElement.style.setProperty("--" + k, v);
  }});
  localStorage.setItem("reader-theme", name);
}}

document.querySelectorAll("#themeBar button").forEach(btn => {{
  btn.onclick = () => {{
    applyTheme(btn.dataset.theme);
    document.getElementById("themeBar").classList.remove("show"); // 👈 自动收起
  }};
}});

// 启动时恢复
applyTheme(localStorage.getItem("reader-theme") || "dark");

let chapterIndex = 0;
let lineIndex = 0;
let playing = false;

toggle.onclick = () => chaptersBox.classList.toggle("show");
rate.oninput = () => {{
  rateLabel.textContent = rate.value + "×";
  player.playbackRate = parseFloat(rate.value); // ⚡ 实时修改当前播放速度
}};
pauseRange.oninput = () => {{
  pauseLabel.textContent = pauseRange.value + " ms";
}};
btn.onclick = () => {{
  playing = !playing;
  if (playing) playCurrentLine();
  else player.pause();
}};

// 渲染章节函数
function renderChapter(ci) {{
    chapterIndex = ci;
    lineIndex = 0;
    content.innerHTML = "";
    content.scrollTop = 0;

    chapters[ci].forEach((l, i) => {{
        // 1️⃣ 删除 text 中的 <br> 或 <br/>
        const cleanText = (l.text || "").replace(/<br\\s*\\/?>/gi, "").replace(/\\[(ch|jp|en)\\]\\s*/g, "").replace(/\\[sound:[^\\]]*\\]/g, "");
        // 临时容器解析 HTML
        const temp = document.createElement("div");
        temp.innerHTML = cleanText;
        // 判断可读文字（中文、英文、数字）
        const visibleText = temp.textContent
        .replace(/\\u00a0/g, "")   // &nbsp;
        .replace(/\\s+/g, "")
        .replace(/[^\\u4e00-\\u9fffA-Za-z0-9]/g, "");

        const lineDiv = document.createElement("div");
        lineDiv.className = "line";
        lineDiv.innerHTML = cleanText;

        if (!visibleText && !lineDiv.querySelector("hr")) {{
            lineDiv.style.display = "none";   // 隐藏空行
            lineDiv.dataset.skip = "1";       // 播放逻辑跳过
        }}

        // 点击行播放
        lineDiv.onclick = () => {{
            lineIndex = i;
            playing = true;
            playCurrentLine();
        }};

        content.appendChild(lineDiv);
    }});
}}

function playCurrentLine() {{
    if (!playing) return;

    const lines = document.querySelectorAll(".line");
    if (!lines[lineIndex]) {{
        autoNextChapter();
        return;
    }}

    // 高亮
    lines.forEach((l, idx) => {{
      l.classList.remove("active");
      if (idx < lineIndex) l.classList.add("done");
    }});
    const currentLine = lines[lineIndex];
    currentLine.classList.add("active");
    currentLine.scrollIntoView({{ block: "center", behavior: "smooth" }});

    const audioSrc = chapters[chapterIndex][lineIndex].audio;
    if (!audioSrc || currentLine.dataset.skip === "1") {{
        // 空音频或空行 → 跳过
        lineIndex++;
        playCurrentLine();
        return;
    }}

    // ⚡ 核心修复：先设置 src，再设置播放速度
    player.src = audioSrc;
    player.playbackRate = parseFloat(rate.value) + parseFloat(speedIncrement.value) * currentRepeat;
    player.play();
}}

player.onended = () => {{
  const delay = Number(pauseRange.value);
  const lines = document.querySelectorAll(".line");
  const maxRepeat = parseInt(repeatCount.value);

  setTimeout(() => {{
    if (chapterRepeatMode === "line") {{
      // 每行重复模式
      if (currentRepeat + 1 < maxRepeat) {{
        currentRepeat++;
      }} else {{
        currentRepeat = 0;
        lineIndex++;
      }}
      playCurrentLine();
    }} else if (chapterRepeatMode === "chapter") {{
      // 整章重复模式
      lineIndex++;
      if (lineIndex >= lines.length) {{
        if (currentRepeat + 1 < maxRepeat) {{
          currentRepeat++;
          lineIndex = 0;
        }} else {{
          currentRepeat = 0;
          autoNextChapter();
          return;
        }}
      }}
      playCurrentLine();
    }}
  }}, delay);
}};

function autoNextChapter() {{
  if (chapterIndex + 1 >= chapters.length) {{
    playing = false;
    return;
  }}
  document.querySelectorAll("#chapters div").forEach(x => x.classList.remove("active"));
  chapterIndex++;
  chaptersBox.children[chapterIndex].classList.add("active");
  renderChapter(chapterIndex);
  playCurrentLine();
}}

chapters.forEach((_, i) => {{
  const d = document.createElement("div");
  d.textContent = "Chapter " + (i + 1);
  d.onclick = () => {{
    document.querySelectorAll("#chapters div").forEach(x => x.classList.remove("active"));
    d.classList.add("active");
    chaptersBox.classList.remove("show");
    renderChapter(i);
  }};
  chaptersBox.appendChild(d);
}});
// ctrl 折叠
document.getElementById("ctrlToggle").onclick = () => {{
  document.getElementById("ctrl").classList.toggle("show");
}};

// themeBar 折叠
document.getElementById("themeToggle").onclick = () => {{
  document.getElementById("themeBar").classList.toggle("show");
}};

renderChapter(0);
chaptersBox.firstChild.classList.add("active");
</script>
</body>
</html>
"""

        # 删除连续空行
        html = re.sub(r'\n\s*\n', '\n', html)

        # 写入文件
        with open(html_name, "w", encoding="utf-8") as f:
            f.write(html)

        print(f"📱 小说朗读器已生成：{html_name}")
        return html_name

    except Exception as e:
        print(f"❌ HTML生成失败 {html_name}: {e}")
        import traceback
        traceback.print_exc()
        return None

# math_processor.py - 数学符号处理模块


In [ ]:
# ============================================================
# 🧮 数学符号处理


In [ ]:
# ============================================================

MATH_SYMBOL_MAP = {
    # 基础运算
    "+": "加",
    "-": "减",
    "−": "减",
    "*": "乘",
    "×": "乘",
    "/": "或",
    "÷": "除以",
    "=": "等于",
    "≠": "不等于",
    ">": "大于",
    "<": "小于",
    "≥": "大于等于",
    "≤": "小于等于",

    # 指数 / 根号
    "^": "的",
    "√": "根号",

    # 高级符号
    "∑": "求和",
    "∏": "连乘",
    "∫": "积分",
    "∞": "无穷大",
    "≈": "约等于",

    # 逻辑 / 箭头
    #"→": "推出",
    "←": "由",
    #"⇒": "因此",
    "⇔": "等价于",

    # 集合
    "∈": "属于",
    "∉": "不属于",
    "∪": "并集",
    "∩": "交集",
}

MATH_FUNC_MAP = {
    r"\bsin\b": "正弦",
    r"\bcos\b": "余弦",
    r"\btan\b": "正切",
    r"\blog\b": "对数",
    r"\bln\b": "自然对数",
    r"\bexp\b": "指数函数",
}

NUM_MAP = {
    "0": "零",
    "1": "一",
    "2": "二",
    "3": "三",
    "4": "四",
    "5": "五",
    "6": "六",
    "7": "七",
    "8": "八",
    "9": "九",
}

def number_to_chinese(num_str: str) -> str:
    """
    简单数字转中文（支持小数）
    70 -> 七十
    3.5 -> 三点五
    """
    if "." in num_str:
        int_part, dec_part = num_str.split(".", 1)
        return number_to_chinese(int_part) + "点" + "".join(NUM_MAP.get(c, c) for c in dec_part)

    num = int(num_str)

    if num < 10:
        return NUM_MAP[str(num)]
    elif num < 20:
        return "十" if num == 10 else "十" + NUM_MAP[str(num % 10)]
    elif num < 100:
        tens = num // 10
        ones = num % 10
        return NUM_MAP[str(tens)] + "十" + (NUM_MAP[str(ones)] if ones else "")
    else:
        # 100, 1000 这种直接读数字
        return "".join(NUM_MAP.get(c, c) for c in num_str)

def normalize_percentage(text: str) -> str:
    """
    70% → 百分之七十
    3.5% → 百分之三点五
    """
    def repl(match):
        num = match.group(1)
        return "百分之" + number_to_chinese(num)

    return re.sub(r"\b(\d+(?:\.\d+)?)\s*%", repl, text)

def normalize_power(text: str) -> str:
    """
    处理 x^2 → x 的 二 次方
    """
    return re.sub(
        r"([A-Za-z0-9]+)\s*\^\s*([A-Za-z0-9]+)",
        r"\1 的 \2 次方",
        text
    )

def normalize_negative_numbers(text: str) -> str:
    """
    把 -1、=-1、(-1) 这样的负数转成「负一」
    避免被误读为「减一」
    """
    def repl(match):
        num = match.group(1)
        return "负" + number_to_chinese(num)

    # 情况 1：等号、括号、空格 后面的 -数字
    text = re.sub(
        r"(?:(?<=^)|(?<=[=\(\s]))-(\d+(?:\.\d+)?)",
        repl,
        text
    )
    return text

GREEK_MAP = {
    "α": "alpha",
    "β": "beta",
    "γ": "gamma",
    "δ": "delta",
    "ε": "epsilon",
    "θ": "theta",
    "λ": "lambda",
    "μ": "mu",
    "σ": "sigma",
    "ω": "omega",
}

SUBSCRIPT_MAP = {
    "₀": "0",
    "₁": "1",
    "₂": "2",
    "₃": "3",
    "₄": "4",
    "₅": "5",
    "₆": "6",
    "₇": "7",
    "₈": "8",
    "₉": "9",
}

def normalize_greek_and_subscript(text: str) -> str:
    # 1️⃣ 下标数字先转普通数字
    for k, v in SUBSCRIPT_MAP.items():
        text = text.replace(k, v)

    # 2️⃣ 希腊字母
    for k, v in GREEK_MAP.items():
        text = text.replace(k, v)

    return text

def normalize_hat_variables(text: str) -> str:
    """
    处理 y^, ŷ, ŷ → y 帽
    """
    # Unicode 组合帽
    text = re.sub(r"([A-Za-z])[\u0302]", r"\1 帽", text)

    # ŷ 这种预组字符
    text = text.replace("ŷ", "y 帽").replace("Ŷ", "Y 帽")

    # y^ 但后面不是数字（避免 y^2）
    text = re.sub(
        r"\b([A-Za-z])\^(?!\d)",
        r"\1 帽",
        text
    )
    return text

def normalize_transpose(text: str) -> str:
    """
    处理 θᵀ, xᵀ → theta 转置 / x 转置
    """
    # 变量 + 上标 T
    text = re.sub(
        r"([A-Za-zα-ωΑ-ΩθλμσΘΛΜΣ])ᵀ",
        r"\1 转置",
        text
    )
    return text

SUBSCRIPT_ALPHA_MAP = {
    "ᵢ": "i",
    "ⱼ": "j",
    "ₖ": "k",
    "ₙ": "n",
}

SUBSCRIPT_DIGIT_MAP = {
    "₀": "0", "₁": "1", "₂": "2", "₃": "3", "₄": "4",
    "₅": "5", "₆": "6", "₇": "7", "₈": "8", "₉": "9",
}

def normalize_subscript(text: str) -> str:
    # Unicode 下标字母
    for k, v in SUBSCRIPT_ALPHA_MAP.items():
        text = text.replace(k, f" 下标 {v}")

    # Unicode 下标数字
    for k, v in SUBSCRIPT_DIGIT_MAP.items():
        text = text.replace(k, f" 下标 {v}")

    # w_i / x_j 形式
    text = re.sub(r"([A-Za-z])_([A-Za-z0-9])", r"\1 下标 \2", text)
    return text

def normalize_implicit_multiply(text: str) -> str:
    return re.sub(
        r"(下标\s+[A-Za-z0-9]+)\s*([A-Za-z])",
        r"\1 乘 \2",
        text
    )

def normalize_summation(text: str) -> str:
    # 简单 ∑
    text = text.replace("∑", "求和 ")

    # ∑ᵢ₌₁ⁿ → 从 i 等于 1 累加到 n
    text = re.sub(
        r"∑\s*ᵢ\s*₌\s*₁\s*ⁿ",
        "从 i 等于 1 累加到 n 的 ",
        text
    )
    return text

def normalize_argminmax(text: str) -> str:
    text = re.sub(r"\barg\s*min\b", "使 最小 的 参数", text, flags=re.I)
    text = re.sub(r"\barg\s*max\b", "使 最大 的 参数", text, flags=re.I)
    return text

def normalize_gradient(text: str) -> str:
    # ∇J → J 的 梯度
    text = re.sub(
        r"∇\s*([A-Za-zθλμσΘΛΜΣ]+)",
        r"\1 的 梯度",
        text
    )

    # ∂J / ∂θᵢ
    text = re.sub(
        r"∂\s*([A-Za-zθλμσΘΛΜΣ]+)\s*/\s*∂\s*([A-Za-zθλμσΘΛΜΣ]+\s*下标\s*[A-Za-z0-9]+)",
        r"\1 对 \2 的 偏导数",
        text
    )

    # ∂J / ∂θ
    text = re.sub(
        r"∂\s*([A-Za-zθλμσΘΛΜΣ]+)\s*/\s*∂\s*([A-Za-zθλμσΘΛΜΣ]+)",
        r"\1 对 \2 的 偏导数",
        text
    )
    return text

def normalize_norm(text: str) -> str:
    # ‖x‖² → x 的 范数 的 平方
    text = re.sub(
        r"‖\s*(.*?)\s*‖\s*²",
        r"\1 的 范数 的 平方",
        text
    )

    # ‖x‖ → x 的 范数
    text = re.sub(
        r"‖\s*(.*?)\s*‖",
        r"\1 的 范数",
        text
    )
    return text

def normalize_math_text0(text: str) -> str:
    """
    主函数：将数学文本转换为适合TTS朗读的文本
    """
    text = normalize_percentage(text)
    text = normalize_negative_numbers(text)

    # y^
    text = normalize_hat_variables(text)

    # 转置
    text = normalize_transpose(text)

    # 求和
    text = normalize_summation(text)

    # 下标
    text = normalize_subscript(text)

    # 隐式乘法
    text = normalize_implicit_multiply(text)

    # 希腊字母
    text = normalize_greek_and_subscript(text)

    # ⭐ 范数（在幂之前）
    text = normalize_norm(text)

    # 幂
    text = normalize_power(text)

    # ⭐ 梯度 / 偏导
    text = normalize_gradient(text)

    # argmin / argmax
    text = normalize_argminmax(text)

    # 数学符号
    for sym, spoken in MATH_SYMBOL_MAP.items():
        text = text.replace(sym, f" {spoken} ")

    return re.sub(r"\s+", " ", text).strip()
def normalize_math_text(text):
    """
    标准化数学文本，但保持原样
    音频生成现在由新的函数处理
    """
    # 对于中日混合文本，我们不再修改
    # 音频生成由专门的函数处理
    return text

# 其他数学处理函数（简化版）
def latex_to_speech(text: str) -> str:
    if not text:
        return ""
    # ...（简化实现）
    return text

def formula_to_speech(text: str) -> str:
    # ...（简化实现）
    return text


In [ ]:
# ============================================================
# 🧩 文本清洗


In [ ]:
# ============================================================

# 包含日语假名的版本
ALLOWED_PATTERN = re.compile(
    r'''[^
        \u4e00-\u9fff    # 中文字符/日文汉字
        \u3040-\u309F    # 日文平假名
        \u30A0-\u30FF    # 日文片假名
        \uFF00-\uFFEF    # 全角字符（包含日文标点）
        A-Za-z0-9        # 英文字母和数字
        ，。！？、；：    # 中文标点
        '"               # 英文引号
        （）《》【】—    # 中文括号和破折号
        \.,!?;:          # 英文标点
        '"
        \(\)             # 英文括号
        \-               # 连字符
        \s               # 空白字符
    ]''', re.VERBOSE
)
def clean_text(text):
    text = ALLOWED_PATTERN.sub(" ", text)
    return re.sub(r"\s+", " ", text).strip()

text_ALLOWED_PATTERN = re.compile(
    r"[^\u4e00-\u9fffA-Za-z0-9]"
)

def clean_text_wordOnly0(text):
    text = text_ALLOWED_PATTERN.sub(" ", text)
    return re.sub(r"\s+", " ", text).strip()

def clean_text_wordOnly(text):
    """
    清理文本，只保留必要的字符
    对于中日混合文本，保持原样
    """
    # 移除HTML标签
    soup = BeautifulSoup(text, "html.parser")
    plain_text = remove_sound_tags(soup.get_text().strip())

    # 检查是否已经有语音标签
    if not re.match(r'^\s*\[(ch|jp|en)\]', plain_text):
        # 没有语音标签，默认是中文
        return f'[ch]{plain_text}'

    return plain_text

#         # # 所有分隔符字符（引号、括号等）
#         # self.all_delimiter_chars = set('()（）""""\'\'「」【】[]“”（）')

#         # 所有分隔符字符（引号、括号等）
#         self.all_delimiter_chars = set('()（）""""\'\'「」【】[]“”（）《》〈〉｛｝〔〕『』')

#         return segments

#         return False

#         analyzed_segments = []

#             analyzed_segments.append((segment_text, language))

#         return analyzed_segments

#             audio_segments.append((segment_text, language, voice_tag))

#         return audio_segments

#         return stats

#         analyzed_segments = analyzer.analyze_text(test_line)

#         audio_segments = analyzer.split_to_audio_segments(test_line)

# !pip install fasttext
# !pip install langid

# !pip install "numpy<2"

# model_path = "/content/drive/MyDrive/Colab Notebooks/lid.176.bin"

# class TextLanguageAnalyzer:

#     def __init__(self):

#         # fastText模型
#         self.ft_model = fasttext.load_model(model_path)

#         # # 所有分隔符字符（引号、括号等）
#         # self.all_delimiter_chars = set('()（）""""\'\'「」【】[]“”（）《》〈〉｛｝〔〕『』')

#         # 所有分隔符字符（引号、括号等）- 添加半角尖括号 <>
#         self.all_delimiter_chars = set('()（）""""\'\'「」【】[]“”（）《》〈〉｛｝〔〕『』<>')

#         # 3 langid
#         li_lang, li_score = langid.classify(text)

#         return segments

#         return False

#         analyzed_segments = []

#             language = self.detect_language_vote(segment_text)

#             analyzed_segments.append((segment_text, language))

#         return analyzed_segments

#             audio_segments.append((segment_text, language, voice_tag))

#         return audio_segments

#         return stats

#         analyzed_segments = analyzer.analyze_text(test_line)

#         audio_segments = analyzer.split_to_audio_segments(test_line)

"""
混合语言文本的逐段语言识别器
支持：中文 (zh)、日语 (ja)、英语 (en)

核心原理：
- 英语：Latin 字母 (A-Z, a-z)
- 日语：平假名 (ぁ-ん) 或 片假名 (ァ-ヶ) 是日语独有标志
- 中文：CJK 统一汉字，且不含假名
- 共享汉字（如 "設計"）：没有假名时默认判中文，有假名时判日语
"""

import re
import unicodedata
from dataclasses import dataclass

@dataclass
class Segment:
    """识别结果"""
    text: str         # 原始文本
    language: str     # zh / ja / en / unknown / punctuation
    confidence: float # 0.0 ~ 1.0


In [ ]:
# ============================================================
# Unicode 范围定义


In [ ]:
# ============================================================

def is_hiragana(ch: str) -> bool:
    """平假名：ぁ-ん (U+3040-U+309F)"""
    return '\u3040' <= ch <= '\u309F'

def is_katakana(ch: str) -> bool:
    """片假名：ァ-ヶ (U+30A0-U+30FF) + 半角片假名 (U+FF65-U+FF9F)"""
    return ('\u30A0' <= ch <= '\u30FF') or ('\uFF65' <= ch <= '\uFF9F')

def is_cjk_unified(ch: str) -> bool:
    """CJK 统一汉字（中日韩共享）"""
    cp = ord(ch)
    return (
        (0x4E00 <= cp <= 0x9FFF) or      # CJK Unified Ideographs
        (0x3400 <= cp <= 0x4DBF) or      # CJK Extension A
        (0x20000 <= cp <= 0x2A6DF) or    # CJK Extension B
        (0xF900 <= cp <= 0xFAFF)         # CJK Compatibility Ideographs
    )

def is_latin(ch: str) -> bool:
    """拉丁字母 + 数字"""
    return ch.isascii() and (ch.isalpha() or ch.isdigit())

def is_japanese_only_kanji(ch: str) -> bool:
    """日语专用汉字（中文里不用的字）"""
    # 日语简化汉字（新字体）中，部分字形与中文不同
    # 这些字在中文里不存在或极罕用
    ja_only = set('込丼畑辻峠枠栃雫匁')
    return ch in ja_only

def is_simplified_chinese(ch: str) -> bool:
    """
    简体中文特有字（日语里绝对不用的简体字）。

    注意：只收录有对应繁体字且日语使用繁体版的简体字。
    像 "行"、"装"、"面" 这种中日共用字不收录！
    """
    # 精选：这些简体字有对应的繁体字，且日语只用繁体版
    # 简体 -> 繁体（日语用）: 设->設, 确->確, 执->執, 处->處, 验->驗 ...
    simplified_only = set(
        # 言字旁简化系列（讠 vs 言）
        '计讨让讲许论设访证评识诊词译诗诚话该详语误说请诸诺读调谈谢谱'
        # 金字旁简化系列（钅 vs 金）
        '钉钢钟钥钦钧钩钮钱钳钻铁铃银铜铝铠铭铲铸链锁锅锋锐错锡键锤锦锻镜'
        # 门字简化系列（门 vs 門）
        '闪闭问闲间闷闸闹闻阀阁阅阔'
        # 车字简化系列（车 vs 車）
        '车轧轨轩转轮软轰轻载较辅辆辈辉辑输'
        # 贝字简化系列（贝 vs 貝）
        '贝贞负贡贤败货质贩贪贫购贮贯贱贴贵贸费赋赏赐赔赖赚赛赠赢'
        # 饣字简化系列（饣 vs 食）
        '饥饭饮饰饱饲饵饶饺饼馅馆馈'
        # 纟字简化系列（纟 vs 糸）
        '纠红纤约级纪纯纲纳纵纷纸纹线练组细织终结绘给络绝统继续缘编缓'
        # 马字简化系列（马 vs 馬）
        '马驮驰驱驳驴驶驹驻驼驾骂验骄骑骗'
        # 鸟字简化系列（鸟 vs 鳥）
        '鸟鸡鸣鸭鹅鹤鹰'
        # 鱼字简化系列（鱼 vs 魚）
        '鱼鲁鲍鲤鲸'
        # 其他高频简化字（有明确繁体对应且日语用繁体）
        '与专东两严举义乐书买亿从仅仓优会传伤体'
        '价华单卖厂厅历压发变号叶员听启呢吗吨团图'
        '国园围垫块坏坚声处备够头奋奖夺奥妇妈'
        '学实宝对导层岁岛币帅师带帮干广庄应废开异弃张弹归当录总战扩'
        '扫扬扰找护报拟拥择挡挤挥损换据掌掳摆撑撰数整斗断无时旷昼显晓'
        '晕暂术机杀杂条来极构标栈样档桥梦检欢欧歼毕毙沟没沪浇济浏涌涡'
        '涣减渐温测湿灭灯灵炉炼烂烦烧热爱牵犹独献猎猪环现玛珐珲瑰璃'
        '产畅疗盏盖监盘县眯码础确磁祸禅离种稳穷穿窃窜竞笔笼签简类粮糊'
        '紧繁纠约级纪纯纲纳纹线组织终结给络统继续编缓'
        '罢网翻耸职联胜脉脑腊脏腻腾临举旧'
        '节范茎荐荣药蒋蓝蔷蕴虏虑蜡补观规视觉览'
        '触订认议讨记许论设访证评识诊词译诗话该详语误说请诸诺读调谈谢谱'
        '贝负贡财责贤败货质贩贪贫购贯贵贸费赋赏赐赔赖赚赛赠赢'
        '趋跃践跄距跨路跳踊踪蹲身躯转轮软轰轻载较辅辆辈辉辑输辩辫'
        '边达迁过运还这进远违连迟选逊递逻遗邮邻'
        '郑酝酱酿释锅锈锋错锡键锤锦锻镇镜'
        '闪闭问闲间闷闸闹闻阅阔队陆陈降险随隐难'
        '雾静页顶项顺须顽顾颁颂预领频颗题颜额颤'
        '风飘飞饥饭饮饰饱馆驰驱驳驶驻驾验骑鱼鲸鸟鸡鸣鹤龙龟'
    )
    return ch in simplified_only

def is_traditional_ja_indicator(ch: str) -> bool:
    """日语中常用的繁体字/旧字体（中文简体中不使用的对应字）"""
    # 日语至今使用这些"旧字体"或繁体形式
    ja_traditional = set(
        '駅険届届届届届届届届届届届届届届届届届届届届届届届届届届届届届届届届届届届届届届届届届届届届届届届届届届届届届届届届届届届届'
    )
    # 更实用的方法：检查是否是日语常用繁体汉字词
    return False  # 单字判断不可靠，改用词级判断


In [ ]:
# ============================================================
# 日语高频纯汉字词典（核心：解决纯汉字段的中日歧义）


In [ ]:
# ============================================================

# 这些词在日语中高频使用，但在中文里要么不用、要么用简体版
# 格式：日语汉字 -> 如果出现则强烈暗示日语
JA_KANJI_WORDS = {
    # 常见日语汉字词（繁体/旧字体形式，中文用简体）
    '設計', '設計書', '確認', '実行', '実装', '処理', '検索', '検証',
    '変換', '変更', '対応', '関数', '関連', '関係', '環境', '機能',
    '構築', '構造', '構成', '接続', '認証', '認識', '開発', '運用',
    '選択', '説明', '調査', '請求', '課題', '議論', '識別', '試験',
    '資料', '質問', '起動', '送信', '受信', '結果', '結合', '経験',
    '経過', '継続', '総合', '統合', '編集', '管理', '簡単', '画面',
    '番号', '発生', '発行', '発表', '登録', '目的', '直接', '相談',
    '研究', '確定', '確保', '社員', '禁止', '種類', '空間', '端末',
    '算出', '組織', '緊急', '練習', '翻訳', '自動', '自動化',
    '複数', '複雑', '見積', '規則', '規格', '規模', '記録', '記述',
    '評価', '詳細', '誤差', '読込', '課金', '調整', '論理', '講義',
    '警告', '負荷', '販売', '購入', '返却', '返信', '通信', '通知',
    '連携', '配置', '配信', '配列', '障害', '電話', '非同期', '項目',
    '顧客', '類似',
    # 动词形式（汉字部分）
    '取得', '取消', '受付', '呼出', '問合', '回答', '回収', '報告',
    '変更', '完了', '対処', '廃止', '排除', '操作', '改善', '更新',
    '検出', '残高', '決定', '決済', '注文', '注意', '活用', '消去',
    '準備', '生成', '申請', '申込', '移行', '納品', '給与', '繰返',
    '脆弱', '表示', '要件', '要求', '計算', '計画', '設定', '証明',
    '転送', '追加', '除外', '集計', '削除', '制御', '印刷',
    # テスト関連
    '正常', '異常', '欠陥', '修正', '合格', '不合格', '再実行',
    '単体', '結合', '総合', '受入', '性能', '負荷',
}

# 中文简体高频词（日语里绝对不会出现简体）
ZH_SIMPLIFIED_WORDS = {
    # 日常用语（中文特有表达）
    '你好', '谢谢', '再见', '大家好', '是的', '不是', '好的', '没问题',
    '什么', '怎么', '为什么', '这个', '那个', '还是', '已经', '虽然',
    '但是', '因为', '所以', '如果', '不是', '没有', '可以', '应该',
    '能够', '需要', '已经', '正在', '通过', '进行', '实现', '处理',
    '开发', '测试', '运行', '执行', '创建', '删除', '修改', '查询',
    '调用', '返回', '请求', '响应', '部署', '配置', '环境', '服务',
    '接口', '模块', '组件', '功能', '数据', '文件', '文档', '项目',
    '系统', '框架', '工具', '方法', '函数', '变量', '参数', '类型',
    '对象', '结构', '数组', '列表', '字典', '字符', '字节',
    '异步', '同步', '并发', '线程', '进程', '协程',
    '缓存', '队列', '堆栈', '索引', '向量', '维度',
    '检索', '匹配', '过滤', '排序', '分页', '聚合',
    '验证', '校验', '认证', '授权', '加密', '解密',
    '稳定', '可靠', '安全', '性能', '效率', '优化',
}

def score_kanji_segment(text: str) -> tuple[float, float]:
    """
    对纯汉字段进行日语/中文打分。

    Returns:
        (ja_score, zh_score): 日语得分和中文得分，范围 0.0~1.0
    """
    ja_score = 0.0
    zh_score = 0.0

    # 1. 检查是否包含简体中文特有字
    for ch in text:
        if is_simplified_chinese(ch):
            zh_score += 2.0  # 简体字是中文的强信号
        if is_japanese_only_kanji(ch):
            ja_score += 2.0  # 日语专用汉字是日语的强信号

    # 2. 检查是否匹配日语高频汉字词
    for word in JA_KANJI_WORDS:
        if word in text:
            ja_score += len(word)  # 匹配越长的词得分越高

    # 3. 检查是否匹配中文简体高频词
    for word in ZH_SIMPLIFIED_WORDS:
        if word in text:
            zh_score += len(word)

    return ja_score, zh_score

def is_punctuation(ch: str) -> bool:
    """标点符号（中英日通用）"""
    # 常见中日标点
    cjk_puncts = set('，。、；：！？「」『』（）【】《》〈〉・…—―～〜""''')
    # ASCII 标点
    ascii_puncts = set(',.;:!?()[]{}<>-–—/\\|@#$%^&*+=~`\'"')
    return ch in cjk_puncts or ch in ascii_puncts or unicodedata.category(ch).startswith('P')


In [ ]:
# ============================================================
# 按标点拆分


In [ ]:
# ============================================================

def split_by_punctuation(text: str) -> list[str]:
    """
    按标点符号拆分文本，保留标点作为独立段。

    例: "Hello，你好。こんにちは！"
    -> ["Hello", "，", "你好", "。", "こんにちは", "！"]
    """
    # 按标点和空格拆分，保留分隔符
    result = []
    current = []
    for ch in text:
        if is_punctuation(ch) or ch.isspace():
            if current:
                result.append(''.join(current))
                current = []
            result.append(ch)
        else:
            current.append(ch)
    if current:
        result.append(''.join(current))

    return [s for s in result if s.strip()]  # 过滤纯空白


In [ ]:
# ============================================================
# 单段语言识别


In [ ]:
# ============================================================

def detect_segment_language(text: str) -> Segment:
    """
    识别单个文本段的语言。

    判断逻辑（优先级从高到低）：
    1. 纯标点 -> "punctuation"
    2. 含平假名或片假名 -> "ja"（假名是日语的唯一标志）
    3. 含 CJK 汉字且无假名 -> "zh"
    4. 含拉丁字母 -> "en"
    5. 以上都不是 -> "unknown"

    对于汉字+假名混合（如 "設計書を確認"），判定为日语。
    对于纯汉字（如 "設計書"），无法100%区分中日，默认判中文。
    """
    text = text.strip()
    if not text:
        return Segment(text=text, language="unknown", confidence=0.0)

    # 统计各类字符数量
    hiragana_count = 0
    katakana_count = 0
    cjk_count = 0
    latin_count = 0
    punct_count = 0
    other_count = 0

    for ch in text:
        if is_hiragana(ch):
            hiragana_count += 1
        elif is_katakana(ch):
            katakana_count += 1
        elif is_cjk_unified(ch):
            cjk_count += 1
        elif is_latin(ch):
            latin_count += 1
        elif is_punctuation(ch) or ch.isspace():
            punct_count += 1
        else:
            other_count += 1

    total_meaningful = hiragana_count + katakana_count + cjk_count + latin_count
    kana_count = hiragana_count + katakana_count

    # 1. 纯标点
    if total_meaningful == 0:
        return Segment(text=text, language="punctuation", confidence=1.0)

    # 2. 有假名 -> 日语（假名是日语的唯一判别标志）
    if kana_count > 0:
        confidence = min(1.0, (kana_count + cjk_count) / total_meaningful)
        return Segment(text=text, language="ja", confidence=confidence)

    # 3. 有 CJK 汉字，无假名 -> 需要区分中日
    if cjk_count > 0:
        # 如果同时有大量拉丁字母，可能是中英混合
        if latin_count > cjk_count:
            return Segment(text=text, language="en", confidence=0.6)

        # 对纯汉字段进行中日打分
        ja_score, zh_score = score_kanji_segment(text)

        if ja_score > 0 and ja_score > zh_score:
            # 匹配到日语汉字词/日语专用字 -> 判日语
            confidence = min(0.9, 0.5 + ja_score / (ja_score + zh_score + 1) * 0.5)
            return Segment(text=text, language="ja", confidence=confidence)
        elif zh_score > 0:
            # 匹配到中文简体字/简体词 -> 判中文（高置信度）
            confidence = min(1.0, 0.7 + zh_score / (ja_score + zh_score + 1) * 0.3)
            return Segment(text=text, language="zh", confidence=confidence)
        else:
            # 两边都没匹配到 -> 默认中文，低置信度（等上下文修正）
            confidence = cjk_count / total_meaningful * 0.6
            return Segment(text=text, language="zh", confidence=confidence)

    # 4. 拉丁字母 -> 英语
    if latin_count > 0:
        return Segment(text=text, language="en", confidence=1.0)

    # 5. 无法识别
    return Segment(text=text, language="unknown", confidence=0.0)


In [ ]:
# ============================================================
# 增强版：上下文感知（解决纯汉字歧义）


In [ ]:
# ============================================================

def _find_neighbor_lang(segments: list[Segment], idx: int, direction: int) -> str | None:
    """跳过标点找到真正的前/后语言段。direction: -1=向前, +1=向后"""
    i = idx + direction
    while 0 <= i < len(segments):
        if segments[i].language not in ("punctuation", "unknown"):
            return segments[i].language
        i += direction
    return None

def detect_with_context(segments: list[Segment]) -> list[Segment]:
    """
    利用上下文修正低置信度汉字段的语言判断。

    规则：
    1. 低置信度中文段（confidence < 0.7），如果前或后是日语 -> 修正为日语
    2. 多轮传播：一次修正后可能影响下一个段，所以跑两轮
    """
    if len(segments) <= 1:
        return segments

    result = list(segments)

    # 跑两轮传播（A 修正为 ja 后，可能让 B 也变成 ja）
    for _ in range(2):
        changed = False
        for i in range(len(result)):
            seg = result[i]
            # 只修正低置信度的中文段（高置信度说明有简体字等强信号，不修正）
            if seg.language == "zh" and seg.confidence < 0.7:
                prev_lang = _find_neighbor_lang(result, i, -1)
                next_lang = _find_neighbor_lang(result, i, +1)

                if prev_lang == "ja" or next_lang == "ja":
                    result[i] = Segment(
                        text=seg.text,
                        language="ja",
                        confidence=0.7
                    )
                    changed = True
        if not changed:
            break

    return result


In [ ]:
# ============================================================
# 主函数：完整检测流程


In [ ]:
# ============================================================

def detect_languages(text: str, use_context: bool = True) -> list[Segment]:
    """
    完整的混合语言检测流程。

    Args:
        text: 包含中日英混合的文本
        use_context: 是否启用上下文修正（推荐开启）

    Returns:
        list[Segment]: 每段的语言识别结果
    """
    # Step 1: 按标点拆分
    raw_segments = split_by_punctuation(text)

    # Step 2: 逐段识别
    segments = [detect_segment_language(s) for s in raw_segments]

    # Step 3: 上下文修正（可选）
    if use_context:
        segments = detect_with_context(segments)

    return segments

def detect_languages_merged(text: str) -> list[Segment]:
    """
    检测后合并相邻的同语言段（更简洁的输出）。

    例：["Hello", " ", "world"] 合并为 ["Hello world"]
    """
    segments = detect_languages(text)

    if not segments:
        return []

    merged = [segments[0]]
    for seg in segments[1:]:
        prev = merged[-1]
        if seg.language == "punctuation":
            # 标点归属到前一段
            merged[-1] = Segment(
                text=prev.text + seg.text,
                language=prev.language,
                confidence=prev.confidence
            )
        elif seg.language == prev.language:
            # 同语言合并（英文之间加空格）
            joiner = " " if prev.language == "en" and not prev.text.endswith(" ") else ""
            merged[-1] = Segment(
                text=prev.text + joiner + seg.text,
                language=seg.language,
                confidence=min(prev.confidence, seg.confidence)
            )
        else:
            merged.append(seg)

    return merged


In [ ]:
# ============================================================
# 演示

# ============================================================


In [ ]:
if __name__ == "__main__":
    test_cases = [
        # 基础测试
        "Hello，你好。こんにちは！",

        # 日语中大量汉字（含假名，容易判）
        "設計書を確認してください，テスト計画を生成します。",

        # 中英混合
        "使用 FastAPI 构建 RESTful API，支持 async/await 异步调用。",

        # 三语混合（真实场景）
        "このプロジェクトでは、LangGraph を使用して Agent を構築し、测试用例を自動生成します。The system uses RAG for knowledge retrieval。",

        # 纯汉字歧义（上下文修正 — 关键难题！）
        "テスト計画の設計書を確認する",

        # 技术术语混合
        "Agent の Tool Use は、MCP プロトコルで標準化されています。中文翻译：Agent 的工具调用通过 MCP 协议标准化。",

        # 带各种标点
        "Q1：什么是AI Agent？A：AIエージェントとは、自律的に行動するシステムです。In English: An AI Agent is an autonomous system.",

        # === 纯汉字段歧义测试（新增） ===

        # 纯日语繁体汉字（无假名！）— 通过词典匹配判日语
        "設計書，確認，実行，処理",

        # 纯中文简体 — 通过简体字判中文
        "设计文档，确认，执行，处理",

        # 日语汉字 + 上下文修正
        "テスト、計画、設計書、実装",

        # 中日汉字混合句（最难的情况）
        "設計書を確認する。然后开始执行测试。Please check the results.",
    ]

    for text in test_cases:
        print(f"\n{'='*70}")
        print(f"输入: {text}")
        print(f"{'─'*70}")

        # 详细模式（逐段）
        segments = detect_languages(text)
        print("逐段识别:")
        for seg in segments:
            if seg.language != "punctuation":
                print(f"  [{seg.language}|{seg.confidence:.1f}] {seg.text}")

        # 合并模式（更简洁）
        merged = detect_languages_merged(text)
        print("合并结果:")
        for seg in merged:
            print(f"  [{seg.language}] {seg.text}")

"""
TextLanguageAnalyzer — Colab 完整版（可直接复制粘贴）

外部调用方式不变：
    segments = text_analyzer.split_to_audio_segments(plain_text)

增强点：
    1. detect_language_vote 加入汉字词典打分（第三投票者）
    2. analyze_text 末尾加上下文修正
    3. 其他所有接口保持原样
"""

model_path = "/content/drive/MyDrive/Colab Notebooks/lid.176.bin"


In [ ]:
# ============================================================
# 【新增】汉字词典 & 简体字集（解决纯汉字段的中日歧义）


In [ ]:
# ============================================================

# 日语独有字（国字 + 日本新字体中与中文简体不同的字）
_JA_ONLY_CHARS = frozenset(
    # 国字（日本独创汉字，中文不使用）
    '込丼畑辻峠枠栃雫匁凧笹榊椛鰯鱈鰤鯰鶯蛯鵜躾咲'
    '腺粁粍竍籵粨甅兪嬶裃麿畠杢膰聟'
    # 日本新字体（与中文简体写法不同的字）
    # 日语用左边的字，中文简体用括号里的字
    # 駅(驿) 気(气) 図(图) 広(广) 芸(艺) 険(险) 剣(剑) 権(权)
    # 観(观) 勧(劝) 歓(欢) 戦(战) 銭(钱) 鉄(铁) 価(价) 絵(绘)
    # 覚(觉) 楽(乐) 帰(归) 拠(据) 挙(举) 経(经) 県(县) 倹(俭)
    # 桜(樱) 済(济) 斎(斋) 蚕(蚕) 残(残) 歯(齿) 児(儿) 湿(湿)
    # 寿(寿) 渋(涩) 獣(兽) 縦(纵) 焼(烧) 証(证) 剰(剩) 譲(让)
    # 嬢(娘) 触(触) 尽(尽) 酔(醉) 随(随) 髄(髓) 斉(齐) 摂(摄)
    # 窃(窃) 専(专) 繕(缮) 騒(骚) 贈(赠) 臓(脏) 堕(堕) 滞(滞)
    # 担(担) 胆(胆) 鋳(铸) 庁(厅) 聴(听) 鎮(镇) 逓(递) 転(转)
    # 伝(传) 灯(灯) 闘(斗) 届(届) 悩(恼) 脳(脑) 拝(拜) 麦(麦)
    # 発(发) 髪(发) 抜(拔) 浜(滨) 瓶(瓶) 仏(佛) 辺(边) 舗(铺)
    # 穂(穗) 翻(翻) 黙(默) 訳(译) 薬(药) 様(样) 竜(龙) 虜(虏)
    # 慮(虑) 猟(猎) 涼(凉) 隣(邻) 礼(礼) 廉(廉) 炉(炉) 労(劳)
    '駅気図広芸険剣権観勧歓戦銭鉄価絵覚楽帰拠挙経県倹'
    '桜済斎蚕歯児湿寿渋獣縦焼証剰譲嬢触尽酔随髄斉摂'
    '窃専繕騒贈臓滞担鋳庁聴鎮逓転伝灯闘届悩脳拝麦'
    '発髪抜浜仏辺舗穂黙訳薬様竜虜慮猟涼隣礼炉労'
    # 日语常用但中文罕用的汉字
    '嵐膳壱弐繋頃溢喧嘩侍忍袴塀瓦藤鈴椿萩蕎蒲柿桐杉檜'
)

_SIMPLIFIED_ONLY_CHARS = frozenset(
    # 言字旁（讠 vs 言）
    '计讨让讲许论设访证评识诊词译诗诚话该详语误说请诸诺读调谈谢谱'
    # 金字旁（钅 vs 金）
    '钉钢钟钥钦钧钩钮钱钳钻铁铃银铜铝铠铭铲铸链锁锅锋锐错锡键锤锦锻镜'
    # 门（门 vs 門）
    '闪闭问闲间闷闸闹闻阀阁阅阔'
    # 车（车 vs 車）
    '车轧轨轩转轮软轰轻载较辅辆辈辉辑输'
    # 贝（贝 vs 貝）
    '贝贞负贡贤败货质贩贪贫购贮贯贱贴贵贸费赋赏赐赔赖赚赛赠赢'
    # 饣（饣 vs 食）
    '饥饭饮饰饱饲饵饶饺饼馅馆馈'
    # 纟（纟 vs 糸）
    '纠红纤约级纪纯纲纳纵纷纸纹线练组细织终结绘给络绝统继续缘编缓'
    # 马（马 vs 馬）
    '马驮驰驱驳驴驶驹驻驼驾骂验骄骑骗'
    # 其他
    '鸟鸡鸣鸭鹅鹤鹰鱼鲁鲍鲤鲸'
    '与专东两严举义乐书买亿从仅仓优会传伤体'
    '价华单卖厂厅历压发变号叶员听启呢吗吨团图'
    '国园围垫块坏坚声处备够头奋奖夺奥妇妈'
    '学实宝对导层岁岛币帅师带帮干广庄应废开异弃张弹归当录总战扩'
    '扫扬扰找护报拟拥择挡挤挥损换据掌掳摆撑撰数整斗断无时旷昼显晓'
    '晕暂术机杀杂条来极构标栈样档桥梦检欢欧歼毕毙沟没沪浇济浏涌涡'
    '涣减渐温测湿灭灯灵炉炼烂烦烧热爱牵犹独献猎猪环现玛珐珲瑰璃'
    '产畅疗盏盖监盘县眯码础确磁祸禅离种稳穷穿窃窜竞笔笼签简类粮糊'
    '紧繁罢网翻耸职联胜脉脑腊脏腻腾临旧'
    '节范茎荐荣药蒋蓝蔷蕴虏虑蜡补观规视觉览触订认议'
    '趋跃践跄距跨跳踊踪蹲躯辩辫'
    '边达迁过运还这进远违连迟选逊递逻遗邮邻'
    '郑酝酱酿释镇队陆陈降险随隐难'
    '雾静页顶项顺须顽顾颁颂预领频颗题颜额颤'
    '风飘飞龙龟'
)

_JA_KANJI_WORDS = frozenset({
    # ── IT・ビジネス ──
    '設計', '設計書', '確認', '実行', '実装', '処理', '検索', '検証',
    '変換', '変更', '対応', '関数', '関連', '関係', '環境', '機能',
    '構築', '構造', '構成', '接続', '認証', '認識', '開発', '運用',
    '選択', '説明', '調査', '請求', '課題', '議論', '識別', '試験',
    '資料', '質問', '起動', '送信', '受信', '結果', '結合', '経験',
    '経過', '継続', '総合', '統合', '編集', '管理', '簡単', '画面',
    '番号', '発生', '発行', '発表', '登録', '目的', '直接', '相談',
    '研究', '確定', '確保', '社員', '禁止', '種類', '空間', '端末',
    '算出', '組織', '緊急', '練習', '翻訳', '自動', '自動化',
    '複数', '複雑', '見積', '規則', '規格', '規模', '記録', '記述',
    '評価', '詳細', '誤差', '読込', '課金', '調整', '論理', '講義',
    '警告', '負荷', '販売', '購入', '返却', '返信', '通信', '通知',
    '連携', '配置', '配信', '配列', '障害', '電話', '非同期', '項目',
    '顧客', '類似', '仕様', '納期', '稟議', '決裁', '進捗', '工程',
    '取得', '取消', '受付', '呼出', '問合', '回答', '回収', '報告',
    '完了', '対処', '廃止', '排除', '操作', '改善', '更新',
    '検出', '残高', '決定', '決済', '注文', '注意', '活用', '消去',
    '準備', '生成', '申請', '申込', '移行', '納品', '給与',
    '脆弱', '表示', '要件', '要求', '計算', '計画', '設定', '証明',
    '転送', '追加', '除外', '集計', '削除', '制御', '印刷',
    '正常', '異常', '欠陥', '修正', '合格', '不合格', '再実行',
    '単体', '性能', '権限', '承認', '却下', '差戻',
    # ── 日常生活 ──
    '油断', '大敵', '勉強', '準備', '復習', '予習', '出張', '会議',
    '仕事', '名前', '友達', '場所', '天気', '大丈夫', '上手',
    '下手', '元気', '本当', '大変', '丁寧', '綺麗', '素敵',
    '全然', '結構', '我慢', '邪魔', '面倒', '無理', '心配',
    '約束', '挨拶', '意味', '理由', '支度', '沢山', '一緒',
    '散歩', '買物', '掃除', '洗濯', '留守', '引越', '荷物',
    '切符', '乗換', '片道', '往復', '定期', '改札', '踏切',
    '信号', '交差点', '横断歩道', '歩道', '駐車場',
    # ── 人・家族 ──
    '家族', '両親', '兄弟', '姉妹', '子供', '彼女', '彼氏',
    '先生', '生徒', '学校', '病院', '会社', '銀行', '空港',
    '駅前', '交番', '図書館', '美術館', '博物館', '役所',
    '大家', '隣人', '同僚', '部下', '上司', '後輩', '先輩',
    # ── 食べ物 ──
    '料理', '弁当', '味噌', '醤油', '豆腐', '納豆', '刺身',
    '寿司', '天麩羅', '焼肉', '丼物', '蕎麦', '饂飩', '煎餅',
    '漬物', '佃煮', '味噌汁', '焼魚', '煮物', '揚物', '和菓子',
    '甘酒', '日本酒', '焼酎', '抹茶', '煎茶', '玄米',
    # ── 自然・季節 ──
    '桜花', '紅葉', '梅雨', '台風', '地震', '津波', '噴火',
    '富士山', '温泉', '露天風呂', '花見', '雪景色', '夕焼',
    # ── 文化・伝統 ──
    '歌舞伎', '能楽', '茶道', '華道', '書道', '柔道', '剣道',
    '弓道', '合気道', '空手道', '相撲', '将棋', '囲碁',
    '着物', '浴衣', '草履', '下駄', '扇子', '風呂敷',
    '神社', '鳥居', '仏閣', '参拝', '御守', '絵馬',
    '祭', '盆踊', '花火大会', '初詣', '七五三', '成人式',
    # ── 感情・性格 ──
    '嬉', '悔', '懐', '恥', '惜', '憧', '怒', '呆',
    '真面目', '几帳面', '頑固', '素直', '優', '厳',
    '寂', '恋', '慕', '憎', '妬', '羨',
    # ── 体・健康 ──
    '風邪', '怪我', '熱中症', '花粉症', '頭痛', '腹痛',
    '虫歯', '骨折', '捻挫', '火傷', '擦傷', '診察',
    '処方箋', '入院', '退院', '手術', '検診', '予防接種',
    # ── 四字熟語 ──
    '油断大敵', '一石二鳥', '一期一会', '以心伝心', '因果応報',
    '温故知新', '花鳥風月', '起死回生', '自業自得', '弱肉強食',
    '十人十色', '前代未聞', '大同小異', '適材適所', '独立独歩',
    '日進月歩', '百発百中', '付和雷同', '無我夢中', '竜頭蛇尾',
    '臨機応変', '老若男女', '我田引水', '異口同音', '一目瞭然',
    '危機一髪', '五里霧中', '四面楚歌', '七転八起', '針小棒大',
    '天真爛漫', '馬耳東風', '半信半疑', '本末転倒', '門外漢',
    '用意周到', '理路整然', '冷静沈着', '有言実行', '単刀直入',
    '一所懸命', '一生懸命', '切磋琢磨', '試行錯誤', '意気投合',
    '喜怒哀楽', '言語道断', '公明正大', '質実剛健', '自給自足',
    '順風満帆', '晴耕雨読', '誠心誠意', '絶体絶命', '創意工夫',
    '二束三文', '八方美人', '品行方正', '不眠不休', '無病息災',
    '明鏡止水', '優柔不断', '落花流水', '和洋折衷',
    # ── 敬語・接尾辞 ──
    '御中', '各位', '殿', '拝啓', '敬具', '前略', '草々',
    '御社', '弊社', '貴社', '当社', '御礼', '御祝', '御見舞',
})

_ZH_SIMPLIFIED_WORDS = frozenset({
    # ── 虚词・连接词・助词（最高频，防止被误判）──
    '是', '的', '了', '在', '和', '与', '或', '而', '但', '也',
    '都', '就', '才', '又', '还', '把', '被', '让', '给', '从',
    '到', '为', '向', '对', '比', '跟', '按', '将', '用', '着',
    '及', '等', '中', '上', '下', '内', '外', '前', '后', '里',
    '即', '若', '虽', '则', '且', '并', '仍', '已', '曾', '正',
    '很', '更', '最', '太', '不', '没', '别', '非', '再', '还',
    '这', '那', '哪', '每', '各', '某', '另', '其', '此', '该',
    '谁', '何', '几', '多', '少', '些', '么', '吗', '呢', '吧',
    '啊', '呀', '哦', '嗯', '哈', '嘛', '罢', '啦', '喽',
    '故', '称', '乃', '之', '于', '以', '者', '所', '然',
    # ── 日常用语 ──
    '你好', '谢谢', '再见', '大家好', '是的', '不是', '好的', '没问题',
    '什么', '怎么', '为什么', '这个', '那个', '还是', '已经', '虽然',
    '但是', '因为', '所以', '如果', '没有', '可以', '应该', '或者',
    '能够', '需要', '正在', '通过', '进行', '实现', '当然', '一定',
    '大概', '也许', '可能', '必须', '千万', '赶紧', '马上', '立刻',
    '然后', '开始', '而且', '就是', '其实', '比如', '例如', '总之',
    '另外', '同时', '反而', '否则', '不过', '尽管', '于是', '因此',
    '提示', '指', '表示', '意思', '用于', '用来', '常用', '属于',
    '构成', '加上', '补助', '相当', '根据', '按照', '关于', '对于',
    # ── IT・技术 ──
    '开发', '测试', '运行', '执行', '创建', '删除', '修改', '查询',
    '调用', '返回', '请求', '响应', '部署', '配置', '环境', '服务',
    '接口', '模块', '组件', '功能', '数据', '文件', '文档', '项目',
    '系统', '框架', '工具', '方法', '函数', '变量', '参数', '类型',
    '对象', '结构', '数组', '列表', '字典', '字符', '字节', '编码',
    '异步', '同步', '并发', '线程', '进程', '协程', '回调', '闭包',
    '缓存', '队列', '堆栈', '索引', '向量', '维度', '矩阵', '算法',
    '检索', '匹配', '过滤', '排序', '分页', '聚合', '遍历', '递归',
    '验证', '校验', '认证', '授权', '加密', '解密', '签名', '令牌',
    '稳定', '可靠', '安全', '效率', '优化', '兼容', '扩展', '维护',
    '浏览器', '服务器', '数据库', '操作系统', '命令行', '终端',
    '网络', '协议', '端口', '域名', '路由', '代理', '防火墙',
    # ── 日语学习相关（用户场景高频词）──
    '代表', '洪亮', '感觉', '印象', '联想', '解析', '记忆', '词根',
    '发音', '音图', '拓展', '扩展', '单词', '汉字', '意思', '含义',
    '例句', '重音', '词性', '位置', '方法', '记', '名', '谐音',
    '假名', '平假名', '片假名', '五十音', '浊音', '半浊音', '拗音',
    '动词', '形容词', '名词', '副词', '助词', '连词', '感叹词',
    '变化', '活用', '原形', '过去式', '否定', '被动', '使役',
    '敬语', '谦语', '尊敬语', '丁宁语', '口语', '书面语',
    '语法', '句型', '主语', '谓语', '宾语', '定语', '状语',
    '翻译', '朗读', '听力', '阅读', '写作', '会话', '发音',
    # ── 形容词 ──
    '明亮', '黑暗', '温暖', '寒冷', '高兴', '悲伤', '快乐', '痛苦',
    '美丽', '丑陋', '聪明', '愚蠢', '勇敢', '胆小', '善良', '邪恶',
    '强大', '弱小', '巨大', '渺小', '宽广', '狭窄', '深远', '浅薄',
    '干净', '肮脏', '整齐', '凌乱', '丰富', '贫乏', '复杂', '简单',
    '热闹', '安静', '拥挤', '空旷', '辽阔', '狭小', '崭新', '陈旧',
    '柔软', '坚硬', '光滑', '粗糙', '锋利', '钝', '甜蜜', '苦涩',
    '特别', '普通', '正常', '异常', '完整', '残缺', '充足', '缺乏',
    # ── 颜色 ──
    '颜色', '红色', '蓝色', '绿色', '黄色', '白色', '黑色', '紫色',
    '橙色', '粉色', '灰色', '棕色', '金色', '银色', '透明',
    # ── 时间・数量 ──
    '东西', '时候', '地方', '方面', '关系', '问题', '情况',
    '今天', '明天', '昨天', '现在', '以前', '以后', '将来', '过去',
    '经常', '偶尔', '总是', '从来', '刚才', '立刻', '渐渐', '忽然',
    '大约', '至少', '最多', '左右', '以上', '以下', '之间', '附近',
    # ── 人物・称谓 ──
    '爸爸', '妈妈', '爷爷', '奶奶', '哥哥', '姐姐', '弟弟', '妹妹',
    '叔叔', '阿姨', '老师', '同学', '朋友', '邻居', '客人', '领导',
    # ── 身体・健康 ──
    '头', '脸', '眼睛', '鼻子', '嘴巴', '耳朵', '手', '脚',
    '身体', '健康', '感冒', '发烧', '咳嗽', '头疼', '肚子疼',
    # ── 自然・天气 ──
    '天空', '太阳', '月亮', '星星', '云彩', '大风', '下雨', '下雪',
    '春天', '夏天', '秋天', '冬天', '温度', '湿度', '雾霾', '彩虹',
    '山', '河', '湖', '海', '森林', '草地', '沙漠', '平原',
    # ── 成语（四字）──
    '一心一意', '三心二意', '半途而废', '全力以赴', '自言自语',
    '异想天开', '画蛇添足', '守株待兔', '掩耳盗铃', '亡羊补牢',
    '刻舟求剑', '叶公好龙', '杯弓蛇影', '对牛弹琴', '狐假虎威',
    '井底之蛙', '盲人摸象', '鹤立鸡群', '画龙点睛', '锦上添花',
    '雪中送炭', '火上浇油', '班门弄斧', '杞人忧天', '胸有成竹',
    '愚公移山', '精卫填海', '铁杵成针', '悬梁刺股', '囊萤映雪',
    '闻鸡起舞', '卧薪尝胆', '破釜沉舟', '负荆请罪', '完璧归赵',
    '纸上谈兵', '围魏救赵', '四面楚歌', '背水一战', '风声鹤唳',
    '开卷有益', '学以致用', '举一反三', '融会贯通', '博览群书',
    '不耻下问', '温故知新', '循序渐进', '持之以恒', '锲而不舍',
    '脚踏实地', '实事求是', '言行一致', '表里如一', '光明磊落',
    '堂堂正正', '理直气壮', '义正辞严', '大公无私', '两袖清风',
    '万众一心', '齐心协力', '同舟共济', '风雨同舟', '众志成城',
    '取长补短', '扬长避短', '因地制宜', '因材施教', '量力而行',
    '随机应变', '审时度势', '知己知彼', '运筹帷幄', '未雨绸缪',
    '防患未然', '居安思危', '有备无患', '未卜先知', '先见之明',
    '心旷神怡', '赏心悦目', '流连忘返', '乐不思蜀', '心花怒放',
    '喜出望外', '欣喜若狂', '眉飞色舞', '手舞足蹈', '欢天喜地',
    '日新月异', '突飞猛进', '蒸蒸日上', '欣欣向荣', '蓬勃发展',
})

def _score_kanji(text: str) -> Tuple[float, float]:
    """
    对文本进行日语/中文打分。
    Returns: (ja_score, zh_score)
    """
    ja_score = 0.0
    zh_score = 0.0

    # 去掉空格后再匹配（原始文本可能有 "表 示" 这样的空格）
    text_no_space = re.sub(r'\s+', '', text)

    # 逐字：简体字 / 日语专用字
    for ch in text_no_space:
        if ch in _SIMPLIFIED_ONLY_CHARS:
            zh_score += 2.0
        if ch in _JA_ONLY_CHARS:
            ja_score += 2.0

    # 词级匹配（用去空格版本匹配）
    for word in _JA_KANJI_WORDS:
        if word in text_no_space:
            ja_score += len(word)
    for word in _ZH_SIMPLIFIED_WORDS:
        if word in text_no_space:
            zh_score += len(word)

    return ja_score, zh_score


In [ ]:
# ============================================================
# TextLanguageAnalyzer（在原版基础上增强，接口完全不变）


In [ ]:
# ============================================================

class TextLanguageAnalyzer:

    def __init__(self):
        # fastText模型
        self.ft_model = fasttext.load_model(model_path)

        # 假名范围
        self.hiragana_range = r'[\u3040-\u309F]'
        self.katakana_range = r'[\u30A0-\u30FF]'

        # 所有分隔符字符（确保覆盖所有 Unicode 引号变体）
        self.all_delimiter_chars = set(
            '()（）'                    # 圆括号
            '[]【】〔〕'               # 方括号
            '{}｛｝'                    # 花括号
            '<>《》〈〉'               # 尖括号
            '「」『』'                  # 日文引号
            '""'                        # 直引号 U+0022
            '\u201C\u201D'             # 弯引号 "" U+201C/U+201D
            '\u2018\u2019'             # 弯单引号 '' U+2018/U+2019
            '\uFF02'                    # 全角引号 ＂ U+FF02
            "'"                         # 直单引号 U+0027
        )

        # 分隔符配对
        self.delimiter_pairs = {
            '(': ')', '（': '）',
            '[': ']', '【': '】', '〔': '〕',
            '{': '}', '｛': '｝',
            '<': '>', '《': '》', '〈': '〉',
            '「': '」', '『': '』',
            '"': '"',                            # 直引号 U+0022
            '\u201C': '\u201D',                  # 弯引号 "" U+201C → U+201D
            '\u2018': '\u2019',                  # 弯单引号 '' U+2018 → U+2019
            '\uFF02': '\uFF02',                  # 全角引号 ＂ U+FF02
            "'": "'",                            # 直单引号 U+0027
        }

    def detect_language_vote(self, text: str) -> str:
        """
        【增强版】四层投票检测：
        1. 假名检测（最快最准）
        2. 汉字词典打分（新增！解决纯汉字歧义）
        3. fastText + langid（长文本兜底）
        4. 默认中文
        """
        if not text:
            return "zh"

        # ---- 第1层：假名检测（100%准确）----
        if re.search(self.hiragana_range, text) or re.search(self.katakana_range, text):
            return "ja"

        # ---- 第2层【新增】：汉字词典打分 ----
        has_cjk = bool(re.search(r'[\u4e00-\u9fff\u3400-\u4dbf]', text))
        if has_cjk:
            ja_score, zh_score = _score_kanji(text)
            if ja_score > 0 or zh_score > 0:
                # 词典有明确信号 -> 直接判定，不走 fastText
                if ja_score > zh_score:
                    return "ja"
                else:
                    return "zh"

        # ---- 第2.5层：纯汉字 + 含简体字 → 直接判中文 ----
        if has_cjk:
            has_kana = bool(re.search(r'[\u3040-\u309F\u30A0-\u30FF]', text))
            if not has_kana:
                # 含简体字 = 100%中文（日语绝不用简体字）
                if any(ch in _SIMPLIFIED_ONLY_CHARS for ch in text):
                    return "zh"

        # ---- 第3层：fastText + langid ----
        clean_text = text.replace('\n', ' ').replace('\r', ' ').strip()
        if not clean_text:
            return "zh"
        ft_label, ft_prob = self.ft_model.predict(clean_text)
        ft_lang = ft_label[0].replace("__label__", "")
        ft_score = ft_prob[0]

        li_lang, li_score = langid.classify(clean_text)

        # 纯汉字无假名的特殊处理：任一模型说日语就判日语
        # （中日混合语境下，纯汉字段如果有模型识别为日语，大概率确实是日语）
        if has_cjk and not bool(re.search(r'[\u3040-\u309F\u30A0-\u30FF]', text)):
            if ft_lang == "ja" or li_lang == "ja":
                return "ja"
            return "zh"

        # 非纯汉字文本（含拉丁字母等）：原逻辑
        if ft_lang == li_lang:
            return ft_lang
        if ft_score > 0.9:
            return ft_lang
        return li_lang

    # --------------------------------------------------------
    # 【新增】上下文修正
    # --------------------------------------------------------

    def _apply_context_correction(
        self, segments: List[Tuple[str, str]]
    ) -> List[Tuple[str, str]]:
        """
        低置信度中文段（词典/简体字都没匹配到的纯共用汉字），
        如果前后是日语段，修正为日语。两轮传播。
        """
        if len(segments) <= 1:
            return segments

        result = list(segments)

        # 标记哪些段可以被上下文修正
        correctable = []
        for text, lang in result:
            if lang == "zh":
                ja_s, zh_s = _score_kanji(text)
                # 两边都没信号 = 纯共用汉字，可被修正
                correctable.append(ja_s == 0 and zh_s == 0)
            else:
                correctable.append(False)

        for _ in range(2):
            changed = False
            for i in range(len(result)):
                if not correctable[i]:
                    continue
                text_i, lang_i = result[i]
                if lang_i != "zh":
                    continue

                prev_lang = self._find_neighbor_lang(result, i, -1)
                next_lang = self._find_neighbor_lang(result, i, +1)

                # 只有两侧都是日语时才修正（避免中日混合文本被过度同化）
                if prev_lang == "ja" and next_lang == "ja":
                    result[i] = (text_i, "ja")
                    changed = True

            if not changed:
                break

        return result

    @staticmethod
    def _find_neighbor_lang(
        segments: List[Tuple[str, str]], idx: int, direction: int
    ) -> str:
        """跳过空段找前/后有效语言"""
        i = idx + direction
        while 0 <= i < len(segments):
            text, lang = segments[i]
            if text.strip() and lang in ('zh', 'ja', 'en'):
                return lang
            i += direction
        return ""

    # --------------------------------------------------------
    # 以下方法：保持原版逻辑不变
    # --------------------------------------------------------

    def split_by_all_delimiters(self, text: str) -> List[Tuple[str, str]]:
        """用所有分隔符拆分文本，返回: [(文本片段, 'quote'|'none')]"""
        if not text:
            return []

        segments = []
        current_segment = ''
        in_delimiter = False
        delimiter_stack = []

        i = 0
        n = len(text)

        while i < n:
            char = text[i]

            if char in self.all_delimiter_chars:
                if current_segment:
                    seg_type = 'quote' if in_delimiter else 'none'
                    segments.append((current_segment.strip(), seg_type))
                    current_segment = ''

                if not in_delimiter:
                    in_delimiter = True
                    delimiter_stack.append(char)
                else:
                    if delimiter_stack:
                        last_start = delimiter_stack[-1]
                        expected_end = self.delimiter_pairs.get(last_start)
                        if char == expected_end:
                            delimiter_stack.pop()
                            if not delimiter_stack:
                                in_delimiter = False
                    else:
                        in_delimiter = False

                i += 1
            else:
                current_segment += char
                i += 1

        if current_segment:
            seg_type = 'quote' if in_delimiter else 'none'
            segments.append((current_segment.strip(), seg_type))

        return [(text, delim_type) for text, delim_type in segments if text]

    # --------------------------------------------------------
    # 【新增】按所有标点进一步拆分
    # --------------------------------------------------------

    # 所有可能的标点分隔符
    _all_punctuation = set(
        '，。、；：！？…—―～〜·'   # 中日标点
        ',.;:!?-_|'                 # 英文标点（含下划线、竖线）
        '→←↑↓'                     # 箭头符号
        '／＼｜＋＝'               # 全角符号
    )

    def _split_further_by_punctuation(
        self, segments: List[Tuple[str, str]]
    ) -> List[Tuple[str, str]]:
        """
        在 split_by_all_delimiters 的结果上，再按逗号/句号等标点进一步拆分。

        例：[("設計書，確認，実行", "none")]
          → [("設計書", "none"), ("確認", "none"), ("実行", "none")]

        注意：标点本身不保留为独立段，避免输出太碎。
        """
        result = []
        for text, delim_type in segments:
            if not text.strip():
                continue

            # 按标点拆分
            current = []
            for ch in text:
                if ch in self._all_punctuation:
                    if current:
                        piece = ''.join(current).strip()
                        if piece:
                            result.append((piece, delim_type))
                        current = []
                    # 标点本身不要（语言检测不需要）
                else:
                    current.append(ch)

            # 最后一段
            if current:
                piece = ''.join(current).strip()
                if piece:
                    result.append((piece, delim_type))

        return result

    # --------------------------------------------------------
    # 【新增】合并相邻同语言段（拆太细后合回来，保留标点）
    # --------------------------------------------------------

    def _merge_same_language(
        self,
        analyzed: List[Tuple[str, str]],
        original_text: str
    ) -> List[Tuple[str, str]]:
        """
        合并相邻的同语言段，并恢复中间被丢掉的标点。
        使用累进搜索定位每个片段在原文中的位置。
        """
        if not analyzed:
            return []

        # 为每个片段在原文中找到位置（累进搜索，不会被重复文本干扰）
        positions = []  # (start, end) in original_text
        search_from = 0
        for seg_text, lang in analyzed:
            pos = original_text.find(seg_text, search_from)
            if pos >= 0:
                positions.append((pos, pos + len(seg_text)))
                search_from = pos + len(seg_text)
            else:
                positions.append((-1, -1))

        # 合并相邻同语言段，恢复中间标点
        merged = []
        i = 0
        while i < len(analyzed):
            text, lang = analyzed[i]
            start_pos = positions[i][0]
            end_pos = positions[i][1]
            # 向后合并同语言段
            j = i + 1
            while j < len(analyzed) and analyzed[j][1] == lang:
                # 恢复中间的标点
                if end_pos >= 0 and positions[j][0] >= 0:
                    between = original_text[end_pos:positions[j][0]]
                    text = text + between + analyzed[j][0]
                    end_pos = positions[j][1]
                else:
                    text = text + analyzed[j][0]
                j += 1
            merged.append((text, lang))
            i = j

        return merged

    def is_japanese_text(self, text: str) -> bool:
        """判断文本是否为日文（增强版：加入词典匹配）"""
        if not text:
            return False

        if re.search(self.hiragana_range, text) or re.search(self.katakana_range, text):
            return True

        # 【新增】日语汉字词匹配
        ja_score, _ = _score_kanji(text)
        if ja_score > 0:
            return True

        return False

    def analyze_text(self, text: str) -> List[Tuple[str, str]]:
        """分析文本，返回每个片段的语言。返回: [(文本片段, 语言标签)]"""

        # 1. 剥离开头的语言标签 [ch] 或 [jp]（不再影响检测结果）
        match = re.match(r'^(\s*\[(ch|jp)\]\s*)+', text)
        if match:
            text = text[match.end():].strip()

        # 2. 第一轮拆分：按引号/括号
        segments = self.split_by_all_delimiters(text)

        # 3.【新增】第二轮拆分：按所有标点进一步拆细
        segments = self._split_further_by_punctuation(segments)

        # 4. 逐段检测语言
        analyzed_segments = []
        for segment_text, delim_type in segments:
            if not segment_text:
                continue

            language = self.detect_language_vote(segment_text)

            # 英文兜底
            if language not in ['zh', 'ja']:
                if re.search(r'[a-zA-Z]', segment_text):
                    language = 'en'
                else:
                    language = 'zh'

            analyzed_segments.append((segment_text, language))

        # 5.【新增】上下文修正（纯共用汉字段被日语邻居同化）
        analyzed_segments = self._apply_context_correction(analyzed_segments)

        # 6.【新增】合并相邻同语言段（恢复标点，避免输出太碎）
        analyzed_segments = self._merge_same_language(analyzed_segments, text)

        for language, segment_text in [(l, t) for t, l in analyzed_segments]:
            print(f"{language}:{segment_text}")

        return analyzed_segments

    def split_to_audio_segments(self, text: str) -> List[Tuple[str, str, str]]:
        """将文本拆分为音频片段。返回: [(文本片段, 语言标签, 语音标签)]"""
        analyzed_segments = self.analyze_text(text)

        audio_segments = []
        for segment_text, language in analyzed_segments:
            if language == 'zh':
                voice_tag = 'ch'
            elif language == 'ja':
                voice_tag = 'jp'
            elif language == 'en':
                voice_tag = 'en'
            else:
                voice_tag = 'ch'
            audio_segments.append((segment_text, language, voice_tag))

        return audio_segments

    def get_language_statistics(self, analyzed_segments: List[Tuple[str, str]]) -> dict:
        """获取语言统计信息"""
        stats = {
            'total_segments': len(analyzed_segments),
            'chinese_segments': 0, 'japanese_segments': 0, 'english_segments': 0,
            'chinese_chars': 0, 'japanese_chars': 0, 'english_chars': 0,
        }
        for text, language in analyzed_segments:
            if language == 'zh':
                stats['chinese_segments'] += 1
                stats['chinese_chars'] += len(text)
            elif language == 'ja':
                stats['japanese_segments'] += 1
                stats['japanese_chars'] += len(text)
            elif language == 'en':
                stats['english_segments'] += 1
                stats['english_chars'] += len(text)
        return stats


In [ ]:
# ============================================================
# 测试


In [ ]:
# ============================================================

def test_text_analyzer():
    """测试"""
    analyzer = TextLanguageAnalyzer()

    test_cases = [
        ('[ch] 提示"コマーシャル"是"流れています"这个动作的主体。', '中日混合'),
        ('[ch] 由动词的"て形"加上补助动词"います"构成。', '中日混合'),
        ('[ch] 流れています → 流れてる', '日文变化'),
        ('[ch] 这是一个纯中文句子。', '纯中文'),
        ('[ch] Hello, this is English text.', '英文'),
        ('[jp] これは日本語です。「ここは日本語です」', '纯日文'),
        # ★ 纯汉字歧义测试
        ('[ch] 設計書，確認，実行', '纯日语繁体汉字（无假名！）'),
        ('[ch] 设计文档，确认，执行', '纯中文简体'),
        ('[ch] 然后开始执行测试', '中文简体词'),
        ('[ch] <あまり>副词', '日文+中文'),
        ('[jp] 「身」就是「もの」，东西', '日文引号内汉字'),
    ]

    print("=" * 70)
    for i, (test_line, description) in enumerate(test_cases, 1):
        print(f"\n测试 {i}: {description}")
        print(f"原文: {test_line}")
        print("-" * 50)
        analyzed_segments = analyzer.analyze_text(test_line)
        for j, (text, language) in enumerate(analyzed_segments, 1):
            print(f"  片段 {j}: [{language}] {text}")

        # 验证 split_to_audio_segments 接口
        audio_segments = analyzer.split_to_audio_segments(test_line)
        print(f"  音频片段: {[(t, v) for t, _, v in audio_segments]}")


In [ ]:
if __name__ == "__main__":
    test_text_analyzer()


In [ ]:
# ============================================================
# 全局实例（保持原有调用方式）


In [ ]:
# ============================================================

text_analyzer = TextLanguageAnalyzer()

# main_program.py - 主程序模块


In [ ]:
# ============================================================
# 🗣 Edge TTS


In [ ]:
# ============================================================

voices = [
    'zh-CN-XiaoxiaoNeural',
    'zh-CN-YunxiNeural',
    'zh-CN-YunjianNeural',
    'zh-CN-YunyangNeural',
    'zh-CN-YunxiaNeural',
    'zh-CN-XiaoxuanNeural',
    'zh-CN-XiaoyiNeural'
]

CHINESE_VOICES = [
    "zh-CN-XiaoxiaoNeural",  # 晓晓 - 女声，默认中文语音
    "zh-CN-YunxiNeural",     # 云希 - 男声
    "zh-CN-YunjianNeural",   # 云健 - 男声，新闻风格
    "zh-CN-XiaoyiNeural",    # 晓伊 - 女声
    "zh-CN-YunyangNeural",   # 云扬 - 男声，体育解说风格
]

JAPANESE_VOICES = [
    "ja-JP-NanamiNeural",    # 七海 - 女声
    "ja-JP-KeitaNeural",     # 庆太 - 男声
    #"ja-JP-AoiNeural",       # 葵 - 女声，动漫风格
    #"ja-JP-DaichiNeural",    # 大地 - 男声
    #"ja-JP-MayuNeural",      # 真由 - 女声，温柔风格
]

ENGLISH_VOICES = [
    "en-US-JennyNeural",     # 珍妮 - 女声，美式英语
    "en-US-GuyNeural",       # 盖伊 - 男声，美式英语
    "en-GB-SoniaNeural",     # 索尼亚 - 女声，英式英语
    "en-AU-NatashaNeural",   # 娜塔莎 - 女声，澳洲英语
    "en-US-AnaNeural",       # 安娜 - 女声，美式英语，儿童声线
]


In [ ]:
# ============================================================
# 🗣 章节级语音选择器（同一章节内语音固定，避免跳跃感）


In [ ]:
# ============================================================
_chapter_voice_rng = random.Random()
_chapter_voices_cache = {}  # {chapter_idx: {"ch": voice, "jp": voice, "en": voice}}

def set_chapter_voice_seed(chapter_idx):
    """为当前章节设置语音种子，同一章节内语音固定"""
    global _chapter_voices_cache
    rng = random.Random(chapter_idx * 31337)
    _chapter_voices_cache = {
        "ch": rng.choice(CHINESE_VOICES),
        "jp": rng.choice(JAPANESE_VOICES),
        "en": rng.choice(ENGLISH_VOICES),
    }

def get_chapter_voice(lang_tag):
    """根据语言标签获取当前章节固定的语音"""
    if not _chapter_voices_cache:
        # 未设置种子时回退到随机
        if lang_tag == "jp":
            return random.choice(JAPANESE_VOICES)
        elif lang_tag == "en":
            return random.choice(ENGLISH_VOICES)
        return random.choice(CHINESE_VOICES)
    if lang_tag in ("jp", "ja"):
        return _chapter_voices_cache.get("jp", random.choice(JAPANESE_VOICES))
    elif lang_tag == "en":
        return _chapter_voices_cache.get("en", random.choice(ENGLISH_VOICES))
    return _chapter_voices_cache.get("ch", random.choice(CHINESE_VOICES))

def should_speak_line(line):
    """
    判断这一行是否应该朗读
    规则：
    1. 必须以 [ch]、[jp]、[en] 开头
    2. 不能是空行
    3. 不能是特定的标签列表内容
    """
    line = line.strip()
    if not line:
        return False

    # 检查是否有语音标签 [ch]、[jp]、[en]
    if not re.match(r'^\s*\[(ch|jp|en)\]', line):
        return False

    # 提取标签后的实际内容
    match = re.match(r'^\s*\[(ch|jp|en)\]\s*(.*)', line)
    if not match:
        return False

    content = match.group(2).strip()

    # 定义需要排除的内容列表
    exclude_list = [
        "整句假名",
        "汉字对应假名",
        "分词假名",
        "假名",
        "词性",
        "原型",
        "当前形",
        "当前活用形",
        "词性",
        "中文含义",
        "中文说明",
        "含义",
        "句型",
        "日语词",
        "是否发生日式语义变化",
        "否",
        "是",
        "完整原型",
        "缩略路径",
        "无",
        "使用语境",
        # 可以继续添加其他需要排除的内容
    ]

    # 如果内容在排除列表中，则不朗读
    if content in exclude_list:
        return False

    # 检查是否有实际的可读内容
    return bool(re.search(r'[A-Za-z0-9\u4e00-\u9fff\u3040-\u309F\u30A0-\u30FF\u3000-\u303F]', content))

def parse_line_and_select_voice(line, default_voice="ch"):
    """
    解析文本行并选择对应的语音

    参数:
    line: 要解析的文本行
    default_voice: 默认语音，当没有匹配时使用

    返回:
    (voice, text, lang): 如果解析成功，返回语音、文本和语言标签
    None: 如果解析失败或文本为空
    """
    line = line.strip()
    if not line:
        return None

    # 只处理有语音标签的行
    match = re.match(r'^\s*\[(ch|jp|en)\]\s*(.*)', line)
    if match:
        lang = match.group(1)  # 获取语言标签
        text = match.group(2).strip()  # 获取实际文本

        if text:  # 确保有文本内容
            # 根据标签选择对应的语音（章节级固定）
            voice = get_chapter_voice(lang)
            if lang not in ("ch", "jp", "en"):
                lang = default_voice

            return voice, text, lang

    return None
import re

def remove_spaces_between_cjk(text):
    """
    只删除中文、日文字符之间的空格
    简单直接的方法
    """
    if not text:
        return text

    # 匹配CJK字符（中文汉字、日文假名、标点）
    cjk_char = r'[\u4e00-\u9fff\u3040-\u309f\u30a0-\u30ff\u3000-\u303f]'

    # 匹配：CJK字符 + 一个或多个空格/制表符 + CJK字符
    pattern = f'({cjk_char})\\s+({cjk_char})'

    # 递归删除所有这样的空格
    while True:
        new_text = re.sub(pattern, r'\1\2', text)
        if new_text == text:
            break
        text = new_text

    return text

def collapse_repeated_punctuation(text):
    """
    将连续重复的相同标点符号折叠为一个。
    例如: "___" → "_", "！！！" → "！", "---" → "-"
    但保留 "……"（中文省略号的标准写法是两个…）。
    """
    if not text:
        return text

    # 需要折叠的标点（不含省略号…）
    punct_chars = set(
        '，。、；：！？—―～〜·'
        ',.;:!?-_'
        '→←↑↓'
        '／＼｜＋＝'
        '()（）""""\'\'「」【】[]""（）《》〈〉｛｝〔〕『』<>'
    )

    result = []
    prev_char = None
    repeat_count = 0
    for ch in text:
        if ch == prev_char and ch in punct_chars:
            # 连续重复的标点，跳过
            continue
        elif ch == '…':
            # 省略号特殊处理：允许最多两个（……是标准写法）
            if prev_char == '…':
                repeat_count += 1
                if repeat_count > 2:
                    continue  # 第3个及以后的…跳过
            else:
                repeat_count = 1
            result.append(ch)
            prev_char = ch
            continue
        else:
            repeat_count = 0
        result.append(ch)
        prev_char = ch

    return ''.join(result)

def process_mixed_language_text(text):
    """
    处理中日混合文本，返回需要单独生成音频的片段列表
    每个片段包含文本和对应的语音标签

    返回: List[Tuple[str, str]]  # (text, voice_tag)
    """
    soup = BeautifulSoup(text, "html.parser")
    plain_text = remove_sound_tags(soup.get_text().strip())

    # 使用分析器处理中日混合文本
    segments = text_analyzer.split_to_audio_segments(plain_text)

    # 转换为需要的格式
    audio_segments = []
    for segment_text, language, voice_tag in segments:
        # 折叠连续重复标点
        segment_text = collapse_repeated_punctuation(segment_text)
        audio_segments.append((segment_text, voice_tag))

    return audio_segments

async def generate_mixed_audio_segments(text, output_dir, base_filename):
    """
    为中日混合文本生成多个音频文件

    参数:
    text: 原始文本
    output_dir: 输出目录
    base_filename: 基础文件名

    返回:
    List[Tuple[str, str, str]]: [(音频文件路径, 文本片段, 语音标签)]
    """
    # 获取音频片段
    audio_segments = process_mixed_language_text(text)

    generated_files = []

    for i, (segment_text, voice_tag) in enumerate(audio_segments):
        # 生成音频文件名
        audio_filename = f"{base_filename}_{voice_tag}_{i:02d}.mp3"
        audio_path = os.path.join(output_dir, audio_filename)

        # 检查是否已存在
        if not os.path.exists(audio_path) or os.path.getsize(audio_path) < 1000:
            print(f"    🔊 生成片段 {i+1}/{len(audio_segments)} ({voice_tag}): {audio_filename}")

            # 根据语音标签选择语音（章节级固定）
            voice = get_chapter_voice(voice_tag)

            # 删除CJK字符间的空格（如果是中文或日文）
            if voice_tag in ["ch", "jp"]:
                segment_text = remove_spaces_between_cjk(segment_text)

            # 生成TTS - 需要await
            for attempt in range(1, 4):  # 重试3次
                try:
                    tts = edge_tts.Communicate(segment_text, voice)
                    await tts.save(audio_path)
                    print(f"    ✅ 片段生成成功")
                    break
                except Exception as e:
                    if attempt < 3:
                        print(f"    ⚠️ 片段生成失败（第{attempt}次）: {e}")
                        await asyncio.sleep(1)
                    else:
                        print(f"    ❌ 片段生成失败: {e}")
                        # 创建静默音频作为替代
                        AudioSegment.silent(duration=500).export(audio_path, format="mp3")
        else:
            print(f"    ⏭ 片段已存在: {audio_filename}")

        generated_files.append((audio_path, segment_text, voice_tag))

    return generated_files

async def gen_tts_mixed(text, out, base_filename=None):
    """
    为中日混合文本生成TTS音频
    如果文本是中日混合，将生成多个音频文件并合并
    """
    # 获取纯文本内容
    soup = BeautifulSoup(text, "html.parser")
    plain_text = remove_sound_tags(soup.get_text().strip())

    # 检查是否有语言标签（[ch] 或 [jp]）→ 走混合拆分路径
    if re.match(r'^\s*\[(ch|jp)\]', plain_text):
        # 中日混合文本：生成多个音频片段
        output_dir = os.path.dirname(out)
        if base_filename is None:
            base_filename = os.path.splitext(os.path.basename(out))[0]

        print(f"      🔊 检测到中日混合文本，将生成多个音频片段")

        # 生成所有音频片段（需要await）
        audio_segments = await generate_mixed_audio_segments(
            text, output_dir, base_filename
        )

        # 合并所有音频文件
        if audio_segments:
            combined = AudioSegment.silent(duration=0)

            for audio_path, segment_text, voice_tag in audio_segments:
                try:
                    audio = AudioSegment.from_file(audio_path, format="mp3")
                    # 根据语言添加不同间隔
                    if voice_tag == "ch":
                        combined += audio + AudioSegment.silent(duration=200)  # 中文间短间隔
                    elif voice_tag == "jp":
                        combined += audio + AudioSegment.silent(duration=300)  # 日文间中长间隔
                    else:
                        combined += audio + AudioSegment.silent(duration=400)  # 其他语言长间隔
                except Exception as e:
                    print(f"      ⚠️ 无法读取音频片段 {audio_path}: {e}")
                    # 添加静默段
                    combined += AudioSegment.silent(duration=1000)

            # 导出合并的音频
            combined.export(out, format="mp3")
            print(f"      ✅ 合并音频完成: {os.path.basename(out)}")

            # 可选：统计信息
            ch_count = sum(1 for _, _, tag in audio_segments if tag == 'ch')
            jp_count = sum(1 for _, _, tag in audio_segments if tag == 'jp')
            en_count = sum(1 for _, _, tag in audio_segments if tag == 'en')
            print(f"      统计: 中文片段={ch_count}, 日文片段={jp_count}, 英文片段={en_count}")
        else:
            # 没有生成音频，创建空音频
            AudioSegment.silent(duration=100).export(out, format="mp3")
            print(f"      ⚠️ 未生成音频片段，创建静默音频")
    else:
        # 单一语言文本：使用原来的逻辑
        await gen_tts(text, out)

async def gen_tts(text, out):
    """
    生成TTS音频文件
    只对有语音标签的行进行朗读
    """
    for attempt in range(1, 10):
        try:
            # 获取纯文本内容，移除HTML标签
            soup = BeautifulSoup(text, "html.parser")
            plain_text = remove_sound_tags(soup.get_text().strip())

            # 解析语音标签并选择语音
            result = parse_line_and_select_voice(plain_text, default_voice="ch")

            if result is None:
                # 没有语音标签，创建空音频文件
                AudioSegment.silent(duration=100).export(out, format="mp3")
                return

            voice, actualText, lang = result

            print(f"      🔊 TTS 生成中（{lang}，第 {attempt} 次） → {os.path.basename(out)}")
            print(f"      原始文本: {plain_text[:100]}...")

            # 应用文本处理（区分中文日文模式下，不使用数学符号处理）
            if lang in ["ch", "jp"]:
                original_text = actualText

                # 删除CJK字符之间的空格
                actualText = remove_spaces_between_cjk(actualText)

                if original_text != actualText:
                    removed_count = len(original_text) - len(actualText)
                    print(f"      🧹 删除了 {removed_count} 个CJK字符间的空格")

            # 折叠连续重复标点
            actualText = collapse_repeated_punctuation(actualText)

            tts = edge_tts.Communicate(actualText, voice)
            await tts.save(out)
            print(f"      ✅ TTS 成功 ({lang}) → {os.path.basename(out)}")
            return
        except Exception as e:
            print(f"      ⚠️ TTS 失败（第 {attempt} 次）：{e}, {voice}")
            await asyncio.sleep(2)

    print(f"      ❌ 放弃生成：{os.path.basename(out)}")


In [ ]:
# ============================================================
# 🗣 TTS 模式配置（在 generate_mp3_and_html 中设定）


In [ ]:
# ============================================================

TTS_MODE = "differentiated"  # 默认值，会被 generate_mp3_and_html 覆盖
BATCH_SIZE = 20              # 默认值，会被 generate_mp3_and_html 覆盖

async def gen_tts_read_all(text, out):
    """
    全文朗读模式：朗读所有文字内容，不区分中文日文，不跳过任何内容。
    统一使用中文语音朗读。
    """
    for attempt in range(1, 10):
        try:
            # 获取纯文本内容，移除HTML标签
            soup = BeautifulSoup(text, "html.parser")
            plain_text = remove_sound_tags(soup.get_text().strip())

            if not plain_text:
                AudioSegment.silent(duration=100).export(out, format="mp3")
                return

            # 移除语言标签 [ch] [jp] [en]（只移除标签，保留后面文本）
            plain_text = re.sub(r'\[(ch|jp|en)\]\s*', '', plain_text).strip()

            if not plain_text:
                AudioSegment.silent(duration=100).export(out, format="mp3")
                return

            # 删除CJK字符之间的空格
            actualText = remove_spaces_between_cjk(plain_text)
            # 折叠连续重复标点
            actualText = collapse_repeated_punctuation(actualText)

            # 统一使用中文语音（章节级固定）
            voice = get_chapter_voice("ch")

            print(f"      🔊 全文朗读 TTS 生成中（第 {attempt} 次） → {os.path.basename(out)}")

            tts = edge_tts.Communicate(actualText, voice)
            await tts.save(out)
            print(f"      ✅ TTS 成功（全文朗读） → {os.path.basename(out)}")
            return
        except Exception as e:
            print(f"      ⚠️ TTS 失败（第 {attempt} 次）：{e}")
            await asyncio.sleep(2)

    print(f"      ❌ 放弃生成：{os.path.basename(out)}")

async def gen_tts_read_all_smart(text, out, base_filename=None):
    """
    智能全文朗读模式：与 gen_tts_mixed 逻辑完全一致（标点拆分→逐块检测语言→
    分别用对应语音朗读），唯一区别是不要求语言标签——无标签的文本自动加 [ch] 前缀
    让分析器正常工作。
    """
    soup = BeautifulSoup(text, "html.parser")
    plain_text = remove_sound_tags(soup.get_text().strip())

    if not plain_text:
        AudioSegment.silent(duration=100).export(out, format="mp3")
        return

    # 无语言标签时，加 [ch] 前缀让 process_mixed_language_text 正常处理
    if not re.match(r'^\s*\[(ch|jp|en)\]', plain_text):
        text_for_process = f'[ch] {plain_text}'
    else:
        text_for_process = plain_text

    # 直接复用 gen_tts_mixed 的完整流程（标点拆分 + 语言检测 + 分片生成 + 合并）
    await gen_tts_mixed(text_for_process, out, base_filename)

async def gen_tts_tag_trust(text, out, base_filename=None):
    """
    标签信任模式：
    - [jp] 开头 → 整行当日语朗读（信任标签，不拆分）
    - [ch] 开头 → 按标点拆分检测语言（与 gen_tts_mixed 一样）
    - [en] 开头 → 整行当英语朗读
    """
    soup = BeautifulSoup(text, "html.parser")
    plain_text = remove_sound_tags(soup.get_text().strip())

    if not plain_text:
        AudioSegment.silent(duration=100).export(out, format="mp3")
        return

    # 检测标签
    tag_match = re.match(r'^\s*\[(ch|jp|en)\]\s*', plain_text)
    if not tag_match:
        # 无标签 → 不朗读
        AudioSegment.silent(duration=100).export(out, format="mp3")
        return

    tag = tag_match.group(1)
    content = plain_text[tag_match.end():].strip()

    if not content:
        AudioSegment.silent(duration=100).export(out, format="mp3")
        return

    if tag == "jp":
        # [jp] → 整行当日语朗读，不拆分
        actualText = collapse_repeated_punctuation(remove_spaces_between_cjk(content))
        voice = get_chapter_voice("jp")
        for attempt in range(1, 10):
            try:
                tts = edge_tts.Communicate(actualText, voice)
                await tts.save(out)
                print(f"      ✅ TTS 成功（标签信任 jp） → {os.path.basename(out)}")
                return
            except Exception as e:
                print(f"      ⚠️ TTS 失败（第 {attempt} 次）：{e}")
                await asyncio.sleep(2)
        print(f"      ❌ 放弃生成：{os.path.basename(out)}")

    elif tag == "en":
        # [en] → 整行当英语朗读
        actualText = collapse_repeated_punctuation(content)
        voice = get_chapter_voice("en")
        for attempt in range(1, 10):
            try:
                tts = edge_tts.Communicate(actualText, voice)
                await tts.save(out)
                print(f"      ✅ TTS 成功（标签信任 en） → {os.path.basename(out)}")
                return
            except Exception as e:
                print(f"      ⚠️ TTS 失败（第 {attempt} 次）：{e}")
                await asyncio.sleep(2)
        print(f"      ❌ 放弃生成：{os.path.basename(out)}")

    else:
        # [ch] → 按标点拆分检测语言（复用 gen_tts_mixed 的完整流程）
        await gen_tts_mixed(text, out, base_filename)

def mp3_b64(path):
    with open(path, "rb") as f:
        return "data:audio/mp3;base64," + base64.b64encode(f.read()).decode()


In [ ]:
# ============================================================
# 📖 处理单个 TXT


In [ ]:
# ============================================================

def convert_br_to_hr_in_html(html_content):
    """
    将HTML中的连续<br>标签转换为包含<hr>的结构
    例如：<br><br> -> <br><hr><br>
          <br><br><br> -> <br><hr><br>
    """
    if not html_content:
        return html_content

    # 方法1：使用正则表达式替换
    # 匹配两个或多个连续的<br>标签（支持各种变体）
    pattern = r'(<br\s*/?>)\s*(<br\s*/?>)+'

    def replacement(match):
        # 获取第一个<br>的确切形式
        first_br = re.search(r'<br\s*/?>', match.group(0)).group(0)
        return f'{first_br}<hr><br>'

    result = re.sub(pattern, replacement, html_content, flags=re.IGNORECASE)

    # 方法2：处理特定情况
    # 也可以先规范化所有<br>标签，然后替换
    normalized = re.sub(r'<br\s*/?>', '<br>', result, flags=re.IGNORECASE)
    result = normalized.replace('<br><br>', '<br><hr><br>')

    return result

def extract_lines_from_html(html_content):
    """
    从HTML中提取行，保留格式
    修复版：正确保留HTML标签和格式
    """
    if not html_content:
        return []

    # 方法1：直接按标签拆分（最简单有效）
    lines = []

    # 处理换行标签 - 将它们作为单独的行
    html_content = html_content.strip()

    # 将连续的换行标签合并
    html_content = re.sub(r'(<br\s*/?>)+', '<br>', html_content, flags=re.IGNORECASE)
    html_content = re.sub(r'(<hr\s*/?>)+', '<hr>', html_content, flags=re.IGNORECASE)

    # 按标签拆分
    i = 0
    n = len(html_content)

    while i < n:
        # 查找标签开始
        if html_content[i] == '<':
            # 找到标签结束
            end_pos = html_content.find('>', i)
            if end_pos != -1:
                tag = html_content[i:end_pos+1]

                # 检查标签类型
                if re.match(r'<br\s*/?>', tag, re.IGNORECASE):
                    # 换行标签 - 如果有前面的内容，先添加
                    if i > 0:
                        prev_text = html_content[:i].strip()
                        if prev_text:
                            lines.append(prev_text)
                    lines.append(tag)
                    html_content = html_content[end_pos+1:]
                    i = 0
                    n = len(html_content)
                elif re.match(r'<hr\s*/?>', tag, re.IGNORECASE):
                    # 水平线标签
                    if i > 0:
                        prev_text = html_content[:i].strip()
                        if prev_text:
                            lines.append(prev_text)
                    lines.append(tag)
                    html_content = html_content[end_pos+1:]
                    i = 0
                    n = len(html_content)
                else:
                    # 其他标签，继续
                    i += 1
            else:
                i += 1
        else:
            i += 1

    # 处理剩余的文本
    if html_content.strip():
        lines.append(html_content.strip())

    # 清理结果
    cleaned_lines = []
    for line in lines:
        # line = line.strip()
        # if line:
        #     cleaned_lines.append(line)
        # else:
        #     # 保留真正的空行（由<br>标签创建）
        #     cleaned_lines.append("")

        if line == "":
            cleaned_lines.append("<hr>")  # 保留空行
        elif line == "<hr>":
            cleaned_lines.append("<hr>")  # 水平线标签
        else:
            # 对于非空行，清理空白但保留HTML
            cleaned_lines.append(line)

    return cleaned_lines

def strip_language_tags_for_display(html_text):
    """
    从 HTML 文本中移除语言标签和 [sound:...] 标记，仅用于最终显示。
    """
    if not html_text:
        return html_text

    # 移除语言标签 [ch]、[jp]、[en]
    result = re.sub(r'(?:^|\G|(?<=>))\s*\[(ch|jp|en)\]\s*', '', html_text)
    # 移除 [sound:...] 标记（包括 URL 格式）
    result = re.sub(r'\[sound:[^\]]*\]', '', result)

    return result.strip() if result.strip() else html_text

def remove_sound_tags(text):
    """
    移除文本中所有 [sound:...] 标记。
    匹配所有格式：
      [sound:xxx.mp3]
      [sound:http://dict.youdao.com/dictvoice?type=0&audio=xxx&le=jap]
      [sound:任意内容]
    """
    if not text:
        return text
    return re.sub(r'\[sound:[^\]]*\]', '', text).strip()

def has_readable_content0(text):
    """判断是否有可朗读内容"""
    text_plain = BeautifulSoup(text, "html.parser").get_text().strip()
    text_plain = remove_sound_tags(text_plain)
    if not text_plain:
        return False
    return bool(re.search(r"[A-Za-z0-9\u4e00-\u9fff]", text_plain))

def has_readable_content(text):
    """判断是否有可朗读内容（含日文）"""
    text_plain = BeautifulSoup(text, "html.parser").get_text().strip()
    text_plain = remove_sound_tags(text_plain)
    if not text_plain:
        return False

    # 检查是否有可读内容：
    # - 中文字符：\u4e00-\u9fff
    # - 英文字母和数字：A-Za-z0-9
    # - 日文假名（平假名和片假名）：\u3040-\u309F \u30A0-\u30FF
    # - 日文汉字：\u4E00-\u9FFF（与中文相同，但包含日文特有汉字）
    # - 日文标点：\u3000-\u303F
    return bool(re.search(r'[A-Za-z0-9\u4e00-\u9fff\u3040-\u309F\u30A0-\u30FF\u3000-\u303F]', text_plain))

from bs4 import BeautifulSoup, NavigableString

def remove_text_before_marker(html, marker="✅ 修正版讲解段落（自然语言版）"):
    soup = BeautifulSoup(html, "html.parser")

    found = False

    for node in soup.descendants:
        if isinstance(node, NavigableString):
            if marker in node:
                # 命中标记：删除标记前的文本，只保留后面的
                new_text = node.split(marker, 1)[1]
                node.replace_with(new_text)
                found = True
            elif not found:
                # 标记之前的所有纯文本 → 清空
                node.replace_with("")

    return str(soup)

def mark_display_only(
    lines,
    markers=("✅ 修正版讲解段落（自然语言版）", "原文纠正版")
):
    """
    markers: iterable[str]
      多个 marker，只对「第一个命中的 marker」生效
    """

    # 🔍 找到第一个命中的 marker 行号
    first_hit_index = None
    for i, line in enumerate(lines):
        text = line.get("text", "")
        if any(m in text for m in markers):
            first_hit_index = i
            break

    # 没命中 → 原样返回
    if first_hit_index is None:
        return lines

    result = []
    for i, line in enumerate(lines):
        new_line = line.copy()

        # 第一个 marker 之前 + marker 行 → 静音
        if i <= first_hit_index:
            new_line["audio"] = ""

        result.append(new_line)

    return result

async def process_table_html(table_html, output_dir, base_name):
    """
    table_html: 含 <table> 的 HTML 字符串
    output_dir: MP3 输出目录
    base_name: 用于生成 MP3 文件名的基准名（例如行索引）
    返回: 合并后的 MP3 base64
    """
    soup = BeautifulSoup(table_html, "html.parser")
    cell_mp3s = []

    # 表格标题
    caption = soup.find("caption")
    if caption and caption.get_text(strip=True):
        cap_text = caption.get_text(strip=True)
        temp_mp3 = os.path.join(output_dir, f"{base_name}_caption.mp3")
        #if not os.path.exists(temp_mp3):
        if not os.path.exists(temp_mp3) or os.path.getsize(temp_mp3) < 1000:
            cap_text = normalize_math_text(cap_text)
            await gen_tts(clean_text_wordOnly(cap_text), temp_mp3)
        cell_mp3s.append(temp_mp3)

    # 所有单元格 <th> 和 <td>
    for cell in soup.find_all(["th", "td"]):
        text = cell.get_text(strip=True)
        if not text:
            continue
        temp_mp3 = os.path.join(output_dir, f"{base_name}_{len(cell_mp3s)}.mp3")
        #if not os.path.exists(temp_mp3):
        if not os.path.exists(temp_mp3) or os.path.getsize(temp_mp3) < 1000:
            text = normalize_math_text(text)
            await gen_tts(clean_text_wordOnly(text), temp_mp3)
        cell_mp3s.append(temp_mp3)

    # 检查 MP3 是否可用，跳过损坏文件
    valid_mp3s = []
    for mp in cell_mp3s:
        try:
            AudioSegment.from_file(mp, format="mp3")
            valid_mp3s.append(mp)
        except CouldntDecodeError:
            print(f"⚠️ 无法读取 MP3，跳过：{mp}")

    # 合并 MP3
    if valid_mp3s:
        final_mp3 = os.path.join(output_dir, f"{base_name}.mp3")
        combined = AudioSegment.silent(duration=0)
        for mp in valid_mp3s:
            combined += AudioSegment.from_file(mp) + AudioSegment.silent(duration=400)
        combined.export(final_mp3, format="mp3")
        return mp3_b64(final_mp3)
    else:
        return ""

import re

def safe_filename(name: str) -> str:
    """
    将文件名中的危险字符统一替换为下划线
    保留：中文、英文、数字、下划线、短横线
    """
    name = name.strip()

    # 把空格统一成下划线
    name = name.replace(" ", "_")

    # 替换 Windows / ffmpeg / 浏览器 不安全字符
    name = re.sub(r'[\\/:*?"<>|()\[\]{}!@#$%^&+=,;\'`~]', "_", name)

    # 多个下划线压缩成一个
    name = re.sub(r'_+', "_", name)

    # 确保文件名不以点结尾
    name = name.rstrip('.')

    return name

import os
import json
from bs4 import BeautifulSoup

def load_all_chapters_from_html(html_path):
    """从HTML文件中提取所有章节数据"""
    try:
        with open(html_path, 'r', encoding='utf-8') as f:
            html_content = f.read()

        # 查找 JavaScript 中的 chapters 变量
        # 匹配 patterns like: const chapters = [...];
        patterns = [
            r'const chapters\s*=\s*(\[.*?\]);',
            r'var chapters\s*=\s*(\[.*?\]);',
            r'let chapters\s*=\s*(\[.*?\]);',
            r'chapters\s*:\s*(\[.*?\])',
        ]

        chapters_json = None
        for pattern in patterns:
            match = re.search(pattern, html_content, re.DOTALL)
            if match:
                chapters_json = match.group(1)
                break

        if not chapters_json:
            print(f"    ⚠️ 未找到chapters变量")
            return None

        # 可能需要清理JSON字符串
        # 移除注释和多余的空白
        chapters_json = re.sub(r'//.*?\n', '', chapters_json)
        chapters_json = re.sub(r'/\*.*?\*/', '', chapters_json, flags=re.DOTALL)

        # 解析JSON
        chapters_data = json.loads(chapters_json)
        print(f"    ✅ 从HTML加载了 {len(chapters_data)} 个章节")
        return chapters_data

    except json.JSONDecodeError as e:
        print(f"    ❌ JSON解析失败: {e}")
        # 尝试调试
        print(f"    JSON片段: {chapters_json[:500]}...")
        return None
    except Exception as e:
        print(f"    ❌ 加载HTML失败: {e}")
        return None


In [ ]:
# ============================================================
# 📄 HTML 文件处理：保留原始格式，逐句生成音频


In [ ]:
# ============================================================

async def process_html_file(html_path, out_dir, name):
    """
    处理 HTML 文件：保留原始格式，为每个文本块生成音频，
    输出一个带音频播放功能的完整 HTML 文件。
    点击任意段落即可朗读，底部控制栏支持连续播放、上下句、速度调节。
    """
    print(f"📄 处理 HTML 文件：{os.path.basename(html_path)}")
    os.makedirs(out_dir, exist_ok=True)

    with open(html_path, encoding="utf-8-sig") as f:
        html_content = f.read()
    soup = BeautifulSoup(html_content, "html.parser")

    # 提取所有包含可朗读文字的块级元素（跳过 style/script）
    block_tags = ['p', 'div', 'li', 'h1', 'h2', 'h3', 'h4', 'h5', 'h6',
                  'td', 'th', 'blockquote', 'pre', 'dt', 'dd', 'figcaption']
    text_elements = []
    seen = set()
    for elem in soup.find_all(block_tags):
        if elem.find_parent(['style', 'script']):
            continue
        if id(elem) in seen:
            continue
        text = remove_sound_tags(elem.get_text().strip())
        text = re.sub(r'\[(ch|jp|en)\]\s*', '', text).strip()
        if text and re.search(r'[A-Za-z0-9\u4e00-\u9fff\u3040-\u309F\u30A0-\u30FF]', text):
            text_elements.append((elem, text))
            for child in elem.find_all(block_tags):
                seen.add(id(child))

    print(f"  📊 找到 {len(text_elements)} 个可朗读文本块")

    # 生成音频
    mp3_dir = os.path.join(out_dir, "mp3")
    os.makedirs(mp3_dir, exist_ok=True)
    audio_data = {}
    for i, (elem, text) in enumerate(text_elements):
        mp3_path = os.path.join(mp3_dir, f"{i:04d}.mp3")
        set_chapter_voice_seed(i)
        if not os.path.exists(mp3_path) or os.path.getsize(mp3_path) < 1000:
            print(f"  🔊 [{i+1}/{len(text_elements)}] {text[:40]}...")
            try:
                if TTS_MODE == "read_all":
                    await gen_tts_read_all(text, mp3_path)
                else:
                    await gen_tts_read_all_smart(text, mp3_path, f"{i:04d}")
            except Exception as e:
                print(f"    ⚠️ 生成失败: {e}")
                AudioSegment.silent(duration=100).export(mp3_path, format="mp3")
        else:
            print(f"  ⏭ [{i+1}/{len(text_elements)}] 音频已存在")
        audio_data[i] = mp3_b64(mp3_path)

    # 给元素添加点击播放
    for i, (elem, text) in enumerate(text_elements):
        if i not in audio_data:
            continue
        elem['data-audio-idx'] = str(i)
        existing_class = elem.get('class', [])
        if isinstance(existing_class, str):
            existing_class = existing_class.split()
        elem['class'] = existing_class + ['tts-line']
        elem['onclick'] = f'playAudio({i})'
        elem['style'] = (elem.get('style', '') or '') + '; cursor: pointer;'

    # 注入播放器
    head = soup.find('head')
    if not head:
        head = soup.new_tag('head')
        if soup.html:
            soup.html.insert(0, head)
        else:
            soup.insert(0, head)
    style_tag = soup.new_tag('style')
    style_tag.string = """
    .tts-line { transition: background 0.3s; border-radius: 4px; }
    .tts-line:hover { background: rgba(0,200,255,0.1) !important; }
    .tts-line.playing { background: rgba(0,200,255,0.2) !important; outline: 2px solid #00c8ff; }
    #tts-controls { position:fixed; bottom:0; left:0; right:0; background:#222; color:#fff;
        padding:8px 16px; display:flex; align-items:center; gap:10px; z-index:9999;
        font-family:sans-serif; font-size:14px; box-shadow: 0 -2px 10px rgba(0,0,0,0.3); }
    #tts-controls button { background:#444; color:#fff; border:none; padding:6px 12px;
        border-radius:4px; cursor:pointer; font-size:14px; }
    #tts-controls button:hover { background:#666; }
    body { padding-bottom: 50px !important; }
    """
    head.append(style_tag)

    audio_json = json.dumps(audio_data, ensure_ascii=False)
    total_count = len(text_elements)
    script_content = """
var audioData = """ + audio_json + """;
var currentAudio = null, currentIdx = -1, isAutoPlay = false;
var allIndices = Object.keys(audioData).map(Number).sort(function(a,b){return a-b;});
function playAudio(idx) {
    if (currentAudio) currentAudio.pause();
    var prev = document.querySelector('.tts-line.playing');
    if (prev) prev.classList.remove('playing');
    if (!audioData[idx]) return;
    currentIdx = idx;
    currentAudio = new Audio(audioData[idx]);
    currentAudio.playbackRate = parseFloat(document.getElementById('tts-rate').value||'1');
    var elem = document.querySelector('[data-audio-idx="'+idx+'"]');
    if (elem) { elem.classList.add('playing'); elem.scrollIntoView({behavior:'smooth',block:'center'}); }
    document.getElementById('tts-progress').textContent = (allIndices.indexOf(idx)+1)+'/""" + str(total_count) + """';
    currentAudio.onended = function() {
        if (elem) elem.classList.remove('playing');
        if (isAutoPlay) {
            var next = allIndices.indexOf(idx)+1;
            if (next < allIndices.length) setTimeout(function(){playAudio(allIndices[next]);}, 500);
            else { isAutoPlay=false; document.getElementById('tts-auto-btn').textContent='\\u25b6 连续播放'; }
        }
    };
    currentAudio.play();
}
function toggleAutoPlay() {
    isAutoPlay = !isAutoPlay;
    var btn = document.getElementById('tts-auto-btn');
    if (isAutoPlay) { btn.textContent='\\u23f8 停止'; var p=allIndices.indexOf(currentIdx); playAudio(allIndices[p>=0?p:0]); }
    else { btn.textContent='\\u25b6 连续播放'; if(currentAudio) currentAudio.pause(); }
}
function playPrev() { var p=allIndices.indexOf(currentIdx); if(p>0) playAudio(allIndices[p-1]); }
function playNext() { var p=allIndices.indexOf(currentIdx); if(p<allIndices.length-1) playAudio(allIndices[p+1]); }
"""
    script_tag = soup.new_tag('script')
    script_tag.string = script_content
    body = soup.find('body')
    if not body:
        body = soup.new_tag('body')
        soup.append(body)
    body.append(script_tag)

    controls_html = ('<div id="tts-controls">'
        '<button onclick="playPrev()">\u23ee</button>'
        '<button id="tts-auto-btn" onclick="toggleAutoPlay()">\u25b6 连续播放</button>'
        '<button onclick="playNext()">\u23ed</button>'
        '<span>速度:</span>'
        '<input id="tts-rate" type="range" min="0.5" max="3" step="0.1" value="1" '
        "onchange=\"document.getElementById('tts-rate-label').textContent=this.value+'x';"
        "if(currentAudio)currentAudio.playbackRate=parseFloat(this.value);\">"
        '<span id="tts-rate-label">1x</span>'
        '<span id="tts-progress">0/' + str(total_count) + '</span>'
        '</div>')
    body.append(BeautifulSoup(controls_html, 'html.parser'))

    output_html = os.path.join(out_dir, f"{name}_朗读版.html")
    with open(output_html, 'w', encoding='utf-8') as f:
        f.write(str(soup))

    print(f"  ✅ 朗读版 HTML 已生成: {output_html}")
    print(f"  📊 共 {len(text_elements)} 段文字，{len(audio_data)} 个音频")


In [ ]:
# ============================================================

async def process_one_txt(txt_path):
    name = os.path.splitext(os.path.basename(txt_path))[0]
    out_dir = os.path.join(ANKI_ROOT, name)
    print(f"\n📘 开始处理 TXT：{os.path.basename(txt_path)}")
    print(f"📂 输出目录：{name}")
    print(f"🗣 TTS 模式：{TTS_MODE}")

    perHtml_dir = os.path.join(out_dir, "perHtml")

    # 先检查是否有批量HTML文件可以提取
    html_dir = os.path.join(out_dir, "html")
    all_chapters = None

    if os.path.exists(html_dir):
        html_files = [f for f in sorted(os.listdir(html_dir))
                     if f.endswith('.html') and f.startswith(safe_filename(name))]

        if html_files:
            print(f"📊 检查批量HTML文件...")
            # 尝试从批量HTML文件加载所有章节
            for html_file in html_files:
                html_path = os.path.join(html_dir, html_file)
                print(f"  检查: {html_file}")
                chapters_from_html = load_all_chapters_from_html(html_path)

                if chapters_from_html:
                    if all_chapters is None:
                        all_chapters = []
                    all_chapters.extend(chapters_from_html)

            if all_chapters:
                print(f"✅ 从批量HTML加载了 {len(all_chapters)} 个章节")
                # 检查章节数是否完整
                # 读取原始TXT文件了解总章节数
                with open(txt_path, encoding="utf-8") as f:
                    lines = [l for l in f if l.strip()]

                df = pd.read_csv(
                    StringIO("".join(lines)),
                    sep="\t",
                    header=None,
                    skiprows=2,
                    engine="python",
                    dtype=str,
                    on_bad_lines="skip"
                ).fillna("")

                total_chapters = len(df)

                if len(all_chapters) >= total_chapters:
                    print(f"🎉 已加载完整的 {len(all_chapters)} 个章节，跳过音频生成")

                    # 确保perHtml目录存在
                    os.makedirs(perHtml_dir, exist_ok=True)

                    # 生成缺失的单章节HTML文件
                    for idx, chapter_data in enumerate(all_chapters[:total_chapters], 1):
                        safe_name = safe_filename(name)
                        perHtml_path = os.path.join(perHtml_dir, f"{safe_name}_{idx:04d}.html")

                        if not os.path.exists(perHtml_path):
                            print(f"  📝 生成单章节HTML {idx:04d}/{total_chapters:04d}")
                            try:
                                build_reader_oppo(out_dir, [chapter_data], perHtml_path)
                            except Exception as e:
                                print(f"  ❌ 生成失败: {e}")

                    print(f"✅ 所有章节HTML已更新")

                    # ========== 在这里生成艾宾浩斯学习系统 ==========
                    print("\n🧠 正在生成艾宾浩斯学习系统...")
                    try:
                        from datetime import datetime, timedelta
                        import hashlib
                        import json

                        ebbinghaus_html = os.path.join(html_dir, f"{safe_filename(name)}_艾宾浩斯学习系统.html")
                        generate_super_language_learning_system(ebbinghaus_html, all_chapters[:total_chapters])
                        print(f"✅ 艾宾浩斯学习系统生成成功: {ebbinghaus_html}")
                    except Exception as e:
                        print(f"❌ 艾宾浩斯学习系统生成失败: {e}")

                    return
                else:
                    print(f"⚠️ 只加载了 {len(all_chapters)}/{total_chapters} 个章节，需要继续处理")

    # ========== HTML 文件 → 单独处理流程 ==========
    if txt_path.lower().endswith(('.html', '.htm')):
        await process_html_file(txt_path, out_dir, name)
        return

    # ========== CSV/TXT 文件处理（原逻辑）==========
    print(f"🔍 继续处理文件...")

    with open(txt_path, encoding="utf-8") as f:
        lines = [l for l in f if l.strip()]

    df = pd.read_csv(
        StringIO("".join(lines)),
        sep="\t",
        header=None,
        skiprows=2,
        engine="python",
        dtype=str,
        on_bad_lines="skip"
    ).fillna("")

    total_chapters = len(df)

    print(f"\n📖 总章节数：{total_chapters:04d}")

    chapters = []
    os.makedirs(out_dir, exist_ok=True)
    os.makedirs(perHtml_dir, exist_ok=True)

    # 如果已经加载了部分章节，从那里继续
    start_idx = len(all_chapters) if all_chapters else 1

    # for idx in range(start_idx, total_chapters + 1):
    #     print(f"\n📖 处理章节 {idx:04d} / {total_chapters:04d}")

    #     # 检查是否已有该章节的HTML文件
    #     safe_name = safe_filename(name)
    #     perHtml_path = os.path.join(perHtml_dir, f"{safe_name}_{idx:04d}.html")

    #     if os.path.exists(perHtml_path) and idx > len(all_chapters or []):
    #         print(f"  ✅ 已有HTML文件，尝试加载")
    #         # 从单章节HTML文件提取数据
    #         chapter_data = load_all_chapters_from_html(perHtml_path)
    #         if chapter_data and len(chapter_data) > 0:
    #             chapters.append(chapter_data[0])
    #             continue

    #     print(f"  🔄 处理新章节 {idx}")
    #     # ... 原来的处理逻辑 ...
    #     row = df.iloc[idx-1]
    #     fields = [str(x) for x in row.tolist() if str(x).strip()]
    #     html = "<br>".join(fields)
    #     html = convert_br_to_hr_in_html(html)

    #     sub = os.path.join(out_dir, f"{idx:04d}")
    #     os.makedirs(sub, exist_ok=True)

    #     texts = extract_lines_from_html(html)
    #     total_lines = len(texts)
    #     chapter_lines = []

    #     for i, t in enumerate(texts):
    #         plain_text = BeautifulSoup(t, "html.parser").get_text().strip()
    #         should_speak = should_speak_line(plain_text)

    #         if not should_speak:
    #             audio = ""
    #         else:
    #             mp3 = os.path.join(sub, f"{i:03d}.mp3")

    #             # 检查音频文件是否已存在
    #             audio_exists = os.path.exists(mp3) and os.path.getsize(mp3) >= 1000

    #             if not audio_exists:
    #                 print(f"   🔊 生成 MP3 {i+1}/{total_lines}：{i:03d}.mp3")
    #                 if plain_text.startswith('[ch]'):
    #                     base_filename = f"{i:03d}"
    #                     await gen_tts_mixed(t, mp3, base_filename)
    #                 else:
    #                     await gen_tts(t, mp3)
    #             else:
    #                 print(f"   ⏭ 音频已存在，跳过：{i:03d}.mp3")

    #             audio = mp3_b64(mp3)

    #         chapter_lines.append({"text": t, "audio": audio})

    #     # 应用静音标记
    #     chapter_lines = mark_display_only(chapter_lines)
    #     chapters.append(chapter_lines)

    #     # 生成单章节HTML文件
    #     print(f"  📝 生成单章节HTML: {os.path.basename(perHtml_path)}")
    #     try:
    #         build_reader_oppo(out_dir, [chapter_lines], perHtml_path)
    #         print(f"  ✅ 单章节HTML生成成功")
    #     except Exception as e:
    #         print(f"  ❌ 单章节HTML生成失败: {e}")
    for idx, (_, row) in enumerate(df.iterrows(), start=1):
        if idx < start_idx:
            continue

        print(f"\n📖 处理章节 {idx:04d} / {total_chapters:04d}")

        # 为当前章节设置固定语音种子（同章节内语音不变）
        set_chapter_voice_seed(idx)

        # 获取原始内容（假设CSV只有一列，即 result 字段）
        fields = [str(x) for x in row.tolist() if str(x).strip()]
        if not fields:
            print(f"  ⚠️ 章节 {idx:04d} 为空，跳过")
            continue
        full_text = fields[0]

        # ------------------ 智能句子分割 ------------------
        # def split_sentences(content):
        #     """根据内容格式返回句子列表（保留原始标签）"""
        #     # 情况1：包含 <br> 标签 -> 按 <br> 分割
        #     if re.search(r'<br\s*/?>', content, re.IGNORECASE):
        #         parts = re.split(r'(<br\s*/?>)', content, flags=re.IGNORECASE)
        #         sentences = []
        #         current = ''
        #         for part in parts:
        #             if re.match(r'<br\s*/?>', part, re.IGNORECASE):
        #                 if current.strip():
        #                     sentences.append(current.strip())
        #                 current = ''
        #             else:
        #                 current += part
        #         if current.strip():
        #             sentences.append(current.strip())
        #         return sentences
        #     # 情况2：无 <br> 标签 -> 按换行符分割（纯文本）
        #     else:
        #         lines = [line.strip() for line in content.split('\n') if line.strip()]
        #         return lines if lines else [content.strip()]
        def split_sentences(content):
            """
            根据内容格式返回句子列表（保留原始标签）
            - 若包含 <br> 标签，则按 <br> 分割
            - 否则按换行符 \n 分割
            """
            if re.search(r'<br\s*/?>', content, re.IGNORECASE):
                # 直接按 <br> 分割（不保留分隔符），去除首尾空格并过滤空串
                sentences = re.split(r'<br\s*/?>', content)
                sentences = [s.strip() for s in sentences if s.strip()]
                return sentences
            else:
                # 无 <br> 标签，按换行符分割（纯文本情况）
                lines = [line.strip() for line in content.split('\n') if line.strip()]
                return lines if lines else [content.strip()]

        sentences = split_sentences(full_text)
        print(f"  📊 本章分割出 {len(sentences)} 个句子")

        # 创建章节子目录（存放MP3）
        sub = os.path.join(out_dir, f"{idx:04d}")
        os.makedirs(sub, exist_ok=True)

        chapter_lines = []
        for i, raw_sentence in enumerate(sentences):
            # 提取纯文本用于TTS判断和生成
            plain_text = remove_sound_tags(BeautifulSoup(raw_sentence, "html.parser").get_text().strip())

            if TTS_MODE in ("read_all", "read_all_smart"):
                # 全文朗读模式：不跳过任何内容
                has_content = bool(re.search(r'[A-Za-z0-9\u4e00-\u9fff\u3040-\u309F\u30A0-\u30FF]', plain_text))
                if not has_content:
                    audio = ""
                    print(f"    ⏭ 句子 {i+1}/{len(sentences)} 无文字内容，跳过")
                else:
                    mp3 = os.path.join(sub, f"{i:03d}.mp3")
                    if not os.path.exists(mp3) or os.path.getsize(mp3) < 1000:
                        if TTS_MODE == "read_all":
                            print(f"   🔊 全文朗读 MP3 {i+1}/{len(sentences)}：{i:03d}.mp3")
                            await gen_tts_read_all(raw_sentence, mp3)
                        else:  # read_all_smart
                            print(f"   🔊 智能全文朗读 MP3 {i+1}/{len(sentences)}：{i:03d}.mp3")
                            base_filename = f"{i:03d}"
                            await gen_tts_read_all_smart(raw_sentence, mp3, base_filename)
                    else:
                        print(f"    ⏭ 音频已存在：{i:03d}.mp3")
                    audio = mp3_b64(mp3)

            elif TTS_MODE == "tag_trust":
                # 标签信任模式：[jp]→整行日语，[ch]→拆分检测，无标签→跳过
                should_speak = should_speak_line(plain_text)
                if not should_speak:
                    audio = ""
                    print(f"    ⏭ 句子 {i+1}/{len(sentences)} 不朗读")
                else:
                    mp3 = os.path.join(sub, f"{i:03d}.mp3")
                    if not os.path.exists(mp3) or os.path.getsize(mp3) < 1000:
                        print(f"   🔊 标签信任 MP3 {i+1}/{len(sentences)}：{i:03d}.mp3")
                        base_filename = f"{i:03d}"
                        await gen_tts_tag_trust(raw_sentence, mp3, base_filename)
                    else:
                        print(f"    ⏭ 音频已存在：{i:03d}.mp3")
                    audio = mp3_b64(mp3)

            else:
                # 区分中文/日文朗读模式（differentiated，默认）
                should_speak = should_speak_line(plain_text)
                if not should_speak:
                    audio = ""
                    print(f"    ⏭ 句子 {i+1}/{len(sentences)} 不朗读 (纯文本: {plain_text[:30]}...)")
                else:
                    mp3 = os.path.join(sub, f"{i:03d}.mp3")
                    if not os.path.exists(mp3) or os.path.getsize(mp3) < 1000:
                        print(f"   🔊 生成 MP3 {i+1}/{len(sentences)}：{i:03d}.mp3")
                        if re.match(r'^\s*\[(ch|jp)\]', plain_text):
                            base_filename = f"{i:03d}"
                            await gen_tts_mixed(raw_sentence, mp3, base_filename)
                        else:
                            await gen_tts(plain_text, mp3)
                    else:
                        print(f"    ⏭ 音频已存在：{i:03d}.mp3")
                    audio = mp3_b64(mp3)

            # 存储原始句子和音频base64
            chapter_lines.append({"text": raw_sentence, "audio": audio})

        # 应用静音标记（如果需要）
        chapter_lines = mark_display_only(chapter_lines)

        # 保存JSON
        json_path = os.path.join(json_dir, f"chapter_{idx:04d}.json")
        with open(json_path, 'w', encoding='utf-8') as jf:
            json.dump(chapter_lines, jf, ensure_ascii=False, indent=2)
        print(f"  💾 保存JSON文件: {os.path.basename(json_path)}")

        # 添加到总集合
        all_chapters.append(chapter_lines)
        current_batch_chapters.append(chapter_lines)

        # 批次处理（与原来相同）
        if len(current_batch_chapters) >= batch_size or idx == total_chapters:
            await process_batch(current_batch_chapters, current_batch_num + 1,
                                out_dir, html_dir, perHtml_dir, ebbinghaus_dir,
                                safe_name, idx, total_chapters)
            current_batch_chapters = []
            current_batch_num += 1

    # 生成批量HTML文件
    print("\n📚 正在生成批量HTML文件 …")
    html_dir = os.path.join(out_dir, "html")
    os.makedirs(html_dir, exist_ok=True)

    batch_size = BATCH_SIZE
    num_batches = (total_chapters + batch_size - 1) // batch_size
    print(f"  总章节数: {total_chapters}, 批次: {num_batches}")

    for batch_idx in range(num_batches):
        start = batch_idx * batch_size
        end = min((batch_idx + 1) * batch_size, total_chapters)
        batch_chapters = chapters[start:end]

        html_name = os.path.join(
            html_dir,
            f"{safe_name}_{batch_idx+1}.html"
        )

        # 检查批量HTML是否已存在
        if os.path.exists(html_name):
            print(f"  ⏭ 批量HTML已存在，跳过: {os.path.basename(html_name)}")
            continue

        print(f"  📝 正在生成批量HTML {batch_idx+1}/{num_batches}: {os.path.basename(html_name)}")
        try:
            build_reader_oppo(out_dir, batch_chapters, html_name)
            print(f"  ✅ 批量HTML生成成功")
        except Exception as e:
            print(f"  ❌ 批量HTML生成失败: {e}")

    # ========== 在这里生成艾宾浩斯学习系统（主要位置） ==========
    print("\n🧠 正在生成艾宾浩斯学习系统...")
    try:
        from datetime import datetime, timedelta
        import hashlib
        import json

        ebbinghaus_html = os.path.join(html_dir, f"{safe_filename(name)}_艾宾浩斯学习系统.html")

        # 确保generate_super_language_learning_system函数已定义
        if 'generate_super_language_learning_system' in globals():
            generate_super_language_learning_system(ebbinghaus_html, chapters)
            print(f"✅ 艾宾浩斯学习系统生成成功!")
            print(f"📁 文件位置: {ebbinghaus_html}")
            print(f"📊 包含 {len(chapters)} 章, {sum(len(c) for c in chapters)} 个学习点")
        else:
            print(f"⚠️ 未找到艾宾浩斯学习系统生成函数，请先定义")
    except Exception as e:
        print(f"❌ 艾宾浩斯学习系统生成失败: {e}")
        import traceback
        traceback.print_exc()

    print(f"\n✅ TXT 完成：{name}")

def extract_data_from_html(html_path):
    """从HTML文件中提取章节数据"""
    try:
        with open(html_path, 'r', encoding='utf-8') as f:
            html_content = f.read()

        # 使用正则表达式查找 chapters 变量
        # 查找 "const chapters = " 之后的内容
        pattern = r'const chapters\s*=\s*(.*?);'
        match = re.search(pattern, html_content, re.DOTALL)

        if not match:
            # 尝试其他可能的模式
            pattern = r'const chapters\s*=\s*(\[.*?\]);'
            match = re.search(pattern, html_content, re.DOTALL)

        if match:
            chapters_json = match.group(1)
            # 解析JSON
            chapters_data = json.loads(chapters_json)

            if chapters_data and len(chapters_data) > 0:
                # 返回第一个章节的数据
                print(f"    ✅ 从HTML成功提取了 {len(chapters_data[0])} 行数据")
                return chapters_data[0]

        print(f"    ⚠️ 未找到chapters数据，尝试备用方法")

        # 备用方法：从页面内容中提取
        soup = BeautifulSoup(html_content, 'html.parser')
        content_div = soup.find('div', id='content')

        if content_div:
            chapter_lines = []
            for line_div in content_div.find_all('div', class_='line'):
                # 获取HTML文本内容
                line_html = str(line_div)

                # 清理文本（去掉外部div标签）
                inner_html = line_html.replace('<div class="line">', '').replace('</div>', '')
                inner_html = inner_html.strip()

                # 查找对应的音频数据
                # 需要从原始HTML中查找这一行的索引
                audio = ""

                chapter_lines.append({"text": inner_html, "audio": audio})

            if chapter_lines:
                print(f"    ✅ 从content提取了 {len(chapter_lines)} 行数据")
                return chapter_lines

        return []

    except Exception as e:
        print(f"    ❌ 提取HTML数据失败: {e}")
        import traceback
        traceback.print_exc()
        return []

#     print(f"📄 行数：{len(lines)}")

#                 audio = mp3_b64(mp3)

#             chapter_lines.append({"text": t, "audio": audio})

#         chapters.append(chapter_lines)

#     print(f"✅ TXT 完成：{name}")

def test_mixed_audio_generation():
    """测试混合音频生成"""
    print("\n测试混合音频生成:")
    print("=" * 80)

    test_cases = [
        '[ch] 提示"コマーシャル"是"流れています"这个动作的主体。',
        '[ch] 由动词的"て形"加上补助动词"います"构成。',
        '[ch] 流れています → 流れてる',
        '[ch] 这是一个纯中文句子。',
        '[ch] "コマーシャル"',
    ]

    for i, test_case in enumerate(test_cases, 1):
        print(f"\n测试 {i}: {test_case}")

        # 处理文本
        audio_segments = process_mixed_language_text(test_case)

        print(f"  将生成 {len(audio_segments)} 个音频片段:")
        for j, (segment_text, voice_tag) in enumerate(audio_segments):
            print(f"    片段 {j+1}: [{voice_tag}] {segment_text}")


In [ ]:
# ============================================================
# 🚀 主入口


In [ ]:
# ============================================================
async def main_batch():
    # 先测试分析器
    test_mixed_audio_generation()

    txts = [f for f in sorted(os.listdir(ANKI_ROOT)) if f.lower().endswith(".txt")]
    print(f"📦 发现 {len(txts)} 个 TXT 文件")
    for i, f in enumerate(txts, 1):
        print(f"\n==============================")
        print(f"🚀 TXT {i}/{len(txts)}")
        print(f"==============================")
        await process_one_txt(os.path.join(ANKI_ROOT, f))
    print("\n🎉 所有 TXT 处理完成")
async def main_single():
    # 先测试分析器
    test_mixed_audio_generation()
    input_file = "/content/drive/MyDrive/ChatGPTAnkiKaraOK/extracted_results.txt"
    await process_one_txt(input_file)
    print("\n🎉 TXT 处理完成")

import glob
import json
import re
from collections import defaultdict

def extract_chapter_data_from_single_html(html_path):
    """从单个章节HTML文件中提取数据"""
    try:
        with open(html_path, 'r', encoding='utf-8') as f:
            content = f.read()

        # 查找 JavaScript 中的 chapters 变量
        pattern = r'const chapters\s*=\s*(\[.*?\]);'
        match = re.search(pattern, content, re.DOTALL)

        if match:
            chapters_json = match.group(1)
            # 清理JSON字符串
            chapters_json = re.sub(r'//.*?\n', '', chapters_json)
            chapters_json = re.sub(r'/\*.*?\*/', '', chapters_json, flags=re.DOTALL)

            # 解析JSON
            data = json.loads(chapters_json)
            if data and len(data) > 0:
                return data[0]  # 返回章节数据

        return None
    except Exception as e:
        print(f"    ❌ 提取数据失败 {os.path.basename(html_path)}: {e}")
        return None

def combine_chapters_to_batch_html(base_dir, chapters_data, batch_name, start_chapter, total_chapters):
    """将多个章节数据合并为一个批量HTML文件"""
    try:
        # 生成章节导航HTML
        chapters_nav_html = ""
        for i in range(len(chapters_data)):
            chapter_num = start_chapter + i
            chapters_nav_html += f'<div onclick="renderChapter({i})">Chapter {chapter_num}</div>\n'

        # 生成完整的HTML
        chapters_json = json.dumps(chapters_data, ensure_ascii=False)

        html_template = """<!DOCTYPE html>
<html lang="zh">
<head>
<meta charset="UTF-8">
<title>{batch_name}</title>
<meta name="viewport" content="width=device-width, initial-scale=1.0, maximum-scale=1.0, user-scalable=no, viewport-fit=cover">
<style>
:root {{
  --bg: #111;
  --text: #eee;
  --panel: #1b1b1b;
  --accent: #00eaff;
  --ui-scale: 0.85;
}}
* {{
  box-sizing: border-box;
  -webkit-tap-highlight-color: transparent;
}}
body {{
  margin: 0;
  display: flex;
  min-height: 100%;
  background: var(--bg);
  color: var(--text);
  font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
}}

#chapters {{
  width: 220px;
  background: var(--panel);
  overflow-y: auto;
  flex-shrink: 0;
}}

#chapters div {{
  padding: 8px 10px;
  font-size: 13px;
  cursor: pointer;
}}
#chapters .active {{
  background: #333;
}}
#content {{
  flex: 1 1 auto;
  padding: 16px;
  padding-bottom: 160px;
  overflow-y: auto;
  font-size: 1.15em;
  line-height: 1.2;
}}
.line {{
  padding: 0px 38px;
  margin: 2px 0;
  border-radius: 6px;
  line-height: 1.2;
}}
.line.active {{
  background: var(--accent);
  color: #000;
  font-weight: bold;
}}
.line.done {{
  opacity: .5;
}}
.line:empty {{
    display: none;
}}
.line[data-skip="1"] {{
    display: none;
}}

/* ... 其他CSS样式与原来相同 ... */

</style>
</head>
<body>
<div id="ui-layer">
  <div id="toggle" class="ui">📖</div>
  <div id="chapters" class="ui">
    {chapters_nav}
  </div>

  <!-- 控制面板等与原来相同 -->
  <div id="ctrlToggle" class="ui">🎛</div>
  <div id="ctrl" class="ui">
    速度 <span id="rateLabel">1.0×</span>
    <input id="rate" type="range" min="0.5" max="4" step="0.1" value="1">
    跟读停顿 <span id="pauseLabel">800 ms</span>
    <input id="pauseRange" type="range" min="0" max="3000" step="100" value="800">

    模式：
    <select id="repeatMode">
      <option value="line">每行重复</option>
      <option value="chapter">整章重复</option>
    </select>

    次数 <span id="repeatLabel">1</span>
    <input id="repeatCount" type="number" min="1" max="10" value="1" style="width:60px">

    速度增量 <span id="speedIncLabel">0</span>
    <input id="speedIncrement" type="number" step="0.1" min="0" max="1" value="0" style="width:60px">

    <button id="btn">▶ 播放 / 暂停</button>
  </div>

  <div id="themeToggle" class="ui">🎨</div>
  <div id="themeBar" class="ui">
    <button data-theme="dark">🌙 夜间</button>
    <button data-theme="paper">📜 纸书</button>
    <button data-theme="green">🌿 护眼</button>
    <button data-theme="blue">🌌 蓝夜</button>
    <button data-theme="coffee">☕ 咖啡</button>
  </div>
</div>

<div id="content"></div>

<div id="edge-left"></div>
<div id="edge-right"></div>

<audio id="player"></audio>
<script>
const chapters = {chapters_json};

const chaptersBox = document.getElementById("chapters");
const content = document.getElementById("content");
const player = document.getElementById("player");
const rate = document.getElementById("rate");
const rateLabel = document.getElementById("rateLabel");
const pauseRange = document.getElementById("pauseRange");
const pauseLabel = document.getElementById("pauseLabel");

const repeatCount = document.getElementById("repeatCount");
const repeatLabel = document.getElementById("repeatLabel");
repeatCount.oninput = () => repeatLabel.textContent = repeatCount.value;

const speedIncrement = document.getElementById("speedIncrement");
const speedIncLabel = document.getElementById("speedIncLabel");
speedIncrement.oninput = () => speedIncLabel.textContent = speedIncrement.value;

const repeatModeSelect = document.getElementById("repeatMode");
repeatModeSelect.onchange = () => {{
  chapterRepeatMode = repeatModeSelect.value;
}};

let chapterRepeatMode = "line";
let currentRepeat = 0;
let chapterIndex = 0;
let lineIndex = 0;
let playing = false;

const btn = document.getElementById("btn");
const toggle = document.getElementById("toggle");

// 点击正文 → 进入沉浸
content.addEventListener("click", () => {{
  document.body.classList.add("immersive");
}});
content.addEventListener("dblclick", () => {{
  document.body.classList.remove("immersive");
}});

// UI 内点击不触发正文
document.querySelectorAll(".ui").forEach(el => {{
  el.addEventListener("click", e => e.stopPropagation());
}});

// 点击边缘 → 召回 UI
document.querySelectorAll("#edge-left, #edge-right").forEach(edge => {{
  edge.addEventListener("click", () => {{
    document.body.classList.remove("immersive");
  }});
}});

const THEMES = {{
  dark: {{ bg: "#111111", text: "#eaeaea", panel: "#1b1b1b", accent: "#00eaff" }},
  paper: {{ bg: "#f4f1ea", text: "#2a2a2a", panel: "#e6e1d6", accent: "#ffd24d" }},
  green: {{ bg: "#0f1f1a", text: "#d9f3ea", panel: "#162a23", accent: "#3ddc97" }},
  blue: {{ bg: "#0d1b2a", text: "#e0e6ed", panel: "#13293d", accent: "#4cc9f0" }},
  coffee: {{ bg: "#1a1410", text: "#f1e7dc", panel: "#261c16", accent: "#c9a66b" }}
}};

function applyTheme(name) {{
  const t = THEMES[name];
  if (!t) return;
  Object.entries(t).forEach(([k, v]) => {{
    document.documentElement.style.setProperty("--" + k, v);
  }});
  localStorage.setItem("reader-theme", name);
}}

document.querySelectorAll("#themeBar button").forEach(btn => {{
  btn.onclick = () => {{
    applyTheme(btn.dataset.theme);
    document.getElementById("themeBar").classList.remove("show");
  }};
}});

applyTheme(localStorage.getItem("reader-theme") || "dark");

toggle.onclick = () => chaptersBox.classList.toggle("show");
rate.oninput = () => {{
  rateLabel.textContent = rate.value + "×";
  player.playbackRate = parseFloat(rate.value);
}};
pauseRange.oninput = () => {{
  pauseLabel.textContent = pauseRange.value + " ms";
}};
btn.onclick = () => {{
  playing = !playing;
  if (playing) playCurrentLine();
  else player.pause();
}};

function renderChapter(ci) {{
    chapterIndex = ci;
    lineIndex = 0;
    content.innerHTML = "";
    content.scrollTop = 0;

    // 更新章节导航高亮
    document.querySelectorAll("#chapters div").forEach((div, i) => {{
        div.classList.remove("active");
        if (i === ci) div.classList.add("active");
    }});

    chapters[ci].forEach((l, i) => {{
        const cleanText = (l.text || "").replace(/<br\\s*\\/?>/gi, "").replace(/\\[(ch|jp|en)\\]\\s*/g, "").replace(/\\[sound:[^\\]]*\\]/g, "");
        const temp = document.createElement("div");
        temp.innerHTML = cleanText;
        const visibleText = temp.textContent
        .replace(/\\u00a0/g, "")
        .replace(/\\s+/g, "")
        .replace(/[^\\u4e00-\\u9fffA-Za-z0-9]/g, "");

        const lineDiv = document.createElement("div");
        lineDiv.className = "line";
        lineDiv.innerHTML = cleanText;

        if (!visibleText && !lineDiv.querySelector("hr")) {{
            lineDiv.style.display = "none";
            lineDiv.dataset.skip = "1";
        }}

        lineDiv.onclick = () => {{
            lineIndex = i;
            playing = true;
            playCurrentLine();
        }};

        content.appendChild(lineDiv);
    }});
}}

function playCurrentLine() {{
    if (!playing) return;

    const lines = document.querySelectorAll(".line");
    if (!lines[lineIndex]) {{
        autoNextChapter();
        return;
    }}

    lines.forEach((l, idx) => {{
      l.classList.remove("active");
      if (idx < lineIndex) l.classList.add("done");
    }});
    const currentLine = lines[lineIndex];
    currentLine.classList.add("active");
    currentLine.scrollIntoView({{ block: "center", behavior: "smooth" }});

    const audioSrc = chapters[chapterIndex][lineIndex].audio;
    if (!audioSrc || currentLine.dataset.skip === "1") {{
        lineIndex++;
        playCurrentLine();
        return;
    }}

    player.src = audioSrc;
    player.playbackRate = parseFloat(rate.value) + parseFloat(speedIncrement.value) * currentRepeat;
    player.play();
}}

player.onended = () => {{
  const delay = Number(pauseRange.value);
  const lines = document.querySelectorAll(".line");
  const maxRepeat = parseInt(repeatCount.value);

  setTimeout(() => {{
    if (chapterRepeatMode === "line") {{
      if (currentRepeat + 1 < maxRepeat) {{
        currentRepeat++;
      }} else {{
        currentRepeat = 0;
        lineIndex++;
      }}
      playCurrentLine();
    }} else if (chapterRepeatMode === "chapter") {{
      lineIndex++;
      if (lineIndex >= lines.length) {{
        if (currentRepeat + 1 < maxRepeat) {{
          currentRepeat++;
          lineIndex = 0;
        }} else {{
          currentRepeat = 0;
          autoNextChapter();
          return;
        }}
      }}
      playCurrentLine();
    }}
  }}, delay);
}};

function autoNextChapter() {{
  if (chapterIndex + 1 >= chapters.length) {{
    playing = false;
    return;
  }}
  document.querySelectorAll("#chapters div").forEach(x => x.classList.remove("active"));
  chapterIndex++;
  chaptersBox.children[chapterIndex].classList.add("active");
  renderChapter(chapterIndex);
  playCurrentLine();
}}

// ctrl 折叠
document.getElementById("ctrlToggle").onclick = () => {{
  document.getElementById("ctrl").classList.toggle("show");
}};

// themeBar 折叠
document.getElementById("themeToggle").onclick = () => {{
  document.getElementById("themeBar").classList.toggle("show");
}};

renderChapter(0);
</script>
</body>
</html>"""

        html = html_template.format(
            batch_name=batch_name,
            chapters_nav=chapters_nav_html
        ).replace("{chapters_json}", chapters_json)

        return html

    except Exception as e:
        print(f"    ❌ 合并HTML失败: {e}")
        import traceback
        traceback.print_exc()
        return None

async def fast_combine_existing_html(txt_path, batch_size=None):
    if batch_size is None:
        batch_size = BATCH_SIZE
    """快速合并已有的单章节HTML文件"""
    name = os.path.splitext(os.path.basename(txt_path))[0]
    out_dir = os.path.join(ANKI_ROOT, name)
    perHtml_dir = os.path.join(out_dir, "perHtml")
    html_dir = os.path.join(out_dir, "html")

    print(f"\n📘 快速合并处理: {name}")
    print(f"📂 单章节目录: {perHtml_dir}")
    print(f"📂 批量目录: {html_dir}")

    # 检查单章节目录是否存在
    if not os.path.exists(perHtml_dir):
        print(f"❌ 单章节目录不存在，需要从头处理")
        return False

    # 获取所有单章节HTML文件
    safe_name = safe_filename(name)
    pattern = os.path.join(perHtml_dir, f"{safe_name}_*.html")
    single_files = sorted(glob.glob(pattern))

    if not single_files:
        print(f"❌ 未找到单章节HTML文件")
        return False

    print(f"📊 找到 {len(single_files)} 个单章节HTML文件")

    # 确定总章节数（从文件名中提取最大编号）
    max_chapter = 0
    for file_path in single_files:
        match = re.search(r'_(\d{4})\.html$', os.path.basename(file_path))
        if match:
            chapter_num = int(match.group(1))
            max_chapter = max(max_chapter, chapter_num)

    # 读取原始TXT文件确定总章节数
    with open(txt_path, encoding="utf-8") as f:
        lines = [l for l in f if l.strip()]

    df = pd.read_csv(
        StringIO("".join(lines)),
        sep="\t",
        header=None,
        skiprows=2,
        engine="python",
        dtype=str,
        on_bad_lines="skip"
    ).fillna("")

    total_chapters = len(df)
    print(f"📖 总章节数: {total_chapters}, 已生成: {max_chapter}")

    # 检查是否需要继续生成
    if max_chapter < total_chapters:
        print(f"⚠️ 需要继续生成 {total_chapters - max_chapter} 个章节")
        # 这里可以调用原来的处理逻辑，但只处理缺失的章节
        return False

    # 创建批量目录
    os.makedirs(html_dir, exist_ok=True)

    # 批量合并
    num_batches = (total_chapters + batch_size - 1) // batch_size
    print(f"\n📚 开始批量合并，共 {num_batches} 批")

    all_chapters_data = []

    # 先加载所有章节数据
    for i in range(1, total_chapters + 1):
        html_path = os.path.join(perHtml_dir, f"{safe_name}_{i:04d}.html")
        if os.path.exists(html_path):
            print(f"  📖 加载章节 {i:04d}/{total_chapters:04d}")
            chapter_data = extract_chapter_data_from_single_html(html_path)
            if chapter_data:
                all_chapters_data.append(chapter_data)
            else:
                print(f"  ❌ 章节 {i} 数据加载失败")
                all_chapters_data.append([])
        else:
            print(f"  ⚠️ 章节 {i} 文件不存在")
            all_chapters_data.append([])

    # 分批生成
    for batch_idx in range(num_batches):
        start = batch_idx * batch_size
        end = min((batch_idx + 1) * batch_size, total_chapters)
        batch_chapters = all_chapters_data[start:end]

        batch_name = f"{safe_name}_{batch_idx+1}"
        html_name = os.path.join(html_dir, f"{batch_name}.html")

        # 检查是否已存在
        if os.path.exists(html_name):
            print(f"  ⏭ 批量文件已存在，跳过: {batch_name}.html")
            continue

        print(f"\n  📝 生成批量 {batch_idx+1}/{num_batches}: {batch_name}.html")
        print(f"    包含章节 {start+1} 到 {end}")

        html_content = combine_chapters_to_batch_html(
            out_dir, batch_chapters, batch_name, start + 1, total_chapters
        )

        if html_content:
            with open(html_name, 'w', encoding='utf-8') as f:
                f.write(html_content)
            print(f"  ✅ 批量HTML生成成功")

    print(f"\n🎉 快速合并完成!")
    return True

async def process_one_txt_smart(txt_path):
    """智能处理TXT文件：优先使用快速合并，失败时回退到完整处理"""
    name = os.path.splitext(os.path.basename(txt_path))[0]

    print(f"\n{'='*50}")
    print(f"🚀 开始处理: {name}")
    print(f"{'='*50}")

    # 首先尝试快速合并
    success = await fast_combine_existing_html(txt_path)

    if not success:
        print(f"\n⚠️ 快速合并失败，使用完整处理模式")
        print(f"{'-'*50}")
        await process_one_txt(txt_path)
    else:
        print(f"\n✅ 已通过快速合并完成!")

async def main_batch_fast():
    """快速批量处理所有TXT文件"""
    txts = [f for f in sorted(os.listdir(ANKI_ROOT)) if f.lower().endswith(".txt")]
    print(f"📦 发现 {len(txts)} 个 TXT 文件")

    for i, f in enumerate(txts, 1):
        print(f"\n{'='*60}")
        print(f"🚀 TXT {i}/{len(txts)}: {f}")
        print(f"{'='*60}")
        await process_one_txt_smart(os.path.join(ANKI_ROOT, f))

    print("\n🎉 所有 TXT 快速处理完成")

# 使用新的快速处理函数

import glob
import json
import re
from pathlib import Path

def merge_single_html_to_batch(txt_path, batch_size=None):
    if batch_size is None:
        batch_size = BATCH_SIZE
    """直接合并单章节HTML文件为批量HTML"""
    name = os.path.splitext(os.path.basename(txt_path))[0]
    safe_name = safe_filename(name)
    out_dir = os.path.join(ANKI_ROOT, name)
    perHtml_dir = os.path.join(out_dir, "perHtml")
    html_dir = os.path.join(out_dir, "html")

    print(f"\n📘 开始合并: {name}")
    print(f"📂 单章节目录: {perHtml_dir}")

    if not os.path.exists(perHtml_dir):
        print(f"❌ 单章节目录不存在: {perHtml_dir}")
        return False

    # 获取所有单章节HTML文件，按数字排序
    html_files = []
    for f in sorted(os.listdir(perHtml_dir)):
        if f.endswith('.html') and f.startswith(safe_name):
            match = re.search(r'_(\d{4})\.html$', f)
            if match:
                chapter_num = int(match.group(1))
                html_files.append((chapter_num, os.path.join(perHtml_dir, f)))

    if not html_files:
        print(f"❌ 未找到单章节HTML文件")
        return False

    print(f"📊 找到 {len(html_files)} 个单章节文件")

    # 读取原始TXT确定总章节数
    try:
        with open(txt_path, encoding="utf-8") as f:
            lines = [l for l in f if l.strip()]

        df = pd.read_csv(
            StringIO("".join(lines)),
            sep="\t",
            header=None,
            skiprows=2,
            engine="python",
            dtype=str,
            on_bad_lines="skip"
        ).fillna("")

        total_chapters = len(df)
        print(f"📖 总章节数: {total_chapters}")
    except:
        total_chapters = len(html_files)
        print(f"⚠️ 无法读取TXT，使用文件数量作为总章节数: {total_chapters}")

    # 确保输出目录存在
    os.makedirs(html_dir, exist_ok=True)

    # 计算需要多少批次
    num_batches = (total_chapters + batch_size - 1) // batch_size
    print(f"📚 需要生成 {num_batches} 个批量文件")

    for batch_idx in range(num_batches):
        start = batch_idx * batch_size
        end = min((batch_idx + 1) * batch_size, total_chapters)

        batch_name = f"{safe_name}_{batch_idx+1}"
        output_path = os.path.join(html_dir, f"{batch_name}.html")

        # 检查是否已存在
        if os.path.exists(output_path):
            print(f"\n⏭ 批量文件已存在: {batch_name}.html")
            continue

        print(f"\n📝 创建批量 {batch_idx+1}/{num_batches}: {batch_name}.html")
        print(f"   包含章节 {start+1} 到 {end}")

        # 收集这个批次的章节数据
        batch_chapters_data = []

        for chapter_num in range(start + 1, end + 1):
            # 查找对应的HTML文件
            html_path = None
            for num, path in html_files:
                if num == chapter_num:
                    html_path = path
                    break

            if html_path and os.path.exists(html_path):
                print(f"   📖 加载章节 {chapter_num:04d}")
                chapter_data = extract_chapter_from_html(html_path)
                if chapter_data:
                    batch_chapters_data.append(chapter_data)
                else:
                    print(f"   ⚠️ 章节 {chapter_num} 数据为空，添加空数组")
                    batch_chapters_data.append([])
            else:
                print(f"   ⚠️ 章节 {chapter_num} 文件不存在，添加空数组")
                batch_chapters_data.append([])

        # 生成批量HTML
        if batch_chapters_data:
            generate_batch_html(output_path, batch_name, batch_chapters_data, start + 1)
        else:
            print(f"   ❌ 批量 {batch_idx+1} 没有数据")

    print(f"\n✅ 合并完成!")
    return True

def extract_chapter_from_html(html_path):
    """从单个HTML文件中提取章节数据（简化版）"""
    try:
        with open(html_path, 'r', encoding='utf-8') as f:
            content = f.read()

        # 查找 chapters 数据
        # 使用正则表达式匹配 JSON 数组
        pattern = r'const chapters\s*=\s*(\[\s*\[.*?\]\s*\])\s*;'
        match = re.search(pattern, content, re.DOTALL)

        if not match:
            # 尝试其他模式
            pattern = r'chapters\s*:\s*(\[\s*\[.*?\]\s*\])'
            match = re.search(pattern, content, re.DOTALL)

        if match:
            json_str = match.group(1)
            # 简单清理
            json_str = re.sub(r'//.*?\n', '', json_str)
            json_str = re.sub(r',\s*\]', ']', json_str)  # 修复尾随逗号
            json_str = re.sub(r',\s*}', '}', json_str)

            try:
                data = json.loads(json_str)
                if isinstance(data, list) and len(data) > 0:
                    return data[0]  # 返回第一个章节
            except json.JSONDecodeError as e:
                print(f"      ❌ JSON解析失败: {e}")
                # 尝试手动提取
                return extract_lines_manually(content)

        return extract_lines_manually(content)

    except Exception as e:
        print(f"      ❌ 提取失败: {e}")
        return []

def extract_lines_manually(html_content):
    """手动从HTML中提取行数据"""
    lines = []

    # 查找所有<div class="line">元素
    line_pattern = r'<div class="line"[^>]*>(.*?)</div>'
    line_matches = re.findall(line_pattern, html_content, re.DOTALL)

    for line_html in line_matches:
        # 查找音频数据
        audio_pattern = r'src="(data:audio/mp3;base64,[^"]+)"'
        audio_match = re.search(audio_pattern, line_html)
        audio = audio_match.group(1) if audio_match else ""

        lines.append({
            "text": line_html.strip(),
            "audio": audio
        })

    return lines

def generate_batch_html(output_path, batch_name, chapters_data, start_chapter_num):
    """生成批量HTML文件"""
    try:
        # 生成章节导航
        chapters_nav = ""
        for i, chapter_data in enumerate(chapters_data):
            chapter_num = start_chapter_num + i
            chapters_nav += f'<div onclick="renderChapter({i})">Chapter {chapter_num}</div>\n'

        # 生成完整的HTML
        chapters_json = json.dumps(chapters_data, ensure_ascii=False)

        # 读取模板文件或使用字符串
        html_template = """<!DOCTYPE html>
<html lang="zh">
<head>
<meta charset="UTF-8">
<title>{batch_name}</title>
<meta name="viewport" content="width=device-width, initial-scale=1.0, maximum-scale=1.0, user-scalable=no, viewport-fit=cover">
<style>
/* CSS样式（与之前相同） */
:root {{ --bg: #111; --text: #eee; --panel: #1b1b1b; --accent: #00eaff; --ui-scale: 0.85; }}
* {{ box-sizing: border-box; -webkit-tap-highlight-color: transparent; }}
body {{ margin: 0; display: flex; min-height: 100%; background: var(--bg); color: var(--text); font-family: -apple-system, sans-serif; }}

.line {{ padding: 0px 38px; margin: 2px 0; border-radius: 6px; line-height: 1.2; }}
.line.active {{ background: var(--accent); color: #000; font-weight: bold; }}
.line.done {{ opacity: .5; }}
.line:empty, .line[data-skip="1"] {{ display: none; }}

.ui {{ opacity: 1; visibility: visible; pointer-events: auto; }}
body.immersive .ui {{ opacity: 0; visibility: hidden; pointer-events: none; }}
#edge-left, #edge-right {{ position: fixed; top: 0; bottom: 0; width: 24px; z-index: 200; }}
#edge-left {{ left: 0; }} #edge-right {{ right: 0; }}
</style>
</head>
<body>
<div id="ui-layer">
  <div id="toggle" class="ui">📖</div>
  <div id="chapters" class="ui">
{chapters_nav}
  </div>

  <div id="ctrlToggle" class="ui">🎛</div>
  <div id="ctrl" class="ui">
    速度 <span id="rateLabel">1.0×</span>
    <input id="rate" type="range" min="0.5" max="4" step="0.1" value="1">
    跟读停顿 <span id="pauseLabel">800 ms</span>
    <input id="pauseRange" type="range" min="0" max="3000" step="100" value="800">
    模式：<select id="repeatMode"><option value="line">每行重复</option><option value="chapter">整章重复</option></select>
    次数 <span id="repeatLabel">1</span>
    <input id="repeatCount" type="number" min="1" max="10" value="1" style="width:60px">
    速度增量 <span id="speedIncLabel">0</span>
    <input id="speedIncrement" type="number" step="0.1" min="0" max="1" value="0" style="width:60px">
    <button id="btn">▶ 播放 / 暂停</button>
  </div>

  <div id="themeToggle" class="ui">🎨</div>
  <div id="themeBar" class="ui">
    <button data-theme="dark">🌙 夜间</button>
    <button data-theme="paper">📜 纸书</button>
    <button data-theme="green">🌿 护眼</button>
    <button data-theme="blue">🌌 蓝夜</button>
    <button data-theme="coffee">☕ 咖啡</button>
  </div>
</div>

<div id="content"></div>

<div id="edge-left"></div>
<div id="edge-right"></div>

<audio id="player"></audio>
<script>
const chapters = {chapters_json};

const chaptersBox = document.getElementById("chapters");
const content = document.getElementById("content");
const player = document.getElementById("player");
const rate = document.getElementById("rate");
const rateLabel = document.getElementById("rateLabel");
const pauseRange = document.getElementById("pauseRange");
const pauseLabel = document.getElementById("pauseLabel");
const repeatCount = document.getElementById("repeatCount");
const repeatLabel = document.getElementById("repeatLabel");
const speedIncrement = document.getElementById("speedIncrement");
const speedIncLabel = document.getElementById("speedIncLabel");
const repeatModeSelect = document.getElementById("repeatMode");

let chapterRepeatMode = "line";
let currentRepeat = 0;
let chapterIndex = 0;
let lineIndex = 0;
let playing = false;

const btn = document.getElementById("btn");
const toggle = document.getElementById("toggle");

// 主题设置
const THEMES = {{
  dark: {{ bg: "#111111", text: "#eaeaea", panel: "#1b1b1b", accent: "#00eaff" }},
  paper: {{ bg: "#f4f1ea", text: "#2a2a2a", panel: "#e6e1d6", accent: "#ffd24d" }},
  green: {{ bg: "#0f1f1a", text: "#d9f3ea", panel: "#162a23", accent: "#3ddc97" }},
  blue: {{ bg: "#0d1b2a", text: "#e0e6ed", panel: "#13293d", accent: "#4cc9f0" }},
  coffee: {{ bg: "#1a1410", text: "#f1e7dc", panel: "#261c16", accent: "#c9a66b" }}
}};

function applyTheme(name) {{
  const t = THEMES[name];
  if (!t) return;
  Object.entries(t).forEach(([k, v]) => {{
    document.documentElement.style.setProperty("--" + k, v);
  }});
  localStorage.setItem("reader-theme", name);
}}

// 事件监听
toggle.onclick = () => chaptersBox.classList.toggle("show");
rate.oninput = () => {{ rateLabel.textContent = rate.value + "×"; player.playbackRate = parseFloat(rate.value); }};
pauseRange.oninput = () => {{ pauseLabel.textContent = pauseRange.value + " ms"; }};
repeatCount.oninput = () => repeatLabel.textContent = repeatCount.value;
speedIncrement.oninput = () => speedIncLabel.textContent = speedIncrement.value;
repeatModeSelect.onchange = () => {{ chapterRepeatMode = repeatModeSelect.value; }};
btn.onclick = () => {{ playing = !playing; if (playing) playCurrentLine(); else player.pause(); }};

// 主题按钮
document.querySelectorAll("#themeBar button").forEach(btn => {{
  btn.onclick = () => {{ applyTheme(btn.dataset.theme); document.getElementById("themeBar").classList.remove("show"); }};
}});

// 控制面板折叠
document.getElementById("ctrlToggle").onclick = () => {{
  document.getElementById("ctrl").classList.toggle("show");
}};
document.getElementById("themeToggle").onclick = () => {{
  document.getElementById("themeBar").classList.toggle("show");
}};

// 沉浸模式
content.addEventListener("click", () => document.body.classList.add("immersive"));
content.addEventListener("dblclick", () => document.body.classList.remove("immersive"));
document.querySelectorAll(".ui").forEach(el => el.addEventListener("click", e => e.stopPropagation()));
document.querySelectorAll("#edge-left, #edge-right").forEach(edge => {{
  edge.addEventListener("click", () => document.body.classList.remove("immersive"));
}});

// 渲染章节
function renderChapter(ci) {{
    chapterIndex = ci;
    lineIndex = 0;
    content.innerHTML = "";
    content.scrollTop = 0;

    document.querySelectorAll("#chapters div").forEach((div, i) => {{
        div.classList.remove("active");
        if (i === ci) div.classList.add("active");
    }});

    chapters[ci].forEach((l, i) => {{
        const cleanText = (l.text || "").replace(/<br\\s*\\/?>/gi, "").replace(/\\[(ch|jp|en)\\]\\s*/g, "").replace(/\\[sound:[^\\]]*\\]/g, "");
        const temp = document.createElement("div");
        temp.innerHTML = cleanText;
        const visibleText = temp.textContent.replace(/\\u00a0|\\s+/g, "").replace(/[^\\u4e00-\\u9fffA-Za-z0-9]/g, "");

        const lineDiv = document.createElement("div");
        lineDiv.className = "line";
        lineDiv.innerHTML = cleanText;

        if (!visibleText && !lineDiv.querySelector("hr")) {{
            lineDiv.style.display = "none";
            lineDiv.dataset.skip = "1";
        }}

        lineDiv.onclick = () => {{ lineIndex = i; playing = true; playCurrentLine(); }};
        content.appendChild(lineDiv);
    }});
}}

function playCurrentLine() {{
    if (!playing) return;
    const lines = document.querySelectorAll(".line");
    if (!lines[lineIndex]) {{ autoNextChapter(); return; }}

    lines.forEach((l, idx) => {{ l.classList.remove("active"); if (idx < lineIndex) l.classList.add("done"); }});
    const currentLine = lines[lineIndex];
    currentLine.classList.add("active");
    currentLine.scrollIntoView({{ block: "center", behavior: "smooth" }});

    const audioSrc = chapters[chapterIndex][lineIndex].audio;
    if (!audioSrc || currentLine.dataset.skip === "1") {{ lineIndex++; playCurrentLine(); return; }}

    player.src = audioSrc;
    player.playbackRate = parseFloat(rate.value) + parseFloat(speedIncrement.value) * currentRepeat;
    player.play();
}}

player.onended = () => {{
  setTimeout(() => {{
    const maxRepeat = parseInt(repeatCount.value);
    if (chapterRepeatMode === "line") {{
      if (currentRepeat + 1 < maxRepeat) currentRepeat++;
      else {{ currentRepeat = 0; lineIndex++; }}
      playCurrentLine();
    }} else if (chapterRepeatMode === "chapter") {{
      lineIndex++;
      if (lineIndex >= document.querySelectorAll(".line").length) {{
        if (currentRepeat + 1 < maxRepeat) {{ currentRepeat++; lineIndex = 0; }}
        else {{ currentRepeat = 0; autoNextChapter(); return; }}
      }}
      playCurrentLine();
    }}
  }}, Number(pauseRange.value));
}};

function autoNextChapter() {{
  if (chapterIndex + 1 >= chapters.length) {{ playing = false; return; }}
  document.querySelectorAll("#chapters div").forEach(x => x.classList.remove("active"));
  chapterIndex++;
  chaptersBox.children[chapterIndex].classList.add("active");
  renderChapter(chapterIndex);
  playCurrentLine();
}}

// 初始化
applyTheme(localStorage.getItem("reader-theme") || "dark");
renderChapter(0);
chaptersBox.firstChild.classList.add("active");
</script>
</body>
</html>"""

        html = html_template.format(
            batch_name=batch_name,
            chapters_nav=chapters_nav,
            chapters_json=chapters_json
        )

        with open(output_path, 'w', encoding='utf-8') as f:
            f.write(html)

        print(f"   ✅ 批量HTML生成成功: {os.path.basename(output_path)}")
        return True

    except Exception as e:
        print(f"   ❌ 生成失败: {e}")
        import traceback
        traceback.print_exc()
        return False

async def process_txt_simple(txt_path):
    """简单的处理流程：先尝试合并，不行就从头开始"""
    name = os.path.splitext(os.path.basename(txt_path))[0]

    print(f"\n{'='*60}")
    print(f"🚀 处理: {name}")
    print(f"{'='*60}")

    # 第一步：尝试直接合并现有HTML
    print("\n1️⃣ 尝试合并现有HTML文件...")
    success = merge_single_html_to_batch(txt_path)

    if not success:
        print("\n2️⃣ 合并失败，需要生成缺失内容...")
        # 调用你的原始process_one_txt函数
        await process_one_txt(txt_path)
    else:
        print("\n✅ 成功通过合并完成!")

async def main_all_simple():
    """处理所有TXT文件的简单版本"""
    txts = [f for f in sorted(os.listdir(ANKI_ROOT)) if f.lower().endswith(".txt")]
    print(f"📦 发现 {len(txts)} 个 TXT 文件")

    for i, f in enumerate(txts, 1):
        await process_txt_simple(os.path.join(ANKI_ROOT, f))

    print("\n🎉 所有文件处理完成!")

#await process_txt_simple(input_file)


In [ ]:
# ============================================================
# 🧠 艾宾浩斯学习系统生成函数 - 完整交互版


In [ ]:
# ============================================================

def generate_super_language_learning_system_word(html_path, chapters):
    """
    生成超级语言学习系统 - 完整交互版
    功能：艾宾浩斯记忆曲线 + 智能单词本 + 语法收藏 + 进度追踪
    """
    import json
    from datetime import datetime
    import os
    import hashlib
    import base64

    print(f"  📊 正在为 {len(chapters)} 章内容生成艾宾浩斯学习系统...")

    if len(chapters) == 0:
        print("  ❌ 错误：章节数据为空，无法生成学习系统")
        return

    # 将章节数据转换为JSON
    chapters_json = json.dumps(chapters, ensure_ascii=False)

    # 计算文件指纹
    file_fingerprint = hashlib.md5(chapters_json.encode()).hexdigest()[:8]

    # 计算统计数据
    total_chapters = len(chapters)
    total_lines = sum(len(c) for c in chapters)
    total_audio = sum(1 for c in chapters for l in c if l.get('audio'))

    # 构建学习项列表
    learning_items = []
    for chap_idx, chapter in enumerate(chapters):
        for line_idx, line in enumerate(chapter):
            if line.get('audio') and line.get('text'):
                # 解析文本
                text = line['text']
                soup = BeautifulSoup(text, "html.parser")
                plain_text = remove_sound_tags(soup.get_text().strip())

                # 判断语言类型
                lang = 'unknown'
                if plain_text.startswith('[ch]'):
                    lang = 'ch'
                elif plain_text.startswith('[jp]'):
                    lang = 'jp'
                elif plain_text.startswith('[en]'):
                    lang = 'en'

                # 提取实际内容（移除所有语言标签）
                content = re.sub(r'\[sound:[^\]]*\]', '', re.sub(r'\[(ch|jp|en)\]\s*', '', plain_text)).strip()

                learning_items.append({
                    'id': f'{chap_idx:04d}_{line_idx:03d}',
                    'chapter': chap_idx + 1,
                    'line': line_idx,
                    'text': text,
                    'content': content[:50] + '...' if len(content) > 50 else content,
                    'audio': line['audio'],
                    'lang': lang,
                    'mastery': 0,
                    'review_count': 0,
                    'last_review': None,
                    'next_review': 0,
                    'added_to_notebook': False,
                    'tags': []
                })

    items_json = json.dumps(learning_items, ensure_ascii=False)

    # ========== HTML内容 ==========
    html_content = f'''<!DOCTYPE html>
<html lang="zh">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0, maximum-scale=1.0, user-scalable=no, viewport-fit=cover">
    <title>🧠 艾宾浩斯·智能语言学习</title>
    <meta name="theme-color" content="#6C5CE7">
    <meta name="mobile-web-app-capable" content="yes">
    <meta name="apple-mobile-web-app-capable" content="yes">
    <style>
        * {{
            margin: 0;
            padding: 0;
            box-sizing: border-box;
            font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, sans-serif;
        }}

        :root {{
            --primary: #6C5CE7;
            --primary-dark: #5649C0;
            --secondary: #00B894;
            --accent: #FD79A8;
            --warning: #FDCB6E;
            --danger: #FF7675;
            --dark: #2D3436;
            --light: #F5F6FA;
            --surface: #FFFFFF;
        }}

        body {{
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            min-height: 100vh;
            padding: 16px;
        }}

        .app {{
            max-width: 1000px;
            margin: 0 auto;
            background: rgba(255, 255, 255, 0.95);
            backdrop-filter: blur(20px);
            border-radius: 40px;
            padding: 24px;
            box-shadow: 0 25px 50px -12px rgba(0,0,0,0.25);
        }}

        /* 导航栏 */
        .nav-tabs {{
            display: flex;
            gap: 8px;
            margin-bottom: 24px;
            padding: 8px;
            background: rgba(0,0,0,0.02);
            border-radius: 30px;
            flex-wrap: wrap;
        }}

        .nav-tab {{
            flex: 1;
            min-width: 80px;
            padding: 12px 16px;
            border: none;
            border-radius: 25px;
            font-size: 15px;
            font-weight: 600;
            display: flex;
            align-items: center;
            justify-content: center;
            gap: 6px;
            background: transparent;
            color: var(--dark);
            cursor: pointer;
            transition: all 0.3s;
        }}

        .nav-tab.active {{
            background: var(--primary);
            color: white;
            box-shadow: 0 10px 20px -5px rgba(108,92,231,0.4);
        }}

        /* 统计卡片 */
        .dashboard {{
            display: grid;
            grid-template-columns: repeat(auto-fit, minmax(160px, 1fr));
            gap: 16px;
            margin-bottom: 24px;
        }}

        .stat-card {{
            background: linear-gradient(135deg, var(--primary), var(--primary-dark));
            padding: 20px;
            border-radius: 24px;
            color: white;
            position: relative;
            overflow: hidden;
        }}

        .stat-card::before {{
            content: '🧠';
            position: absolute;
            right: 16px;
            bottom: 16px;
            font-size: 40px;
            opacity: 0.2;
        }}

        .stat-label {{
            font-size: 14px;
            opacity: 0.9;
            margin-bottom: 8px;
        }}

        .stat-value {{
            font-size: 32px;
            font-weight: 700;
            margin-bottom: 4px;
        }}

        /* 学习模式选择 */
        .mode-selector {{
            display: grid;
            grid-template-columns: repeat(auto-fit, minmax(140px, 1fr));
            gap: 12px;
            margin-bottom: 24px;
            padding: 16px;
            background: rgba(108,92,231,0.1);
            border-radius: 20px;
        }}

        .mode-btn {{
            padding: 16px;
            border: 2px solid transparent;
            border-radius: 16px;
            background: white;
            font-weight: 600;
            display: flex;
            flex-direction: column;
            align-items: center;
            gap: 8px;
            cursor: pointer;
            transition: all 0.2s;
        }}

        .mode-btn.active {{
            border-color: var(--primary);
            background: rgba(108,92,231,0.05);
        }}

        .mode-icon {{
            font-size: 28px;
        }}

        /* 内容区域 */
        .content-area {{
            background: white;
            border-radius: 24px;
            padding: 20px;
            margin-bottom: 20px;
            min-height: 300px;
        }}

        /* 单词卡片 */
        .word-card {{
            display: flex;
            align-items: center;
            padding: 16px;
            border-radius: 16px;
            background: var(--light);
            margin-bottom: 12px;
            transition: all 0.2s;
            border-left: 6px solid var(--primary);
        }}

        .word-info {{
            flex: 1;
        }}

        .word-text {{
            font-size: 18px;
            font-weight: 600;
            margin-bottom: 4px;
        }}

        .word-meta {{
            display: flex;
            gap: 12px;
            font-size: 12px;
            color: #666;
        }}

        .play-btn {{
            width: 44px;
            height: 44px;
            border: none;
            border-radius: 50%;
            background: var(--secondary);
            color: white;
            font-size: 20px;
            display: flex;
            align-items: center;
            justify-content: center;
            cursor: pointer;
            margin-right: 12px;
            flex-shrink: 0;
        }}

        .play-btn.playing {{
            background: var(--warning);
            animation: pulse 2s infinite;
        }}

        .add-btn {{
            padding: 8px 16px;
            border: none;
            border-radius: 20px;
            background: var(--primary);
            color: white;
            font-size: 14px;
            cursor: pointer;
            margin-left: 8px;
        }}

        /* 复习面板 */
        .review-panel {{
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            border-radius: 24px;
            padding: 24px;
            color: white;
            margin-bottom: 24px;
        }}

        .review-item {{
            background: rgba(255,255,255,0.15);
            backdrop-filter: blur(10px);
            border-radius: 20px;
            padding: 20px;
            margin-bottom: 16px;
        }}

        .review-content {{
            font-size: 20px;
            font-weight: 600;
            margin-bottom: 16px;
        }}

        .review-actions {{
            display: flex;
            gap: 8px;
            flex-wrap: wrap;
        }}

        .review-btn {{
            flex: 1;
            min-width: 80px;
            padding: 12px;
            border: none;
            border-radius: 12px;
            font-weight: 600;
            display: flex;
            align-items: center;
            justify-content: center;
            gap: 6px;
            cursor: pointer;
        }}

        .btn-easy {{ background: var(--secondary); color: white; }}
        .btn-good {{ background: var(--primary); color: white; }}
        .btn-hard {{ background: var(--warning); color: var(--dark); }}
        .btn-again {{ background: var(--danger); color: white; }}

        /* 单词本 */
        .notebook-header {{
            display: flex;
            justify-content: space-between;
            align-items: center;
            margin-bottom: 20px;
        }}

        .mastery-bar {{
            width: 100px;
            height: 6px;
            background: rgba(0,0,0,0.1);
            border-radius: 3px;
            margin-top: 8px;
        }}

        .mastery-fill {{
            height: 100%;
            background: linear-gradient(90deg, var(--secondary), #00CEC9);
            border-radius: 3px;
            transition: width 0.3s;
        }}

        .status-bar {{
            position: sticky;
            bottom: 0;
            background: white;
            padding: 16px;
            border-radius: 30px;
            box-shadow: 0 -4px 20px rgba(0,0,0,0.1);
            display: flex;
            justify-content: space-between;
            align-items: center;
            margin-top: 20px;
            font-size: 14px;
        }}

        @keyframes pulse {{
            0% {{ box-shadow: 0 0 0 0 rgba(255,152,0,0.4); }}
            70% {{ box-shadow: 0 0 0 10px rgba(255,152,0,0); }}
            100% {{ box-shadow: 0 0 0 0 rgba(255,152,0,0); }}
        }}

        @media (max-width: 600px) {{
            .app {{ padding: 16px; }}
            .nav-tab {{ padding: 10px; font-size: 14px; }}
            .stat-value {{ font-size: 28px; }}
        }}
    </style>
</head>
<body>
    <div class="app">
        <!-- 头部 -->
        <div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 20px;">
            <h1 style="font-size: 26px; background: linear-gradient(135deg, var(--primary), var(--accent)); -webkit-background-clip: text; -webkit-text-fill-color: transparent;">
                🧠 艾宾浩斯·智能学习
            </h1>
            <span style="padding: 8px 16px; background: var(--light); border-radius: 20px; font-size: 12px;">
                🔑 {file_fingerprint}
            </span>
        </div>

        <!-- 导航标签 -->
        <div class="nav-tabs">
            <button class="nav-tab active" onclick="switchTab('learn')">📚 学习</button>
            <button class="nav-tab" onclick="switchTab('review')">
                🔄 复习
                <span id="reviewBadge" style="background: var(--danger); color: white; padding: 2px 8px; border-radius: 12px; font-size: 12px;">0</span>
            </button>
            <button class="nav-tab" onclick="switchTab('notebook')">
                📔 单词本
                <span id="notebookBadge" style="background: var(--secondary); color: white; padding: 2px 8px; border-radius: 12px; font-size: 12px;">0</span>
            </button>
            <button class="nav-tab" onclick="switchTab('stats')">📊 统计</button>
        </div>

        <!-- 学习面板 -->
        <div id="learnPanel">
            <!-- 统计卡片 -->
            <div class="dashboard">
                <div class="stat-card">
                    <div class="stat-label">今日复习</div>
                    <div class="stat-value" id="todayCount">0</div>
                    <div class="stat-sub">艾宾浩斯曲线</div>
                </div>
                <div class="stat-card" style="background: linear-gradient(135deg, #00B894, #00CEC9);">
                    <div class="stat-label">单词本</div>
                    <div class="stat-value" id="notebookCount">0</div>
                    <div class="stat-sub">已掌握 <span id="masteredCount">0</span></div>
                </div>
                <div class="stat-card" style="background: linear-gradient(135deg, #FD79A8, #E84342);">
                    <div class="stat-label">记忆保持</div>
                    <div class="stat-value" id="retentionRate">0%</div>
                    <div class="stat-sub">艾宾浩斯预测</div>
                </div>
            </div>

            <!-- 学习模式 -->
            <div class="mode-selector">
                <button class="mode-btn active" onclick="setMode('ebb')">
                    <span class="mode-icon">🧠</span>
                    <span>艾宾浩斯模式</span>
                </button>
                <button class="mode-btn" onclick="setMode('sequential')">
                    <span class="mode-icon">📖</span>
                    <span>顺序模式</span>
                </button>
                <button class="mode-btn" onclick="setMode('random')">
                    <span class="mode-icon">🎲</span>
                    <span>随机模式</span>
                </button>
                <button class="mode-btn" onclick="setMode('weakness')">
                    <span class="mode-icon">🎯</span>
                    <span>薄弱突破</span>
                </button>
            </div>

            <!-- 学习内容 -->
            <div class="content-area" id="learningContent">
                <div style="text-align: center; padding: 40px; color: #666;">
                    🎯 选择学习模式开始
                </div>
            </div>
        </div>

        <!-- 复习面板（隐藏） -->
        <div id="reviewPanel" style="display: none;">
            <div class="review-panel">
                <h2 style="margin-bottom: 20px;">📅 今日艾宾浩斯复习</h2>
                <div id="reviewList"></div>
            </div>
        </div>

        <!-- 单词本面板（隐藏） -->
        <div id="notebookPanel" style="display: none;">
            <div style="padding: 20px;">
                <div class="notebook-header">
                    <h2>📔 我的单词本</h2>
                    <span id="notebookStats">0/0</span>
                </div>
                <div id="wordList"></div>
            </div>
        </div>

        <!-- 统计面板（隐藏） -->
        <div id="statsPanel" style="display: none;">
            <div style="padding: 20px;">
                <h2>📊 学习统计</h2>
                <div style="margin-top: 20px;">
                    <p>总学习次数: <span id="totalReviews">0</span></p>
                    <p>掌握单词: <span id="masteredWords">0</span></p>
                    <p>学习天数: <span id="studyDays">0</span></p>
                </div>
            </div>
        </div>

        <!-- 状态栏 -->
        <div class="status-bar">
            <span>📚 {total_chapters}章 · {total_lines}句</span>
            <span>🎵 {total_audio}音频</span>
            <span id="currentTime">{datetime.now().strftime('%H:%M')}</span>
        </div>
    </div>

    <script>
        // ========== 学习数据 ==========
        const LEARNING_ITEMS = {items_json};
        const TOTAL_CHAPTERS = {total_chapters};
        const FILE_ID = "{file_fingerprint}";

        // ========== 本地存储 ==========
        class StorageManager {{
            constructor() {{
                this.key = `EBBINGHAUS_${{FILE_ID}}`;
                this.load();
            }}

            load() {{
                const saved = localStorage.getItem(this.key);
                if (saved) {{
                    try {{
                        const data = JSON.parse(saved);
                        LEARNING_ITEMS.forEach((item, index) => {{
                            if (data[index]) {{
                                item.mastery = data[index].mastery || 0;
                                item.review_count = data[index].review_count || 0;
                                item.added_to_notebook = data[index].added_to_notebook || false;
                                item.last_review = data[index].last_review;
                                item.next_review = data[index].next_review || 0;
                            }}
                        }});
                    }} catch (e) {{
                        console.error('加载失败:', e);
                    }}
                }}
                this.save();
            }}

            save() {{
                const data = LEARNING_ITEMS.map(item => ({{
                    mastery: item.mastery,
                    review_count: item.review_count,
                    added_to_notebook: item.added_to_notebook,
                    last_review: item.last_review,
                    next_review: item.next_review
                }}));
                localStorage.setItem(this.key, JSON.stringify(data));
            }}

            addToNotebook(itemId) {{
                const item = LEARNING_ITEMS.find(i => i.id === itemId);
                if (item) {{
                    item.added_to_notebook = true;
                    item.added_date = new Date().toISOString();
                    this.save();
                    this.showNotification('✅ 已加入单词本');
                    return true;
                }}
                return false;
            }}

            removeFromNotebook(itemId) {{
                const item = LEARNING_ITEMS.find(i => i.id === itemId);
                if (item) {{
                    item.added_to_notebook = false;
                    this.save();
                    this.showNotification('🗑️ 已从单词本移除');
                    return true;
                }}
                return false;
            }}

            recordReview(itemId, difficulty) {{
                const item = LEARNING_ITEMS.find(i => i.id === itemId);
                if (!item) return;

                item.review_count += 1;
                item.last_review = new Date().toISOString();

                // 根据难度调整掌握度
                switch(difficulty) {{
                    case 'easy': item.mastery = Math.min(100, item.mastery + 20); break;
                    case 'good': item.mastery = Math.min(100, item.mastery + 15); break;
                    case 'hard': item.mastery = Math.min(100, item.mastery + 5); break;
                    case 'again': item.mastery = Math.max(0, item.mastery - 10); break;
                }}

                // 艾宾浩斯复习周期
                if (item.mastery < 30) item.next_review = 0;
                else if (item.mastery < 50) item.next_review = 1;
                else if (item.mastery < 70) item.next_review = 2;
                else if (item.mastery < 85) item.next_review = 4;
                else if (item.mastery < 95) item.next_review = 7;
                else item.next_review = 15;

                this.save();
                this.updateUI();
            }}

            getTodayReviews() {{
                const today = new Date();
                today.setHours(0, 0, 0, 0);

                return LEARNING_ITEMS.filter(item => {{
                    if (!item.last_review) return false;
                    if (item.mastery >= 100) return false;
                    const lastReview = new Date(item.last_review);
                    const daysDiff = Math.floor((today - lastReview) / (1000 * 60 * 60 * 24));
                    return daysDiff >= item.next_review;
                }});
            }}

            showNotification(msg) {{
                alert(msg);
            }}

            updateUI() {{
                updateStats();
                updateReviewBadge();
                loadLearningContent();
            }}
        }}

        // ========== 全局变量 ==========
        const storage = new StorageManager();
        let currentMode = 'ebb';
        let currentAudio = null;

        // ========== UI函数 ==========
        function switchTab(tab) {{
            document.querySelectorAll('.nav-tab').forEach(t => t.classList.remove('active'));
            event.currentTarget.classList.add('active');

            document.getElementById('learnPanel').style.display = tab === 'learn' ? 'block' : 'none';
            document.getElementById('reviewPanel').style.display = tab === 'review' ? 'block' : 'none';
            document.getElementById('notebookPanel').style.display = tab === 'notebook' ? 'block' : 'none';
            document.getElementById('statsPanel').style.display = tab === 'stats' ? 'block' : 'none';

            if (tab === 'review') loadReviews();
            if (tab === 'notebook') loadNotebook();
            if (tab === 'stats') loadStats();
        }}

        function setMode(mode) {{
            currentMode = mode;
            document.querySelectorAll('.mode-btn').forEach(btn => btn.classList.remove('active'));
            event.currentTarget.classList.add('active');
            loadLearningContent();
        }}

        function loadLearningContent() {{
            const content = document.getElementById('learningContent');
            let items = [];

            switch(currentMode) {{
                case 'ebb':
                    items = storage.getTodayReviews().slice(0, 10);
                    break;
                case 'sequential':
                    items = LEARNING_ITEMS.filter(item => item.chapter === 1).slice(0, 10);
                    break;
                case 'random':
                    items = [...LEARNING_ITEMS].sort(() => Math.random() - 0.5).slice(0, 10);
                    break;
                case 'weakness':
                    items = LEARNING_ITEMS.filter(item => item.mastery < 30).slice(0, 10);
                    break;
            }}

            if (items.length === 0) {{
                content.innerHTML = '<div style="text-align: center; padding: 40px; color: #666;">🎉 今天没有学习任务，休息一下吧！</div>';
                return;
            }}

            let html = '';
            items.forEach(item => {{
                html += `
                    <div class="word-card">
                        <button class="play-btn" onclick="playAudio('${{item.audio}}', this)">▶</button>
                        <div class="word-info">
                            <div class="word-text">${{item.content}}</div>
                            <div class="word-meta">
                                <span>📌 第${{item.chapter}}章</span>
                                <span>🎯 ${{item.mastery}}%</span>
                                <span>🔄 ${{item.review_count}}次</span>
                            </div>
                        </div>
                        ${{item.added_to_notebook ?
                            '<button class="add-btn" style="background: #666;" onclick="removeFromNotebook(\\'' + item.id + '\\')">✓ 已收藏</button>' :
                            '<button class="add-btn" onclick="addToNotebook(\\'' + item.id + '\\')">📚 收藏</button>'
                        }}
                    </div>
                `;
            }});

            content.innerHTML = html;
        }}

        function loadReviews() {{
            const reviews = storage.getTodayReviews();
            const list = document.getElementById('reviewList');

            if (reviews.length === 0) {{
                list.innerHTML = '<div style="text-align: center; padding: 40px; color: white;">🎯 今日复习已完成！</div>';
                return;
            }}

            let html = '';
            reviews.slice(0, 5).forEach(item => {{
                html += `
                    <div class="review-item">
                        <div class="review-content">${{item.content}}</div>
                        <div class="word-meta" style="color: rgba(255,255,255,0.8); margin-bottom: 16px;">
                            <span>📌 第${{item.chapter}}章</span>
                            <span>🎯 掌握度 ${{item.mastery}}%</span>
                        </div>
                        <div class="review-actions">
                            <button class="review-btn btn-again" onclick="recordReview('${{item.id}}', 'again')">🔁 再记不住</button>
                            <button class="review-btn btn-hard" onclick="recordReview('${{item.id}}', 'hard')">😓 有点难</button>
                            <button class="review-btn btn-good" onclick="recordReview('${{item.id}}', 'good')">😊 记住了</button>
                            <button class="review-btn btn-easy" onclick="recordReview('${{item.id}}', 'easy')">🌟 太简单</button>
                        </div>
                    </div>
                `;
            }});

            list.innerHTML = html;
        }}

        function loadNotebook() {{
            const notebook = LEARNING_ITEMS.filter(item => item.added_to_notebook);
            const list = document.getElementById('wordList');

            document.getElementById('notebookCount').textContent = notebook.length;
            document.getElementById('notebookBadge').textContent = notebook.length;

            let html = '';
            notebook.forEach(item => {{
                html += `
                    <div class="word-card">
                        <button class="play-btn" onclick="playAudio('${{item.audio}}', this)">▶</button>
                        <div class="word-info">
                            <div class="word-text">${{item.content}}</div>
                            <div class="word-meta">
                                <span>📌 第${{item.chapter}}章</span>
                                <span>🎯 掌握度 ${{item.mastery}}%</span>
                            </div>
                            <div class="mastery-bar">
                                <div class="mastery-fill" style="width: ${{item.mastery}}%;"></div>
                            </div>
                        </div>
                        <button class="add-btn" style="background: var(--danger);" onclick="removeFromNotebook('${{item.id}}')">🗑️ 移除</button>
                    </div>
                `;
            }});

            list.innerHTML = html || '<div style="text-align: center; padding: 40px; color: #666;">📭 单词本还是空的，在学习页面点击📚收藏</div>';
        }}

        function loadStats() {{
            const mastered = LEARNING_ITEMS.filter(i => i.mastery >= 80).length;
            const totalReviews = LEARNING_ITEMS.reduce((sum, i) => sum + i.review_count, 0);

            document.getElementById('totalReviews').textContent = totalReviews;
            document.getElementById('masteredWords').textContent = mastered;
            document.getElementById('studyDays').textContent = Math.floor(totalReviews / 10) || 1;
        }}

        function updateStats() {{
            const todayReviews = storage.getTodayReviews();
            const notebook = LEARNING_ITEMS.filter(i => i.added_to_notebook);
            const mastered = notebook.filter(i => i.mastery >= 80).length;
            const avgMastery = notebook.length ?
                Math.round(notebook.reduce((sum, i) => sum + i.mastery, 0) / notebook.length) : 0;

            document.getElementById('todayCount').textContent = todayReviews.length;
            document.getElementById('reviewBadge').textContent = todayReviews.length;
            document.getElementById('notebookCount').textContent = notebook.length;
            document.getElementById('notebookBadge').textContent = notebook.length;
            document.getElementById('masteredCount').textContent = mastered;
            document.getElementById('retentionRate').textContent = avgMastery + '%';
        }}

        function updateReviewBadge() {{
            const count = storage.getTodayReviews().length;
            document.getElementById('reviewBadge').textContent = count;
        }}

        // ========== 音频播放 ==========
        function playAudio(base64Data, btn) {{
            if (currentAudio) {{
                currentAudio.pause();
                currentAudio.currentTime = 0;
            }}

            const audio = new Audio(base64Data);
            audio.play();
            currentAudio = audio;

            document.querySelectorAll('.play-btn.playing').forEach(b => b.classList.remove('playing'));
            if (btn) btn.classList.add('playing');

            audio.onended = () => {{
                if (btn) btn.classList.remove('playing');
                currentAudio = null;
            }};
        }}

        // ========== 全局函数 ==========
        window.switchTab = switchTab;
        window.setMode = setMode;
        window.playAudio = playAudio;
        window.addToNotebook = (id) => {{
            storage.addToNotebook(id);
            loadLearningContent();
            loadNotebook();
            updateStats();
        }};
        window.removeFromNotebook = (id) => {{
            storage.removeFromNotebook(id);
            loadLearningContent();
            loadNotebook();
            updateStats();
        }};
        window.recordReview = (id, difficulty) => {{
            storage.recordReview(id, difficulty);
            loadReviews();
            loadLearningContent();
            updateStats();
        }};

        // ========== 初始化 ==========
        window.addEventListener('load', () => {{
            updateStats();
            loadLearningContent();
            console.log('✅ 艾宾浩斯学习系统已启动', FILE_ID);
        }});
    </script>
</body>
</html>
    '''

    # 写入文件
    with open(html_path, 'w', encoding='utf-8') as f:
        f.write(html_content)

    # 计算文件大小
    file_size = os.path.getsize(html_path) // 1024

    print(f"""
╔══════════════════════════════════════════════════════════════╗
║  ✅ 艾宾浩斯学习系统 - 完整交互版 生成成功                  ║
╠══════════════════════════════════════════════════════════════╣
║  📁 文件: {os.path.basename(html_path)}
║  📊 章节: {total_chapters} 章
║  📝 学习点: {len(learning_items)} 个
║  🎵 音频: {total_audio} 个
║  💾 大小: {file_size} KB
║  🔑 指纹: {file_fingerprint}
║
║  ✨ 已启用功能:
║     • 🧠 艾宾浩斯记忆曲线 - 智能复习提醒
║     • 📔 智能单词本 - 点击📚收藏
║     • 🔄 4种学习模式 - 艾宾浩斯/顺序/随机/薄弱
║     • 📊 掌握度追踪 - 复习反馈自动调整
║     • 💾 本地存储 - 数据永久保存在手机
║
║  📱 使用说明:
║     1. 复制到手机用浏览器打开
║     2. 点击「学习」标签开始
║     3. 点击📚收藏生词到单词本
║     4. 点击「复习」标签完成今日任务
║     5. 数据自动保存，下次打开不丢失
║
╚══════════════════════════════════════════════════════════════╝
    """)

    return html_path


In [ ]:
# ============================================================
# 🧠 艾宾浩斯学习系统生成函数 - 变量定义修复版


In [ ]:
# ============================================================

def generate_super_language_learning_system_total(html_path, chapters):
    """
    生成超级语言学习系统 - 变量定义修复版
    功能：自动朗读 + 手动收藏生词/语法 + 艾宾浩斯复习
    """
    import json
    from datetime import datetime
    import os
    import hashlib
    import base64
    import re
    from bs4 import BeautifulSoup

    print(f"  📊 正在为 {len(chapters)} 章内容生成艾宾浩斯学习系统...")

    if len(chapters) == 0:
        print("  ❌ 错误：章节数据为空，无法生成学习系统")
        return

    # 计算统计数据
    total_chapters = len(chapters)
    total_lines = sum(len(c) for c in chapters)
    total_audio = sum(1 for c in chapters for l in c if l.get('audio') and len(l.get('audio', '')) > 100)

    print(f"  📊 总章节: {total_chapters}, 总行数: {total_lines}, 总音频: {total_audio}")

    # ========== 构建完整的章节数据，区分可学习行和只读行 ==========
    simplified_chapters = []

    # 定义变量用于统计
    total_learnable = 0
    total_readonly = 0
    total_valid_audio = 0

    for chap_idx, chapter in enumerate(chapters):
        simplified_lines = []
        for line_idx, line in enumerate(chapter):
            # 提取纯文本内容
            text = line.get('text', '')
            audio = line.get('audio', '')

            # 使用BeautifulSoup提取纯文本
            soup = BeautifulSoup(text, "html.parser")
            plain_text = remove_sound_tags(soup.get_text().strip())

            # 判断语言类型 - 只有以 [ch]、[jp]、[en] 开头的行才是可学习行
            lang = 'unknown'
            is_learnable = False

            if plain_text.startswith('[ch]'):
                lang = 'ch'
                is_learnable = True
            elif plain_text.startswith('[jp]'):
                lang = 'jp'
                is_learnable = True
            elif plain_text.startswith('[en]'):
                lang = 'en'
                is_learnable = True

            # 移除所有语言标签（不仅是行首的）
            content = re.sub(r'\[sound:[^\]]*\]', '', re.sub(r'\[(ch|jp|en)\]\s*', '', plain_text)).strip()

            # 如果内容为空，跳过
            if not content:
                continue

            # 确保音频数据完整
            has_valid_audio = False
            if audio and len(audio) > 100:
                if not audio.startswith('data:audio'):
                    audio = 'data:audio/mp3;base64,' + audio
                has_valid_audio = True
                if is_learnable:  # 只统计可学习行的音频
                    total_valid_audio += 1

            # 更新统计
            if is_learnable:
                total_learnable += 1
            else:
                total_readonly += 1

            simplified_lines.append({
                'id': f'{chap_idx:04d}_{line_idx:03d}',
                'content': content,
                'audio': audio if has_valid_audio else '',
                'lang': lang,
                'chapter': chap_idx + 1,
                'has_audio': has_valid_audio,
                'is_learnable': is_learnable  # 标记是否可学习
            })

        if simplified_lines:
            simplified_chapters.append(simplified_lines)

    print(f"  📝 处理后章节数: {len(simplified_chapters)}")
    print(f"  📝 可学习语句: {total_learnable} 句, 只读语句: {total_readonly} 句")
    print(f"  📝 有效音频: {total_valid_audio} 个")

    # ========== 构建学习项列表（只有可学习行） ==========
    learning_items = []
    for chap_idx, chapter in enumerate(simplified_chapters):
        for line in chapter:
            if line.get('is_learnable'):  # 只有可学习行才加入学习项
                learning_items.append({
                    'id': line['id'],
                    'chapter': line['chapter'],
                    'content': line['content'][:100] + '...' if len(line['content']) > 100 else line['content'],
                    'lang': line['lang'],
                    'has_audio': line.get('has_audio', False),
                    'mastery': 0,
                    'review_count': 0,
                    'added_to_notebook': False,
                    'added_to_grammar': False,
                    'grammar_note': '',
                    'last_review': None,
                    'next_review': 0
                })

    # ========== 将数据转换为JSON字符串 ==========
    chapters_json_str = json.dumps(simplified_chapters, ensure_ascii=False)
    items_json_str = json.dumps(learning_items, ensure_ascii=False)

    # ========== 生成章节统计信息字符串 ==========
    chapter_stats = []
    for i, c in enumerate(simplified_chapters[:5]):
        learnable_count = sum(1 for line in c if line.get('is_learnable'))
        readonly_count = sum(1 for line in c if not line.get('is_learnable'))
        audio_count = sum(1 for line in c if line.get('has_audio'))
        chapter_stats.append(f'     第{i+1}章: 可学习{learnable_count}句, 只读{readonly_count}句, 音频{audio_count}个')
    if len(simplified_chapters) > 5:
        chapter_stats.append('      ...')
    chapter_stats_str = '\n'.join(chapter_stats)

    # ========== HTML内容 ==========
    html_content = f'''<!DOCTYPE html>
<html lang="zh">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0, maximum-scale=1.0, user-scalable=no, viewport-fit=cover">
    <title>🧠 艾宾浩斯·自动学习系统</title>
    <meta name="theme-color" content="#6C5CE7">
    <meta name="mobile-web-app-capable" content="yes">
    <meta name="apple-mobile-web-app-capable" content="yes">
    <style>
        /* 样式代码 - 与之前相同 */
        * {{
            margin: 0;
            padding: 0;
            box-sizing: border-box;
            font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, sans-serif;
        }}

        :root {{
            --primary: #6C5CE7;
            --primary-dark: #5649C0;
            --secondary: #00B894;
            --accent: #FD79A8;
            --warning: #FDCB6E;
            --danger: #FF7675;
            --dark: #2D3436;
            --light: #F5F6FA;
            --gray: #B2BEC3;
        }}

        body {{
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            min-height: 100vh;
            padding: 16px;
            margin: 0;
        }}

        .app {{
            max-width: 1000px;
            margin: 0 auto;
            background: rgba(255, 255, 255, 0.98);
            backdrop-filter: blur(20px);
            border-radius: 40px;
            padding: 24px;
            box-shadow: 0 25px 50px -12px rgba(0,0,0,0.25);
            display: flex;
            flex-direction: column;
            height: calc(100vh - 32px);
            max-height: 1000px;
        }}

        .app-header {{
            flex-shrink: 0;
        }}

        h1 {{
            font-size: 24px;
            color: var(--primary);
            margin-bottom: 16px;
            display: flex;
            justify-content: space-between;
            align-items: center;
        }}

        .nav-tabs {{
            display: flex;
            gap: 8px;
            margin-bottom: 16px;
            padding: 8px;
            background: rgba(0,0,0,0.02);
            border-radius: 30px;
            flex-wrap: wrap;
        }}

        .nav-tab {{
            flex: 1;
            min-width: 80px;
            padding: 12px 16px;
            border: none;
            border-radius: 25px;
            font-size: 15px;
            font-weight: 600;
            display: flex;
            align-items: center;
            justify-content: center;
            gap: 6px;
            background: transparent;
            color: var(--dark);
            cursor: pointer;
            transition: all 0.3s;
        }}

        .nav-tab.active {{
            background: var(--primary);
            color: white;
            box-shadow: 0 10px 20px -5px rgba(108,92,231,0.4);
        }}

        .chapter-selector {{
            display: flex;
            align-items: center;
            gap: 12px;
            margin-bottom: 16px;
            padding: 16px;
            background: white;
            border-radius: 30px;
            box-shadow: 0 4px 12px rgba(0,0,0,0.05);
            border: 1px solid var(--light);
        }}

        .chapter-select {{
            flex: 1;
            padding: 14px 20px;
            border: 2px solid #e0e0e0;
            border-radius: 30px;
            font-size: 16px;
            background: white;
            color: var(--dark);
            font-weight: 500;
            outline: none;
        }}

        .chapter-select:focus {{
            border-color: var(--primary);
        }}

        .nav-btn {{
            width: 48px;
            height: 48px;
            border: none;
            border-radius: 50%;
            background: white;
            color: var(--primary);
            font-size: 20px;
            display: flex;
            align-items: center;
            justify-content: center;
            cursor: pointer;
            border: 2px solid var(--light);
            transition: all 0.2s;
        }}

        .auto-play-bar {{
            display: flex;
            align-items: center;
            gap: 16px;
            margin-bottom: 16px;
            padding: 16px 20px;
            background: linear-gradient(135deg, var(--primary), var(--primary-dark));
            border-radius: 40px;
            color: white;
        }}

        .play-control-btn {{
            width: 56px;
            height: 56px;
            border: none;
            border-radius: 50%;
            background: white;
            color: var(--primary);
            font-size: 24px;
            display: flex;
            align-items: center;
            justify-content: center;
            cursor: pointer;
            box-shadow: 0 8px 16px rgba(0,0,0,0.2);
        }}

        .play-control-btn.playing {{
            background: var(--warning);
            color: white;
        }}

        .progress-info {{
            flex: 1;
        }}

        .progress-bar {{
            height: 8px;
            background: rgba(255,255,255,0.3);
            border-radius: 4px;
            margin-top: 8px;
            overflow: hidden;
        }}

        .progress-fill {{
            height: 100%;
            background: white;
            border-radius: 4px;
            transition: width 0.3s;
        }}

        .content-area {{
            flex: 1;
            background: white;
            border-radius: 24px;
            padding: 20px;
            overflow-y: auto;
            position: relative;
            min-height: 0;
            box-shadow: inset 0 4px 12px rgba(0,0,0,0.02);
            scroll-behavior: smooth;
        }}

        .chapter-title {{
            font-size: 20px;
            font-weight: bold;
            color: var(--primary);
            margin-bottom: 20px;
            padding-bottom: 12px;
            border-bottom: 2px solid var(--primary);
            position: sticky;
            top: 0;
            background: white;
            z-index: 10;
        }}

        .lines-container {{
            display: flex;
            flex-direction: column;
            gap: 8px;
            padding: 4px 0;
        }}

        /* ========== 可学习的行 ========== */
        .learning-line {{
            display: flex;
            align-items: flex-start;
            padding: 16px;
            border-radius: 16px;
            background: white;
            border: 1px solid transparent;
            transition: all 0.3s ease;
            cursor: pointer;
            box-shadow: 0 2px 4px rgba(0,0,0,0.02);
        }}

        .learning-line:hover {{
            background: rgba(108,92,231,0.05);
            border-color: rgba(108,92,231,0.2);
        }}

        .learning-line.active {{
            background: rgba(108,92,231,0.12);
            border: 2px solid var(--primary);
            box-shadow: 0 8px 20px rgba(108,92,231,0.2);
            transform: scale(1.02);
            margin: 8px 0;
        }}

        .line-number {{
            width: 36px;
            height: 36px;
            display: flex;
            align-items: center;
            justify-content: center;
            background: var(--light);
            border-radius: 50%;
            font-size: 14px;
            font-weight: 600;
            color: var(--dark);
            margin-right: 12px;
            flex-shrink: 0;
        }}

        .learning-line.active .line-number {{
            background: var(--primary);
            color: white;
        }}

        /* ========== 只读的行 ========== */
        .readonly-line {{
            display: flex;
            align-items: flex-start;
            padding: 12px 16px;
            border-radius: 12px;
            background: #fafafa;
            color: #666;
            font-size: 15px;
            line-height: 1.5;
            border-left: 4px solid var(--gray);
            margin-left: 8px;
            cursor: default;
            user-select: none;
        }}

        .readonly-line .text-content {{
            color: #666;
            font-style: italic;
            padding: 4px 0;
        }}

        .readonly-indicator {{
            width: 24px;
            height: 24px;
            display: flex;
            align-items: center;
            justify-content: center;
            color: #999;
            font-size: 14px;
            margin-right: 12px;
            flex-shrink: 0;
        }}

        .text-content {{
            flex: 1;
            font-size: 17px;
            line-height: 1.6;
            word-break: break-word;
            padding: 6px 0;
        }}

        .learning-line .text-content {{
            color: var(--dark);
        }}

        .readonly-line .text-content {{
            color: #666;
            font-style: italic;
        }}

        /* 语言标签 */
        .lang-tag {{
            font-size: 12px;
            padding: 4px 8px;
            border-radius: 12px;
            background: rgba(108,92,231,0.1);
            color: var(--primary);
            margin-left: 8px;
            white-space: nowrap;
            font-weight: 600;
        }}

        .lang-tag.ch {{ background: rgba(0,184,148,0.1); color: #00B894; }}
        .lang-tag.jp {{ background: rgba(253,121,168,0.1); color: #FD79A8; }}
        .lang-tag.en {{ background: rgba(108,92,231,0.1); color: #6C5CE7; }}

        /* 音频指示器 */
        .audio-indicator {{
            width: 24px;
            height: 24px;
            border-radius: 50%;
            background: var(--secondary);
            color: white;
            display: inline-flex;
            align-items: center;
            justify-content: center;
            font-size: 12px;
            margin-left: 8px;
            flex-shrink: 0;
        }}

        /* 操作按钮组 - 只出现在可学习行 */
        .action-buttons {{
            display: flex;
            gap: 8px;
            margin-left: 12px;
            flex-shrink: 0;
        }}

        .action-btn {{
            width: 40px;
            height: 40px;
            border: none;
            border-radius: 50%;
            background: var(--light);
            color: var(--dark);
            font-size: 18px;
            display: flex;
            align-items: center;
            justify-content: center;
            cursor: pointer;
            transition: all 0.2s;
        }}

        .action-btn:hover {{
            transform: scale(1.1);
        }}

        .action-btn.added {{
            background: var(--secondary);
            color: white;
        }}

        .action-btn.grammar.added {{
            background: var(--accent);
            color: white;
        }}

        .panel {{
            flex: 1;
            overflow-y: auto;
            padding: 20px;
            background: white;
            border-radius: 24px;
        }}

        .panel h2 {{
            font-size: 20px;
            color: var(--primary);
            margin-bottom: 20px;
            padding-bottom: 12px;
            border-bottom: 2px solid var(--primary);
        }}

        .word-card {{
            padding: 16px;
            border-radius: 16px;
            background: var(--light);
            margin-bottom: 12px;
            border-left: 6px solid var(--primary);
        }}

        .word-card.grammar-card {{
            border-left-color: var(--accent);
        }}

        .word-text {{
            font-size: 17px;
            font-weight: 600;
            margin-bottom: 8px;
            color: var(--dark);
        }}

        .word-meta {{
            display: flex;
            gap: 16px;
            font-size: 12px;
            color: #666;
            flex-wrap: wrap;
            margin-bottom: 8px;
        }}

        .grammar-note {{
            background: var(--accent);
            color: white;
            padding: 8px 12px;
            border-radius: 8px;
            margin-top: 8px;
            font-size: 14px;
        }}

        .empty-message {{
            text-align: center;
            padding: 40px;
            color: #666;
            font-size: 16px;
        }}

        .status-bar {{
            flex-shrink: 0;
            margin-top: 16px;
            padding: 16px 24px;
            background: white;
            border-radius: 30px;
            box-shadow: 0 -4px 20px rgba(0,0,0,0.05);
            display: flex;
            justify-content: space-between;
            align-items: center;
            font-size: 14px;
            color: var(--dark);
            border: 1px solid var(--light);
        }}

        .badge {{
            background: var(--primary);
            color: white;
            padding: 4px 12px;
            border-radius: 20px;
            font-size: 12px;
        }}

        .hidden {{
            display: none !important;
        }}

        @media (max-width: 600px) {{
            .app {{ padding: 16px; height: calc(100vh - 32px); }}
            .text-content {{ font-size: 16px; }}
        }}
    </style>
</head>
<body>
    <div class="app">
        <div class="app-header">
            <h1>
                <span>🧠 艾宾浩斯·自动学习</span>
                <span class="badge">🔑 {hashlib.md5(str(total_chapters).encode()).hexdigest()[:8]}</span>
            </h1>

            <div class="nav-tabs">
                <button class="nav-tab active" onclick="switchTab('learn')">📚 自动学习</button>
                <button class="nav-tab" onclick="switchTab('notebook')">
                    📔 单词本
                    <span id="notebookBadge" style="background: var(--secondary); color: white; padding: 2px 8px; border-radius: 12px; font-size: 12px;">0</span>
                </button>
                <button class="nav-tab" onclick="switchTab('grammar')">
                    🔤 语法本
                    <span id="grammarBadge" style="background: var(--accent); color: white; padding: 2px 8px; border-radius: 12px; font-size: 12px;">0</span>
                </button>
                <button class="nav-tab" onclick="switchTab('review')">
                    🧠 复习
                    <span id="reviewBadge" style="background: var(--danger); color: white; padding: 2px 8px; border-radius: 12px; font-size: 12px;">0</span>
                </button>
            </div>
        </div>

        <!-- 学习面板 -->
        <div id="learnPanel">
            <div class="chapter-selector">
                <button class="nav-btn" onclick="prevChapter()">◀</button>
                <select id="chapterSelect" class="chapter-select" onchange="loadChapter(parseInt(this.value))">
                    <option value="">📚 请选择章节</option>
                </select>
                <button class="nav-btn" onclick="nextChapter()">▶</button>
            </div>

            <div class="auto-play-bar">
                <button id="playPauseBtn" class="play-control-btn" onclick="togglePlayPause()">▶</button>
                <div class="progress-info">
                    <div style="display: flex; justify-content: space-between; margin-bottom: 4px;">
                        <span id="currentLineInfo">第 0/0 句</span>
                        <span id="chapterInfo">第 0/{total_chapters} 章</span>
                    </div>
                    <div class="progress-bar">
                        <div id="progressFill" class="progress-fill" style="width: 0%;"></div>
                    </div>
                </div>
            </div>

            <div class="content-area" id="learningContent">
                <div id="chapterTitle" class="chapter-title">请选择章节</div>
                <div id="linesContainer" class="lines-container"></div>
            </div>
        </div>

        <!-- 单词本面板 -->
        <div id="notebookPanel" class="panel hidden">
            <h2>📔 我的单词本</h2>
            <div id="wordList"></div>
        </div>

        <!-- 语法本面板 -->
        <div id="grammarPanel" class="panel hidden">
            <h2>🔤 我的语法本</h2>
            <div id="grammarList"></div>
        </div>

        <!-- 复习面板 -->
        <div id="reviewPanel" class="panel hidden">
            <h2>🧠 今日复习</h2>
            <div id="reviewList"></div>
        </div>

        <div class="status-bar">
            <span>📚 {total_chapters}章 · {total_lines}句</span>
            <span>🎵 {total_audio}音频</span>
            <span>📝 可学习 {total_learnable}句</span>
            <span id="currentTime">{datetime.now().strftime('%H:%M')}</span>
        </div>
    </div>

    <script>
        // ========== 🌟 真实章节数据 ==========
        const CHAPTERS_DATA = {chapters_json_str};
        const LEARNING_ITEMS = {items_json_str};
        const TOTAL_CHAPTERS = {total_chapters};

        console.log('=================================');
        console.log('✅ 艾宾浩斯学习系统启动');
        console.log('📊 总章节数:', TOTAL_CHAPTERS);
        console.log('📊 处理后章节数:', CHAPTERS_DATA ? CHAPTERS_DATA.length : 0);
        console.log('=================================');

        // ========== 全局变量 ==========
        let currentChapter = 0;
        let currentLineIndex = 0;
        let isPlaying = false;
        let playTimer = null;
        let currentAudio = null;

        // ========== 页面初始化 ==========
        window.addEventListener('load', function() {{
            console.log('页面加载完成，开始初始化...');

            if (!CHAPTERS_DATA || !Array.isArray(CHAPTERS_DATA) || CHAPTERS_DATA.length === 0) {{
                console.error('❌ 错误：章节数据为空！');
                return;
            }}

            initChapterSelect();
            loadFromStorage();

            if (CHAPTERS_DATA.length > 0) {{
                loadChapter(0);
            }}

            updateAllPanels();
        }});

        // ========== 初始化章节选择器 ==========
        function initChapterSelect() {{
            const select = document.getElementById('chapterSelect');
            if (!select) return;

            select.innerHTML = '<option value="">📚 请选择章节</option>';

            for (let i = 1; i <= CHAPTERS_DATA.length; i++) {{
                const option = document.createElement('option');
                option.value = i - 1;
                const chapter = CHAPTERS_DATA[i-1];
                const learnableCount = chapter ? chapter.filter(line => line.is_learnable).length : 0;
                const readonlyCount = chapter ? chapter.filter(line => !line.is_learnable).length : 0;
                const audioCount = chapter ? chapter.filter(line => line.audio && line.audio.length > 100).length : 0;
                option.textContent = '第 ' + i + ' 章 (可学习' + learnableCount + '句, 只读' + readonlyCount + '句, 音频' + audioCount + '个)';
                select.appendChild(option);
            }}

            console.log('✅ 章节选择器初始化完成，共', select.options.length - 1, '个章节');
        }}

        // ========== 本地存储 ==========
        function loadFromStorage() {{
            const saved = localStorage.getItem('ebbinghaus_notebook');
            if (saved) {{
                try {{
                    const notebook = JSON.parse(saved);
                    if (LEARNING_ITEMS) {{
                        LEARNING_ITEMS.forEach(item => {{
                            const savedItem = notebook.find(n => n.id === item.id);
                            if (savedItem) {{
                                item.added_to_notebook = savedItem.added_to_notebook || false;
                                item.added_to_grammar = savedItem.added_to_grammar || false;
                                item.grammar_note = savedItem.grammar_note || '';
                                item.mastery = savedItem.mastery || 0;
                                item.review_count = savedItem.review_count || 0;
                                item.last_review = savedItem.last_review;
                                item.next_review = savedItem.next_review || 0;
                            }}
                        }});
                    }}
                    console.log('✅ 本地存储加载成功');
                }} catch (e) {{
                    console.error('❌ 本地存储加载失败:', e);
                }}
            }}
        }}

        function saveToStorage() {{
            if (!LEARNING_ITEMS) return;
            const notebook = LEARNING_ITEMS.map(item => ({{
                id: item.id,
                added_to_notebook: item.added_to_notebook || false,
                added_to_grammar: item.added_to_grammar || false,
                grammar_note: item.grammar_note || '',
                mastery: item.mastery || 0,
                review_count: item.review_count || 0,
                last_review: item.last_review,
                next_review: item.next_review || 0
            }})).filter(item => item.added_to_notebook || item.added_to_grammar);
            localStorage.setItem('ebbinghaus_notebook', JSON.stringify(notebook));
        }}

        // ========== 章节加载 ==========
        function loadChapter(index) {{
            if (index < 0 || index >= CHAPTERS_DATA.length) return;

            currentChapter = index;
            currentLineIndex = 0;

            const select = document.getElementById('chapterSelect');
            if (select) select.value = index;

            const titleEl = document.getElementById('chapterTitle');
            if (titleEl) titleEl.innerHTML = '第 ' + (index + 1) + ' 章';

            const infoEl = document.getElementById('chapterInfo');
            if (infoEl) infoEl.innerHTML = '第 ' + (index + 1) + '/' + CHAPTERS_DATA.length + ' 章';

            renderChapter();
            stopPlay();
        }}

        // ========== 渲染章节内容 ==========
        function renderChapter() {{
            const chapterData = CHAPTERS_DATA[currentChapter];
            if (!chapterData || !Array.isArray(chapterData)) return;

            const container = document.getElementById('linesContainer');
            let html = '';

            chapterData.forEach((line, idx) => {{
                if (!line) return;

                if (line.is_learnable) {{
                    // ===== 可学习的行 =====
                    const hasAudio = line.audio && line.audio.length > 100;
                    const learningItem = LEARNING_ITEMS?.find(item => item && item.id === line.id);
                    const isNotebook = learningItem ? learningItem.added_to_notebook : false;
                    const isGrammar = learningItem ? learningItem.added_to_grammar : false;

                    html += '<div id="line-' + line.id + '" class="learning-line" onclick="playThisLine(\\'' + line.id + '\\', ' + idx + ')">';
                    html += '<span class="line-number">' + (idx + 1) + '</span>';
                    html += '<div class="text-content">' + (line.content || '').replace(/\\[(ch|jp|en)\\]\\s*/g, '').replace(/\\[sound:[^\\]]*\\]/g, '') + '</div>';

                    // 显示语言标签
                    let langClass = '';
                    let langText = '';
                    if (line.lang === 'ch') {{ langClass = 'ch'; langText = '中文'; }}
                    else if (line.lang === 'jp') {{ langClass = 'jp'; langText = '日语'; }}
                    else if (line.lang === 'en') {{ langClass = 'en'; langText = '英语'; }}
                    html += '<span class="lang-tag ' + langClass + '">' + langText + '</span>';

                    if (hasAudio) {{
                        html += '<span class="audio-indicator">🔊</span>';
                    }}

                    // 显示操作按钮
                    html += '<div class="action-buttons">';
                    html += '<button class="action-btn' + (isNotebook ? ' added' : '') + '" onclick="event.stopPropagation(); toggleNotebook(\\'' + line.id + '\\', this)">' + (isNotebook ? '✓' : '📚') + '</button>';
                    html += '<button class="action-btn grammar' + (isGrammar ? ' added' : '') + '" onclick="event.stopPropagation(); addGrammar(\\'' + line.id + '\\', this)">🔤</button>';
                    html += '</div>';
                    html += '</div>';

                }} else {{
                    // ===== 只读的行 =====
                    html += '<div class="readonly-line">';
                    html += '<span class="readonly-indicator">📄</span>';
                    html += '<div class="text-content">' + (line.content || '').replace(/\\[(ch|jp|en)\\]\\s*/g, '').replace(/\\[sound:[^\\]]*\\]/g, '') + '</div>';
                    html += '</div>';
                }}
            }});

            container.innerHTML = html;
            updateProgress();
        }}

        // ========== 播放指定行 ==========
        function playThisLine(lineId, index) {{
            console.log('播放行:', lineId, index);

            const chapterData = CHAPTERS_DATA[currentChapter];
            if (!chapterData || index >= chapterData.length) return;

            const line = chapterData[index];

            // 只播放可学习的行
            if (!line.is_learnable) return;

            // 更新当前行索引
            currentLineIndex = index;

            // 移除所有active类
            document.querySelectorAll('.learning-line').forEach(el => el.classList.remove('active'));

            // 为当前行添加active类
            const currentLine = document.getElementById('line-' + lineId);
            if (currentLine) {{
                currentLine.classList.add('active');

                // 滚动到中间位置
                const contentArea = document.getElementById('learningContent');
                const lineRect = currentLine.getBoundingClientRect();
                const contentRect = contentArea.getBoundingClientRect();
                const scrollTop = currentLine.offsetTop - (contentRect.height / 2) + (lineRect.height / 2);

                contentArea.scrollTo({{
                    top: Math.max(0, scrollTop),
                    behavior: 'smooth'
                }});
            }}

            // 播放音频
            if (line && line.audio && line.audio.length > 100) {{
                playAudio(line.audio);
            }}

            updateProgress();
        }}

        // ========== 音频播放函数 ==========
        function playAudio(audioData) {{
            return new Promise((resolve, reject) => {{
                try {{
                    if (!audioData || audioData.length < 100) {{
                        console.log('无效的音频数据');
                        reject('无效的音频数据');
                        return;
                    }}

                    let audioSrc = audioData;
                    if (!audioSrc.startsWith('data:audio')) {{
                        audioSrc = 'data:audio/mp3;base64,' + audioSrc;
                    }}

                    if (currentAudio) {{
                        currentAudio.pause();
                        currentAudio.currentTime = 0;
                    }}

                    currentAudio = new Audio(audioSrc);

                    currentAudio.onended = function() {{
                        console.log('音频播放完成');
                        currentAudio = null;
                        resolve();
                    }};

                    currentAudio.onerror = function(e) {{
                        console.error('音频播放错误:', e);
                        currentAudio = null;
                        reject(e);
                    }};

                    currentAudio.play().catch(e => {{
                        console.error('播放失败:', e);
                        currentAudio = null;
                        reject(e);
                    }});

                }} catch (e) {{
                    console.error('音频创建失败:', e);
                    reject(e);
                }}
            }});
        }}

        // ========== 自动播放控制 ==========
        function togglePlayPause() {{
            const btn = document.getElementById('playPauseBtn');
            if (isPlaying) {{
                stopPlay();
                btn.innerHTML = '▶';
                btn.classList.remove('playing');
            }} else {{
                startPlay();
                btn.innerHTML = '⏸';
                btn.classList.add('playing');
            }}
        }}

        function startPlay() {{
            if (isPlaying) return;
            if (!CHAPTERS_DATA[currentChapter]) return;
            isPlaying = true;
            playCurrentLine();
        }}

        function stopPlay() {{
            isPlaying = false;
            if (playTimer) {{
                clearTimeout(playTimer);
                playTimer = null;
            }}
            if (currentAudio) {{
                currentAudio.pause();
                currentAudio.currentTime = 0;
                currentAudio = null;
            }}
            const btn = document.getElementById('playPauseBtn');
            if (btn) {{
                btn.innerHTML = '▶';
                btn.classList.remove('playing');
            }}
        }}

        async function playCurrentLine() {{
            if (!isPlaying) return;

            const chapterData = CHAPTERS_DATA[currentChapter];
            if (!chapterData) return;

            // 跳过只读行，找到下一个可学习行
            while (currentLineIndex < chapterData.length && !chapterData[currentLineIndex].is_learnable) {{
                currentLineIndex++;
                updateProgress();
            }}

            if (currentLineIndex >= chapterData.length) {{
                if (currentChapter + 1 < CHAPTERS_DATA.length) {{
                    loadChapter(currentChapter + 1);
                    currentLineIndex = 0;
                    setTimeout(() => playCurrentLine(), 800);
                }} else {{
                    stopPlay();
                }}
                return;
            }}

            const line = chapterData[currentLineIndex];
            const lineId = line.id;

            // 高亮当前行
            document.querySelectorAll('.learning-line').forEach(el => el.classList.remove('active'));
            const currentLine = document.getElementById('line-' + lineId);
            if (currentLine) {{
                currentLine.classList.add('active');

                // 滚动到中间位置
                const contentArea = document.getElementById('learningContent');
                const lineRect = currentLine.getBoundingClientRect();
                const contentRect = contentArea.getBoundingClientRect();
                const scrollTop = currentLine.offsetTop - (contentRect.height / 2) + (lineRect.height / 2);

                contentArea.scrollTo({{
                    top: Math.max(0, scrollTop),
                    behavior: 'smooth'
                }});
            }}

            // 播放音频
            if (line && line.audio && line.audio.length > 100) {{
                try {{
                    await playAudio(line.audio);
                    currentLineIndex++;
                    updateProgress();
                    if (isPlaying) {{
                        playTimer = setTimeout(() => playCurrentLine(), 300);
                    }}
                }} catch (e) {{
                    console.log('音频播放失败，跳过:', e);
                    currentLineIndex++;
                    updateProgress();
                    if (isPlaying) {{
                        playTimer = setTimeout(() => playCurrentLine(), 300);
                    }}
                }}
            }} else {{
                currentLineIndex++;
                updateProgress();
                if (isPlaying) {{
                    playTimer = setTimeout(() => playCurrentLine(), 300);
                }}
            }}
        }}

        function updateProgress() {{
            const chapterData = CHAPTERS_DATA[currentChapter];
            if (!chapterData) return;

            // 计算可学习行的总数量和当前进度
            const learnableCount = chapterData.filter(line => line.is_learnable).length;
            const currentLearnableIndex = chapterData.slice(0, currentLineIndex).filter(line => line.is_learnable).length;

            const progress = learnableCount > 0 ? (currentLearnableIndex / learnableCount) * 100 : 0;
            const fill = document.getElementById('progressFill');
            if (fill) fill.style.width = progress + '%';

            const lineInfo = document.getElementById('currentLineInfo');
            if (lineInfo) {{
                lineInfo.innerHTML = '第 ' + (currentLearnableIndex + 1) + '/' + learnableCount + ' 句';
            }}
        }}

        // ========== 单词本操作 ==========
        function toggleNotebook(itemId, btn) {{
            const item = LEARNING_ITEMS?.find(i => i.id === itemId);
            if (item) {{
                item.added_to_notebook = !item.added_to_notebook;
                btn.innerHTML = item.added_to_notebook ? '✓' : '📚';
                btn.classList.toggle('added');
                saveToStorage();
                updateNotebookPanel();
                updateBadges();
            }}
        }}

        function addGrammar(itemId, btn) {{
            const item = LEARNING_ITEMS?.find(i => i.id === itemId);
            if (item) {{
                const note = prompt('添加语法笔记:', item.grammar_note || '');
                if (note !== null) {{
                    item.added_to_grammar = true;
                    item.grammar_note = note;
                    btn.classList.add('added');
                    saveToStorage();
                    updateGrammarPanel();
                    updateBadges();
                }}
            }}
        }}

        // ========== 单词本面板 ==========
        function updateNotebookPanel() {{
            const notebook = LEARNING_ITEMS?.filter(item => item && item.added_to_notebook) || [];
            const list = document.getElementById('wordList');

            let html = '';
            notebook.forEach(item => {{
                html += '<div class="word-card">';
                html += '<div class="word-text">' + (item.content || '').replace(/\\[(ch|jp|en)\\]\\s*/g, '').replace(/\\[sound:[^\\]]*\\]/g, '') + '</div>';
                html += '<div class="word-meta">';
                html += '<span>📌 第' + (item.chapter || 0) + '章</span>';
                html += '<span>🎯 掌握度 ' + (item.mastery || 0) + '%</span>';
                html += '<span>🔄 复习' + (item.review_count || 0) + '次</span>';
                html += '</div>';
                html += '<button class="action-btn" style="background: var(--danger); color: white; margin-top: 8px;" onclick="toggleNotebook(\\'' + item.id + '\\', this)">🗑️ 移除</button>';
                html += '</div>';
            }});

            if (list) {{
                list.innerHTML = html || '<div class="empty-message">📭 单词本还是空的，点击句子旁边的📚收藏</div>';
            }}

            document.getElementById('notebookBadge').innerHTML = notebook.length;
        }}

        // ========== 语法本面板 ==========
        function updateGrammarPanel() {{
            const grammar = LEARNING_ITEMS?.filter(item => item && item.added_to_grammar) || [];
            const list = document.getElementById('grammarList');

            let html = '';
            grammar.forEach(item => {{
                html += '<div class="word-card grammar-card">';
                html += '<div class="word-text">' + (item.content || '').replace(/\\[(ch|jp|en)\\]\\s*/g, '').replace(/\\[sound:[^\\]]*\\]/g, '') + '</div>';
                html += '<div class="grammar-note">📝 ' + (item.grammar_note || '无备注') + '</div>';
                html += '<div class="word-meta" style="margin-top: 8px;">';
                html += '<span>📌 第' + (item.chapter || 0) + '章</span>';
                html += '</div>';
                html += '</div>';
            }});

            if (list) {{
                list.innerHTML = html || '<div class="empty-message">🔤 语法本还是空的，点击句子旁边的🔤添加笔记</div>';
            }}

            document.getElementById('grammarBadge').innerHTML = grammar.length;
        }}

        // ========== 更新徽章 ==========
        function updateBadges() {{
            if (!LEARNING_ITEMS) return;

            const notebook = LEARNING_ITEMS.filter(item => item && item.added_to_notebook).length || 0;
            const grammar = LEARNING_ITEMS.filter(item => item && item.added_to_grammar).length || 0;

            document.getElementById('notebookBadge').innerHTML = notebook;
            document.getElementById('grammarBadge').innerHTML = grammar;
        }}

        // ========== 更新所有面板 ==========
        function updateAllPanels() {{
            if (!LEARNING_ITEMS) return;
            updateNotebookPanel();
            updateGrammarPanel();
            updateBadges();
        }}

        // ========== 章节导航 ==========
        function prevChapter() {{
            if (currentChapter > 0) {{
                loadChapter(currentChapter - 1);
                stopPlay();
            }}
        }}

        function nextChapter() {{
            if (currentChapter + 1 < CHAPTERS_DATA.length) {{
                loadChapter(currentChapter + 1);
                stopPlay();
            }}
        }}

        // ========== 标签页切换 ==========
        function switchTab(tab) {{
            document.querySelectorAll('.nav-tab').forEach(t => t.classList.remove('active'));
            event.currentTarget.classList.add('active');

            const learnPanel = document.getElementById('learnPanel');
            const notebookPanel = document.getElementById('notebookPanel');
            const grammarPanel = document.getElementById('grammarPanel');
            const reviewPanel = document.getElementById('reviewPanel');

            if (learnPanel) learnPanel.classList.toggle('hidden', tab !== 'learn');
            if (notebookPanel) notebookPanel.classList.toggle('hidden', tab !== 'notebook');
            if (grammarPanel) grammarPanel.classList.toggle('hidden', tab !== 'grammar');
            if (reviewPanel) reviewPanel.classList.toggle('hidden', tab !== 'review');

            if (tab === 'notebook') updateNotebookPanel();
            if (tab === 'grammar') updateGrammarPanel();
        }}

        // ========== 暴露全局函数 ==========
        window.switchTab = switchTab;
        window.loadChapter = loadChapter;
        window.prevChapter = prevChapter;
        window.nextChapter = nextChapter;
        window.togglePlayPause = togglePlayPause;
        window.toggleNotebook = toggleNotebook;
        window.addGrammar = addGrammar;
        window.playThisLine = playThisLine;
    </script>
</body>
</html>
    '''

    # 写入主HTML文件
    with open(html_path, 'w', encoding='utf-8') as f:
        f.write(html_content)

    file_size = os.path.getsize(html_path) // 1024

    print(f"""
╔══════════════════════════════════════════════════════════════╗
║  ✅ 艾宾浩斯·自动学习系统 生成成功                          ║
╠══════════════════════════════════════════════════════════════╣
║  📁 文件: {os.path.basename(html_path)}
║  📊 总章节: {len(simplified_chapters)} 章
║  📝 可学习语句: {total_learnable} 句
║  📄 只读语句: {total_readonly} 句
║  🎵 有效音频: {total_valid_audio} 个
║  💾 文件大小: {file_size} KB
║
║  📖 章节详情:
{chapter_stats_str}
║
║  🎯 功能状态:
║     • ✅ 可学习行 - 彩色，可点击，有操作按钮
║     • ✅ 只读行 - 灰色，不可点击，无按钮
║     • ✅ 自动播放 - 自动跳过只读行
║     • ✅ 单词本 - 正常收藏和显示
║     • ✅ 语法本 - 正常添加笔记
║     • ✅ 音频播放 - 点击句子或自动播放
║
║  📱 现在可以正常使用了！
║
╚══════════════════════════════════════════════════════════════╝
    """)

    return html_path


In [ ]:
# ============================================================
# 🧠 艾宾浩斯学习系统生成函数 - 滚动容器优化版


In [ ]:
# ============================================================

def generate_super_language_learning_system_oneColor(html_path, chapters):
    """
    生成超级语言学习系统 - 滚动容器优化版
    功能：固定高度滚动容器 + 自动居中 + 可学习/只读行区分
    """
    import json
    from datetime import datetime
    import os
    import hashlib
    import base64
    import re
    from bs4 import BeautifulSoup

    print(f"  📊 正在为 {len(chapters)} 章内容生成艾宾浩斯学习系统...")

    if len(chapters) == 0:
        print("  ❌ 错误：章节数据为空，无法生成学习系统")
        return

    # 计算统计数据
    total_chapters = len(chapters)
    total_lines = sum(len(c) for c in chapters)
    total_audio = sum(1 for c in chapters for l in c if l.get('audio') and len(l.get('audio', '')) > 100)

    print(f"  📊 总章节: {total_chapters}, 总行数: {total_lines}, 总音频: {total_audio}")

    # ========== 构建完整的章节数据 ==========
    simplified_chapters = []

    # 定义变量用于统计
    total_learnable = 0
    total_readonly = 0
    total_valid_audio = 0

    for chap_idx, chapter in enumerate(chapters):
        simplified_lines = []
        for line_idx, line in enumerate(chapter):
            # 提取纯文本内容
            text = line.get('text', '')
            audio = line.get('audio', '')

            # 使用BeautifulSoup提取纯文本
            soup = BeautifulSoup(text, "html.parser")
            plain_text = remove_sound_tags(soup.get_text().strip())

            # 判断语言类型 - 只有以 [ch]、[jp]、[en] 开头的行才是可学习行
            lang = 'unknown'
            is_learnable = False

            if plain_text.startswith('[ch]'):
                lang = 'ch'
                is_learnable = True
            elif plain_text.startswith('[jp]'):
                lang = 'jp'
                is_learnable = True
            elif plain_text.startswith('[en]'):
                lang = 'en'
                is_learnable = True

            # 移除所有语言标签（不仅是行首的）
            content = re.sub(r'\[sound:[^\]]*\]', '', re.sub(r'\[(ch|jp|en)\]\s*', '', plain_text)).strip()

            # 如果内容为空，跳过
            if not content:
                continue

            # 确保音频数据完整
            has_valid_audio = False
            if audio and len(audio) > 100:
                if not audio.startswith('data:audio'):
                    audio = 'data:audio/mp3;base64,' + audio
                has_valid_audio = True
                if is_learnable:
                    total_valid_audio += 1

            # 更新统计
            if is_learnable:
                total_learnable += 1
            else:
                total_readonly += 1

            simplified_lines.append({
                'id': f'{chap_idx:04d}_{line_idx:03d}',
                'content': content,
                'audio': audio if has_valid_audio else '',
                'lang': lang,
                'chapter': chap_idx + 1,
                'has_audio': has_valid_audio,
                'is_learnable': is_learnable
            })

        if simplified_lines:
            simplified_chapters.append(simplified_lines)

    print(f"  📝 处理后章节数: {len(simplified_chapters)}")
    print(f"  📝 可学习语句: {total_learnable} 句, 只读语句: {total_readonly} 句")
    print(f"  📝 有效音频: {total_valid_audio} 个")

    # ========== 构建学习项列表 ==========
    learning_items = []
    for chap_idx, chapter in enumerate(simplified_chapters):
        for line in chapter:
            if line.get('is_learnable'):
                learning_items.append({
                    'id': line['id'],
                    'chapter': line['chapter'],
                    'content': line['content'][:100] + '...' if len(line['content']) > 100 else line['content'],
                    'lang': line['lang'],
                    'has_audio': line.get('has_audio', False),
                    'mastery': 0,
                    'review_count': 0,
                    'added_to_notebook': False,
                    'added_to_grammar': False,
                    'grammar_note': '',
                    'last_review': None,
                    'next_review': 0
                })

    # ========== 将数据转换为JSON字符串 ==========
    chapters_json_str = json.dumps(simplified_chapters, ensure_ascii=False)
    items_json_str = json.dumps(learning_items, ensure_ascii=False)

    # ========== 生成章节统计信息字符串 ==========
    chapter_stats = []
    for i, c in enumerate(simplified_chapters[:5]):
        learnable_count = sum(1 for line in c if line.get('is_learnable'))
        readonly_count = sum(1 for line in c if not line.get('is_learnable'))
        audio_count = sum(1 for line in c if line.get('has_audio'))
        chapter_stats.append(f'     第{i+1}章: 可学习{learnable_count}句, 只读{readonly_count}句, 音频{audio_count}个')
    if len(simplified_chapters) > 5:
        chapter_stats.append('      ...')
    chapter_stats_str = '\n'.join(chapter_stats)

    # ========== HTML内容 - 滚动容器优化版 ==========
    html_content = f'''<!DOCTYPE html>
<html lang="zh">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0, maximum-scale=1.0, user-scalable=no, viewport-fit=cover">
    <title>🧠 艾宾浩斯·自动学习系统</title>
    <meta name="theme-color" content="#6C5CE7">
    <meta name="mobile-web-app-capable" content="yes">
    <meta name="apple-mobile-web-app-capable" content="yes">
    <style>
        /* ========== 全局样式 ========== */
        * {{
            margin: 0;
            padding: 0;
            box-sizing: border-box;
            font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, sans-serif;
        }}

        :root {{
            --primary: #6C5CE7;
            --primary-dark: #5649C0;
            --secondary: #00B894;
            --accent: #FD79A8;
            --warning: #FDCB6E;
            --danger: #FF7675;
            --dark: #2D3436;
            --light: #F5F6FA;
            --gray: #B2BEC3;
        }}

        body {{
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            min-height: 100vh;
            padding: 16px;
            margin: 0;
        }}

        .app {{
            max-width: 1000px;
            margin: 0 auto;
            background: rgba(255, 255, 255, 0.98);
            backdrop-filter: blur(20px);
            border-radius: 40px;
            padding: 24px;
            box-shadow: 0 25px 50px -12px rgba(0,0,0,0.25);
            display: flex;
            flex-direction: column;
            height: calc(100vh - 32px);
            max-height: 1000px;
        }}

        .app-header {{
            flex-shrink: 0;
        }}

        h1 {{
            font-size: 24px;
            color: var(--primary);
            margin-bottom: 16px;
            display: flex;
            justify-content: space-between;
            align-items: center;
        }}

        /* ========== 导航标签 ========== */
        .nav-tabs {{
            display: flex;
            gap: 8px;
            margin-bottom: 16px;
            padding: 8px;
            background: rgba(0,0,0,0.02);
            border-radius: 30px;
            flex-wrap: wrap;
        }}

        .nav-tab {{
            flex: 1;
            min-width: 80px;
            padding: 12px 16px;
            border: none;
            border-radius: 25px;
            font-size: 15px;
            font-weight: 600;
            display: flex;
            align-items: center;
            justify-content: center;
            gap: 6px;
            background: transparent;
            color: var(--dark);
            cursor: pointer;
            transition: all 0.3s;
        }}

        .nav-tab.active {{
            background: var(--primary);
            color: white;
            box-shadow: 0 10px 20px -5px rgba(108,92,231,0.4);
        }}

        /* ========== 章节选择器 ========== */
        .chapter-selector {{
            display: flex;
            align-items: center;
            gap: 12px;
            margin-bottom: 16px;
            padding: 16px;
            background: white;
            border-radius: 30px;
            box-shadow: 0 4px 12px rgba(0,0,0,0.05);
            border: 1px solid var(--light);
        }}

        .chapter-select {{
            flex: 1;
            padding: 14px 20px;
            border: 2px solid #e0e0e0;
            border-radius: 30px;
            font-size: 16px;
            background: white;
            color: var(--dark);
            font-weight: 500;
            outline: none;
            cursor: pointer;
        }}

        .chapter-select:focus {{
            border-color: var(--primary);
        }}

        .nav-btn {{
            width: 48px;
            height: 48px;
            border: none;
            border-radius: 50%;
            background: white;
            color: var(--primary);
            font-size: 20px;
            display: flex;
            align-items: center;
            justify-content: center;
            cursor: pointer;
            border: 2px solid var(--light);
            transition: all 0.2s;
        }}

        .nav-btn:active {{
            background: var(--primary);
            color: white;
            border-color: var(--primary);
            transform: scale(0.95);
        }}

        /* ========== 自动播放控制栏 ========== */
        .auto-play-bar {{
            display: flex;
            align-items: center;
            gap: 16px;
            margin-bottom: 16px;
            padding: 16px 20px;
            background: linear-gradient(135deg, var(--primary), var(--primary-dark));
            border-radius: 40px;
            color: white;
            flex-shrink: 0;
        }}

        .play-control-btn {{
            width: 56px;
            height: 56px;
            border: none;
            border-radius: 50%;
            background: white;
            color: var(--primary);
            font-size: 24px;
            display: flex;
            align-items: center;
            justify-content: center;
            cursor: pointer;
            box-shadow: 0 8px 16px rgba(0,0,0,0.2);
            transition: all 0.2s;
        }}

        .play-control-btn:active {{
            transform: scale(0.9);
        }}

        .play-control-btn.playing {{
            background: var(--warning);
            color: white;
        }}

        .progress-info {{
            flex: 1;
        }}

        .progress-bar {{
            height: 8px;
            background: rgba(255,255,255,0.3);
            border-radius: 4px;
            margin-top: 8px;
            overflow: hidden;
        }}

        .progress-fill {{
            height: 100%;
            background: white;
            border-radius: 4px;
            transition: width 0.3s;
        }}

        /* ========== 滚动容器 - 这才是关键 ========== */
        .scroll-container {{
            flex: 1;
            background: white;
            border-radius: 24px;
            padding: 20px;
            overflow-y: auto;
            position: relative;
            min-height: 0;
            height: 0;  /* 触发flex收缩 */
            box-shadow: inset 0 4px 12px rgba(0,0,0,0.02);
            scroll-behavior: smooth;
        }}

        /* 自定义滚动条 */
        .scroll-container::-webkit-scrollbar {{
            width: 8px;
        }}

        .scroll-container::-webkit-scrollbar-track {{
            background: var(--light);
            border-radius: 4px;
        }}

        .scroll-container::-webkit-scrollbar-thumb {{
            background: var(--primary);
            border-radius: 4px;
        }}

        .scroll-container::-webkit-scrollbar-thumb:hover {{
            background: var(--primary-dark);
        }}

        .chapter-title {{
            font-size: 20px;
            font-weight: bold;
            color: var(--primary);
            margin-bottom: 20px;
            padding-bottom: 12px;
            border-bottom: 2px solid var(--primary);
            position: sticky;
            top: 0;
            background: white;
            z-index: 10;
        }}

        .lines-container {{
            display: flex;
            flex-direction: column;
            gap: 8px;
            padding: 4px 0 20px 0;
        }}

        /* ========== 可学习的行 ========== */
        .learning-line {{
            display: flex;
            align-items: flex-start;
            padding: 16px;
            border-radius: 16px;
            background: white;
            border: 1px solid transparent;
            transition: all 0.3s ease;
            cursor: pointer;
            box-shadow: 0 2px 4px rgba(0,0,0,0.02);
        }}

        .learning-line:hover {{
            background: rgba(108,92,231,0.05);
            border-color: rgba(108,92,231,0.2);
            transform: translateX(4px);
        }}

        .learning-line.active {{
            background: rgba(108,92,231,0.12);
            border: 2px solid var(--primary);
            box-shadow: 0 8px 20px rgba(108,92,231,0.2);
            transform: scale(1.02);
            margin: 8px 0;
        }}

        .line-number {{
            width: 36px;
            height: 36px;
            display: flex;
            align-items: center;
            justify-content: center;
            background: var(--light);
            border-radius: 50%;
            font-size: 14px;
            font-weight: 600;
            color: var(--dark);
            margin-right: 12px;
            flex-shrink: 0;
        }}

        .learning-line.active .line-number {{
            background: var(--primary);
            color: white;
        }}

        /* ========== 只读的行 ========== */
        .readonly-line {{
            display: flex;
            align-items: flex-start;
            padding: 12px 16px;
            border-radius: 12px;
            background: #fafafa;
            color: #666;
            font-size: 15px;
            line-height: 1.5;
            border-left: 4px solid var(--gray);
            margin-left: 8px;
            cursor: default;
            user-select: none;
        }}

        .readonly-line .text-content {{
            color: #666;
            font-style: italic;
        }}

        .readonly-indicator {{
            width: 24px;
            height: 24px;
            display: flex;
            align-items: center;
            justify-content: center;
            color: #999;
            font-size: 14px;
            margin-right: 12px;
            flex-shrink: 0;
        }}

        .text-content {{
            flex: 1;
            font-size: 17px;
            line-height: 1.6;
            word-break: break-word;
            padding: 6px 0;
        }}

        .learning-line .text-content {{
            color: var(--dark);
        }}

        /* 语言标签 */
        .lang-tag {{
            font-size: 12px;
            padding: 4px 8px;
            border-radius: 12px;
            background: rgba(108,92,231,0.1);
            color: var(--primary);
            margin-left: 8px;
            white-space: nowrap;
            font-weight: 600;
        }}

        .lang-tag.ch {{ background: rgba(0,184,148,0.1); color: #00B894; }}
        .lang-tag.jp {{ background: rgba(253,121,168,0.1); color: #FD79A8; }}
        .lang-tag.en {{ background: rgba(108,92,231,0.1); color: #6C5CE7; }}

        /* 音频指示器 */
        .audio-indicator {{
            width: 24px;
            height: 24px;
            border-radius: 50%;
            background: var(--secondary);
            color: white;
            display: inline-flex;
            align-items: center;
            justify-content: center;
            font-size: 12px;
            margin-left: 8px;
            flex-shrink: 0;
            animation: pulse 2s infinite;
        }}

        @keyframes pulse {{
            0% {{ box-shadow: 0 0 0 0 rgba(0,184,148,0.4); }}
            70% {{ box-shadow: 0 0 0 6px rgba(0,184,148,0); }}
            100% {{ box-shadow: 0 0 0 0 rgba(0,184,148,0); }}
        }}

        /* 操作按钮组 */
        .action-buttons {{
            display: flex;
            gap: 8px;
            margin-left: 12px;
            flex-shrink: 0;
        }}

        .action-btn {{
            width: 40px;
            height: 40px;
            border: none;
            border-radius: 50%;
            background: var(--light);
            color: var(--dark);
            font-size: 18px;
            display: flex;
            align-items: center;
            justify-content: center;
            cursor: pointer;
            transition: all 0.2s;
        }}

        .action-btn:hover {{
            transform: scale(1.1);
            box-shadow: 0 4px 8px rgba(0,0,0,0.1);
        }}

        .action-btn:active {{
            transform: scale(0.95);
        }}

        .action-btn.added {{
            background: var(--secondary);
            color: white;
        }}

        .action-btn.grammar.added {{
            background: var(--accent);
            color: white;
        }}

        /* ========== 面板样式 ========== */
        .panel {{
            flex: 1;
            overflow-y: auto;
            padding: 20px;
            background: white;
            border-radius: 24px;
            height: 0;
        }}

        .panel h2 {{
            font-size: 20px;
            color: var(--primary);
            margin-bottom: 20px;
            padding-bottom: 12px;
            border-bottom: 2px solid var(--primary);
            position: sticky;
            top: 0;
            background: white;
            z-index: 10;
        }}

        .word-card {{
            padding: 16px;
            border-radius: 16px;
            background: var(--light);
            margin-bottom: 12px;
            border-left: 6px solid var(--primary);
            transition: all 0.2s;
        }}

        .word-card:hover {{
            transform: translateX(4px);
            box-shadow: 0 4px 12px rgba(0,0,0,0.1);
        }}

        .word-card.grammar-card {{
            border-left-color: var(--accent);
        }}

        .word-text {{
            font-size: 17px;
            font-weight: 600;
            margin-bottom: 8px;
            color: var(--dark);
        }}

        .word-meta {{
            display: flex;
            gap: 16px;
            font-size: 12px;
            color: #666;
            flex-wrap: wrap;
            margin-bottom: 8px;
        }}

        .grammar-note {{
            background: var(--accent);
            color: white;
            padding: 8px 12px;
            border-radius: 8px;
            margin-top: 8px;
            font-size: 14px;
        }}

        .empty-message {{
            text-align: center;
            padding: 40px;
            color: #666;
            font-size: 16px;
        }}

        /* ========== 状态栏 ========== */
        .status-bar {{
            flex-shrink: 0;
            margin-top: 16px;
            padding: 16px 24px;
            background: white;
            border-radius: 30px;
            box-shadow: 0 -4px 20px rgba(0,0,0,0.05);
            display: flex;
            justify-content: space-between;
            align-items: center;
            font-size: 14px;
            color: var(--dark);
            border: 1px solid var(--light);
        }}

        .badge {{
            background: var(--primary);
            color: white;
            padding: 4px 12px;
            border-radius: 20px;
            font-size: 12px;
        }}

        .hidden {{
            display: none !important;
        }}

        @media (max-width: 600px) {{
            .app {{ padding: 16px; height: calc(100vh - 32px); }}
            .text-content {{ font-size: 16px; }}
            .action-btn {{ width: 36px; height: 36px; font-size: 16px; }}
        }}
    </style>
</head>
<body>
    <div class="app">
        <div class="app-header">
            <h1>
                <span>🧠 艾宾浩斯·自动学习</span>
                <span class="badge">🔑 {hashlib.md5(str(total_chapters).encode()).hexdigest()[:8]}</span>
            </h1>

            <div class="nav-tabs">
                <button class="nav-tab active" onclick="switchTab('learn')">📚 自动学习</button>
                <button class="nav-tab" onclick="switchTab('notebook')">
                    📔 单词本
                    <span id="notebookBadge" style="background: var(--secondary); color: white; padding: 2px 8px; border-radius: 12px; font-size: 12px;">0</span>
                </button>
                <button class="nav-tab" onclick="switchTab('grammar')">
                    🔤 语法本
                    <span id="grammarBadge" style="background: var(--accent); color: white; padding: 2px 8px; border-radius: 12px; font-size: 12px;">0</span>
                </button>
                <button class="nav-tab" onclick="switchTab('review')">
                    🧠 复习
                    <span id="reviewBadge" style="background: var(--danger); color: white; padding: 2px 8px; border-radius: 12px; font-size: 12px;">0</span>
                </button>
            </div>
        </div>

        <!-- 学习面板 -->
        <div id="learnPanel" style="display: flex; flex-direction: column; flex: 1; min-height: 0;">
            <div class="chapter-selector">
                <button class="nav-btn" onclick="prevChapter()">◀</button>
                <select id="chapterSelect" class="chapter-select" onchange="loadChapter(parseInt(this.value))">
                    <option value="">📚 请选择章节</option>
                </select>
                <button class="nav-btn" onclick="nextChapter()">▶</button>
            </div>

            <div class="auto-play-bar">
                <button id="playPauseBtn" class="play-control-btn" onclick="togglePlayPause()">▶</button>
                <div class="progress-info">
                    <div style="display: flex; justify-content: space-between; margin-bottom: 4px;">
                        <span id="currentLineInfo">第 0/0 句</span>
                        <span id="chapterInfo">第 0/{total_chapters} 章</span>
                    </div>
                    <div class="progress-bar">
                        <div id="progressFill" class="progress-fill" style="width: 0%;"></div>
                    </div>
                </div>
            </div>

            <!-- ========== 滚动容器 ========== -->
            <div id="scrollContainer" class="scroll-container">
                <div id="chapterTitle" class="chapter-title">请选择章节</div>
                <div id="linesContainer" class="lines-container"></div>
            </div>
        </div>

        <!-- 单词本面板 -->
        <div id="notebookPanel" class="panel hidden">
            <h2>📔 我的单词本</h2>
            <div id="wordList"></div>
        </div>

        <!-- 语法本面板 -->
        <div id="grammarPanel" class="panel hidden">
            <h2>🔤 我的语法本</h2>
            <div id="grammarList"></div>
        </div>

        <!-- 复习面板 -->
        <div id="reviewPanel" class="panel hidden">
            <h2>🧠 今日复习</h2>
            <div id="reviewList"></div>
        </div>

        <div class="status-bar">
            <span>📚 {total_chapters}章 · {total_lines}句</span>
            <span>🎵 {total_audio}音频</span>
            <span>📝 可学习 {total_learnable}句</span>
            <span id="currentTime">{datetime.now().strftime('%H:%M')}</span>
        </div>
    </div>

    <script>
        // ========== 数据 ==========
        const CHAPTERS_DATA = {chapters_json_str};
        const LEARNING_ITEMS = {items_json_str};
        const TOTAL_CHAPTERS = {total_chapters};

        console.log('=================================');
        console.log('✅ 艾宾浩斯学习系统启动');
        console.log('📊 总章节数:', TOTAL_CHAPTERS);
        console.log('📊 处理后章节数:', CHAPTERS_DATA ? CHAPTERS_DATA.length : 0);
        console.log('=================================');

        // ========== 全局变量 ==========
        let currentChapter = 0;
        let currentLineIndex = 0;
        let isPlaying = false;
        let playTimer = null;
        let currentAudio = null;

        // ========== 页面初始化 ==========
        window.addEventListener('load', function() {{
            console.log('页面加载完成，开始初始化...');

            if (!CHAPTERS_DATA || !Array.isArray(CHAPTERS_DATA) || CHAPTERS_DATA.length === 0) {{
                console.error('❌ 错误：章节数据为空！');
                return;
            }}

            initChapterSelect();
            loadFromStorage();

            if (CHAPTERS_DATA.length > 0) {{
                loadChapter(0);
            }}

            updateAllPanels();
        }});

        // ========== 初始化章节选择器 ==========
        function initChapterSelect() {{
            const select = document.getElementById('chapterSelect');
            if (!select) return;

            select.innerHTML = '<option value="">📚 请选择章节</option>';

            for (let i = 1; i <= CHAPTERS_DATA.length; i++) {{
                const option = document.createElement('option');
                option.value = i - 1;
                const chapter = CHAPTERS_DATA[i-1];
                const learnableCount = chapter ? chapter.filter(line => line.is_learnable).length : 0;
                const readonlyCount = chapter ? chapter.filter(line => !line.is_learnable).length : 0;
                const audioCount = chapter ? chapter.filter(line => line.audio && line.audio.length > 100).length : 0;
                option.textContent = '第 ' + i + ' 章 (可学习' + learnableCount + '句, 只读' + readonlyCount + '句, 音频' + audioCount + '个)';
                select.appendChild(option);
            }}

            console.log('✅ 章节选择器初始化完成，共', select.options.length - 1, '个章节');
        }}

        // ========== 本地存储 ==========
        function loadFromStorage() {{
            const saved = localStorage.getItem('ebbinghaus_notebook');
            if (saved) {{
                try {{
                    const notebook = JSON.parse(saved);
                    if (LEARNING_ITEMS) {{
                        LEARNING_ITEMS.forEach(item => {{
                            const savedItem = notebook.find(n => n.id === item.id);
                            if (savedItem) {{
                                item.added_to_notebook = savedItem.added_to_notebook || false;
                                item.added_to_grammar = savedItem.added_to_grammar || false;
                                item.grammar_note = savedItem.grammar_note || '';
                                item.mastery = savedItem.mastery || 0;
                                item.review_count = savedItem.review_count || 0;
                                item.last_review = savedItem.last_review;
                                item.next_review = savedItem.next_review || 0;
                            }}
                        }});
                    }}
                    console.log('✅ 本地存储加载成功');
                }} catch (e) {{
                    console.error('❌ 本地存储加载失败:', e);
                }}
            }}
        }}

        function saveToStorage() {{
            if (!LEARNING_ITEMS) return;
            const notebook = LEARNING_ITEMS.map(item => ({{
                id: item.id,
                added_to_notebook: item.added_to_notebook || false,
                added_to_grammar: item.added_to_grammar || false,
                grammar_note: item.grammar_note || '',
                mastery: item.mastery || 0,
                review_count: item.review_count || 0,
                last_review: item.last_review,
                next_review: item.next_review || 0
            }})).filter(item => item.added_to_notebook || item.added_to_grammar);
            localStorage.setItem('ebbinghaus_notebook', JSON.stringify(notebook));
        }}

        // ========== 章节加载 ==========
        function loadChapter(index) {{
            if (index < 0 || index >= CHAPTERS_DATA.length) return;

            currentChapter = index;
            currentLineIndex = 0;

            const select = document.getElementById('chapterSelect');
            if (select) select.value = index;

            const titleEl = document.getElementById('chapterTitle');
            if (titleEl) titleEl.innerHTML = '第 ' + (index + 1) + ' 章';

            const infoEl = document.getElementById('chapterInfo');
            if (infoEl) infoEl.innerHTML = '第 ' + (index + 1) + '/' + CHAPTERS_DATA.length + ' 章';

            renderChapter();
            stopPlay();
        }}

        // ========== 渲染章节内容 ==========
        function renderChapter() {{
            const chapterData = CHAPTERS_DATA[currentChapter];
            if (!chapterData || !Array.isArray(chapterData)) return;

            const container = document.getElementById('linesContainer');
            let html = '';

            chapterData.forEach((line, idx) => {{
                if (!line) return;

                if (line.is_learnable) {{
                    // ===== 可学习的行 =====
                    const hasAudio = line.audio && line.audio.length > 100;
                    const learningItem = LEARNING_ITEMS?.find(item => item && item.id === line.id);
                    const isNotebook = learningItem ? learningItem.added_to_notebook : false;
                    const isGrammar = learningItem ? learningItem.added_to_grammar : false;

                    html += '<div id="line-' + line.id + '" class="learning-line" onclick="playThisLine(\\'' + line.id + '\\', ' + idx + ')">';
                    html += '<span class="line-number">' + (idx + 1) + '</span>';
                    html += '<div class="text-content">' + (line.content || '').replace(/\\[(ch|jp|en)\\]\\s*/g, '').replace(/\\[sound:[^\\]]*\\]/g, '') + '</div>';

                    // 显示语言标签
                    let langClass = '';
                    let langText = '';
                    if (line.lang === 'ch') {{ langClass = 'ch'; langText = '中文'; }}
                    else if (line.lang === 'jp') {{ langClass = 'jp'; langText = '日语'; }}
                    else if (line.lang === 'en') {{ langClass = 'en'; langText = '英语'; }}
                    html += '<span class="lang-tag ' + langClass + '">' + langText + '</span>';

                    if (hasAudio) {{
                        html += '<span class="audio-indicator">🔊</span>';
                    }}

                    // 显示操作按钮
                    html += '<div class="action-buttons">';
                    html += '<button class="action-btn' + (isNotebook ? ' added' : '') + '" onclick="event.stopPropagation(); toggleNotebook(\\'' + line.id + '\\', this)">' + (isNotebook ? '✓' : '📚') + '</button>';
                    html += '<button class="action-btn grammar' + (isGrammar ? ' added' : '') + '" onclick="event.stopPropagation(); addGrammar(\\'' + line.id + '\\', this)">🔤</button>';
                    html += '</div>';
                    html += '</div>';

                }} else {{
                    // ===== 只读的行 =====
                    html += '<div class="readonly-line">';
                    html += '<span class="readonly-indicator">📄</span>';
                    html += '<div class="text-content">' + (line.content || '').replace(/\\[(ch|jp|en)\\]\\s*/g, '').replace(/\\[sound:[^\\]]*\\]/g, '') + '</div>';
                    html += '</div>';
                }}
            }});

            container.innerHTML = html;
            updateProgress();

            // 滚动到顶部
            const scrollContainer = document.getElementById('scrollContainer');
            if (scrollContainer) {{
                scrollContainer.scrollTop = 0;
            }}
        }}

        // ========== 居中滚动函数 ==========
        function scrollToCenter(element) {{
            const container = document.getElementById('scrollContainer');
            if (!container || !element) return;

            const containerHeight = container.clientHeight;
            const elementTop = element.offsetTop;
            const elementHeight = element.clientHeight;

            // 计算滚动位置，使元素居中
            const scrollTop = elementTop - (containerHeight / 2) + (elementHeight / 2);

            container.scrollTo({{
                top: Math.max(0, scrollTop),
                behavior: 'smooth'
            }});
        }}

        // ========== 播放指定行 ==========
        function playThisLine(lineId, index) {{
            console.log('播放行:', lineId, index);

            const chapterData = CHAPTERS_DATA[currentChapter];
            if (!chapterData || index >= chapterData.length) return;

            const line = chapterData[index];

            // 只播放可学习的行
            if (!line.is_learnable) return;

            // 更新当前行索引
            currentLineIndex = index;

            // 移除所有active类
            document.querySelectorAll('.learning-line').forEach(el => el.classList.remove('active'));

            // 为当前行添加active类
            const currentLine = document.getElementById('line-' + lineId);
            if (currentLine) {{
                currentLine.classList.add('active');
                // 滚动到居中位置
                scrollToCenter(currentLine);
            }}

            // 播放音频
            if (line && line.audio && line.audio.length > 100) {{
                playAudio(line.audio);
            }}

            updateProgress();
        }}

        // ========== 音频播放函数 ==========
        function playAudio(audioData) {{
            return new Promise((resolve, reject) => {{
                try {{
                    if (!audioData || audioData.length < 100) {{
                        console.log('无效的音频数据');
                        reject('无效的音频数据');
                        return;
                    }}

                    let audioSrc = audioData;
                    if (!audioSrc.startsWith('data:audio')) {{
                        audioSrc = 'data:audio/mp3;base64,' + audioSrc;
                    }}

                    if (currentAudio) {{
                        currentAudio.pause();
                        currentAudio.currentTime = 0;
                    }}

                    currentAudio = new Audio(audioSrc);

                    currentAudio.onended = function() {{
                        console.log('音频播放完成');
                        currentAudio = null;
                        resolve();
                    }};

                    currentAudio.onerror = function(e) {{
                        console.error('音频播放错误:', e);
                        currentAudio = null;
                        reject(e);
                    }};

                    currentAudio.play().catch(e => {{
                        console.error('播放失败:', e);
                        currentAudio = null;
                        reject(e);
                    }});

                }} catch (e) {{
                    console.error('音频创建失败:', e);
                    reject(e);
                }}
            }});
        }}

        // ========== 自动播放控制 ==========
        function togglePlayPause() {{
            const btn = document.getElementById('playPauseBtn');
            if (isPlaying) {{
                stopPlay();
                btn.innerHTML = '▶';
                btn.classList.remove('playing');
            }} else {{
                startPlay();
                btn.innerHTML = '⏸';
                btn.classList.add('playing');
            }}
        }}

        function startPlay() {{
            if (isPlaying) return;
            if (!CHAPTERS_DATA[currentChapter]) return;
            isPlaying = true;
            playCurrentLine();
        }}

        function stopPlay() {{
            isPlaying = false;
            if (playTimer) {{
                clearTimeout(playTimer);
                playTimer = null;
            }}
            if (currentAudio) {{
                currentAudio.pause();
                currentAudio.currentTime = 0;
                currentAudio = null;
            }}
            const btn = document.getElementById('playPauseBtn');
            if (btn) {{
                btn.innerHTML = '▶';
                btn.classList.remove('playing');
            }}
        }}

        async function playCurrentLine() {{
            if (!isPlaying) return;

            const chapterData = CHAPTERS_DATA[currentChapter];
            if (!chapterData) return;

            // 跳过只读行，找到下一个可学习行
            while (currentLineIndex < chapterData.length && !chapterData[currentLineIndex].is_learnable) {{
                currentLineIndex++;
                updateProgress();
            }}

            if (currentLineIndex >= chapterData.length) {{
                if (currentChapter + 1 < CHAPTERS_DATA.length) {{
                    loadChapter(currentChapter + 1);
                    currentLineIndex = 0;
                    setTimeout(() => playCurrentLine(), 800);
                }} else {{
                    stopPlay();
                }}
                return;
            }}

            const line = chapterData[currentLineIndex];
            const lineId = line.id;

            // 高亮当前行并居中
            document.querySelectorAll('.learning-line').forEach(el => el.classList.remove('active'));
            const currentLine = document.getElementById('line-' + lineId);
            if (currentLine) {{
                currentLine.classList.add('active');
                scrollToCenter(currentLine);
            }}

            // 播放音频
            if (line && line.audio && line.audio.length > 100) {{
                try {{
                    await playAudio(line.audio);
                    currentLineIndex++;
                    updateProgress();
                    if (isPlaying) {{
                        playTimer = setTimeout(() => playCurrentLine(), 300);
                    }}
                }} catch (e) {{
                    console.log('音频播放失败，跳过:', e);
                    currentLineIndex++;
                    updateProgress();
                    if (isPlaying) {{
                        playTimer = setTimeout(() => playCurrentLine(), 300);
                    }}
                }}
            }} else {{
                currentLineIndex++;
                updateProgress();
                if (isPlaying) {{
                    playTimer = setTimeout(() => playCurrentLine(), 300);
                }}
            }}
        }}

        function updateProgress() {{
            const chapterData = CHAPTERS_DATA[currentChapter];
            if (!chapterData) return;

            const learnableCount = chapterData.filter(line => line.is_learnable).length;
            const currentLearnableIndex = chapterData.slice(0, currentLineIndex).filter(line => line.is_learnable).length;

            const progress = learnableCount > 0 ? (currentLearnableIndex / learnableCount) * 100 : 0;
            const fill = document.getElementById('progressFill');
            if (fill) fill.style.width = progress + '%';

            const lineInfo = document.getElementById('currentLineInfo');
            if (lineInfo) {{
                lineInfo.innerHTML = '第 ' + (currentLearnableIndex + 1) + '/' + learnableCount + ' 句';
            }}
        }}

        // ========== 单词本操作 ==========
        function toggleNotebook(itemId, btn) {{
            const item = LEARNING_ITEMS?.find(i => i.id === itemId);
            if (item) {{
                item.added_to_notebook = !item.added_to_notebook;
                btn.innerHTML = item.added_to_notebook ? '✓' : '📚';
                btn.classList.toggle('added');
                saveToStorage();
                updateNotebookPanel();
                updateBadges();
            }}
        }}

        function addGrammar(itemId, btn) {{
            const item = LEARNING_ITEMS?.find(i => i.id === itemId);
            if (item) {{
                const note = prompt('添加语法笔记:', item.grammar_note || '');
                if (note !== null) {{
                    item.added_to_grammar = true;
                    item.grammar_note = note;
                    btn.classList.add('added');
                    saveToStorage();
                    updateGrammarPanel();
                    updateBadges();
                }}
            }}
        }}

        // ========== 单词本面板 ==========
        function updateNotebookPanel() {{
            const notebook = LEARNING_ITEMS?.filter(item => item && item.added_to_notebook) || [];
            const list = document.getElementById('wordList');

            let html = '';
            notebook.forEach(item => {{
                html += '<div class="word-card">';
                html += '<div class="word-text">' + (item.content || '').replace(/\\[(ch|jp|en)\\]\\s*/g, '').replace(/\\[sound:[^\\]]*\\]/g, '') + '</div>';
                html += '<div class="word-meta">';
                html += '<span>📌 第' + (item.chapter || 0) + '章</span>';
                html += '<span>🎯 掌握度 ' + (item.mastery || 0) + '%</span>';
                html += '<span>🔄 复习' + (item.review_count || 0) + '次</span>';
                html += '</div>';
                html += '<button class="action-btn" style="background: var(--danger); color: white; margin-top: 8px;" onclick="toggleNotebook(\\'' + item.id + '\\', this)">🗑️ 移除</button>';
                html += '</div>';
            }});

            if (list) {{
                list.innerHTML = html || '<div class="empty-message">📭 单词本还是空的，点击句子旁边的📚收藏</div>';
            }}

            document.getElementById('notebookBadge').innerHTML = notebook.length;
        }}

        // ========== 语法本面板 ==========
        function updateGrammarPanel() {{
            const grammar = LEARNING_ITEMS?.filter(item => item && item.added_to_grammar) || [];
            const list = document.getElementById('grammarList');

            let html = '';
            grammar.forEach(item => {{
                html += '<div class="word-card grammar-card">';
                html += '<div class="word-text">' + (item.content || '').replace(/\\[(ch|jp|en)\\]\\s*/g, '').replace(/\\[sound:[^\\]]*\\]/g, '') + '</div>';
                html += '<div class="grammar-note">📝 ' + (item.grammar_note || '无备注') + '</div>';
                html += '<div class="word-meta" style="margin-top: 8px;">';
                html += '<span>📌 第' + (item.chapter || 0) + '章</span>';
                html += '</div>';
                html += '</div>';
            }});

            if (list) {{
                list.innerHTML = html || '<div class="empty-message">🔤 语法本还是空的，点击句子旁边的🔤添加笔记</div>';
            }}

            document.getElementById('grammarBadge').innerHTML = grammar.length;
        }}

        // ========== 更新徽章 ==========
        function updateBadges() {{
            if (!LEARNING_ITEMS) return;

            const notebook = LEARNING_ITEMS.filter(item => item && item.added_to_notebook).length || 0;
            const grammar = LEARNING_ITEMS.filter(item => item && item.added_to_grammar).length || 0;

            document.getElementById('notebookBadge').innerHTML = notebook;
            document.getElementById('grammarBadge').innerHTML = grammar;
        }}

        // ========== 更新所有面板 ==========
        function updateAllPanels() {{
            if (!LEARNING_ITEMS) return;
            updateNotebookPanel();
            updateGrammarPanel();
            updateBadges();
        }}

        // ========== 章节导航 ==========
        function prevChapter() {{
            if (currentChapter > 0) {{
                loadChapter(currentChapter - 1);
                stopPlay();
            }}
        }}

        function nextChapter() {{
            if (currentChapter + 1 < CHAPTERS_DATA.length) {{
                loadChapter(currentChapter + 1);
                stopPlay();
            }}
        }}

        // ========== 标签页切换 ==========
        function switchTab(tab) {{
            document.querySelectorAll('.nav-tab').forEach(t => t.classList.remove('active'));
            event.currentTarget.classList.add('active');

            const learnPanel = document.getElementById('learnPanel');
            const notebookPanel = document.getElementById('notebookPanel');
            const grammarPanel = document.getElementById('grammarPanel');
            const reviewPanel = document.getElementById('reviewPanel');

            if (learnPanel) learnPanel.classList.toggle('hidden', tab !== 'learn');
            if (notebookPanel) notebookPanel.classList.toggle('hidden', tab !== 'notebook');
            if (grammarPanel) grammarPanel.classList.toggle('hidden', tab !== 'grammar');
            if (reviewPanel) reviewPanel.classList.toggle('hidden', tab !== 'review');

            if (tab === 'notebook') updateNotebookPanel();
            if (tab === 'grammar') updateGrammarPanel();
        }}

        // ========== 暴露全局函数 ==========
        window.switchTab = switchTab;
        window.loadChapter = loadChapter;
        window.prevChapter = prevChapter;
        window.nextChapter = nextChapter;
        window.togglePlayPause = togglePlayPause;
        window.toggleNotebook = toggleNotebook;
        window.addGrammar = addGrammar;
        window.playThisLine = playThisLine;
    </script>
</body>
</html>
    '''

    # 写入主HTML文件
    with open(html_path, 'w', encoding='utf-8') as f:
        f.write(html_content)

    file_size = os.path.getsize(html_path) // 1024

    print(f"""
╔══════════════════════════════════════════════════════════════╗
║  ✅ 艾宾浩斯·自动学习系统 生成成功                          ║
╠══════════════════════════════════════════════════════════════╣
║  📁 文件: {os.path.basename(html_path)}
║  📊 总章节: {len(simplified_chapters)} 章
║  📝 可学习语句: {total_learnable} 句
║  📄 只读语句: {total_readonly} 句
║  🎵 有效音频: {total_valid_audio} 个
║  💾 文件大小: {file_size} KB
║
║  📖 章节详情:
{chapter_stats_str}
║
║  🎯 滚动容器优化：
║     • 📦 固定高度滚动容器 - 所有内容都在框内滚动
║     • 🎯 自动居中 - 点击或自动播放时当前行居中显示
║     • 🖱️ 平滑滚动 - 滚动动画平滑自然
║     • 🎨 自定义滚动条 - 紫色主题滚动条
║     • 📱 自适应高度 - 自动适配屏幕
║
║  ✨ 其他功能：
║     • 可学习行/只读行清晰区分
║     • 点击句子播放音频并居中
║     • 自动播放跳过只读行
║     • 单词本/语法本正常显示
║
║  📱 现在所有内容都在滚动框里，当前行自动居中！
║
╚══════════════════════════════════════════════════════════════╝
    """)

    return html_path


In [ ]:
# ============================================================
# 🧠 艾宾浩斯·自动学习系统 - 四色主题版


In [ ]:
# ============================================================

def generate_super_language_learning_system_desktop(html_path, chapters):
    """
    生成超级语言学习系统 - 四色主题版
    功能：四种阅读主题 + 自动居中 + 可学习/只读行区分
    """
    import json
    from datetime import datetime
    import os
    import hashlib
    import base64
    import re
    from bs4 import BeautifulSoup

    print(f"  📊 正在为 {len(chapters)} 章内容生成艾宾浩斯学习系统...")

    if len(chapters) == 0:
        print("  ❌ 错误：章节数据为空，无法生成学习系统")
        return

    # 计算统计数据
    total_chapters = len(chapters)
    total_lines = sum(len(c) for c in chapters)
    total_audio = sum(1 for c in chapters for l in c if l.get('audio') and len(l.get('audio', '')) > 100)

    print(f"  📊 总章节: {total_chapters}, 总行数: {total_lines}, 总音频: {total_audio}")

    # ========== 构建完整的章节数据 ==========
    simplified_chapters = []

    # 定义变量用于统计
    total_learnable = 0
    total_readonly = 0
    total_valid_audio = 0

    for chap_idx, chapter in enumerate(chapters):
        simplified_lines = []
        for line_idx, line in enumerate(chapter):
            # 提取纯文本内容
            text = line.get('text', '')
            audio = line.get('audio', '')

            # 使用BeautifulSoup提取纯文本
            soup = BeautifulSoup(text, "html.parser")
            plain_text = remove_sound_tags(soup.get_text().strip())

            # 判断语言类型 - 只有以 [ch]、[jp]、[en] 开头的行才是可学习行
            lang = 'unknown'
            is_learnable = False

            if plain_text.startswith('[ch]'):
                lang = 'ch'
                is_learnable = True
            elif plain_text.startswith('[jp]'):
                lang = 'jp'
                is_learnable = True
            elif plain_text.startswith('[en]'):
                lang = 'en'
                is_learnable = True

            # 移除所有语言标签（不仅是行首的）
            content = re.sub(r'\[sound:[^\]]*\]', '', re.sub(r'\[(ch|jp|en)\]\s*', '', plain_text)).strip()

            # 如果内容为空，跳过
            if not content:
                continue

            # 确保音频数据完整
            has_valid_audio = False
            if audio and len(audio) > 100:
                if not audio.startswith('data:audio'):
                    audio = 'data:audio/mp3;base64,' + audio
                has_valid_audio = True
                if is_learnable:
                    total_valid_audio += 1

            # 更新统计
            if is_learnable:
                total_learnable += 1
            else:
                total_readonly += 1

            simplified_lines.append({
                'id': f'{chap_idx:04d}_{line_idx:03d}',
                'content': content,
                'audio': audio if has_valid_audio else '',
                'lang': lang,
                'chapter': chap_idx + 1,
                'has_audio': has_valid_audio,
                'is_learnable': is_learnable
            })

        if simplified_lines:
            simplified_chapters.append(simplified_lines)

    print(f"  📝 处理后章节数: {len(simplified_chapters)}")
    print(f"  📝 可学习语句: {total_learnable} 句, 只读语句: {total_readonly} 句")
    print(f"  📝 有效音频: {total_valid_audio} 个")

    # ========== 构建学习项列表 ==========
    learning_items = []
    for chap_idx, chapter in enumerate(simplified_chapters):
        for line in chapter:
            if line.get('is_learnable'):
                learning_items.append({
                    'id': line['id'],
                    'chapter': line['chapter'],
                    'content': line['content'][:100] + '...' if len(line['content']) > 100 else line['content'],
                    'lang': line['lang'],
                    'has_audio': line.get('has_audio', False),
                    'mastery': 0,
                    'review_count': 0,
                    'added_to_notebook': False,
                    'added_to_grammar': False,
                    'grammar_note': '',
                    'last_review': None,
                    'next_review': 0
                })

    # ========== 将数据转换为JSON字符串 ==========
    chapters_json_str = json.dumps(simplified_chapters, ensure_ascii=False)
    items_json_str = json.dumps(learning_items, ensure_ascii=False)

    # ========== 生成章节统计信息字符串 ==========
    chapter_stats = []
    for i, c in enumerate(simplified_chapters[:5]):
        learnable_count = sum(1 for line in c if line.get('is_learnable'))
        readonly_count = sum(1 for line in c if not line.get('is_learnable'))
        audio_count = sum(1 for line in c if line.get('has_audio'))
        chapter_stats.append(f'     第{i+1}章: 可学习{learnable_count}句, 只读{readonly_count}句, 音频{audio_count}个')
    if len(simplified_chapters) > 5:
        chapter_stats.append('      ...')
    chapter_stats_str = '\n'.join(chapter_stats)

    # ========== HTML内容 - 四色主题版 ==========
    html_content = f'''<!DOCTYPE html>
<html lang="zh">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0, maximum-scale=1.0, user-scalable=no, viewport-fit=cover">
    <title>🧠 艾宾浩斯·自动学习系统</title>
    <meta name="theme-color" content="#6C5CE7">
    <meta name="mobile-web-app-capable" content="yes">
    <meta name="apple-mobile-web-app-capable" content="yes">
    <style>
        /* ========== 全局样式 ========== */
        * {{
            margin: 0;
            padding: 0;
            box-sizing: border-box;
            font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, "PingFang SC", "Microsoft YaHei", sans-serif;
            transition: background-color 0.3s, color 0.3s, border-color 0.3s;
        }}

        body {{
            min-height: 100vh;
            padding: 16px;
            margin: 0;
            transition: background 0.3s;
        }}

        .app {{
            max-width: 1000px;
            margin: 0 auto;
            border-radius: 40px;
            padding: 24px;
            box-shadow: 0 25px 50px -12px rgba(0,0,0,0.25);
            display: flex;
            flex-direction: column;
            height: calc(100vh - 32px);
            max-height: 1000px;
            transition: background-color 0.3s, box-shadow 0.3s;
        }}

        .app-header {{
            flex-shrink: 0;
        }}

        h1 {{
            font-size: 24px;
            margin-bottom: 16px;
            display: flex;
            justify-content: space-between;
            align-items: center;
        }}

        /* ========== 四种主题 ========== */
        /* 主题1: 经典紫 - 默认 */
        body.theme-purple {{
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
        }}
        body.theme-purple .app {{
            background: rgba(255, 255, 255, 0.98);
            backdrop-filter: blur(20px);
            color: #2D3436;
        }}
        body.theme-purple .chapter-selector,
        body.theme-purple .status-bar,
        body.theme-purple .scroll-container,
        body.theme-purple .panel {{
            background: white;
            border-color: #F5F6FA;
        }}

        /* 主题2: 护眼绿 */
        body.theme-green {{
            background: #e8f5e9;
        }}
        body.theme-green .app {{
            background: #f1f8e9;
            box-shadow: 0 25px 50px -12px rgba(46, 125, 50, 0.2);
            color: #1e3a1e;
        }}
        body.theme-green .chapter-selector,
        body.theme-green .status-bar,
        body.theme-green .scroll-container,
        body.theme-green .panel {{
            background: #f9fff9;
            border-color: #c8e6c9;
        }}
        body.theme-green .nav-tab.active {{
            background: #2e7d32;
        }}
        body.theme-green .auto-play-bar {{
            background: linear-gradient(135deg, #2e7d32, #1b5e20);
        }}
        body.theme-green .play-control-btn {{
            color: #2e7d32;
        }}
        body.theme-green .badge {{
            background: #2e7d32;
        }}

        /* 主题3: 深夜黑 */
        body.theme-dark {{
            background: #121212;
        }}
        body.theme-dark .app {{
            background: #1e1e1e;
            color: #e0e0e0;
            border: 1px solid #333;
        }}
        body.theme-dark .chapter-selector,
        body.theme-dark .status-bar,
        body.theme-dark .scroll-container,
        body.theme-dark .panel {{
            background: #2d2d2d;
            border-color: #404040;
            color: #e0e0e0;
        }}
        body.theme-dark .chapter-title,
        body.theme-dark h1,
        body.theme-dark h2 {{
            color: #bb86fc;
        }}
        body.theme-dark .nav-tab {{
            color: #e0e0e0;
        }}
        body.theme-dark .nav-tab.active {{
            background: #bb86fc;
            color: #121212;
        }}
        body.theme-dark .auto-play-bar {{
            background: linear-gradient(135deg, #bb86fc, #9a67ea);
        }}
        body.theme-dark .play-control-btn {{
            color: #bb86fc;
        }}
        body.theme-dark .badge {{
            background: #bb86fc;
            color: #121212;
        }}
        body.theme-dark .learning-line {{
            background: #2d2d2d;
            color: #e0e0e0;
        }}
        body.theme-dark .readonly-line {{
            background: #252525;
            color: #999;
        }}

        /* 主题4: 纸质米黄 */
        body.theme-sepia {{
            background: #dac0a1;
        }}
        body.theme-sepia .app {{
            background: #fdf5e6;
            color: #5d3a1a;
        }}
        body.theme-sepia .chapter-selector,
        body.theme-sepia .status-bar,
        body.theme-sepia .scroll-container,
        body.theme-sepia .panel {{
            background: #fff8e7;
            border-color: #e6d3b5;
        }}
        body.theme-sepia .chapter-title {{
            color: #8b4513;
            border-bottom-color: #8b4513;
        }}
        body.theme-sepia .nav-tab.active {{
            background: #8b4513;
        }}
        body.theme-sepia .auto-play-bar {{
            background: linear-gradient(135deg, #8b4513, #a0522d);
        }}
        body.theme-sepia .play-control-btn {{
            color: #8b4513;
        }}
        body.theme-sepia .badge {{
            background: #8b4513;
        }}
        body.theme-sepia .learning-line.active {{
            border-color: #8b4513;
        }}

        /* ========== 主题切换栏 ========== */
        .theme-switcher {{
            display: flex;
            gap: 8px;
            margin-bottom: 16px;
            padding: 8px;
            background: rgba(0,0,0,0.05);
            border-radius: 30px;
            justify-content: flex-end;
        }}

        .theme-btn {{
            width: 36px;
            height: 36px;
            border: 2px solid transparent;
            border-radius: 50%;
            cursor: pointer;
            transition: all 0.2s;
            display: flex;
            align-items: center;
            justify-content: center;
            font-size: 18px;
        }}

        .theme-btn:hover {{
            transform: scale(1.1);
        }}

        .theme-btn.active {{
            border-color: white;
            box-shadow: 0 0 0 2px var(--primary);
        }}

        .theme-btn.purple {{
            background: linear-gradient(135deg, #667eea, #764ba2);
            color: white;
        }}
        .theme-btn.green {{
            background: #2e7d32;
            color: white;
        }}
        .theme-btn.dark {{
            background: #121212;
            color: #bb86fc;
        }}
        .theme-btn.sepia {{
            background: #8b4513;
            color: #fdf5e6;
        }}

        /* ========== 导航标签 ========== */
        .nav-tabs {{
            display: flex;
            gap: 8px;
            margin-bottom: 16px;
            padding: 8px;
            background: rgba(0,0,0,0.02);
            border-radius: 30px;
            flex-wrap: wrap;
        }}

        .nav-tab {{
            flex: 1;
            min-width: 80px;
            padding: 12px 16px;
            border: none;
            border-radius: 25px;
            font-size: 15px;
            font-weight: 600;
            display: flex;
            align-items: center;
            justify-content: center;
            gap: 6px;
            background: transparent;
            cursor: pointer;
            transition: all 0.3s;
        }}

        .nav-tab.active {{
            color: white;
            box-shadow: 0 10px 20px -5px rgba(0,0,0,0.2);
        }}

        /* ========== 章节选择器 ========== */
        .chapter-selector {{
            display: flex;
            align-items: center;
            gap: 12px;
            margin-bottom: 16px;
            padding: 16px;
            border-radius: 30px;
            box-shadow: 0 4px 12px rgba(0,0,0,0.05);
            border: 1px solid;
            transition: all 0.3s;
        }}

        .chapter-select {{
            flex: 1;
            padding: 14px 20px;
            border: 2px solid #e0e0e0;
            border-radius: 30px;
            font-size: 16px;
            background: white;
            font-weight: 500;
            outline: none;
            cursor: pointer;
        }}

        .chapter-select:focus {{
            border-color: var(--primary);
        }}

        body.theme-dark .chapter-select,
        body.theme-dark .nav-btn {{
            background: #2d2d2d;
            color: #e0e0e0;
            border-color: #404040;
        }}

        body.theme-sepia .chapter-select,
        body.theme-sepia .nav-btn {{
            background: #fff8e7;
            color: #5d3a1a;
            border-color: #e6d3b5;
        }}

        .nav-btn {{
            width: 48px;
            height: 48px;
            border: none;
            border-radius: 50%;
            background: white;
            font-size: 20px;
            display: flex;
            align-items: center;
            justify-content: center;
            cursor: pointer;
            border: 2px solid;
            transition: all 0.2s;
        }}

        .nav-btn:active {{
            transform: scale(0.95);
        }}

        /* ========== 自动播放控制栏 ========== */
        .auto-play-bar {{
            display: flex;
            align-items: center;
            gap: 16px;
            margin-bottom: 16px;
            padding: 16px 20px;
            border-radius: 40px;
            color: white;
            flex-shrink: 0;
        }}

        .play-control-btn {{
            width: 56px;
            height: 56px;
            border: none;
            border-radius: 50%;
            background: white;
            font-size: 24px;
            display: flex;
            align-items: center;
            justify-content: center;
            cursor: pointer;
            box-shadow: 0 8px 16px rgba(0,0,0,0.2);
            transition: all 0.2s;
        }}

        .play-control-btn:active {{
            transform: scale(0.9);
        }}

        .play-control-btn.playing {{
            background: var(--warning);
            color: white;
        }}

        .progress-info {{
            flex: 1;
        }}

        .progress-bar {{
            height: 8px;
            background: rgba(255,255,255,0.3);
            border-radius: 4px;
            margin-top: 8px;
            overflow: hidden;
        }}

        .progress-fill {{
            height: 100%;
            background: white;
            border-radius: 4px;
            transition: width 0.3s;
        }}

        /* ========== 滚动容器 ========== */
        .scroll-container {{
            flex: 1;
            border-radius: 24px;
            padding: 20px;
            overflow-y: auto;
            position: relative;
            min-height: 0;
            height: 0;
            box-shadow: inset 0 4px 12px rgba(0,0,0,0.02);
            scroll-behavior: smooth;
            transition: all 0.3s;
        }}

        /* 自定义滚动条 */
        .scroll-container::-webkit-scrollbar {{
            width: 8px;
        }}

        .scroll-container::-webkit-scrollbar-track {{
            background: rgba(0,0,0,0.05);
            border-radius: 4px;
        }}

        .scroll-container::-webkit-scrollbar-thumb {{
            background: var(--primary);
            border-radius: 4px;
        }}

        .scroll-container::-webkit-scrollbar-thumb:hover {{
            background: var(--primary-dark);
        }}

        .chapter-title {{
            font-size: 20px;
            font-weight: bold;
            margin-bottom: 20px;
            padding-bottom: 12px;
            border-bottom: 2px solid;
            position: sticky;
            top: 0;
            z-index: 10;
            transition: all 0.3s;
        }}

        body.theme-purple .chapter-title {{
            background: white;
        }}
        body.theme-green .chapter-title {{
            background: #f9fff9;
        }}
        body.theme-dark .chapter-title {{
            background: #2d2d2d;
        }}
        body.theme-sepia .chapter-title {{
            background: #fff8e7;
        }}

        .lines-container {{
            display: flex;
            flex-direction: column;
            gap: 8px;
            padding: 4px 0 20px 0;
        }}

        /* ========== 可学习的行 ========== */
        .learning-line {{
            display: flex;
            align-items: flex-start;
            padding: 16px;
            border-radius: 16px;
            border: 1px solid transparent;
            transition: all 0.3s ease;
            cursor: pointer;
            box-shadow: 0 2px 4px rgba(0,0,0,0.02);
        }}

        .learning-line:hover {{
            background: rgba(108,92,231,0.05);
            border-color: rgba(108,92,231,0.2);
            transform: translateX(4px);
        }}

        body.theme-green .learning-line:hover {{
            background: rgba(46,125,50,0.05);
            border-color: rgba(46,125,50,0.2);
        }}

        body.theme-dark .learning-line:hover {{
            background: rgba(187,134,252,0.1);
            border-color: rgba(187,134,252,0.3);
        }}

        body.theme-sepia .learning-line:hover {{
            background: rgba(139,69,19,0.05);
            border-color: rgba(139,69,19,0.2);
        }}

        .learning-line.active {{
            background: rgba(108,92,231,0.12);
            border: 2px solid #6C5CE7;
            box-shadow: 0 8px 20px rgba(108,92,231,0.2);
            transform: scale(1.02);
            margin: 8px 0;
        }}

        body.theme-green .learning-line.active {{
            background: rgba(46,125,50,0.12);
            border-color: #2e7d32;
        }}

        body.theme-dark .learning-line.active {{
            background: rgba(187,134,252,0.2);
            border-color: #bb86fc;
        }}

        body.theme-sepia .learning-line.active {{
            background: rgba(139,69,19,0.12);
            border-color: #8b4513;
        }}

        .line-number {{
            width: 36px;
            height: 36px;
            display: flex;
            align-items: center;
            justify-content: center;
            background: var(--light);
            border-radius: 50%;
            font-size: 14px;
            font-weight: 600;
            margin-right: 12px;
            flex-shrink: 0;
            transition: all 0.3s;
        }}

        body.theme-purple .line-number {{
            background: #F5F6FA;
            color: #2D3436;
        }}
        body.theme-green .line-number {{
            background: #c8e6c9;
            color: #1e3a1e;
        }}
        body.theme-dark .line-number {{
            background: #404040;
            color: #e0e0e0;
        }}
        body.theme-sepia .line-number {{
            background: #e6d3b5;
            color: #5d3a1a;
        }}

        .learning-line.active .line-number {{
            background: #6C5CE7;
            color: white;
        }}

        body.theme-green .learning-line.active .line-number {{
            background: #2e7d32;
        }}

        body.theme-dark .learning-line.active .line-number {{
            background: #bb86fc;
            color: #121212;
        }}

        body.theme-sepia .learning-line.active .line-number {{
            background: #8b4513;
            color: #fdf5e6;
        }}

        /* ========== 只读的行 ========== */
        .readonly-line {{
            display: flex;
            align-items: flex-start;
            padding: 12px 16px;
            border-radius: 12px;
            font-size: 15px;
            line-height: 1.5;
            border-left: 4px solid;
            margin-left: 8px;
            cursor: default;
            user-select: none;
        }}

        body.theme-purple .readonly-line {{
            background: #fafafa;
            color: #666;
            border-left-color: #B2BEC3;
        }}
        body.theme-green .readonly-line {{
            background: #f1f8e9;
            color: #4e6b4e;
            border-left-color: #9ccc9c;
        }}
        body.theme-dark .readonly-line {{
            background: #252525;
            color: #999;
            border-left-color: #666;
        }}
        body.theme-sepia .readonly-line {{
            background: #f5ebd5;
            color: #7d5e3a;
            border-left-color: #c3a78a;
        }}

        .readonly-indicator {{
            width: 24px;
            height: 24px;
            display: flex;
            align-items: center;
            justify-content: center;
            color: #999;
            font-size: 14px;
            margin-right: 12px;
            flex-shrink: 0;
        }}

        .text-content {{
            flex: 1;
            font-size: 17px;
            line-height: 1.6;
            word-break: break-word;
            padding: 6px 0;
        }}

        /* 语言标签 */
        .lang-tag {{
            font-size: 12px;
            padding: 4px 8px;
            border-radius: 12px;
            margin-left: 8px;
            white-space: nowrap;
            font-weight: 600;
        }}

        body.theme-purple .lang-tag.ch {{ background: rgba(0,184,148,0.1); color: #00B894; }}
        body.theme-purple .lang-tag.jp {{ background: rgba(253,121,168,0.1); color: #FD79A8; }}
        body.theme-purple .lang-tag.en {{ background: rgba(108,92,231,0.1); color: #6C5CE7; }}

        body.theme-green .lang-tag.ch {{ background: #c8e6c9; color: #2e7d32; }}
        body.theme-green .lang-tag.jp {{ background: #ffcdd2; color: #c62828; }}
        body.theme-green .lang-tag.en {{ background: #bbdefb; color: #1565c0; }}

        body.theme-dark .lang-tag.ch {{ background: #1e3a1e; color: #a5d6a5; }}
        body.theme-dark .lang-tag.jp {{ background: #4a1a1a; color: #ef9a9a; }}
        body.theme-dark .lang-tag.en {{ background: #1a2e4a; color: #90caf9; }}

        body.theme-sepia .lang-tag.ch {{ background: #d4b48c; color: #5d3a1a; }}
        body.theme-sepia .lang-tag.jp {{ background: #e6b8af; color: #7a4a3a; }}
        body.theme-sepia .lang-tag.en {{ background: #b8c9b0; color: #4a6a3a; }}

        /* 音频指示器 */
        .audio-indicator {{
            width: 24px;
            height: 24px;
            border-radius: 50%;
            background: #00B894;
            color: white;
            display: inline-flex;
            align-items: center;
            justify-content: center;
            font-size: 12px;
            margin-left: 8px;
            flex-shrink: 0;
            animation: pulse 2s infinite;
        }}

        body.theme-green .audio-indicator {{
            background: #2e7d32;
        }}
        body.theme-dark .audio-indicator {{
            background: #bb86fc;
            color: #121212;
        }}
        body.theme-sepia .audio-indicator {{
            background: #8b4513;
        }}

        @keyframes pulse {{
            0% {{ box-shadow: 0 0 0 0 rgba(0,184,148,0.4); }}
            70% {{ box-shadow: 0 0 0 6px rgba(0,184,148,0); }}
            100% {{ box-shadow: 0 0 0 0 rgba(0,184,148,0); }}
        }}

        /* 操作按钮组 */
        .action-buttons {{
            display: flex;
            gap: 8px;
            margin-left: 12px;
            flex-shrink: 0;
        }}

        .action-btn {{
            width: 40px;
            height: 40px;
            border: none;
            border-radius: 50%;
            font-size: 18px;
            display: flex;
            align-items: center;
            justify-content: center;
            cursor: pointer;
            transition: all 0.2s;
        }}

        .action-btn:hover {{
            transform: scale(1.1);
            box-shadow: 0 4px 8px rgba(0,0,0,0.1);
        }}

        .action-btn:active {{
            transform: scale(0.95);
        }}

        body.theme-purple .action-btn {{
            background: #F5F6FA;
            color: #2D3436;
        }}
        body.theme-green .action-btn {{
            background: #c8e6c9;
            color: #1e3a1e;
        }}
        body.theme-dark .action-btn {{
            background: #404040;
            color: #e0e0e0;
        }}
        body.theme-sepia .action-btn {{
            background: #e6d3b5;
            color: #5d3a1a;
        }}

        .action-btn.added {{
            background: #00B894;
            color: white;
        }}

        body.theme-green .action-btn.added {{
            background: #2e7d32;
        }}
        body.theme-dark .action-btn.added {{
            background: #bb86fc;
            color: #121212;
        }}
        body.theme-sepia .action-btn.added {{
            background: #8b4513;
            color: #fdf5e6;
        }}

        .action-btn.grammar.added {{
            background: #FD79A8;
            color: white;
        }}

        body.theme-green .action-btn.grammar.added {{
            background: #c62828;
        }}
        body.theme-dark .action-btn.grammar.added {{
            background: #cf6679;
            color: #121212;
        }}
        body.theme-sepia .action-btn.grammar.added {{
            background: #7a4a3a;
            color: #fdf5e6;
        }}

        /* ========== 面板样式 ========== */
        .panel {{
            flex: 1;
            overflow-y: auto;
            padding: 20px;
            border-radius: 24px;
            height: 0;
            transition: all 0.3s;
        }}

        .panel h2 {{
            font-size: 20px;
            margin-bottom: 20px;
            padding-bottom: 12px;
            border-bottom: 2px solid;
            position: sticky;
            top: 0;
            z-index: 10;
        }}

        body.theme-purple .panel h2 {{
            background: white;
        }}
        body.theme-green .panel h2 {{
            background: #f9fff9;
        }}
        body.theme-dark .panel h2 {{
            background: #2d2d2d;
        }}
        body.theme-sepia .panel h2 {{
            background: #fff8e7;
        }}

        .word-card {{
            padding: 16px;
            border-radius: 16px;
            margin-bottom: 12px;
            border-left: 6px solid;
            transition: all 0.2s;
        }}

        .word-card:hover {{
            transform: translateX(4px);
            box-shadow: 0 4px 12px rgba(0,0,0,0.1);
        }}

        body.theme-purple .word-card {{
            background: #F5F6FA;
            border-left-color: #6C5CE7;
        }}
        body.theme-green .word-card {{
            background: #c8e6c9;
            border-left-color: #2e7d32;
        }}
        body.theme-dark .word-card {{
            background: #404040;
            border-left-color: #bb86fc;
        }}
        body.theme-sepia .word-card {{
            background: #e6d3b5;
            border-left-color: #8b4513;
        }}

        .word-card.grammar-card {{
            border-left-color: #FD79A8;
        }}

        body.theme-green .word-card.grammar-card {{
            border-left-color: #c62828;
        }}
        body.theme-dark .word-card.grammar-card {{
            border-left-color: #cf6679;
        }}
        body.theme-sepia .word-card.grammar-card {{
            border-left-color: #7a4a3a;
        }}

        .word-text {{
            font-size: 17px;
            font-weight: 600;
            margin-bottom: 8px;
        }}

        .word-meta {{
            display: flex;
            gap: 16px;
            font-size: 12px;
            flex-wrap: wrap;
            margin-bottom: 8px;
        }}

        .grammar-note {{
            color: white;
            padding: 8px 12px;
            border-radius: 8px;
            margin-top: 8px;
            font-size: 14px;
        }}

        body.theme-purple .grammar-note {{
            background: #FD79A8;
        }}
        body.theme-green .grammar-note {{
            background: #c62828;
        }}
        body.theme-dark .grammar-note {{
            background: #cf6679;
            color: #121212;
        }}
        body.theme-sepia .grammar-note {{
            background: #7a4a3a;
        }}

        .empty-message {{
            text-align: center;
            padding: 40px;
            color: #666;
            font-size: 16px;
        }}

        /* ========== 状态栏 ========== */
        .status-bar {{
            flex-shrink: 0;
            margin-top: 16px;
            padding: 16px 24px;
            border-radius: 30px;
            box-shadow: 0 -4px 20px rgba(0,0,0,0.05);
            display: flex;
            justify-content: space-between;
            align-items: center;
            font-size: 14px;
            border: 1px solid;
            transition: all 0.3s;
        }}

        .badge {{
            padding: 4px 12px;
            border-radius: 20px;
            font-size: 12px;
            color: white;
        }}

        .hidden {{
            display: none !important;
        }}

        @media (max-width: 600px) {{
            .app {{ padding: 16px; height: calc(100vh - 32px); }}
            .text-content {{ font-size: 16px; }}
            .action-btn {{ width: 36px; height: 36px; font-size: 16px; }}
            .theme-switcher {{ justify-content: center; }}
        }}
    </style>
</head>
<body class="theme-purple">
    <div class="app">
        <div class="app-header">
            <h1>
                <span>🧠 艾宾浩斯·自动学习</span>
                <span class="badge" style="background: #6C5CE7;">🔑 {hashlib.md5(str(total_chapters).encode()).hexdigest()[:8]}</span>
            </h1>

            <!-- ========== 四色主题切换器 ========== -->
            <div class="theme-switcher">
                <button class="theme-btn purple active" onclick="setTheme('purple')" title="经典紫">🟣</button>
                <button class="theme-btn green" onclick="setTheme('green')" title="护眼绿">🟢</button>
                <button class="theme-btn dark" onclick="setTheme('dark')" title="深夜黑">⚫</button>
                <button class="theme-btn sepia" onclick="setTheme('sepia')" title="纸质米黄">🟤</button>
            </div>

            <div class="nav-tabs">
                <button class="nav-tab active" onclick="switchTab('learn')">📚 自动学习</button>
                <button class="nav-tab" onclick="switchTab('notebook')">
                    📔 单词本
                    <span id="notebookBadge" style="background: #00B894; color: white; padding: 2px 8px; border-radius: 12px; font-size: 12px;">0</span>
                </button>
                <button class="nav-tab" onclick="switchTab('grammar')">
                    🔤 语法本
                    <span id="grammarBadge" style="background: #FD79A8; color: white; padding: 2px 8px; border-radius: 12px; font-size: 12px;">0</span>
                </button>
                <button class="nav-tab" onclick="switchTab('review')">
                    🧠 复习
                    <span id="reviewBadge" style="background: #FF7675; color: white; padding: 2px 8px; border-radius: 12px; font-size: 12px;">0</span>
                </button>
            </div>
        </div>

        <!-- 学习面板 -->
        <div id="learnPanel" style="display: flex; flex-direction: column; flex: 1; min-height: 0;">
            <div class="chapter-selector">
                <button class="nav-btn" onclick="prevChapter()">◀</button>
                <select id="chapterSelect" class="chapter-select" onchange="loadChapter(parseInt(this.value))">
                    <option value="">📚 请选择章节</option>
                </select>
                <button class="nav-btn" onclick="nextChapter()">▶</button>
            </div>

            <div class="auto-play-bar">
                <button id="playPauseBtn" class="play-control-btn" onclick="togglePlayPause()">▶</button>
                <div class="progress-info">
                    <div style="display: flex; justify-content: space-between; margin-bottom: 4px;">
                        <span id="currentLineInfo">第 0/0 句</span>
                        <span id="chapterInfo">第 0/{total_chapters} 章</span>
                    </div>
                    <div class="progress-bar">
                        <div id="progressFill" class="progress-fill" style="width: 0%;"></div>
                    </div>
                </div>
            </div>

            <!-- 滚动容器 -->
            <div id="scrollContainer" class="scroll-container">
                <div id="chapterTitle" class="chapter-title">请选择章节</div>
                <div id="linesContainer" class="lines-container"></div>
            </div>
        </div>

        <!-- 单词本面板 -->
        <div id="notebookPanel" class="panel hidden">
            <h2>📔 我的单词本</h2>
            <div id="wordList"></div>
        </div>

        <!-- 语法本面板 -->
        <div id="grammarPanel" class="panel hidden">
            <h2>🔤 我的语法本</h2>
            <div id="grammarList"></div>
        </div>

        <!-- 复习面板 -->
        <div id="reviewPanel" class="panel hidden">
            <h2>🧠 今日复习</h2>
            <div id="reviewList"></div>
        </div>

        <div class="status-bar">
            <span>📚 {total_chapters}章 · {total_lines}句</span>
            <span>🎵 {total_audio}音频</span>
            <span>📝 可学习 {total_learnable}句</span>
            <span id="currentTime">{datetime.now().strftime('%H:%M')}</span>
        </div>
    </div>

    <script>
        // ========== 数据 ==========
        const CHAPTERS_DATA = {chapters_json_str};
        const LEARNING_ITEMS = {items_json_str};
        const TOTAL_CHAPTERS = {total_chapters};

        console.log('=================================');
        console.log('✅ 艾宾浩斯学习系统启动');
        console.log('📊 总章节数:', TOTAL_CHAPTERS);
        console.log('📊 处理后章节数:', CHAPTERS_DATA ? CHAPTERS_DATA.length : 0);
        console.log('=================================');

        // ========== 全局变量 ==========
        let currentChapter = 0;
        let currentLineIndex = 0;
        let isPlaying = false;
        let playTimer = null;
        let currentAudio = null;

        // ========== 主题切换函数 ==========
        function setTheme(theme) {{
            // 移除所有主题类
            document.body.classList.remove('theme-purple', 'theme-green', 'theme-dark', 'theme-sepia');
            // 添加选中的主题
            document.body.classList.add(`theme-${{theme}}`);

            // 更新主题按钮状态
            document.querySelectorAll('.theme-btn').forEach(btn => {{
                btn.classList.remove('active');
            }});
            event.currentTarget.classList.add('active');

            // 保存主题到本地存储
            localStorage.setItem('ebbinghaus_theme', theme);
            console.log('🎨 主题切换到:', theme);
        }}

        // ========== 加载保存的主题 ==========
        function loadTheme() {{
            const savedTheme = localStorage.getItem('ebbinghaus_theme');
            if (savedTheme) {{
                document.body.classList.remove('theme-purple', 'theme-green', 'theme-dark', 'theme-sepia');
                document.body.classList.add(`theme-${{savedTheme}}`);

                // 更新主题按钮状态
                document.querySelectorAll('.theme-btn').forEach(btn => {{
                    btn.classList.remove('active');
                    if (btn.classList.contains(savedTheme)) {{
                        btn.classList.add('active');
                    }}
                }});
            }}
        }}

        // ========== 页面初始化 ==========
        window.addEventListener('load', function() {{
            console.log('页面加载完成，开始初始化...');

            if (!CHAPTERS_DATA || !Array.isArray(CHAPTERS_DATA) || CHAPTERS_DATA.length === 0) {{
                console.error('❌ 错误：章节数据为空！');
                return;
            }}

            initChapterSelect();
            loadFromStorage();
            loadTheme(); // 加载保存的主题

            if (CHAPTERS_DATA.length > 0) {{
                loadChapter(0);
            }}

            updateAllPanels();
        }});

        // ========== 初始化章节选择器 ==========
        function initChapterSelect() {{
            const select = document.getElementById('chapterSelect');
            if (!select) return;

            select.innerHTML = '<option value="">📚 请选择章节</option>';

            for (let i = 1; i <= CHAPTERS_DATA.length; i++) {{
                const option = document.createElement('option');
                option.value = i - 1;
                const chapter = CHAPTERS_DATA[i-1];
                const learnableCount = chapter ? chapter.filter(line => line.is_learnable).length : 0;
                const readonlyCount = chapter ? chapter.filter(line => !line.is_learnable).length : 0;
                const audioCount = chapter ? chapter.filter(line => line.audio && line.audio.length > 100).length : 0;
                option.textContent = '第 ' + i + ' 章 (可学习' + learnableCount + '句, 只读' + readonlyCount + '句, 音频' + audioCount + '个)';
                select.appendChild(option);
            }}

            console.log('✅ 章节选择器初始化完成，共', select.options.length - 1, '个章节');
        }}

        // ========== 本地存储 ==========
        function loadFromStorage() {{
            const saved = localStorage.getItem('ebbinghaus_notebook');
            if (saved) {{
                try {{
                    const notebook = JSON.parse(saved);
                    if (LEARNING_ITEMS) {{
                        LEARNING_ITEMS.forEach(item => {{
                            const savedItem = notebook.find(n => n.id === item.id);
                            if (savedItem) {{
                                item.added_to_notebook = savedItem.added_to_notebook || false;
                                item.added_to_grammar = savedItem.added_to_grammar || false;
                                item.grammar_note = savedItem.grammar_note || '';
                                item.mastery = savedItem.mastery || 0;
                                item.review_count = savedItem.review_count || 0;
                                item.last_review = savedItem.last_review;
                                item.next_review = savedItem.next_review || 0;
                            }}
                        }});
                    }}
                    console.log('✅ 本地存储加载成功');
                }} catch (e) {{
                    console.error('❌ 本地存储加载失败:', e);
                }}
            }}
        }}

        function saveToStorage() {{
            if (!LEARNING_ITEMS) return;
            const notebook = LEARNING_ITEMS.map(item => ({{
                id: item.id,
                added_to_notebook: item.added_to_notebook || false,
                added_to_grammar: item.added_to_grammar || false,
                grammar_note: item.grammar_note || '',
                mastery: item.mastery || 0,
                review_count: item.review_count || 0,
                last_review: item.last_review,
                next_review: item.next_review || 0
            }})).filter(item => item.added_to_notebook || item.added_to_grammar);
            localStorage.setItem('ebbinghaus_notebook', JSON.stringify(notebook));
        }}

        // ========== 章节加载 ==========
        function loadChapter(index) {{
            if (index < 0 || index >= CHAPTERS_DATA.length) return;

            currentChapter = index;
            currentLineIndex = 0;

            const select = document.getElementById('chapterSelect');
            if (select) select.value = index;

            const titleEl = document.getElementById('chapterTitle');
            if (titleEl) titleEl.innerHTML = '第 ' + (index + 1) + ' 章';

            const infoEl = document.getElementById('chapterInfo');
            if (infoEl) infoEl.innerHTML = '第 ' + (index + 1) + '/' + CHAPTERS_DATA.length + ' 章';

            renderChapter();
            stopPlay();
        }}

        // ========== 渲染章节内容 ==========
        function renderChapter() {{
            const chapterData = CHAPTERS_DATA[currentChapter];
            if (!chapterData || !Array.isArray(chapterData)) return;

            const container = document.getElementById('linesContainer');
            let html = '';

            chapterData.forEach((line, idx) => {{
                if (!line) return;

                if (line.is_learnable) {{
                    // ===== 可学习的行 =====
                    const hasAudio = line.audio && line.audio.length > 100;
                    const learningItem = LEARNING_ITEMS?.find(item => item && item.id === line.id);
                    const isNotebook = learningItem ? learningItem.added_to_notebook : false;
                    const isGrammar = learningItem ? learningItem.added_to_grammar : false;

                    html += '<div id="line-' + line.id + '" class="learning-line" onclick="playThisLine(\\'' + line.id + '\\', ' + idx + ')">';
                    html += '<span class="line-number">' + (idx + 1) + '</span>';
                    html += '<div class="text-content">' + (line.content || '').replace(/\\[(ch|jp|en)\\]\\s*/g, '').replace(/\\[sound:[^\\]]*\\]/g, '') + '</div>';

                    // 显示语言标签
                    let langClass = '';
                    let langText = '';
                    if (line.lang === 'ch') {{ langClass = 'ch'; langText = '中文'; }}
                    else if (line.lang === 'jp') {{ langClass = 'jp'; langText = '日语'; }}
                    else if (line.lang === 'en') {{ langClass = 'en'; langText = '英语'; }}
                    html += '<span class="lang-tag ' + langClass + '">' + langText + '</span>';

                    if (hasAudio) {{
                        html += '<span class="audio-indicator">🔊</span>';
                    }}

                    // 显示操作按钮
                    html += '<div class="action-buttons">';
                    html += '<button class="action-btn' + (isNotebook ? ' added' : '') + '" onclick="event.stopPropagation(); toggleNotebook(\\'' + line.id + '\\', this)">' + (isNotebook ? '✓' : '📚') + '</button>';
                    html += '<button class="action-btn grammar' + (isGrammar ? ' added' : '') + '" onclick="event.stopPropagation(); addGrammar(\\'' + line.id + '\\', this)">🔤</button>';
                    html += '</div>';
                    html += '</div>';

                }} else {{
                    // ===== 只读的行 =====
                    html += '<div class="readonly-line">';
                    html += '<span class="readonly-indicator">📄</span>';
                    html += '<div class="text-content">' + (line.content || '').replace(/\\[(ch|jp|en)\\]\\s*/g, '').replace(/\\[sound:[^\\]]*\\]/g, '') + '</div>';
                    html += '</div>';
                }}
            }});

            container.innerHTML = html;
            updateProgress();

            // 滚动到顶部
            const scrollContainer = document.getElementById('scrollContainer');
            if (scrollContainer) {{
                scrollContainer.scrollTop = 0;
            }}
        }}

        // ========== 居中滚动函数 ==========
        function scrollToCenter(element) {{
            const container = document.getElementById('scrollContainer');
            if (!container || !element) return;

            const containerHeight = container.clientHeight;
            const elementTop = element.offsetTop;
            const elementHeight = element.clientHeight;

            // 计算滚动位置，使元素居中
            const scrollTop = elementTop - (containerHeight / 2) + (elementHeight / 2);

            container.scrollTo({{
                top: Math.max(0, scrollTop),
                behavior: 'smooth'
            }});
        }}

        // ========== 播放指定行 ==========
        function playThisLine(lineId, index) {{
            console.log('播放行:', lineId, index);

            const chapterData = CHAPTERS_DATA[currentChapter];
            if (!chapterData || index >= chapterData.length) return;

            const line = chapterData[index];

            // 只播放可学习的行
            if (!line.is_learnable) return;

            // 更新当前行索引
            currentLineIndex = index;

            // 移除所有active类
            document.querySelectorAll('.learning-line').forEach(el => el.classList.remove('active'));

            // 为当前行添加active类
            const currentLine = document.getElementById('line-' + lineId);
            if (currentLine) {{
                currentLine.classList.add('active');
                // 滚动到居中位置
                scrollToCenter(currentLine);
            }}

            // 播放音频
            if (line && line.audio && line.audio.length > 100) {{
                playAudio(line.audio);
            }}

            updateProgress();
        }}

        // ========== 音频播放函数 ==========
        function playAudio(audioData) {{
            return new Promise((resolve, reject) => {{
                try {{
                    if (!audioData || audioData.length < 100) {{
                        console.log('无效的音频数据');
                        reject('无效的音频数据');
                        return;
                    }}

                    let audioSrc = audioData;
                    if (!audioSrc.startsWith('data:audio')) {{
                        audioSrc = 'data:audio/mp3;base64,' + audioSrc;
                    }}

                    if (currentAudio) {{
                        currentAudio.pause();
                        currentAudio.currentTime = 0;
                    }}

                    currentAudio = new Audio(audioSrc);

                    currentAudio.onended = function() {{
                        console.log('音频播放完成');
                        currentAudio = null;
                        resolve();
                    }};

                    currentAudio.onerror = function(e) {{
                        console.error('音频播放错误:', e);
                        currentAudio = null;
                        reject(e);
                    }};

                    currentAudio.play().catch(e => {{
                        console.error('播放失败:', e);
                        currentAudio = null;
                        reject(e);
                    }});

                }} catch (e) {{
                    console.error('音频创建失败:', e);
                    reject(e);
                }}
            }});
        }}

        // ========== 自动播放控制 ==========
        function togglePlayPause() {{
            const btn = document.getElementById('playPauseBtn');
            if (isPlaying) {{
                stopPlay();
                btn.innerHTML = '▶';
                btn.classList.remove('playing');
            }} else {{
                startPlay();
                btn.innerHTML = '⏸';
                btn.classList.add('playing');
            }}
        }}

        function startPlay() {{
            if (isPlaying) return;
            if (!CHAPTERS_DATA[currentChapter]) return;
            isPlaying = true;
            playCurrentLine();
        }}

        function stopPlay() {{
            isPlaying = false;
            if (playTimer) {{
                clearTimeout(playTimer);
                playTimer = null;
            }}
            if (currentAudio) {{
                currentAudio.pause();
                currentAudio.currentTime = 0;
                currentAudio = null;
            }}
            const btn = document.getElementById('playPauseBtn');
            if (btn) {{
                btn.innerHTML = '▶';
                btn.classList.remove('playing');
            }}
        }}

        async function playCurrentLine() {{
            if (!isPlaying) return;

            const chapterData = CHAPTERS_DATA[currentChapter];
            if (!chapterData) return;

            // 跳过只读行，找到下一个可学习行
            while (currentLineIndex < chapterData.length && !chapterData[currentLineIndex].is_learnable) {{
                currentLineIndex++;
                updateProgress();
            }}

            if (currentLineIndex >= chapterData.length) {{
                if (currentChapter + 1 < CHAPTERS_DATA.length) {{
                    loadChapter(currentChapter + 1);
                    currentLineIndex = 0;
                    setTimeout(() => playCurrentLine(), 800);
                }} else {{
                    stopPlay();
                }}
                return;
            }}

            const line = chapterData[currentLineIndex];
            const lineId = line.id;

            // 高亮当前行并居中
            document.querySelectorAll('.learning-line').forEach(el => el.classList.remove('active'));
            const currentLine = document.getElementById('line-' + lineId);
            if (currentLine) {{
                currentLine.classList.add('active');
                scrollToCenter(currentLine);
            }}

            // 播放音频
            if (line && line.audio && line.audio.length > 100) {{
                try {{
                    await playAudio(line.audio);
                    currentLineIndex++;
                    updateProgress();
                    if (isPlaying) {{
                        playTimer = setTimeout(() => playCurrentLine(), 300);
                    }}
                }} catch (e) {{
                    console.log('音频播放失败，跳过:', e);
                    currentLineIndex++;
                    updateProgress();
                    if (isPlaying) {{
                        playTimer = setTimeout(() => playCurrentLine(), 300);
                    }}
                }}
            }} else {{
                currentLineIndex++;
                updateProgress();
                if (isPlaying) {{
                    playTimer = setTimeout(() => playCurrentLine(), 300);
                }}
            }}
        }}

        function updateProgress() {{
            const chapterData = CHAPTERS_DATA[currentChapter];
            if (!chapterData) return;

            const learnableCount = chapterData.filter(line => line.is_learnable).length;
            const currentLearnableIndex = chapterData.slice(0, currentLineIndex).filter(line => line.is_learnable).length;

            const progress = learnableCount > 0 ? (currentLearnableIndex / learnableCount) * 100 : 0;
            const fill = document.getElementById('progressFill');
            if (fill) fill.style.width = progress + '%';

            const lineInfo = document.getElementById('currentLineInfo');
            if (lineInfo) {{
                lineInfo.innerHTML = '第 ' + (currentLearnableIndex + 1) + '/' + learnableCount + ' 句';
            }}
        }}

        // ========== 单词本操作 ==========
        function toggleNotebook(itemId, btn) {{
            const item = LEARNING_ITEMS?.find(i => i.id === itemId);
            if (item) {{
                item.added_to_notebook = !item.added_to_notebook;
                btn.innerHTML = item.added_to_notebook ? '✓' : '📚';
                btn.classList.toggle('added');
                saveToStorage();
                updateNotebookPanel();
                updateBadges();
            }}
        }}

        function addGrammar(itemId, btn) {{
            const item = LEARNING_ITEMS?.find(i => i.id === itemId);
            if (item) {{
                const note = prompt('添加语法笔记:', item.grammar_note || '');
                if (note !== null) {{
                    item.added_to_grammar = true;
                    item.grammar_note = note;
                    btn.classList.add('added');
                    saveToStorage();
                    updateGrammarPanel();
                    updateBadges();
                }}
            }}
        }}

        // ========== 单词本面板 ==========
        function updateNotebookPanel() {{
            const notebook = LEARNING_ITEMS?.filter(item => item && item.added_to_notebook) || [];
            const list = document.getElementById('wordList');

            let html = '';
            notebook.forEach(item => {{
                html += '<div class="word-card">';
                html += '<div class="word-text">' + (item.content || '').replace(/\\[(ch|jp|en)\\]\\s*/g, '').replace(/\\[sound:[^\\]]*\\]/g, '') + '</div>';
                html += '<div class="word-meta">';
                html += '<span>📌 第' + (item.chapter || 0) + '章</span>';
                html += '<span>🎯 掌握度 ' + (item.mastery || 0) + '%</span>';
                html += '<span>🔄 复习' + (item.review_count || 0) + '次</span>';
                html += '</div>';
                html += '<button class="action-btn" style="background: var(--danger); color: white; margin-top: 8px;" onclick="toggleNotebook(\\'' + item.id + '\\', this)">🗑️ 移除</button>';
                html += '</div>';
            }});

            if (list) {{
                list.innerHTML = html || '<div class="empty-message">📭 单词本还是空的，点击句子旁边的📚收藏</div>';
            }}

            document.getElementById('notebookBadge').innerHTML = notebook.length;
        }}

        // ========== 语法本面板 ==========
        function updateGrammarPanel() {{
            const grammar = LEARNING_ITEMS?.filter(item => item && item.added_to_grammar) || [];
            const list = document.getElementById('grammarList');

            let html = '';
            grammar.forEach(item => {{
                html += '<div class="word-card grammar-card">';
                html += '<div class="word-text">' + (item.content || '').replace(/\\[(ch|jp|en)\\]\\s*/g, '').replace(/\\[sound:[^\\]]*\\]/g, '') + '</div>';
                html += '<div class="grammar-note">📝 ' + (item.grammar_note || '无备注') + '</div>';
                html += '<div class="word-meta" style="margin-top: 8px;">';
                html += '<span>📌 第' + (item.chapter || 0) + '章</span>';
                html += '</div>';
                html += '</div>';
            }});

            if (list) {{
                list.innerHTML = html || '<div class="empty-message">🔤 语法本还是空的，点击句子旁边的🔤添加笔记</div>';
            }}

            document.getElementById('grammarBadge').innerHTML = grammar.length;
        }}

        // ========== 更新徽章 ==========
        function updateBadges() {{
            if (!LEARNING_ITEMS) return;

            const notebook = LEARNING_ITEMS.filter(item => item && item.added_to_notebook).length || 0;
            const grammar = LEARNING_ITEMS.filter(item => item && item.added_to_grammar).length || 0;

            document.getElementById('notebookBadge').innerHTML = notebook;
            document.getElementById('grammarBadge').innerHTML = grammar;
        }}

        // ========== 更新所有面板 ==========
        function updateAllPanels() {{
            if (!LEARNING_ITEMS) return;
            updateNotebookPanel();
            updateGrammarPanel();
            updateBadges();
        }}

        // ========== 章节导航 ==========
        function prevChapter() {{
            if (currentChapter > 0) {{
                loadChapter(currentChapter - 1);
                stopPlay();
            }}
        }}

        function nextChapter() {{
            if (currentChapter + 1 < CHAPTERS_DATA.length) {{
                loadChapter(currentChapter + 1);
                stopPlay();
            }}
        }}

        // ========== 标签页切换 ==========
        function switchTab(tab) {{
            document.querySelectorAll('.nav-tab').forEach(t => t.classList.remove('active'));
            event.currentTarget.classList.add('active');

            const learnPanel = document.getElementById('learnPanel');
            const notebookPanel = document.getElementById('notebookPanel');
            const grammarPanel = document.getElementById('grammarPanel');
            const reviewPanel = document.getElementById('reviewPanel');

            if (learnPanel) learnPanel.classList.toggle('hidden', tab !== 'learn');
            if (notebookPanel) notebookPanel.classList.toggle('hidden', tab !== 'notebook');
            if (grammarPanel) grammarPanel.classList.toggle('hidden', tab !== 'grammar');
            if (reviewPanel) reviewPanel.classList.toggle('hidden', tab !== 'review');

            if (tab === 'notebook') updateNotebookPanel();
            if (tab === 'grammar') updateGrammarPanel();
        }}

        // ========== 暴露全局函数 ==========
        window.switchTab = switchTab;
        window.loadChapter = loadChapter;
        window.prevChapter = prevChapter;
        window.nextChapter = nextChapter;
        window.togglePlayPause = togglePlayPause;
        window.toggleNotebook = toggleNotebook;
        window.addGrammar = addGrammar;
        window.playThisLine = playThisLine;
        window.setTheme = setTheme;
    </script>
</body>
</html>
    '''

    # 写入主HTML文件
    with open(html_path, 'w', encoding='utf-8') as f:
        f.write(html_content)

    file_size = os.path.getsize(html_path) // 1024

    print(f"""
╔══════════════════════════════════════════════════════════════╗
║  ✅ 艾宾浩斯·自动学习系统 生成成功                          ║
╠══════════════════════════════════════════════════════════════╣
║  📁 文件: {os.path.basename(html_path)}
║  📊 总章节: {len(simplified_chapters)} 章
║  📝 可学习语句: {total_learnable} 句
║  📄 只读语句: {total_readonly} 句
║  🎵 有效音频: {total_valid_audio} 个
║  💾 文件大小: {file_size} KB
║
║  📖 章节详情:
{chapter_stats_str}
║
║  🎨 四色主题系统：
║     • 🟣 经典紫 - 活力紫白，默认主题
║     • 🟢 护眼绿 - 柔和绿色，保护视力
║     • ⚫ 深夜黑 - 暗色模式，夜间阅读
║     • 🟤 纸质米黄 - 仿纸质书，复古温馨
║
║  ✨ 主题特性：
║     • 一键切换，实时生效
║     • 自动保存，下次打开保留选择
║     • 所有组件颜色自适应
║     • 平滑过渡动画
║
║  📱 其他功能：
║     • 滚动容器自动居中
║     • 可学习/只读行清晰区分
║     • 单词本/语法本收藏
║     • 艾宾浩斯复习计划
║
╚══════════════════════════════════════════════════════════════╝
    """)

    return html_path


In [ ]:
# ============================================================
# 🧠 艾宾浩斯·自动学习系统 - 手机终极优化版


In [ ]:
# ============================================================

def generate_super_language_learning_system(html_path, chapters):
    """
    生成超级语言学习系统 - 手机终极优化版
    功能：四色主题 + 手机自适应 + 按钮缩小 + 文字最大化
    """
    import json
    from datetime import datetime
    import os
    import hashlib
    import base64
    import re
    from bs4 import BeautifulSoup

    print(f"  📊 正在为 {len(chapters)} 章内容生成艾宾浩斯学习系统...")

    if len(chapters) == 0:
        print("  ❌ 错误：章节数据为空，无法生成学习系统")
        return

    # 计算统计数据
    total_chapters = len(chapters)
    total_lines = sum(len(c) for c in chapters)
    total_audio = sum(1 for c in chapters for l in c if l.get('audio') and len(l.get('audio', '')) > 100)

    print(f"  📊 总章节: {total_chapters}, 总行数: {total_lines}, 总音频: {total_audio}")

    # ========== 构建完整的章节数据 ==========
    simplified_chapters = []

    # 定义变量用于统计
    total_learnable = 0
    total_readonly = 0
    total_valid_audio = 0

    for chap_idx, chapter in enumerate(chapters):
        simplified_lines = []
        for line_idx, line in enumerate(chapter):
            # 提取纯文本内容
            text = line.get('text', '')
            audio = line.get('audio', '')

            # 使用BeautifulSoup提取纯文本
            soup = BeautifulSoup(text, "html.parser")
            plain_text = remove_sound_tags(soup.get_text().strip())

            # 判断语言类型 - 只有以 [ch]、[jp]、[en] 开头的行才是可学习行
            lang = 'unknown'
            is_learnable = False

            if plain_text.startswith('[ch]'):
                lang = 'ch'
                is_learnable = True
            elif plain_text.startswith('[jp]'):
                lang = 'jp'
                is_learnable = True
            elif plain_text.startswith('[en]'):
                lang = 'en'
                is_learnable = True

            # 移除所有语言标签（不仅是行首的）
            content = re.sub(r'\[sound:[^\]]*\]', '', re.sub(r'\[(ch|jp|en)\]\s*', '', plain_text)).strip()

            # 如果内容为空，跳过
            if not content:
                continue

            # 确保音频数据完整
            has_valid_audio = False
            if audio and len(audio) > 100:
                if not audio.startswith('data:audio'):
                    audio = 'data:audio/mp3;base64,' + audio
                has_valid_audio = True
                if is_learnable:
                    total_valid_audio += 1

            # 更新统计
            if is_learnable:
                total_learnable += 1
            else:
                total_readonly += 1

            simplified_lines.append({
                'id': f'{chap_idx:04d}_{line_idx:03d}',
                'content': content,
                'audio': audio if has_valid_audio else '',
                'lang': lang,
                'chapter': chap_idx + 1,
                'has_audio': has_valid_audio,
                'is_learnable': is_learnable
            })

        if simplified_lines:
            simplified_chapters.append(simplified_lines)

    print(f"  📝 处理后章节数: {len(simplified_chapters)}")
    print(f"  📝 可学习语句: {total_learnable} 句, 只读语句: {total_readonly} 句")
    print(f"  📝 有效音频: {total_valid_audio} 个")

    # ========== 构建学习项列表 ==========
    learning_items = []
    for chap_idx, chapter in enumerate(simplified_chapters):
        for line in chapter:
            if line.get('is_learnable'):
                learning_items.append({
                    'id': line['id'],
                    'chapter': line['chapter'],
                    'content': line['content'][:100] + '...' if len(line['content']) > 100 else line['content'],
                    'lang': line['lang'],
                    'has_audio': line.get('has_audio', False),
                    'mastery': 0,
                    'review_count': 0,
                    'added_to_notebook': False,
                    'added_to_grammar': False,
                    'grammar_note': '',
                    'last_review': None,
                    'next_review': 0
                })

    # ========== 将数据转换为JSON字符串 ==========
    chapters_json_str = json.dumps(simplified_chapters, ensure_ascii=False)
    items_json_str = json.dumps(learning_items, ensure_ascii=False)

    # ========== 生成章节统计信息字符串 ==========
    chapter_stats = []
    for i, c in enumerate(simplified_chapters[:5]):
        learnable_count = sum(1 for line in c if line.get('is_learnable'))
        readonly_count = sum(1 for line in c if not line.get('is_learnable'))
        audio_count = sum(1 for line in c if line.get('has_audio'))
        chapter_stats.append(f'     第{i+1}章: 可学习{learnable_count}句, 只读{readonly_count}句, 音频{audio_count}个')
    if len(simplified_chapters) > 5:
        chapter_stats.append('      ...')
    chapter_stats_str = '\n'.join(chapter_stats)

    # ========== HTML内容 - 手机终极优化版 ==========
    html_content = f'''<!DOCTYPE html>
<html lang="zh">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0, maximum-scale=1.0, user-scalable=no, viewport-fit=cover">
    <title>🧠 艾宾浩斯·学习</title>
    <meta name="theme-color" content="#6C5CE7">
    <meta name="mobile-web-app-capable" content="yes">
    <meta name="apple-mobile-web-app-capable" content="yes">
    <meta name="apple-mobile-web-app-status-bar-style" content="black-translucent">
    <style>
        /* ========== 全局样式 - 手机终极优化 ========== */
        * {{
            margin: 0;
            padding: 0;
            box-sizing: border-box;
            font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, "PingFang SC", "Microsoft YaHei", sans-serif;
            transition: background-color 0.2s, color 0.2s, border-color 0.2s;
            -webkit-tap-highlight-color: transparent;
        }}

        body {{
            min-height: 100vh;
            padding: 8px;
            margin: 0;
            transition: background 0.2s;
        }}

        .app {{
            max-width: 800px;
            margin: 0 auto;
            border-radius: 20px;
            padding: 12px;
            box-shadow: 0 8px 20px -8px rgba(0,0,0,0.2);
            display: flex;
            flex-direction: column;
            height: calc(100vh - 16px);
            max-height: 1000px;
            transition: background-color 0.2s, box-shadow 0.2s;
        }}

        .app-header {{
            flex-shrink: 0;
        }}

        h1 {{
            font-size: 20px;
            margin-bottom: 8px;
            display: flex;
            justify-content: space-between;
            align-items: center;
        }}

        /* ========== 四色主题 - 修复版 ========== */

        /* 主题1: 经典紫 */
        body.theme-purple {{
            background: linear-gradient(135deg, #5a4b8c 0%, #3f2e5f 100%);
        }}
        body.theme-purple .app {{
            background: #ffffff;
            color: #2c3e50;
        }}
        body.theme-purple .chapter-selector,
        body.theme-purple .status-bar,
        body.theme-purple .scroll-container,
        body.theme-purple .panel {{
            background: #ffffff;
            border-color: #e9ecef;
        }}
        body.theme-purple .chapter-title {{
            color: #5a4b8c;
            border-bottom-color: #5a4b8c;
            background: #ffffff;
        }}
        body.theme-purple .nav-tab.active {{
            background: #5a4b8c;
            color: white;
        }}

        /* 主题2: 护眼绿 */
        body.theme-green {{
            background: #c8e6c9;
        }}
        body.theme-green .app {{
            background: #f1f8e9;
            color: #1e3a1e;
        }}
        body.theme-green .chapter-title {{
            color: #2e7d32;
            border-bottom-color: #2e7d32;
            background: #ffffff;
        }}
        body.theme-green .nav-tab.active {{
            background: #2e7d32;
            color: white;
        }}

        /* 主题3: 深夜黑 */
        body.theme-dark {{
            background: #0a0a0a;
        }}
        body.theme-dark .app {{
            background: #1e1e1e;
            color: #e0e0e0;
            border: 1px solid #333;
        }}
        body.theme-dark .chapter-title {{
            color: #bb86fc;
            border-bottom-color: #bb86fc;
            background: #2a2a2a;
        }}
        body.theme-dark .nav-tab.active {{
            background: #bb86fc;
            color: #121212;
        }}

        /* 主题4: 纸质米黄 */
        body.theme-sepia {{
            background: #b68b6d;
        }}
        body.theme-sepia .app {{
            background: #fdf5e6;
            color: #5d3a1a;
        }}
        body.theme-sepia .chapter-title {{
            color: #8b4513;
            border-bottom-color: #8b4513;
            background: #fff8e7;
        }}
        body.theme-sepia .nav-tab.active {{
            background: #8b4513;
            color: #fdf5e6;
        }}

        /* ========== 主题切换器 - 手机优化 ========== */
        .theme-switcher {{
            display: flex;
            gap: 6px;
            margin-bottom: 8px;
            padding: 4px;
            background: rgba(0,0,0,0.05);
            border-radius: 30px;
            justify-content: flex-end;
            overflow-x: auto;
            -webkit-overflow-scrolling: touch;
        }}

        .theme-btn {{
            width: 36px;
            height: 36px;
            border: 2px solid transparent;
            border-radius: 50%;
            cursor: pointer;
            transition: all 0.2s;
            display: flex;
            align-items: center;
            justify-content: center;
            font-size: 18px;
            flex-shrink: 0;
        }}

        .theme-btn:active {{
            transform: scale(0.9);
        }}

        .theme-btn.active {{
            border-color: white;
            box-shadow: 0 0 0 2px rgba(255,255,255,0.5);
        }}

        .theme-btn.purple {{
            background: linear-gradient(135deg, #5a4b8c, #3f2e5f);
            color: white;
        }}
        .theme-btn.green {{
            background: #2e7d32;
            color: white;
        }}
        .theme-btn.dark {{
            background: #121212;
            color: #bb86fc;
        }}
        .theme-btn.sepia {{
            background: #8b4513;
            color: #fdf5e6;
        }}

        /* ========== 导航标签 - 手机极简 ========== */
        .nav-tabs {{
            display: flex;
            gap: 4px;
            margin-bottom: 8px;
            padding: 4px;
            background: rgba(0,0,0,0.03);
            border-radius: 30px;
            flex-wrap: wrap;
        }}

        .nav-tab {{
            flex: 1;
            min-width: 60px;
            padding: 8px 4px;
            border: none;
            border-radius: 30px;
            font-size: 13px;
            font-weight: 600;
            display: flex;
            align-items: center;
            justify-content: center;
            gap: 3px;
            background: transparent;
            cursor: pointer;
            transition: all 0.2s;
            white-space: nowrap;
        }}

        .nav-tab span {{
            margin-left: 2px;
        }}

        .badge {{
            padding: 2px 6px;
            border-radius: 12px;
            font-size: 10px;
            color: white;
            margin-left: 2px;
        }}

        /* ========== 章节选择器 - 手机紧凑 ========== */
        .chapter-selector {{
            display: flex;
            align-items: center;
            gap: 6px;
            margin-bottom: 8px;
            padding: 8px;
            border-radius: 30px;
            box-shadow: 0 2px 6px rgba(0,0,0,0.05);
            border: 1px solid;
            transition: all 0.2s;
        }}

        .chapter-select {{
            flex: 1;
            padding: 10px 12px;
            border: 1px solid #e0e0e0;
            border-radius: 30px;
            font-size: 14px;
            background: white;
            font-weight: 500;
            outline: none;
            cursor: pointer;
        }}

        .nav-btn {{
            width: 38px;
            height: 38px;
            border: none;
            border-radius: 50%;
            background: white;
            font-size: 16px;
            display: flex;
            align-items: center;
            justify-content: center;
            cursor: pointer;
            border: 1px solid;
            transition: all 0.2s;
            flex-shrink: 0;
        }}

        .nav-btn:active {{
            transform: scale(0.9);
        }}

        /* ========== 自动播放控制栏 - 手机紧凑 ========== */
        .auto-play-bar {{
            display: flex;
            align-items: center;
            gap: 8px;
            margin-bottom: 8px;
            padding: 8px 12px;
            border-radius: 30px;
            color: white;
            flex-shrink: 0;
        }}

        .play-control-btn {{
            width: 42px;
            height: 42px;
            border: none;
            border-radius: 50%;
            background: white;
            font-size: 20px;
            display: flex;
            align-items: center;
            justify-content: center;
            cursor: pointer;
            box-shadow: 0 4px 8px rgba(0,0,0,0.2);
            transition: all 0.2s;
            flex-shrink: 0;
        }}

        .play-control-btn:active {{
            transform: scale(0.9);
        }}

        .play-control-btn.playing {{
            background: #FDCB6E;
            color: white;
        }}

        .progress-info {{
            flex: 1;
            min-width: 0;
        }}

        .progress-info div {{
            font-size: 12px;
        }}

        .progress-bar {{
            height: 4px;
            background: rgba(255,255,255,0.3);
            border-radius: 2px;
            margin-top: 4px;
            overflow: hidden;
        }}

        .progress-fill {{
            height: 100%;
            background: white;
            border-radius: 2px;
            transition: width 0.3s;
        }}

        /* ========== 滚动容器 - 手机优化 ========== */
        .scroll-container {{
            flex: 1;
            border-radius: 16px;
            padding: 12px;
            overflow-y: auto;
            position: relative;
            min-height: 0;
            height: 0;
            box-shadow: inset 0 2px 6px rgba(0,0,0,0.02);
            scroll-behavior: smooth;
            -webkit-overflow-scrolling: touch;
        }}

        .chapter-title {{
            font-size: 16px;
            font-weight: bold;
            margin-bottom: 12px;
            padding-bottom: 8px;
            border-bottom: 2px solid;
            position: sticky;
            top: 0;
            z-index: 10;
            transition: all 0.2s;
        }}

        .lines-container {{
            display: flex;
            flex-direction: column;
            gap: 6px;
            padding: 2px 0 12px 0;
        }}

        /* ========== 🎯 核心修复：可学习的行 - 手机终极优化 ========== */
        .learning-line {{
            display: flex;
            align-items: flex-start;
            padding: 10px 8px;
            border-radius: 12px;
            border: 1px solid transparent;
            transition: all 0.2s ease;
            cursor: pointer;
            box-shadow: 0 1px 2px rgba(0,0,0,0.02);
            width: 100%;
        }}

        .learning-line:active {{
            background: rgba(108,92,231,0.1);
            transform: scale(0.99);
        }}

        .learning-line.active {{
            background: rgba(108,92,231,0.12);
            border: 2px solid #5a4b8c;
            box-shadow: 0 4px 10px rgba(90,75,140,0.15);
            transform: scale(1.01);
            margin: 4px 0;
        }}

        /* 行号 - 更小 */
        .line-number {{
            width: 26px;
            height: 26px;
            display: flex;
            align-items: center;
            justify-content: center;
            border-radius: 50%;
            font-size: 12px;
            font-weight: 600;
            margin-right: 8px;
            flex-shrink: 0;
            transition: all 0.2s;
        }}

        /* 文字内容 - 占据尽可能多的空间 */
        .text-content {{
            flex: 1 1 auto;
            font-size: 15px;
            line-height: 1.5;
            word-break: break-word;
            padding: 3px 0;
            min-width: 0; /* 允许收缩 */
            max-width: 100%;
        }}

        /* 语言标签 - 更小 */
        .lang-tag {{
            font-size: 10px;
            padding: 2px 6px;
            border-radius: 10px;
            margin-left: 4px;
            white-space: nowrap;
            font-weight: 600;
            flex-shrink: 0;
        }}

        /* 音频指示器 - 更小 */
        .audio-indicator {{
            width: 20px;
            height: 20px;
            border-radius: 50%;
            background: #00B894;
            color: white;
            display: inline-flex;
            align-items: center;
            justify-content: center;
            font-size: 10px;
            margin-left: 4px;
            flex-shrink: 0;
            animation: pulse 2s infinite;
        }}

        @keyframes pulse {{
            0% {{ box-shadow: 0 0 0 0 rgba(0,184,148,0.4); }}
            70% {{ box-shadow: 0 0 0 3px rgba(0,184,148,0); }}
            100% {{ box-shadow: 0 0 0 0 rgba(0,184,148,0); }}
        }}

        /* ========== 🎯 核心修复：操作按钮组 - 大幅缩小 ========== */
        .action-buttons {{
            display: flex;
            gap: 4px;
            margin-left: 6px;
            flex-shrink: 0;
        }}

        .action-btn {{
            width: 28px;
            height: 28px;
            border: none;
            border-radius: 50%;
            font-size: 14px;
            display: flex;
            align-items: center;
            justify-content: center;
            cursor: pointer;
            transition: all 0.2s;
            flex-shrink: 0;
            padding: 0;
        }}

        .action-btn:active {{
            transform: scale(0.9);
        }}

        .action-btn.added {{
            background: #00B894;
            color: white;
        }}

        .action-btn.grammar.added {{
            background: #FD79A8;
            color: white;
        }}

        /* ========== 只读的行 - 更紧凑 ========== */
        .readonly-line {{
            display: flex;
            align-items: flex-start;
            padding: 8px 10px;
            border-radius: 10px;
            font-size: 14px;
            line-height: 1.5;
            border-left: 3px solid;
            margin-left: 4px;
            cursor: default;
            user-select: none;
        }}

        .readonly-indicator {{
            width: 20px;
            height: 20px;
            display: flex;
            align-items: center;
            justify-content: center;
            color: #999;
            font-size: 12px;
            margin-right: 8px;
            flex-shrink: 0;
        }}

        .readonly-line .text-content {{
            font-size: 14px;
            color: #666;
            font-style: italic;
        }}

        /* ========== 面板 - 手机紧凑 ========== */
        .panel {{
            flex: 1;
            overflow-y: auto;
            padding: 12px;
            border-radius: 16px;
            height: 0;
            transition: all 0.2s;
            -webkit-overflow-scrolling: touch;
        }}

        .panel h2 {{
            font-size: 16px;
            margin-bottom: 12px;
            padding-bottom: 8px;
            border-bottom: 2px solid;
            position: sticky;
            top: 0;
            z-index: 10;
        }}

        .word-card {{
            padding: 12px;
            border-radius: 12px;
            margin-bottom: 8px;
            border-left: 4px solid;
            transition: all 0.2s;
        }}

        .word-card:active {{
            transform: translateX(2px);
        }}

        .word-text {{
            font-size: 15px;
            font-weight: 600;
            margin-bottom: 4px;
        }}

        .word-meta {{
            display: flex;
            gap: 10px;
            font-size: 10px;
            flex-wrap: wrap;
            margin-bottom: 4px;
        }}

        .grammar-note {{
            color: white;
            padding: 6px 8px;
            border-radius: 6px;
            margin-top: 6px;
            font-size: 12px;
        }}

        .empty-message {{
            text-align: center;
            padding: 30px;
            color: #999;
            font-size: 14px;
        }}

        /* ========== 状态栏 - 手机极简 ========== */
        .status-bar {{
            flex-shrink: 0;
            margin-top: 8px;
            padding: 8px 12px;
            border-radius: 30px;
            box-shadow: 0 -2px 8px rgba(0,0,0,0.03);
            display: flex;
            justify-content: space-between;
            align-items: center;
            font-size: 11px;
            border: 1px solid;
            transition: all 0.2s;
            flex-wrap: wrap;
            gap: 4px;
        }}

        .hidden {{
            display: none !important;
        }}

        /* ========== 超小屏幕专用优化 ========== */
        @media (max-width: 400px) {{
            .learning-line {{
                padding: 8px 6px;
            }}

            .line-number {{
                width: 24px;
                height: 24px;
                font-size: 11px;
                margin-right: 6px;
            }}

            .text-content {{
                font-size: 14px;
            }}

            .lang-tag {{
                font-size: 9px;
                padding: 2px 4px;
                margin-left: 3px;
            }}

            .audio-indicator {{
                width: 18px;
                height: 18px;
                font-size: 9px;
                margin-left: 3px;
            }}

            .action-btn {{
                width: 26px;
                height: 26px;
                font-size: 13px;
            }}

            .nav-tab {{
                min-width: 50px;
                padding: 6px 2px;
                font-size: 12px;
            }}

            .nav-tab span {{
                display: none; /* 超小屏隐藏文字，只留图标 */
            }}
        }}

        @media (max-width: 360px) {{
            .action-btn {{
                width: 24px;
                height: 24px;
                font-size: 12px;
            }}

            .line-number {{
                width: 22px;
                height: 22px;
                font-size: 10px;
            }}

            .text-content {{
                font-size: 13px;
            }}
        }}

        /* 安全区域适配 */
        @supports (padding: max(0px)) {{
            .app {{
                padding-left: max(12px, env(safe-area-inset-left));
                padding-right: max(12px, env(safe-area-inset-right));
                padding-bottom: max(12px, env(safe-area-inset-bottom));
            }}
        }}
    </style>
</head>
<body class="theme-purple">
    <div class="app">
        <div class="app-header">
            <h1>
                <span>🧠 艾宾浩斯·学习</span>
                <span class="badge" style="background: #5a4b8c;">🔑 {hashlib.md5(str(total_chapters).encode()).hexdigest()[:6]}</span>
            </h1>

            <!-- 四色主题切换器 -->
            <div class="theme-switcher">
                <button class="theme-btn purple active" onclick="setTheme('purple')" title="经典紫">🟣</button>
                <button class="theme-btn green" onclick="setTheme('green')" title="护眼绿">🟢</button>
                <button class="theme-btn dark" onclick="setTheme('dark')" title="深夜黑">⚫</button>
                <button class="theme-btn sepia" onclick="setTheme('sepia')" title="纸质米黄">🟤</button>
            </div>

            <div class="nav-tabs">
                <button class="nav-tab active" onclick="switchTab('learn')">📚<span>学习</span></button>
                <button class="nav-tab" onclick="switchTab('notebook')">
                    📔<span>单词</span>
                    <span id="notebookBadge" class="badge" style="background: #00B894;">0</span>
                </button>
                <button class="nav-tab" onclick="switchTab('grammar')">
                    🔤<span>语法</span>
                    <span id="grammarBadge" class="badge" style="background: #FD79A8;">0</span>
                </button>
                <button class="nav-tab" onclick="switchTab('review')">
                    🧠<span>复习</span>
                    <span id="reviewBadge" class="badge" style="background: #FF7675;">0</span>
                </button>
            </div>
        </div>

        <!-- 学习面板 -->
        <div id="learnPanel" style="display: flex; flex-direction: column; flex: 1; min-height: 0;">
            <div class="chapter-selector">
                <button class="nav-btn" onclick="prevChapter()">◀</button>
                <select id="chapterSelect" class="chapter-select" onchange="loadChapter(parseInt(this.value))">
                    <option value="">📚 选择章节</option>
                </select>
                <button class="nav-btn" onclick="nextChapter()">▶</button>
            </div>

            <div class="auto-play-bar">
                <button id="playPauseBtn" class="play-control-btn" onclick="togglePlayPause()">▶</button>
                <div class="progress-info">
                    <div style="display: flex; justify-content: space-between;">
                        <span id="currentLineInfo">0/0</span>
                        <span id="chapterInfo">0/{total_chapters}</span>
                    </div>
                    <div class="progress-bar">
                        <div id="progressFill" class="progress-fill" style="width: 0%;"></div>
                    </div>
                </div>
            </div>

            <!-- 滚动容器 -->
            <div id="scrollContainer" class="scroll-container">
                <div id="chapterTitle" class="chapter-title">选择章节</div>
                <div id="linesContainer" class="lines-container"></div>
            </div>
        </div>

        <!-- 单词本面板 -->
        <div id="notebookPanel" class="panel hidden">
            <h2>📔 单词本</h2>
            <div id="wordList"></div>
        </div>

        <!-- 语法本面板 -->
        <div id="grammarPanel" class="panel hidden">
            <h2>🔤 语法本</h2>
            <div id="grammarList"></div>
        </div>

        <!-- 复习面板 -->
        <div id="reviewPanel" class="panel hidden">
            <h2>🧠 复习</h2>
            <div id="reviewList"></div>
        </div>

        <div class="status-bar">
            <span>📚 {total_chapters}章</span>
            <span>🎵 {total_audio}音频</span>
            <span>📝 {total_learnable}句</span>
            <span id="currentTime">{datetime.now().strftime('%H:%M')}</span>
        </div>
    </div>

    <script>
        // ========== 数据 ==========
        const CHAPTERS_DATA = {chapters_json_str};
        const LEARNING_ITEMS = {items_json_str};
        const TOTAL_CHAPTERS = {total_chapters};

        // ========== 全局变量 ==========
        let currentChapter = 0;
        let currentLineIndex = 0;
        let isPlaying = false;
        let playTimer = null;
        let currentAudio = null;

        // ========== 主题切换 ==========
        function setTheme(theme) {{
            document.body.classList.remove('theme-purple', 'theme-green', 'theme-dark', 'theme-sepia');
            document.body.classList.add(`theme-${{theme}}`);

            document.querySelectorAll('.theme-btn').forEach(btn => {{
                btn.classList.remove('active');
            }});
            event.currentTarget.classList.add('active');

            localStorage.setItem('ebbinghaus_theme', theme);
        }}

        function loadTheme() {{
            const savedTheme = localStorage.getItem('ebbinghaus_theme');
            if (savedTheme) {{
                document.body.classList.remove('theme-purple', 'theme-green', 'theme-dark', 'theme-sepia');
                document.body.classList.add(`theme-${{savedTheme}}`);

                document.querySelectorAll('.theme-btn').forEach(btn => {{
                    btn.classList.remove('active');
                    if (btn.classList.contains(savedTheme)) {{
                        btn.classList.add('active');
                    }}
                }});
            }}
        }}

        // ========== 初始化 ==========
        window.addEventListener('load', function() {{
            if (!CHAPTERS_DATA || !Array.isArray(CHAPTERS_DATA) || CHAPTERS_DATA.length === 0) {{
                return;
            }}

            initChapterSelect();
            loadFromStorage();
            loadTheme();

            if (CHAPTERS_DATA.length > 0) {{
                loadChapter(0);
            }}

            updateAllPanels();
        }});

        function initChapterSelect() {{
            const select = document.getElementById('chapterSelect');
            if (!select) return;

            select.innerHTML = '<option value="">📚 选择章节</option>';

            for (let i = 1; i <= CHAPTERS_DATA.length; i++) {{
                const option = document.createElement('option');
                option.value = i - 1;
                const chapter = CHAPTERS_DATA[i-1];
                const learnableCount = chapter ? chapter.filter(line => line.is_learnable).length : 0;
                option.textContent = '第 ' + i + ' 章 (' + learnableCount + '句)';
                select.appendChild(option);
            }}
        }}

        // ========== 本地存储 ==========
        function loadFromStorage() {{
            const saved = localStorage.getItem('ebbinghaus_notebook');
            if (saved) {{
                try {{
                    const notebook = JSON.parse(saved);
                    if (LEARNING_ITEMS) {{
                        LEARNING_ITEMS.forEach(item => {{
                            const savedItem = notebook.find(n => n.id === item.id);
                            if (savedItem) {{
                                item.added_to_notebook = savedItem.added_to_notebook || false;
                                item.added_to_grammar = savedItem.added_to_grammar || false;
                                item.grammar_note = savedItem.grammar_note || '';
                                item.mastery = savedItem.mastery || 0;
                                item.review_count = savedItem.review_count || 0;
                                item.next_review = savedItem.next_review || 0;
                            }}
                        }});
                    }}
                }} catch (e) {{}}
            }}
        }}

        function saveToStorage() {{
            if (!LEARNING_ITEMS) return;
            const notebook = LEARNING_ITEMS.map(item => ({{
                id: item.id,
                added_to_notebook: item.added_to_notebook || false,
                added_to_grammar: item.added_to_grammar || false,
                grammar_note: item.grammar_note || '',
                mastery: item.mastery || 0,
                review_count: item.review_count || 0,
                next_review: item.next_review || 0
            }})).filter(item => item.added_to_notebook || item.added_to_grammar);
            localStorage.setItem('ebbinghaus_notebook', JSON.stringify(notebook));
        }}

        // ========== 章节加载 ==========
        function loadChapter(index) {{
            if (index < 0 || index >= CHAPTERS_DATA.length) return;

            currentChapter = index;
            currentLineIndex = 0;

            const select = document.getElementById('chapterSelect');
            if (select) select.value = index;

            const titleEl = document.getElementById('chapterTitle');
            if (titleEl) titleEl.innerHTML = '第 ' + (index + 1) + ' 章';

            const infoEl = document.getElementById('chapterInfo');
            if (infoEl) infoEl.innerHTML = (index + 1) + '/' + CHAPTERS_DATA.length;

            renderChapter();
            stopPlay();
        }}

        // ========== 渲染章节 ==========
        function renderChapter() {{
            const chapterData = CHAPTERS_DATA[currentChapter];
            if (!chapterData || !Array.isArray(chapterData)) return;

            const container = document.getElementById('linesContainer');
            let html = '';

            chapterData.forEach((line, idx) => {{
                if (!line) return;

                if (line.is_learnable) {{
                    const hasAudio = line.audio && line.audio.length > 100;
                    const learningItem = LEARNING_ITEMS?.find(item => item && item.id === line.id);
                    const isNotebook = learningItem ? learningItem.added_to_notebook : false;
                    const isGrammar = learningItem ? learningItem.added_to_grammar : false;

                    html += '<div id="line-' + line.id + '" class="learning-line" onclick="playThisLine(\\'' + line.id + '\\', ' + idx + ')">';
                    html += '<span class="line-number">' + (idx + 1) + '</span>';
                    html += '<div class="text-content">' + (line.content || '').replace(/\\[(ch|jp|en)\\]\\s*/g, '').replace(/\\[sound:[^\\]]*\\]/g, '') + '</div>';

                    let langClass = '', langText = '';
                    if (line.lang === 'ch') {{ langClass = 'ch'; langText = '中'; }}
                    else if (line.lang === 'jp') {{ langClass = 'jp'; langText = '日'; }}
                    else if (line.lang === 'en') {{ langClass = 'en'; langText = '英'; }}
                    html += '<span class="lang-tag ' + langClass + '">' + langText + '</span>';

                    if (hasAudio) {{
                        html += '<span class="audio-indicator">🔊</span>';
                    }}

                    html += '<div class="action-buttons">';
                    html += '<button class="action-btn' + (isNotebook ? ' added' : '') + '" onclick="event.stopPropagation(); toggleNotebook(\\'' + line.id + '\\', this)">' + (isNotebook ? '✓' : '📚') + '</button>';
                    html += '<button class="action-btn grammar' + (isGrammar ? ' added' : '') + '" onclick="event.stopPropagation(); addGrammar(\\'' + line.id + '\\', this)">🔤</button>';
                    html += '</div>';
                    html += '</div>';

                }} else {{
                    html += '<div class="readonly-line">';
                    html += '<span class="readonly-indicator">📄</span>';
                    html += '<div class="text-content">' + (line.content || '').replace(/\\[(ch|jp|en)\\]\\s*/g, '').replace(/\\[sound:[^\\]]*\\]/g, '') + '</div>';
                    html += '</div>';
                }}
            }});

            container.innerHTML = html;
            updateProgress();

            const scrollContainer = document.getElementById('scrollContainer');
            if (scrollContainer) scrollContainer.scrollTop = 0;
        }}

        // ========== 滚动居中 ==========
        function scrollToCenter(element) {{
            const container = document.getElementById('scrollContainer');
            if (!container || !element) return;

            const containerHeight = container.clientHeight;
            const elementTop = element.offsetTop;
            const elementHeight = element.clientHeight;

            container.scrollTo({{
                top: Math.max(0, elementTop - (containerHeight / 2) + (elementHeight / 2)),
                behavior: 'smooth'
            }});
        }}

        // ========== 播放指定行 ==========
        function playThisLine(lineId, index) {{
            const chapterData = CHAPTERS_DATA[currentChapter];
            if (!chapterData || index >= chapterData.length) return;

            const line = chapterData[index];
            if (!line.is_learnable) return;

            currentLineIndex = index;

            document.querySelectorAll('.learning-line').forEach(el => el.classList.remove('active'));

            const currentLine = document.getElementById('line-' + lineId);
            if (currentLine) {{
                currentLine.classList.add('active');
                scrollToCenter(currentLine);
            }}

            if (line && line.audio && line.audio.length > 100) {{
                playAudio(line.audio);
            }}

            updateProgress();
        }}

        // ========== 音频播放 ==========
        function playAudio(audioData) {{
            return new Promise((resolve, reject) => {{
                try {{
                    if (!audioData || audioData.length < 100) {{
                        reject();
                        return;
                    }}

                    let audioSrc = audioData;
                    if (!audioSrc.startsWith('data:audio')) {{
                        audioSrc = 'data:audio/mp3;base64,' + audioSrc;
                    }}

                    if (currentAudio) currentAudio.pause();

                    currentAudio = new Audio(audioSrc);
                    currentAudio.play().catch(() => {{}});
                    currentAudio.onended = () => {{
                        currentAudio = null;
                        resolve();
                    }};
                }} catch (e) {{
                    reject(e);
                }}
            }});
        }}

        // ========== 自动播放 ==========
        function togglePlayPause() {{
            const btn = document.getElementById('playPauseBtn');
            if (isPlaying) {{
                stopPlay();
                btn.innerHTML = '▶';
                btn.classList.remove('playing');
            }} else {{
                startPlay();
                btn.innerHTML = '⏸';
                btn.classList.add('playing');
            }}
        }}

        function startPlay() {{
            if (isPlaying || !CHAPTERS_DATA[currentChapter]) return;
            isPlaying = true;
            playCurrentLine();
        }}

        function stopPlay() {{
            isPlaying = false;
            if (playTimer) clearTimeout(playTimer);
            if (currentAudio) {{
                currentAudio.pause();
                currentAudio = null;
            }}
            const btn = document.getElementById('playPauseBtn');
            if (btn) {{
                btn.innerHTML = '▶';
                btn.classList.remove('playing');
            }}
        }}

        async function playCurrentLine() {{
            if (!isPlaying) return;

            const chapterData = CHAPTERS_DATA[currentChapter];
            if (!chapterData) return;

            while (currentLineIndex < chapterData.length && !chapterData[currentLineIndex].is_learnable) {{
                currentLineIndex++;
            }}

            if (currentLineIndex >= chapterData.length) {{
                if (currentChapter + 1 < CHAPTERS_DATA.length) {{
                    loadChapter(currentChapter + 1);
                    currentLineIndex = 0;
                    setTimeout(() => playCurrentLine(), 800);
                }} else {{
                    stopPlay();
                }}
                return;
            }}

            const line = chapterData[currentLineIndex];
            const lineId = line.id;

            document.querySelectorAll('.learning-line').forEach(el => el.classList.remove('active'));
            const currentLine = document.getElementById('line-' + lineId);
            if (currentLine) {{
                currentLine.classList.add('active');
                scrollToCenter(currentLine);
            }}

            if (line && line.audio && line.audio.length > 100) {{
                try {{
                    await playAudio(line.audio);
                    currentLineIndex++;
                    updateProgress();
                    if (isPlaying) playTimer = setTimeout(() => playCurrentLine(), 300);
                }} catch (e) {{
                    currentLineIndex++;
                    updateProgress();
                    if (isPlaying) playTimer = setTimeout(() => playCurrentLine(), 300);
                }}
            }} else {{
                currentLineIndex++;
                updateProgress();
                if (isPlaying) playTimer = setTimeout(() => playCurrentLine(), 300);
            }}
        }}

        function updateProgress() {{
            const chapterData = CHAPTERS_DATA[currentChapter];
            if (!chapterData) return;

            const learnableCount = chapterData.filter(line => line.is_learnable).length;
            const currentLearnableIndex = chapterData.slice(0, currentLineIndex).filter(line => line.is_learnable).length;

            const progress = learnableCount > 0 ? (currentLearnableIndex / learnableCount) * 100 : 0;
            const fill = document.getElementById('progressFill');
            if (fill) fill.style.width = progress + '%';

            const lineInfo = document.getElementById('currentLineInfo');
            if (lineInfo) lineInfo.innerHTML = (currentLearnableIndex + 1) + '/' + learnableCount;
        }}

        // ========== 单词本操作 ==========
        function toggleNotebook(itemId, btn) {{
            const item = LEARNING_ITEMS?.find(i => i.id === itemId);
            if (item) {{
                item.added_to_notebook = !item.added_to_notebook;
                btn.innerHTML = item.added_to_notebook ? '✓' : '📚';
                btn.classList.toggle('added');
                saveToStorage();
                updateNotebookPanel();
                updateBadges();
            }}
        }}

        function addGrammar(itemId, btn) {{
            const item = LEARNING_ITEMS?.find(i => i.id === itemId);
            if (item) {{
                const note = prompt('笔记:', item.grammar_note || '');
                if (note !== null) {{
                    item.added_to_grammar = true;
                    item.grammar_note = note;
                    btn.classList.add('added');
                    saveToStorage();
                    updateGrammarPanel();
                    updateBadges();
                }}
            }}
        }}

        // ========== 面板更新 ==========
        function updateNotebookPanel() {{
            const notebook = LEARNING_ITEMS?.filter(item => item && item.added_to_notebook) || [];
            const list = document.getElementById('wordList');

            let html = '';
            notebook.forEach(item => {{
                html += '<div class="word-card">';
                html += '<div class="word-text">' + (item.content || '').replace(/\\[(ch|jp|en)\\]\\s*/g, '').replace(/\\[sound:[^\\]]*\\]/g, '') + '</div>';
                html += '<div class="word-meta">';
                html += '<span>📌 ' + (item.chapter || 0) + '章</span>';
                html += '<span>🎯 ' + (item.mastery || 0) + '%</span>';
                html += '</div>';
                html += '<button class="action-btn" style="background: #FF7675; color: white;" onclick="toggleNotebook(\\'' + item.id + '\\', this)">🗑️</button>';
                html += '</div>';
            }});

            if (list) list.innerHTML = html || '<div class="empty-message">📭 空</div>';
            document.getElementById('notebookBadge').innerHTML = notebook.length;
        }}

        function updateGrammarPanel() {{
            const grammar = LEARNING_ITEMS?.filter(item => item && item.added_to_grammar) || [];
            const list = document.getElementById('grammarList');

            let html = '';
            grammar.forEach(item => {{
                html += '<div class="word-card grammar-card">';
                html += '<div class="word-text">' + (item.content || '').replace(/\\[(ch|jp|en)\\]\\s*/g, '').replace(/\\[sound:[^\\]]*\\]/g, '') + '</div>';
                html += '<div class="grammar-note">📝 ' + (item.grammar_note || '') + '</div>';
                html += '<div class="word-meta">📌 ' + (item.chapter || 0) + '章</div>';
                html += '</div>';
            }});

            if (list) list.innerHTML = html || '<div class="empty-message">🔤 空</div>';
            document.getElementById('grammarBadge').innerHTML = grammar.length;
        }}

        function updateBadges() {{
            if (!LEARNING_ITEMS) return;
            document.getElementById('notebookBadge').innerHTML = LEARNING_ITEMS.filter(item => item.added_to_notebook).length || 0;
            document.getElementById('grammarBadge').innerHTML = LEARNING_ITEMS.filter(item => item.added_to_grammar).length || 0;
        }}

        function updateAllPanels() {{
            if (!LEARNING_ITEMS) return;
            updateNotebookPanel();
            updateGrammarPanel();
            updateBadges();
        }}

        // ========== 导航 ==========
        function prevChapter() {{
            if (currentChapter > 0) {{
                loadChapter(currentChapter - 1);
                stopPlay();
            }}
        }}

        function nextChapter() {{
            if (currentChapter + 1 < CHAPTERS_DATA.length) {{
                loadChapter(currentChapter + 1);
                stopPlay();
            }}
        }}

        // ========== 标签切换 ==========
        function switchTab(tab) {{
            document.querySelectorAll('.nav-tab').forEach(t => t.classList.remove('active'));
            event.currentTarget.classList.add('active');

            document.getElementById('learnPanel').classList.toggle('hidden', tab !== 'learn');
            document.getElementById('notebookPanel').classList.toggle('hidden', tab !== 'notebook');
            document.getElementById('grammarPanel').classList.toggle('hidden', tab !== 'grammar');
            document.getElementById('reviewPanel').classList.toggle('hidden', tab !== 'review');

            if (tab === 'notebook') updateNotebookPanel();
            if (tab === 'grammar') updateGrammarPanel();
        }}

        // ========== 全局函数 ==========
        window.switchTab = switchTab;
        window.loadChapter = loadChapter;
        window.prevChapter = prevChapter;
        window.nextChapter = nextChapter;
        window.togglePlayPause = togglePlayPause;
        window.toggleNotebook = toggleNotebook;
        window.addGrammar = addGrammar;
        window.playThisLine = playThisLine;
        window.setTheme = setTheme;
    </script>
</body>
</html>
    '''

    # 写入主HTML文件
    with open(html_path, 'w', encoding='utf-8') as f:
        f.write(html_content)

    file_size = os.path.getsize(html_path) // 1024

    print(f"""
╔══════════════════════════════════════════════════════════════╗
║  ✅ 艾宾浩斯·自动学习系统 生成成功                          ║
╠══════════════════════════════════════════════════════════════╣
║  📁 文件: {os.path.basename(html_path)}
║  📊 总章节: {len(simplified_chapters)} 章
║  📝 可学习语句: {total_learnable} 句
║  📄 只读语句: {total_readonly} 句
║  🎵 有效音频: {total_valid_audio} 个
║  💾 文件大小: {file_size} KB
║
║  📱 手机终极优化完成：
║     • 🎯 按钮大幅缩小 - 从40px → 28px
║     • 📝 文字区域最大化 - flex: 1 1 auto
║     • 🔤 语言标签缩写 - 中文→中, 日语→日, 英语→英
║     • 📊 状态栏精简 - 只保留关键数字
║     • 📏 行高压缩 - 更紧凑的排版
║     • 🖐️ 点击区域优化 - 合适的触控大小
║
║  📱 超小屏(≤400px)二次优化：
║     • 按钮进一步缩小 - 26px
║     • 超小屏(≤360px) - 24px
║     • 导航栏隐藏文字 - 只留图标
║
║  📊 空间分配：
║     • 行号: 26px → 8%
║     • 文字: 75% → 75%
║     • 标签: 20px → 6%
║     • 按钮组: 28px×2 + 4px → 11%
║
╚══════════════════════════════════════════════════════════════╝
    """)

    return html_path

import os

def rename_files_in_folder(folder_path: str, old_pattern: str, new_pattern: str,
                          file_extension: str = ".txt", preview_only: bool = False):
    """
    批量重命名文件夹中的文件

    Args:
        folder_path: 文件夹路径
        old_pattern: 要替换的旧字符串
        new_pattern: 替换成的新字符串
        file_extension: 文件扩展名（默认 .txt）
        preview_only: 是否只预览不执行（True=只预览，False=执行重命名）
    """
    print(f"\n📁 扫描文件夹: {folder_path}")
    print(f"🔍 查找包含 '{old_pattern}' 的 {file_extension} 文件")
    print("-" * 60)

    renamed_count = 0
    skipped_count = 0

    for filename in os.listdir(folder_path):
        if filename.endswith(file_extension) and old_pattern in filename:
            old_path = os.path.join(folder_path, filename)
            new_filename = filename.replace(old_pattern, new_pattern)
            new_path = os.path.join(folder_path, new_filename)

            if preview_only:
                print(f"🔄 将重命名: {filename} → {new_filename}")
                renamed_count += 1
            else:
                try:
                    os.rename(old_path, new_path)
                    print(f"✅ 已重命名: {filename} → {new_filename}")
                    renamed_count += 1
                except Exception as e:
                    print(f"❌ 重命名失败 {filename}: {e}")
                    skipped_count += 1

    print("-" * 60)
    if preview_only:
        print(f"📊 预览: 找到 {renamed_count} 个文件将重命名")
    else:
        print(f"📊 完成: 重命名 {renamed_count} 个文件, 跳过 {skipped_count} 个")

def rename_single_file(file_path: str, new_name: str):
    """
    重命名单个文件

    Args:
        file_path: 原文件完整路径
        new_name: 新文件名（不含路径）
    """
    folder = os.path.dirname(file_path)
    new_path = os.path.join(folder, new_name)

    try:
        os.rename(file_path, new_path)
        print(f"✅ 已重命名: {os.path.basename(file_path)} → {new_name}")
        return new_path
    except Exception as e:
        print(f"❌ 重命名失败: {e}")
        return new_path

def add_prefix_to_files(folder_path: str, prefix: str, file_extension: str = ".txt",
                        exclude_patterns: list = None, preview_only: bool = False):
    """
    给文件添加前缀

    Args:
        folder_path: 文件夹路径
        prefix: 要添加的前缀
        file_extension: 文件扩展名
        exclude_patterns: 排除包含特定模式的文件
        preview_only: 是否只预览
    """
    if exclude_patterns is None:
        exclude_patterns = []

    print(f"\n📁 扫描文件夹: {folder_path}")
    print(f"🔍 给 {file_extension} 文件添加前缀: {prefix}")
    print("-" * 60)

    renamed_count = 0
    skipped_count = 0

    for filename in os.listdir(folder_path):
        if not filename.endswith(file_extension):
            continue

        # 检查是否应该排除
        should_exclude = any(pattern in filename for pattern in exclude_patterns)
        if should_exclude:
            print(f"⏭️ 跳过: {filename} (匹配排除模式)")
            skipped_count += 1
            continue

        # 如果已经以该前缀开头，跳过
        if filename.startswith(prefix):
            print(f"⏭️ 跳过: {filename} (已有该前缀)")
            skipped_count += 1
            continue

        old_path = os.path.join(folder_path, filename)
        new_filename = prefix + filename
        new_path = os.path.join(folder_path, new_filename)

        if preview_only:
            print(f"🔄 将添加前缀: {filename} → {new_filename}")
            renamed_count += 1
        else:
            try:
                os.rename(old_path, new_path)
                print(f"✅ 已添加前缀: {filename} → {new_filename}")
                renamed_count += 1
            except Exception as e:
                print(f"❌ 失败 {filename}: {e}")
                skipped_count += 1

    print("-" * 60)
    print(f"📊 完成: {renamed_count} 个文件添加前缀, {skipped_count} 个跳过")

def add_suffix_to_files(folder_path: str, suffix: str, file_extension: str = ".txt",
                       before_extension: bool = True, preview_only: bool = False):
    """
    给文件添加后缀

    Args:
        folder_path: 文件夹路径
        suffix: 要添加的后缀
        file_extension: 文件扩展名
        before_extension: True=在扩展名前添加, False=在扩展名后添加
        preview_only: 是否只预览
    """
    print(f"\n📁 扫描文件夹: {folder_path}")
    print(f"🔍 给 {file_extension} 文件添加后缀: {suffix}")
    print("-" * 60)

    renamed_count = 0
    skipped_count = 0

    for filename in os.listdir(folder_path):
        if not filename.endswith(file_extension):
            continue

        old_path = os.path.join(folder_path, filename)

        if before_extension:
            # 在扩展名前添加后缀
            name_without_ext = filename[:-len(file_extension)]
            new_filename = name_without_ext + suffix + file_extension
        else:
            # 在扩展名后添加后缀
            new_filename = filename + suffix

        if preview_only:
            print(f"🔄 将添加后缀: {filename} → {new_filename}")
            renamed_count += 1
        else:
            try:
                os.rename(old_path, new_path)
                print(f"✅ 已添加后缀: {filename} → {new_filename}")
                renamed_count += 1
            except Exception as e:
                print(f"❌ 失败 {filename}: {e}")
                skipped_count += 1

    print("-" * 60)
    print(f"📊 完成: {renamed_count} 个文件添加后缀, {skipped_count} 个跳过")

def remove_pattern_from_files(folder_path: str, pattern: str, file_extension: str = ".txt",
                             preview_only: bool = False):
    """
    从文件名中删除指定模式

    Args:
        folder_path: 文件夹路径
        pattern: 要删除的模式
        file_extension: 文件扩展名
        preview_only: 是否只预览
    """
    print(f"\n📁 扫描文件夹: {folder_path}")
    print(f"🔍 从文件名中删除: '{pattern}'")
    print("-" * 60)

    renamed_count = 0
    skipped_count = 0

    for filename in os.listdir(folder_path):
        if not filename.endswith(file_extension) or pattern not in filename:
            continue

        old_path = os.path.join(folder_path, filename)
        new_filename = filename.replace(pattern, "")
        new_path = os.path.join(folder_path, new_filename)

        if preview_only:
            print(f"🔄 将重命名: {filename} → {new_filename}")
            renamed_count += 1
        else:
            try:
                os.rename(old_path, new_path)
                print(f"✅ 已重命名: {filename} → {new_filename}")
                renamed_count += 1
            except Exception as e:
                print(f"❌ 失败 {filename}: {e}")
                skipped_count += 1

    print("-" * 60)
    print(f"📊 完成: 重命名 {renamed_count} 个文件, 跳过 {skipped_count} 个")

def rename_with_regex(folder_path: str, pattern: str, replacement: str,
                      file_extension: str = ".txt", preview_only: bool = False):
    """
    使用正则表达式重命名文件

    Args:
        folder_path: 文件夹路径
        pattern: 正则表达式模式
        replacement: 替换字符串
        file_extension: 文件扩展名
        preview_only: 是否只预览
    """
    import re

    print(f"\n📁 扫描文件夹: {folder_path}")
    print(f"🔍 使用正则表达式替换: '{pattern}' → '{replacement}'")
    print("-" * 60)

    renamed_count = 0
    skipped_count = 0

    for filename in os.listdir(folder_path):
        if not filename.endswith(file_extension):
            continue

        old_path = os.path.join(folder_path, filename)
        name_without_ext = filename[:-len(file_extension)]
        new_name_without_ext = re.sub(pattern, replacement, name_without_ext)
        new_filename = new_name_without_ext + file_extension

        if new_filename == filename:
            continue

        if preview_only:
            print(f"🔄 将重命名: {filename} → {new_filename}")
            renamed_count += 1
        else:
            try:
                os.rename(old_path, os.path.join(folder_path, new_filename))
                print(f"✅ 已重命名: {filename} → {new_filename}")
                renamed_count += 1
            except Exception as e:
                print(f"❌ 失败 {filename}: {e}")
                skipped_count += 1

    print("-" * 60)
    print(f"📊 完成: 重命名 {renamed_count} 个文件, 跳过 {skipped_count} 个")

def rename_files_case(folder_path: str, case: str = "lower", file_extension: str = ".txt",
                     preview_only: bool = False):
    """
    修改文件名的大小写

    Args:
        folder_path: 文件夹路径
        case: 'lower'=小写, 'upper'=大写, 'title'=首字母大写
        file_extension: 文件扩展名
        preview_only: 是否只预览
    """
    print(f"\n📁 扫描文件夹: {folder_path}")
    print(f"🔍 转换文件名大小写: {case}")
    print("-" * 60)

    renamed_count = 0
    skipped_count = 0

    for filename in os.listdir(folder_path):
        if not filename.endswith(file_extension):
            continue

        old_path = os.path.join(folder_path, filename)
        name_without_ext = filename[:-len(file_extension)]

        if case == "lower":
            new_name_without_ext = name_without_ext.lower()
        elif case == "upper":
            new_name_without_ext = name_without_ext.upper()
        elif case == "title":
            new_name_without_ext = name_without_ext.title()
        else:
            new_name_without_ext = name_without_ext

        new_filename = new_name_without_ext + file_extension

        if new_filename == filename:
            skipped_count += 1
            continue

        if preview_only:
            print(f"🔄 将重命名: {filename} → {new_filename}")
            renamed_count += 1
        else:
            try:
                os.rename(old_path, os.path.join(folder_path, new_filename))
                print(f"✅ 已重命名: {filename} → {new_filename}")
                renamed_count += 1
            except Exception as e:
                print(f"❌ 失败 {filename}: {e}")
                skipped_count += 1

    print("-" * 60)
    print(f"📊 完成: 重命名 {renamed_count} 个文件, {skipped_count} 个无需修改")


In [ ]:
# ================== 使用示例 ==================

# main_program.py - 主程序模块（重复部分）


In [ ]:
# ============================================================
# 🗣 Edge TTS


In [ ]:
# ============================================================

voices = [
    'zh-CN-XiaoxiaoNeural',
    'zh-CN-YunxiNeural',
    'zh-CN-YunjianNeural',
    'zh-CN-YunyangNeural',
    'zh-CN-YunxiaNeural',
    'zh-CN-XiaoxuanNeural',
    'zh-CN-XiaoyiNeural'
]

CHINESE_VOICES = [
    "zh-CN-XiaoxiaoNeural",  # 晓晓 - 女声，默认中文语音
    "zh-CN-YunxiNeural",     # 云希 - 男声
    "zh-CN-YunjianNeural",   # 云健 - 男声，新闻风格
    "zh-CN-XiaoyiNeural",    # 晓伊 - 女声
    "zh-CN-YunyangNeural",   # 云扬 - 男声，体育解说风格
]

JAPANESE_VOICES = [
    "ja-JP-NanamiNeural",    # 七海 - 女声
    "ja-JP-KeitaNeural",     # 庆太 - 男声
    #"ja-JP-AoiNeural",       # 葵 - 女声，动漫风格
    #"ja-JP-DaichiNeural",    # 大地 - 男声
    #"ja-JP-MayuNeural",      # 真由 - 女声，温柔风格
]

ENGLISH_VOICES = [
    "en-US-JennyNeural",     # 珍妮 - 女声，美式英语
    "en-US-GuyNeural",       # 盖伊 - 男声，美式英语
    "en-GB-SoniaNeural",     # 索尼亚 - 女声，英式英语
    "en-AU-NatashaNeural",   # 娜塔莎 - 女声，澳洲英语
    "en-US-AnaNeural",       # 安娜 - 女声，美式英语，儿童声线
]


In [ ]:
# ============================================================
# 🗣 章节级语音选择器（同一章节内语音固定，避免跳跃感）


In [ ]:
# ============================================================
_chapter_voice_rng = random.Random()
_chapter_voices_cache = {}  # {chapter_idx: {"ch": voice, "jp": voice, "en": voice}}

def set_chapter_voice_seed(chapter_idx):
    """为当前章节设置语音种子，同一章节内语音固定"""
    global _chapter_voices_cache
    rng = random.Random(chapter_idx * 31337)
    _chapter_voices_cache = {
        "ch": rng.choice(CHINESE_VOICES),
        "jp": rng.choice(JAPANESE_VOICES),
        "en": rng.choice(ENGLISH_VOICES),
    }

def get_chapter_voice(lang_tag):
    """根据语言标签获取当前章节固定的语音"""
    if not _chapter_voices_cache:
        # 未设置种子时回退到随机
        if lang_tag == "jp":
            return random.choice(JAPANESE_VOICES)
        elif lang_tag == "en":
            return random.choice(ENGLISH_VOICES)
        return random.choice(CHINESE_VOICES)
    if lang_tag in ("jp", "ja"):
        return _chapter_voices_cache.get("jp", random.choice(JAPANESE_VOICES))
    elif lang_tag == "en":
        return _chapter_voices_cache.get("en", random.choice(ENGLISH_VOICES))
    return _chapter_voices_cache.get("ch", random.choice(CHINESE_VOICES))

def should_speak_line(line):
    """
    判断这一行是否应该朗读
    规则：
    1. 必须以 [ch]、[jp]、[en] 开头
    2. 不能是空行
    3. 不能是特定的标签列表内容
    """
    line = line.strip()
    if not line:
        return False

    # 检查是否有语音标签 [ch]、[jp]、[en]
    if not re.match(r'^\s*\[(ch|jp|en)\]', line):
        return False

    # 提取标签后的实际内容
    match = re.match(r'^\s*\[(ch|jp|en)\]\s*(.*)', line)
    if not match:
        return False

    content = match.group(2).strip()

    # 定义需要排除的内容列表
    exclude_list = [
        "整句假名",
        "汉字对应假名",
        "分词假名",
        "假名",
        "词性",
        "原型",
        "当前形",
        "当前活用形",
        "词性",
        "中文含义",
        "中文说明",
        "含义",
        "句型",
        "日语词",
        "是否发生日式语义变化",
        "否",
        "是",
        "完整原型",
        "缩略路径",
        "无",
        "使用语境",
        # 可以继续添加其他需要排除的内容
    ]

    # 如果内容在排除列表中，则不朗读
    if content in exclude_list:
        return False

    # 检查是否有实际的可读内容
    return bool(re.search(r'[A-Za-z0-9\u4e00-\u9fff\u3040-\u309F\u30A0-\u30FF\u3000-\u303F]', content))

def parse_line_and_select_voice(line, default_voice="ch"):
    """
    解析文本行并选择对应的语音

    参数:
    line: 要解析的文本行
    default_voice: 默认语音，当没有匹配时使用

    返回:
    (voice, text, lang): 如果解析成功，返回语音、文本和语言标签
    None: 如果解析失败或文本为空
    """
    line = line.strip()
    if not line:
        return None

    # 只处理有语音标签的行
    match = re.match(r'^\s*\[(ch|jp|en)\]\s*(.*)', line)
    if match:
        lang = match.group(1)  # 获取语言标签
        text = match.group(2).strip()  # 获取实际文本

        if text:  # 确保有文本内容
            # 根据标签选择对应的语音（章节级固定）
            voice = get_chapter_voice(lang)
            if lang not in ("ch", "jp", "en"):
                lang = default_voice

            return voice, text, lang

    return None
import re

def remove_spaces_between_cjk(text):
    """
    只删除中文、日文字符之间的空格
    简单直接的方法
    """
    if not text:
        return text

    # 匹配CJK字符（中文汉字、日文假名、标点）
    cjk_char = r'[\u4e00-\u9fff\u3040-\u309f\u30a0-\u30ff\u3000-\u303f]'

    # 匹配：CJK字符 + 一个或多个空格/制表符 + CJK字符
    pattern = f'({cjk_char})\\s+({cjk_char})'

    # 递归删除所有这样的空格
    while True:
        new_text = re.sub(pattern, r'\1\2', text)
        if new_text == text:
            break
        text = new_text

    return text

def collapse_repeated_punctuation(text):
    """
    将连续重复的相同标点符号折叠为一个。
    例如: "___" → "_", "！！！" → "！", "---" → "-"
    但保留 "……"（中文省略号的标准写法是两个…）。
    """
    if not text:
        return text

    # 需要折叠的标点（不含省略号…）
    punct_chars = set(
        '，。、；：！？—―～〜·'
        ',.;:!?-_'
        '→←↑↓'
        '／＼｜＋＝'
        '()（）""""\'\'「」【】[]""（）《》〈〉｛｝〔〕『』<>'
    )

    result = []
    prev_char = None
    repeat_count = 0
    for ch in text:
        if ch == prev_char and ch in punct_chars:
            # 连续重复的标点，跳过
            continue
        elif ch == '…':
            # 省略号特殊处理：允许最多两个（……是标准写法）
            if prev_char == '…':
                repeat_count += 1
                if repeat_count > 2:
                    continue  # 第3个及以后的…跳过
            else:
                repeat_count = 1
            result.append(ch)
            prev_char = ch
            continue
        else:
            repeat_count = 0
        result.append(ch)
        prev_char = ch

    return ''.join(result)

def process_mixed_language_text(text):
    """
    处理中日混合文本，返回需要单独生成音频的片段列表
    每个片段包含文本和对应的语音标签

    返回: List[Tuple[str, str]]  # (text, voice_tag)
    """
    soup = BeautifulSoup(text, "html.parser")
    plain_text = remove_sound_tags(soup.get_text().strip())

    # 使用分析器处理中日混合文本
    segments = text_analyzer.split_to_audio_segments(plain_text)

    # 转换为需要的格式
    audio_segments = []
    for segment_text, language, voice_tag in segments:
        # 折叠连续重复标点
        segment_text = collapse_repeated_punctuation(segment_text)
        audio_segments.append((segment_text, voice_tag))

    return audio_segments

async def generate_mixed_audio_segments(text, output_dir, base_filename):
    """
    为中日混合文本生成多个音频文件

    参数:
    text: 原始文本
    output_dir: 输出目录
    base_filename: 基础文件名

    返回:
    List[Tuple[str, str, str]]: [(音频文件路径, 文本片段, 语音标签)]
    """
    # 获取音频片段
    audio_segments = process_mixed_language_text(text)

    generated_files = []

    for i, (segment_text, voice_tag) in enumerate(audio_segments):
        # 生成音频文件名
        audio_filename = f"{base_filename}_{voice_tag}_{i:02d}.mp3"
        audio_path = os.path.join(output_dir, audio_filename)

        # 检查是否已存在
        if not os.path.exists(audio_path) or os.path.getsize(audio_path) < 1000:
            print(f"    🔊 生成片段 {i+1}/{len(audio_segments)} ({voice_tag}): {audio_filename}")

            # 根据语音标签选择语音（章节级固定）
            voice = get_chapter_voice(voice_tag)

            # 删除CJK字符间的空格（如果是中文或日文）
            if voice_tag in ["ch", "jp"]:
                segment_text = remove_spaces_between_cjk(segment_text)

            # 生成TTS - 需要await
            for attempt in range(1, 4):  # 重试3次
                try:
                    tts = edge_tts.Communicate(segment_text, voice)
                    await tts.save(audio_path)
                    print(f"    ✅ 片段生成成功")
                    break
                except Exception as e:
                    if attempt < 3:
                        print(f"    ⚠️ 片段生成失败（第{attempt}次）: {e}")
                        await asyncio.sleep(1)
                    else:
                        print(f"    ❌ 片段生成失败: {e}")
                        # 创建静默音频作为替代
                        AudioSegment.silent(duration=500).export(audio_path, format="mp3")
        else:
            print(f"    ⏭ 片段已存在: {audio_filename}")

        generated_files.append((audio_path, segment_text, voice_tag))

    return generated_files

async def gen_tts_mixed(text, out, base_filename=None):
    """
    为中日混合文本生成TTS音频
    如果文本是中日混合，将生成多个音频文件并合并
    """
    # 获取纯文本内容
    soup = BeautifulSoup(text, "html.parser")
    plain_text = remove_sound_tags(soup.get_text().strip())

    # 检查是否有语言标签（[ch] 或 [jp]）→ 走混合拆分路径
    if re.match(r'^\s*\[(ch|jp)\]', plain_text):
        # 中日混合文本：生成多个音频片段
        output_dir = os.path.dirname(out)
        if base_filename is None:
            base_filename = os.path.splitext(os.path.basename(out))[0]

        print(f"      🔊 检测到中日混合文本，将生成多个音频片段")

        # 生成所有音频片段（需要await）
        audio_segments = await generate_mixed_audio_segments(
            text, output_dir, base_filename
        )

        # 合并所有音频文件
        if audio_segments:
            combined = AudioSegment.silent(duration=0)

            for audio_path, segment_text, voice_tag in audio_segments:
                try:
                    audio = AudioSegment.from_file(audio_path, format="mp3")
                    # 根据语言添加不同间隔
                    if voice_tag == "ch":
                        combined += audio + AudioSegment.silent(duration=200)  # 中文间短间隔
                    elif voice_tag == "jp":
                        combined += audio + AudioSegment.silent(duration=300)  # 日文间中长间隔
                    else:
                        combined += audio + AudioSegment.silent(duration=400)  # 其他语言长间隔
                except Exception as e:
                    print(f"      ⚠️ 无法读取音频片段 {audio_path}: {e}")
                    # 添加静默段
                    combined += AudioSegment.silent(duration=1000)

            # 导出合并的音频
            combined.export(out, format="mp3")
            print(f"      ✅ 合并音频完成: {os.path.basename(out)}")

            # 可选：统计信息
            ch_count = sum(1 for _, _, tag in audio_segments if tag == 'ch')
            jp_count = sum(1 for _, _, tag in audio_segments if tag == 'jp')
            en_count = sum(1 for _, _, tag in audio_segments if tag == 'en')
            print(f"      统计: 中文片段={ch_count}, 日文片段={jp_count}, 英文片段={en_count}")
        else:
            # 没有生成音频，创建空音频
            AudioSegment.silent(duration=100).export(out, format="mp3")
            print(f"      ⚠️ 未生成音频片段，创建静默音频")
    else:
        # 单一语言文本：使用原来的逻辑
        await gen_tts(text, out)

async def gen_tts(text, out):
    """
    生成TTS音频文件
    只对有语音标签的行进行朗读
    """
    for attempt in range(1, 10):
        try:
            # 获取纯文本内容，移除HTML标签
            soup = BeautifulSoup(text, "html.parser")
            plain_text = remove_sound_tags(soup.get_text().strip())

            # 解析语音标签并选择语音
            result = parse_line_and_select_voice(plain_text, default_voice="ch")

            if result is None:
                # 没有语音标签，创建空音频文件
                AudioSegment.silent(duration=100).export(out, format="mp3")
                return

            voice, actualText, lang = result

            print(f"      🔊 TTS 生成中（{lang}，第 {attempt} 次） → {os.path.basename(out)}")
            print(f"      原始文本: {plain_text[:100]}...")

            # 应用文本处理（区分中文日文模式下，不使用数学符号处理）
            if lang in ["ch", "jp"]:
                original_text = actualText

                # 删除CJK字符之间的空格
                actualText = remove_spaces_between_cjk(actualText)

                if original_text != actualText:
                    removed_count = len(original_text) - len(actualText)
                    print(f"      🧹 删除了 {removed_count} 个CJK字符间的空格")

            # 折叠连续重复标点
            actualText = collapse_repeated_punctuation(actualText)

            tts = edge_tts.Communicate(actualText, voice)
            await tts.save(out)
            print(f"      ✅ TTS 成功 ({lang}) → {os.path.basename(out)}")
            return
        except Exception as e:
            print(f"      ⚠️ TTS 失败（第 {attempt} 次）：{e}, {voice}")
            await asyncio.sleep(2)

    print(f"      ❌ 放弃生成：{os.path.basename(out)}")


In [ ]:
# ============================================================
# 🗣 TTS 模式配置（在 generate_mp3_and_html 中设定）


In [ ]:
# ============================================================

TTS_MODE = "differentiated"  # 默认值，会被 generate_mp3_and_html 覆盖
BATCH_SIZE = 20              # 默认值，会被 generate_mp3_and_html 覆盖

async def gen_tts_read_all(text, out):
    """
    全文朗读模式：朗读所有文字内容，不区分中文日文，不跳过任何内容。
    统一使用中文语音朗读。
    """
    for attempt in range(1, 10):
        try:
            # 获取纯文本内容，移除HTML标签
            soup = BeautifulSoup(text, "html.parser")
            plain_text = remove_sound_tags(soup.get_text().strip())

            if not plain_text:
                AudioSegment.silent(duration=100).export(out, format="mp3")
                return

            # 移除语言标签 [ch] [jp] [en]（只移除标签，保留后面文本）
            plain_text = re.sub(r'\[(ch|jp|en)\]\s*', '', plain_text).strip()

            if not plain_text:
                AudioSegment.silent(duration=100).export(out, format="mp3")
                return

            # 删除CJK字符之间的空格
            actualText = remove_spaces_between_cjk(plain_text)
            # 折叠连续重复标点
            actualText = collapse_repeated_punctuation(actualText)

            # 统一使用中文语音（章节级固定）
            voice = get_chapter_voice("ch")

            print(f"      🔊 全文朗读 TTS 生成中（第 {attempt} 次） → {os.path.basename(out)}")

            tts = edge_tts.Communicate(actualText, voice)
            await tts.save(out)
            print(f"      ✅ TTS 成功（全文朗读） → {os.path.basename(out)}")
            return
        except Exception as e:
            print(f"      ⚠️ TTS 失败（第 {attempt} 次）：{e}")
            await asyncio.sleep(2)

    print(f"      ❌ 放弃生成：{os.path.basename(out)}")

async def gen_tts_read_all_smart(text, out, base_filename=None):
    """
    智能全文朗读模式：与 gen_tts_mixed 逻辑完全一致（标点拆分→逐块检测语言→
    分别用对应语音朗读），唯一区别是不要求语言标签——无标签的文本自动加 [ch] 前缀
    让分析器正常工作。
    """
    soup = BeautifulSoup(text, "html.parser")
    plain_text = remove_sound_tags(soup.get_text().strip())

    if not plain_text:
        AudioSegment.silent(duration=100).export(out, format="mp3")
        return

    # 无语言标签时，加 [ch] 前缀让 process_mixed_language_text 正常处理
    if not re.match(r'^\s*\[(ch|jp|en)\]', plain_text):
        text_for_process = f'[ch] {plain_text}'
    else:
        text_for_process = plain_text

    # 直接复用 gen_tts_mixed 的完整流程（标点拆分 + 语言检测 + 分片生成 + 合并）
    await gen_tts_mixed(text_for_process, out, base_filename)

async def gen_tts_tag_trust(text, out, base_filename=None):
    """
    标签信任模式：
    - [jp] 开头 → 整行当日语朗读（信任标签，不拆分）
    - [ch] 开头 → 按标点拆分检测语言（与 gen_tts_mixed 一样）
    - [en] 开头 → 整行当英语朗读
    """
    soup = BeautifulSoup(text, "html.parser")
    plain_text = remove_sound_tags(soup.get_text().strip())

    if not plain_text:
        AudioSegment.silent(duration=100).export(out, format="mp3")
        return

    # 检测标签
    tag_match = re.match(r'^\s*\[(ch|jp|en)\]\s*', plain_text)
    if not tag_match:
        # 无标签 → 不朗读
        AudioSegment.silent(duration=100).export(out, format="mp3")
        return

    tag = tag_match.group(1)
    content = plain_text[tag_match.end():].strip()

    if not content:
        AudioSegment.silent(duration=100).export(out, format="mp3")
        return

    if tag == "jp":
        # [jp] → 整行当日语朗读，不拆分
        actualText = collapse_repeated_punctuation(remove_spaces_between_cjk(content))
        voice = get_chapter_voice("jp")
        for attempt in range(1, 10):
            try:
                tts = edge_tts.Communicate(actualText, voice)
                await tts.save(out)
                print(f"      ✅ TTS 成功（标签信任 jp） → {os.path.basename(out)}")
                return
            except Exception as e:
                print(f"      ⚠️ TTS 失败（第 {attempt} 次）：{e}")
                await asyncio.sleep(2)
        print(f"      ❌ 放弃生成：{os.path.basename(out)}")

    elif tag == "en":
        # [en] → 整行当英语朗读
        actualText = collapse_repeated_punctuation(content)
        voice = get_chapter_voice("en")
        for attempt in range(1, 10):
            try:
                tts = edge_tts.Communicate(actualText, voice)
                await tts.save(out)
                print(f"      ✅ TTS 成功（标签信任 en） → {os.path.basename(out)}")
                return
            except Exception as e:
                print(f"      ⚠️ TTS 失败（第 {attempt} 次）：{e}")
                await asyncio.sleep(2)
        print(f"      ❌ 放弃生成：{os.path.basename(out)}")

    else:
        # [ch] → 按标点拆分检测语言（复用 gen_tts_mixed 的完整流程）
        await gen_tts_mixed(text, out, base_filename)

def mp3_b64(path):
    with open(path, "rb") as f:
        return "data:audio/mp3;base64," + base64.b64encode(f.read()).decode()


In [ ]:
# ============================================================
# 📖 处理单个 TXT


In [ ]:
# ============================================================

def convert_br_to_hr_in_html(html_content):
    """
    将HTML中的连续<br>标签转换为包含<hr>的结构
    例如：<br><br> -> <br><hr><br>
          <br><br><br> -> <br><hr><br>
    """
    if not html_content:
        return html_content

    # 方法1：使用正则表达式替换
    # 匹配两个或多个连续的<br>标签（支持各种变体）
    pattern = r'(<br\s*/?>)\s*(<br\s*/?>)+'

    def replacement(match):
        # 获取第一个<br>的确切形式
        first_br = re.search(r'<br\s*/?>', match.group(0)).group(0)
        return f'{first_br}<hr><br>'

    result = re.sub(pattern, replacement, html_content, flags=re.IGNORECASE)

    # 方法2：处理特定情况
    # 也可以先规范化所有<br>标签，然后替换
    normalized = re.sub(r'<br\s*/?>', '<br>', result, flags=re.IGNORECASE)
    result = normalized.replace('<br><br>', '<br><hr><br>')

    return result

def extract_lines_from_html(html_content):
    """
    从HTML中提取行，保留格式
    修复版：正确保留HTML标签和格式
    """
    if not html_content:
        return []

    # 方法1：直接按标签拆分（最简单有效）
    lines = []

    # 处理换行标签 - 将它们作为单独的行
    html_content = html_content.strip()

    # 将连续的换行标签合并
    html_content = re.sub(r'(<br\s*/?>)+', '<br>', html_content, flags=re.IGNORECASE)
    html_content = re.sub(r'(<hr\s*/?>)+', '<hr>', html_content, flags=re.IGNORECASE)

    # 按标签拆分
    i = 0
    n = len(html_content)

    while i < n:
        # 查找标签开始
        if html_content[i] == '<':
            # 找到标签结束
            end_pos = html_content.find('>', i)
            if end_pos != -1:
                tag = html_content[i:end_pos+1]

                # 检查标签类型
                if re.match(r'<br\s*/?>', tag, re.IGNORECASE):
                    # 换行标签 - 如果有前面的内容，先添加
                    if i > 0:
                        prev_text = html_content[:i].strip()
                        if prev_text:
                            lines.append(prev_text)
                    lines.append(tag)
                    html_content = html_content[end_pos+1:]
                    i = 0
                    n = len(html_content)
                elif re.match(r'<hr\s*/?>', tag, re.IGNORECASE):
                    # 水平线标签
                    if i > 0:
                        prev_text = html_content[:i].strip()
                        if prev_text:
                            lines.append(prev_text)
                    lines.append(tag)
                    html_content = html_content[end_pos+1:]
                    i = 0
                    n = len(html_content)
                else:
                    # 其他标签，继续
                    i += 1
            else:
                i += 1
        else:
            i += 1

    # 处理剩余的文本
    if html_content.strip():
        lines.append(html_content.strip())

    # 清理结果
    cleaned_lines = []
    for line in lines:
        # line = line.strip()
        # if line:
        #     cleaned_lines.append(line)
        # else:
        #     # 保留真正的空行（由<br>标签创建）
        #     cleaned_lines.append("")

        if line == "":
            cleaned_lines.append("<hr>")  # 保留空行
        elif line == "<hr>":
            cleaned_lines.append("<hr>")  # 水平线标签
        else:
            # 对于非空行，清理空白但保留HTML
            cleaned_lines.append(line)

    return cleaned_lines

def strip_language_tags_for_display(html_text):
    """
    从 HTML 文本中移除语言标签和 [sound:...] 标记，仅用于最终显示。
    """
    if not html_text:
        return html_text

    # 移除语言标签 [ch]、[jp]、[en]
    result = re.sub(r'(?:^|\G|(?<=>))\s*\[(ch|jp|en)\]\s*', '', html_text)
    # 移除 [sound:...] 标记（包括 URL 格式）
    result = re.sub(r'\[sound:[^\]]*\]', '', result)

    return result.strip() if result.strip() else html_text

def remove_sound_tags(text):
    """
    移除文本中所有 [sound:...] 标记。
    匹配所有格式：
      [sound:xxx.mp3]
      [sound:http://dict.youdao.com/dictvoice?type=0&audio=xxx&le=jap]
      [sound:任意内容]
    """
    if not text:
        return text
    return re.sub(r'\[sound:[^\]]*\]', '', text).strip()

def has_readable_content0(text):
    """判断是否有可朗读内容"""
    text_plain = BeautifulSoup(text, "html.parser").get_text().strip()
    text_plain = remove_sound_tags(text_plain)
    if not text_plain:
        return False
    return bool(re.search(r"[A-Za-z0-9\u4e00-\u9fff]", text_plain))

def has_readable_content(text):
    """判断是否有可朗读内容（含日文）"""
    text_plain = BeautifulSoup(text, "html.parser").get_text().strip()
    text_plain = remove_sound_tags(text_plain)
    if not text_plain:
        return False

    # 检查是否有可读内容：
    # - 中文字符：\u4e00-\u9fff
    # - 英文字母和数字：A-Za-z0-9
    # - 日文假名（平假名和片假名）：\u3040-\u309F \u30A0-\u30FF
    # - 日文汉字：\u4E00-\u9FFF（与中文相同，但包含日文特有汉字）
    # - 日文标点：\u3000-\u303F
    return bool(re.search(r'[A-Za-z0-9\u4e00-\u9fff\u3040-\u309F\u30A0-\u30FF\u3000-\u303F]', text_plain))

from bs4 import BeautifulSoup, NavigableString

def remove_text_before_marker(html, marker="✅ 修正版讲解段落（自然语言版）"):
    soup = BeautifulSoup(html, "html.parser")

    found = False

    for node in soup.descendants:
        if isinstance(node, NavigableString):
            if marker in node:
                # 命中标记：删除标记前的文本，只保留后面的
                new_text = node.split(marker, 1)[1]
                node.replace_with(new_text)
                found = True
            elif not found:
                # 标记之前的所有纯文本 → 清空
                node.replace_with("")

    return str(soup)

def mark_display_only(
    lines,
    markers=("✅ 修正版讲解段落（自然语言版）", "原文纠正版")
):
    """
    markers: iterable[str]
      多个 marker，只对「第一个命中的 marker」生效
    """

    # 🔍 找到第一个命中的 marker 行号
    first_hit_index = None
    for i, line in enumerate(lines):
        text = line.get("text", "")
        if any(m in text for m in markers):
            first_hit_index = i
            break

    # 没命中 → 原样返回
    if first_hit_index is None:
        return lines

    result = []
    for i, line in enumerate(lines):
        new_line = line.copy()

        # 第一个 marker 之前 + marker 行 → 静音
        if i <= first_hit_index:
            new_line["audio"] = ""

        result.append(new_line)

    return result

async def process_table_html(table_html, output_dir, base_name):
    """
    table_html: 含 <table> 的 HTML 字符串
    output_dir: MP3 输出目录
    base_name: 用于生成 MP3 文件名的基准名（例如行索引）
    返回: 合并后的 MP3 base64
    """
    soup = BeautifulSoup(table_html, "html.parser")
    cell_mp3s = []

    # 表格标题
    caption = soup.find("caption")
    if caption and caption.get_text(strip=True):
        cap_text = caption.get_text(strip=True)
        temp_mp3 = os.path.join(output_dir, f"{base_name}_caption.mp3")
        #if not os.path.exists(temp_mp3):
        if not os.path.exists(temp_mp3) or os.path.getsize(temp_mp3) < 1000:
            cap_text = normalize_math_text(cap_text)
            await gen_tts(clean_text_wordOnly(cap_text), temp_mp3)
        cell_mp3s.append(temp_mp3)

    # 所有单元格 <th> 和 <td>
    for cell in soup.find_all(["th", "td"]):
        text = cell.get_text(strip=True)
        if not text:
            continue
        temp_mp3 = os.path.join(output_dir, f"{base_name}_{len(cell_mp3s)}.mp3")
        #if not os.path.exists(temp_mp3):
        if not os.path.exists(temp_mp3) or os.path.getsize(temp_mp3) < 1000:
            text = normalize_math_text(text)
            await gen_tts(clean_text_wordOnly(text), temp_mp3)
        cell_mp3s.append(temp_mp3)

    # 检查 MP3 是否可用，跳过损坏文件
    valid_mp3s = []
    for mp in cell_mp3s:
        try:
            AudioSegment.from_file(mp, format="mp3")
            valid_mp3s.append(mp)
        except CouldntDecodeError:
            print(f"⚠️ 无法读取 MP3，跳过：{mp}")

    # 合并 MP3
    if valid_mp3s:
        final_mp3 = os.path.join(output_dir, f"{base_name}.mp3")
        combined = AudioSegment.silent(duration=0)
        for mp in valid_mp3s:
            combined += AudioSegment.from_file(mp) + AudioSegment.silent(duration=400)
        combined.export(final_mp3, format="mp3")
        return mp3_b64(final_mp3)
    else:
        return ""

import re

def safe_filename(name: str) -> str:
    """
    将文件名中的危险字符统一替换为下划线
    保留：中文、英文、数字、下划线、短横线
    """
    name = name.strip()

    # 把空格统一成下划线
    name = name.replace(" ", "_")

    # 替换 Windows / ffmpeg / 浏览器 不安全字符
    name = re.sub(r'[\\/:*?"<>|()\[\]{}!@#$%^&+=,;\'`~]', "_", name)

    # 多个下划线压缩成一个
    name = re.sub(r'_+', "_", name)

    # 确保文件名不以点结尾
    name = name.rstrip('.')

    return name


In [ ]:
# ============================================================
# 🏗️ HTML 生成函数（需要在外部定义）

# ============================================================


In [ ]:
# ============================================================
# 🔄 增量处理函数


In [ ]:
# ============================================================

def find_last_completed_chapter(html_dir, safe_name, batch_size=None):
    if batch_size is None:
        batch_size = BATCH_SIZE
    """
    查找最后一个已完成的章节
    通过检查HTML文件来判断
    """
    if not os.path.exists(html_dir):
        return 0

    # 查找所有批次HTML文件
    html_files = []
    for f in os.listdir(html_dir):
        if f.startswith(f"{safe_name}_") and f.endswith(".html"):
            try:
                # 提取批次号
                batch_num = int(f.split("_")[-1].split(".")[0])
                html_files.append((batch_num, f))
            except:
                continue

    if not html_files:
        return 0

    # 按批次号排序
    html_files.sort()

    # 获取最大批次号
    max_batch = max(batch_num for batch_num, _ in html_files)

    # 计算已完成的章节数
    return max_batch * batch_size

#     print(f"📄 行数：{len(lines)}")

#     safe_name = safe_filename(name)

#                 audio = mp3_b64(mp3)

#             chapter_lines.append({"text": t, "audio": audio})

#     print(f"📄 行数：{len(lines)}")

#     safe_name = safe_filename(name)

#                 audio = mp3_b64(mp3)

#             chapter_lines.append({"text": t, "audio": audio})

#     print(f"📄 行数：{len(lines)}")

#     safe_name = safe_filename(name)

#                 audio = mp3_b64(mp3)

#             chapter_lines.append({"text": t, "audio": audio})

#     total_batches = (total_chapters + batch_size - 1) // batch_size

#     html_content += f'''
#         </div>

#     print(f"   📋 索引页面: {index_path}")

# 在 process_one_txt 函数中修改，添加 perHtml 目录的创建和处理

async def process_one_txt(txt_path):
    name = os.path.splitext(os.path.basename(txt_path))[0]
    out_dir = os.path.join(ANKI_ROOT, name)
    print(f"\n📘 开始处理 TXT：{os.path.basename(txt_path)}")
    print(f"📂 输出目录：{name}")
    print(f"🗣 TTS 模式：{TTS_MODE}")
    os.makedirs(out_dir, exist_ok=True)

    # ========== HTML 文件 → 单独处理流程 ==========
    if txt_path.lower().endswith(('.html', '.htm')):
        await process_html_file(txt_path, out_dir, name)
        return

    # ========== CSV/TXT 文件处理 ==========
    with open(txt_path, encoding="utf-8-sig") as f:
        lines = [l for l in f if l.strip()]
    print(f"📄 行数：{len(lines)}")
    df = pd.read_csv(
        StringIO("".join(lines)),
        sep="\t",
        header=None,
        skiprows=0,
        engine="python",
        dtype=str,
        on_bad_lines="skip"
    ).fillna("")
    total_chapters = len(df)

    print(f"\n📖 总章节数：{total_chapters:04d}")

    # 创建目录
    html_dir = os.path.join(out_dir, "html")           # 批次HTML目录
    perHtml_dir = os.path.join(out_dir, "perHtml")     # 单句HTML目录
    ebbinghaus_dir = os.path.join(out_dir, "ebbinghaus")
    json_dir = os.path.join(out_dir, "json")

    os.makedirs(html_dir, exist_ok=True)
    os.makedirs(perHtml_dir, exist_ok=True)            # 创建单句HTML目录
    os.makedirs(ebbinghaus_dir, exist_ok=True)
    os.makedirs(json_dir, exist_ok=True)

    safe_name = safe_filename(name)

    # ========== 检查已处理的章节（通过JSON文件） ==========
    existing_chapters = {}
    if os.path.exists(json_dir):
        for f in os.listdir(json_dir):
            if f.startswith(f"chapter_") and f.endswith(".json"):
                try:
                    chap_num = int(f.split('_')[1].split('.')[0])
                    json_path = os.path.join(json_dir, f)
                    with open(json_path, 'r', encoding='utf-8') as jf:
                        chapter_data = json.load(jf)
                        existing_chapters[chap_num] = chapter_data
                        print(f"  ✅ 加载已处理章节 {chap_num:04d}: {json_path}")
                except Exception as e:
                    print(f"  ⚠️ 加载章节文件失败 {f}: {e}")

    print(f"\n📊 已有JSON数据: {len(existing_chapters)} 章 / {total_chapters} 总章")

    # 确定起始章节
    if existing_chapters:
        start_idx = max(existing_chapters.keys()) + 1
        print(f"🔍 发现断点，从第 {start_idx} 章开始继续处理")
    else:
        start_idx = 1
        print(f"🆕 没有找到已处理的JSON文件，从头开始处理")

    # 批次大小
    batch_size = BATCH_SIZE

    # 收集所有章节数据（优先从JSON加载）
    all_chapters = []

    # 1. 先加载已存在的JSON数据
    for chap_num in sorted(existing_chapters.keys()):
        all_chapters.append(existing_chapters[chap_num])

    # 2. 当前批次的数据
    current_batch_chapters = []
    current_batch_num = 0

    # 计算当前应该从哪个批次开始
    if existing_chapters:
        current_batch_num = (start_idx - 1) // batch_size
        # 加载当前批次已有的数据
        for chap_num in range(current_batch_num * batch_size + 1, start_idx):
            if chap_num in existing_chapters:
                current_batch_chapters.append(existing_chapters[chap_num])
        print(f"📦 当前批次已有 {len(current_batch_chapters)} 章")

    # for idx, (_, row) in enumerate(df.iterrows(), start=1):
    #     # 跳过已处理的章节（通过JSON文件判断）
    #     if idx < start_idx:
    #         continue

    #     print(f"\n📖 处理章节 {idx:04d} / {total_chapters:04d}")

    #     # ========== 检查是否已有该章节的JSON文件 ==========
    #     json_path = os.path.join(json_dir, f"chapter_{idx:04d}.json")

    #     if os.path.exists(json_path):
    #         print(f"  ✅ 章节JSON已存在，直接加载: {os.path.basename(json_path)}")
    #         try:
    #             with open(json_path, 'r', encoding='utf-8') as jf:
    #                 chapter_lines = json.load(jf)
    #             print(f"  📊 加载了 {len(chapter_lines)} 行数据")
    #             all_chapters.append(chapter_lines)
    #             current_batch_chapters.append(chapter_lines)

    #             # 检查是否达到批次大小
    #             if len(current_batch_chapters) >= batch_size or idx == total_chapters:
    #                 await process_batch(current_batch_chapters, current_batch_num + 1,
    #                                   out_dir, html_dir, perHtml_dir, ebbinghaus_dir,
    #                                   safe_name, idx, total_chapters)
    #                 current_batch_chapters = []
    #                 current_batch_num += 1
    #             continue
    #         except Exception as e:
    #             print(f"  ⚠️ 加载JSON失败，重新生成: {e}")
    #             # 如果加载失败，重新生成

    #     # ========== 没有JSON文件，需要处理原始数据 ==========
    #     fields = [str(x) for x in row.tolist() if str(x).strip()]
    #     html = "<br>".join(fields)
    #     html = convert_br_to_hr_in_html(html)

    #     sub = os.path.join(out_dir, f"{idx:04d}")
    #     os.makedirs(sub, exist_ok=True)

    #     texts = extract_lines_from_html(html)
    #     total_lines = len(texts)
    #     chapter_lines = []

    #     for i, t in enumerate(texts):
    #         plain_text = BeautifulSoup(t, "html.parser").get_text().strip()
    #         should_speak = should_speak_line(plain_text)
    #         print(f"    行 {i:03d}: plain='{plain_text[:50]}...', should_speak={should_speak}")

    #         if not should_speak:
    #             audio = ""
    #         else:
    #             mp3 = os.path.join(sub, f"{i:03d}.mp3")

    #             if not os.path.exists(mp3) or os.path.getsize(mp3) < 1000:
    #                 print(f"   🔊 生成 MP3 {i+1}/{total_lines}：{i:03d}.mp3")
    #                 if plain_text.startswith('[ch]'):
    #                     base_filename = f"{i:03d}"
    #                     await gen_tts_mixed(t, mp3, base_filename)
    #                 else:
    #                     await gen_tts(t, mp3)
    #             else:
    #                 print(f"   ⏭ 音频已存在：{i:03d}.mp3")

    #             audio = mp3_b64(mp3)

    #         chapter_lines.append({"text": t, "audio": audio})

    #     # 应用静音标记
    #     chapter_lines = mark_display_only(chapter_lines)

    #     # ========== 保存章节数据到JSON文件 ==========
    #     try:
    #         with open(json_path, 'w', encoding='utf-8') as jf:
    #             json.dump(chapter_lines, jf, ensure_ascii=False, indent=2)
    #         print(f"  💾 保存JSON文件: {os.path.basename(json_path)}")
    #     except Exception as e:
    #         print(f"  ❌ 保存JSON失败: {e}")

    #     # 添加到总集合
    #     all_chapters.append(chapter_lines)
    #     current_batch_chapters.append(chapter_lines)

    #     # # 检查是否达到批次大小或最后一章
    #     # if len(current_batch_chapters) >= batch_size or idx == total_chapters:
    #     #     await process_batch(current_batch_chapters, current_batch_num + 1,
    #     #                       out_dir, html_dir, perHtml_dir, ebbinghaus_dir,
    #     #                       safe_name, idx, total_chapters)
    #     #     current_batch_chapters = []
    #     #     current_batch_num += 1

    #     # 检查是否达到批次大小或最后一章
    #     if len(current_batch_chapters) >= batch_size or idx == total_chapters:
    #         await process_batch(
    #             current_batch_chapters,
    #             current_batch_num + 1,
    #             out_dir,
    #             html_dir,
    #             perHtml_dir,      # 添加 perHtml_dir 参数
    #             ebbinghaus_dir,
    #             safe_name,
    #             idx,
    #             total_chapters
    #         )
    #         current_batch_chapters = []
    #         current_batch_num += 1
    for idx, (_, row) in enumerate(df.iterrows(), start=1):
        if idx < start_idx:
            continue

        print(f"\n📖 处理章节 {idx:04d} / {total_chapters:04d}")

        # 为当前章节设置固定语音种子（同章节内语音不变）
        set_chapter_voice_seed(idx)

        # 获取原始内容（假设CSV只有一列，即 result 字段）
        fields = [str(x) for x in row.tolist() if str(x).strip()]
        if not fields:
            print(f"  ⚠️ 章节 {idx:04d} 为空，跳过")
            continue
        full_text = "<br>".join(fields)
        # full_text = fields[0]

        # ------------------ 智能句子分割 ------------------
        def split_sentences(content):
            """根据内容格式返回句子列表（保留原始标签）"""
            # 情况1：包含 <br> 标签 -> 按 <br> 分割
            if re.search(r'<br\s*/?>', content, re.IGNORECASE):
                parts = re.split(r'(<br\s*/?>)', content, flags=re.IGNORECASE)
                sentences = []
                current = ''
                for part in parts:
                    if re.match(r'<br\s*/?>', part, re.IGNORECASE):
                        if current.strip():
                            sentences.append(current.strip())
                        current = ''
                    else:
                        current += part
                if current.strip():
                    sentences.append(current.strip())
                return sentences
            # 情况2：无 <br> 标签 -> 按换行符分割（纯文本）
            else:
                lines = [line.strip() for line in content.split('\n') if line.strip()]
                return lines if lines else [content.strip()]

        sentences = split_sentences(full_text)
        print(f"  📊 本章分割出 {len(sentences)} 个句子")

        # 创建章节子目录（存放MP3）
        sub = os.path.join(out_dir, f"{idx:04d}")
        os.makedirs(sub, exist_ok=True)

        chapter_lines = []
        for i, raw_sentence in enumerate(sentences):
            # 提取纯文本用于TTS判断和生成
            plain_text = remove_sound_tags(BeautifulSoup(raw_sentence, "html.parser").get_text().strip())

            if TTS_MODE in ("read_all", "read_all_smart"):
                # 全文朗读模式：不跳过任何内容
                has_content = bool(re.search(r'[A-Za-z0-9\u4e00-\u9fff\u3040-\u309F\u30A0-\u30FF]', plain_text))
                if not has_content:
                    audio = ""
                    print(f"    ⏭ 句子 {i+1}/{len(sentences)} 无文字内容，跳过")
                else:
                    mp3 = os.path.join(sub, f"{i:03d}.mp3")
                    if not os.path.exists(mp3) or os.path.getsize(mp3) < 1000:
                        if TTS_MODE == "read_all":
                            print(f"   🔊 全文朗读 MP3 {i+1}/{len(sentences)}：{i:03d}.mp3")
                            await gen_tts_read_all(raw_sentence, mp3)
                        else:  # read_all_smart
                            print(f"   🔊 智能全文朗读 MP3 {i+1}/{len(sentences)}：{i:03d}.mp3")
                            base_filename = f"{i:03d}"
                            await gen_tts_read_all_smart(raw_sentence, mp3, base_filename)
                    else:
                        print(f"    ⏭ 音频已存在：{i:03d}.mp3")
                    audio = mp3_b64(mp3)

            elif TTS_MODE == "tag_trust":
                # 标签信任模式：[jp]→整行日语，[ch]→拆分检测，无标签→跳过
                should_speak = should_speak_line(plain_text)
                if not should_speak:
                    audio = ""
                    print(f"    ⏭ 句子 {i+1}/{len(sentences)} 不朗读")
                else:
                    mp3 = os.path.join(sub, f"{i:03d}.mp3")
                    if not os.path.exists(mp3) or os.path.getsize(mp3) < 1000:
                        print(f"   🔊 标签信任 MP3 {i+1}/{len(sentences)}：{i:03d}.mp3")
                        base_filename = f"{i:03d}"
                        await gen_tts_tag_trust(raw_sentence, mp3, base_filename)
                    else:
                        print(f"    ⏭ 音频已存在：{i:03d}.mp3")
                    audio = mp3_b64(mp3)

            else:
                # 区分中文/日文朗读模式（differentiated，默认）
                should_speak = should_speak_line(plain_text)
                if not should_speak:
                    audio = ""
                    print(f"    ⏭ 句子 {i+1}/{len(sentences)} 不朗读 (纯文本: {plain_text[:30]}...)")
                else:
                    mp3 = os.path.join(sub, f"{i:03d}.mp3")
                    if not os.path.exists(mp3) or os.path.getsize(mp3) < 1000:
                        print(f"   🔊 生成 MP3 {i+1}/{len(sentences)}：{i:03d}.mp3")
                        if re.match(r'^\s*\[(ch|jp)\]', plain_text):
                            base_filename = f"{i:03d}"
                            await gen_tts_mixed(raw_sentence, mp3, base_filename)
                        else:
                            await gen_tts(plain_text, mp3)
                    else:
                        print(f"    ⏭ 音频已存在：{i:03d}.mp3")
                    audio = mp3_b64(mp3)

            # 存储原始句子和音频base64
            chapter_lines.append({"text": raw_sentence, "audio": audio})

        # 应用静音标记（如果需要）
        chapter_lines = mark_display_only(chapter_lines)

        # 保存JSON
        json_path = os.path.join(json_dir, f"chapter_{idx:04d}.json")
        with open(json_path, 'w', encoding='utf-8') as jf:
            json.dump(chapter_lines, jf, ensure_ascii=False, indent=2)
        print(f"  💾 保存JSON文件: {os.path.basename(json_path)}")

        # 添加到总集合
        all_chapters.append(chapter_lines)
        current_batch_chapters.append(chapter_lines)

        # 批次处理（与原来相同）
        if len(current_batch_chapters) >= batch_size or idx == total_chapters:
            await process_batch(current_batch_chapters, current_batch_num + 1,
                                out_dir, html_dir, perHtml_dir, ebbinghaus_dir,
                                safe_name, idx, total_chapters)
            current_batch_chapters = []
            current_batch_num += 1
    # ========== 生成完整的艾宾浩斯学习系统（所有章节） ==========
    print("\n" + "="*60)
    print("🧠 正在生成完整的艾宾浩斯记忆曲线学习系统...")
    print("="*60)

    try:
        from datetime import datetime, timedelta
        import hashlib

        # 确保all_chapters包含了所有章节
        if len(all_chapters) < total_chapters:
            print(f"⚠️ 警告：当前只有 {len(all_chapters)} 章，总章节是 {total_chapters} 章")
            print("📚 尝试从JSON文件加载缺失的章节...")

            # 尝试从JSON文件加载缺失的章节
            for chap_num in range(1, total_chapters + 1):
                if chap_num > len(all_chapters):
                    json_path = os.path.join(json_dir, f"chapter_{chap_num:04d}.json")
                    if os.path.exists(json_path):
                        try:
                            with open(json_path, 'r', encoding='utf-8') as jf:
                                chapter_data = json.load(jf)
                                all_chapters.append(chapter_data)
                                print(f"  ✅ 从JSON加载章节 {chap_num:04d}")
                        except Exception as e:
                            print(f"  ⚠️ 加载章节 {chap_num:04d} 失败: {e}")

        # 生成完整的艾宾浩斯学习系统
        ebbinghaus_complete = os.path.join(
            ebbinghaus_dir,
            f"{safe_name}_艾宾浩斯_完整版.html"
        )

        if os.path.exists(ebbinghaus_complete):
            print(f"⏭ 完整的艾宾浩斯学习系统已存在，跳过生成: {os.path.basename(ebbinghaus_complete)}")
        else:
            if 'generate_super_language_learning_system' in globals():
                generate_super_language_learning_system(ebbinghaus_complete, all_chapters)
                print(f"✅ 完整的艾宾浩斯学习系统生成成功!")
                print(f"📁 文件位置: {ebbinghaus_complete}")
                print(f"📊 包含数据: {len(all_chapters)} 章, {sum(len(c) for c in all_chapters)} 个学习点")
            else:
                print(f"⚠️ 未找到艾宾浩斯学习系统生成函数")

    except Exception as e:
        print(f"❌ 完整的艾宾浩斯学习系统生成失败: {e}")
        import traceback
        traceback.print_exc()

    # ========== 生成汇总索引页面 ==========
    try:
        generate_ebbinghaus_index(ebbinghaus_dir, safe_name, total_chapters, batch_size, json_dir)
        print(f"✅ 索引页面生成成功!")
    except Exception as e:
        print(f"❌ 索引页面生成失败: {e}")

    print(f"\n✅ TXT 处理完成：{name}")
    print(f"📂 输出目录：{out_dir}")
    print(f"📁 JSON数据：{json_dir}")
    print(f"📁 阅读器HTML：{html_dir}")
    print(f"🧠 艾宾浩斯系统：{ebbinghaus_dir}")
    print(f"\n📊 生成统计：")
    print(f"   • 总章节数: {total_chapters} 章")
    print(f"   • 已处理JSON: {len(existing_chapters)} 章")
    print(f"   • 新增JSON: {len(all_chapters) - len(existing_chapters)} 章")
    print(f"   • 阅读器批次: {(total_chapters + batch_size - 1) // batch_size} 个")
    print(f"   • 艾宾浩斯批次: {(total_chapters + batch_size - 1) // batch_size} 个")

async def process_batch(batch_chapters, batch_num, out_dir, html_dir, perHtml_dir,
                       ebbinghaus_dir, safe_name, current_idx, total_chapters):
    """
    处理单个批次：
    1. 生成批次阅读器HTML（每batch_size章一个文件）
    2. 生成单句HTML（每句话一个文件到perHtml文件夹）
    3. 生成艾宾浩斯学习系统
    """
    # 1. 生成普通阅读器HTML（批次）
    html_name = os.path.join(
        html_dir,
        f"{safe_name}_{batch_num:02d}.html"
    )

    print(f"\n📝 正在生成阅读器批次 {batch_num:02d}：{html_name}")
    try:
        build_reader_oppo(out_dir, batch_chapters, html_name)
        print(f"✅ 阅读器批次 {batch_num:02d} 生成成功")
    except Exception as e:
        print(f"❌ 阅读器批次 {batch_num:02d} 生成失败: {e}")
        import traceback
        traceback.print_exc()

    # 2. 生成单句HTML（每句话单独一个文件）
    print(f"\n📄 正在生成单句HTML文件到 perHtml 文件夹...")

    # 计算当前批次的起始章节号
    start_chapter = (batch_num - 1) * 10 + 1

    for chap_idx, chapter in enumerate(batch_chapters, 1):
        global_chap_num = start_chapter + chap_idx - 1

        # 为当前章节生成单句HTML
        per_html_name = os.path.join(
            perHtml_dir,
            f"{safe_name}_chapter_{global_chap_num:04d}.html"
        )

        try:
            # 将当前章节（单句话）包装成一个批处理（只有一句话）
            build_reader_oppo(out_dir, [chapter], per_html_name)
            print(f"  ✅ 生成单句HTML: 第{global_chap_num:04d}章 → {os.path.basename(per_html_name)}")
        except Exception as e:
            print(f"  ❌ 生成单句HTML失败 第{global_chap_num:04d}章: {e}")
            import traceback
            traceback.print_exc()

    # 3. 生成艾宾浩斯学习系统（每10章）
    ebbinghaus_name = os.path.join(
        ebbinghaus_dir,
        f"{safe_name}_艾宾浩斯_{batch_num:02d}.html"
    )

    print(f"\n🧠 正在生成艾宾浩斯学习系统批次 {batch_num:02d}...")
    try:
        if os.path.exists(ebbinghaus_name):
            print(f"⏭ 艾宾浩斯学习系统批次 {batch_num:02d} 已存在，跳过")
        else:
            if 'generate_super_language_learning_system' in globals():
                generate_super_language_learning_system(ebbinghaus_name, batch_chapters)
                end_chapter = min(start_chapter + 9, total_chapters)
                print(f"✅ 艾宾浩斯学习系统批次 {batch_num:02d} 生成成功!")
                print(f"   📁 章节: {start_chapter}-{end_chapter}章")
                print(f"   📊 数据: {len(batch_chapters)} 章, {sum(len(c) for c in batch_chapters)} 个学习点")
            else:
                print(f"⚠️ 未找到艾宾浩斯学习系统生成函数")
    except Exception as e:
        print(f"❌ 艾宾浩斯学习系统批次 {batch_num:02d} 生成失败: {e}")
        import traceback
        traceback.print_exc()

def generate_ebbinghaus_index(ebbinghaus_dir, safe_name, total_chapters, batch_size, json_dir):
    """
    生成艾宾浩斯学习系统的索引页面
    """
    import os
    from datetime import datetime

    total_batches = (total_chapters + batch_size - 1) // batch_size

    # 统计JSON文件数量
    json_count = 0
    if os.path.exists(json_dir):
        json_count = len([f for f in os.listdir(json_dir) if f.startswith('chapter_') and f.endswith('.json')])

    html_content = f'''<!DOCTYPE html>
<html lang="zh">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0, maximum-scale=1.0, user-scalable=no">
    <title>🧠 艾宾浩斯学习系统 · 索引</title>
    <style>
        * {{
            margin: 0;
            padding: 0;
            box-sizing: border-box;
            font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, sans-serif;
        }}

        body {{
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            min-height: 100vh;
            padding: 20px;
        }}

        .container {{
            max-width: 900px;
            margin: 0 auto;
            background: white;
            border-radius: 30px;
            padding: 30px;
            box-shadow: 0 20px 40px rgba(0,0,0,0.1);
        }}

        h1 {{
            font-size: 26px;
            color: #6C5CE7;
            margin-bottom: 20px;
            display: flex;
            align-items: center;
            gap: 10px;
        }}

        .info-grid {{
            display: grid;
            grid-template-columns: repeat(auto-fit, minmax(200px, 1fr));
            gap: 15px;
            margin-bottom: 30px;
        }}

        .info-card {{
            background: #F5F6FA;
            border-radius: 16px;
            padding: 20px;
        }}

        .info-label {{
            font-size: 14px;
            color: #666;
            margin-bottom: 8px;
        }}

        .info-value {{
            font-size: 28px;
            font-weight: bold;
            color: #6C5CE7;
        }}

        .info-sub {{
            font-size: 14px;
            color: #999;
            margin-top: 5px;
        }}

        .batch-grid {{
            display: grid;
            grid-template-columns: repeat(auto-fill, minmax(160px, 1fr));
            gap: 15px;
            margin-top: 20px;
        }}

        .batch-card {{
            background: linear-gradient(135deg, #6C5CE7, #5649C0);
            border-radius: 16px;
            padding: 20px;
            color: white;
            text-align: center;
            text-decoration: none;
            transition: all 0.3s;
            display: block;
        }}

        .batch-card:hover {{
            transform: translateY(-5px);
            box-shadow: 0 10px 20px rgba(108,92,231,0.3);
        }}

        .batch-number {{
            font-size: 24px;
            font-weight: bold;
            margin-bottom: 8px;
        }}

        .batch-desc {{
            font-size: 13px;
            opacity: 0.9;
        }}

        .complete-card {{
            background: linear-gradient(135deg, #00B894, #00CEC9);
            margin-top: 20px;
            padding: 25px;
            border-radius: 20px;
            color: white;
            text-decoration: none;
            display: block;
            transition: all 0.3s;
        }}

        .complete-card:hover {{
            transform: translateY(-5px);
            box-shadow: 0 10px 20px rgba(0,184,148,0.3);
        }}

        .json-stats {{
            background: #FFF3E0;
            border-radius: 16px;
            padding: 15px;
            margin-top: 20px;
            display: flex;
            justify-content: space-between;
            align-items: center;
        }}

        .footer {{
            margin-top: 30px;
            text-align: center;
            color: #666;
            font-size: 14px;
            border-top: 1px solid #eee;
            padding-top: 20px;
        }}
    </style>
</head>
<body>
    <div class="container">
        <h1>
            🧠 艾宾浩斯学习系统
            <span style="font-size: 16px; background: #6C5CE7; color: white; padding: 4px 12px; border-radius: 20px;">{safe_name}</span>
        </h1>

        <div class="info-grid">
            <div class="info-card">
                <div class="info-label">📚 总章节</div>
                <div class="info-value">{total_chapters}</div>
                <div class="info-sub">章</div>
            </div>
            <div class="info-card">
                <div class="info-label">📦 批次大小</div>
                <div class="info-value">{batch_size}</div>
                <div class="info-sub">章/批</div>
            </div>
            <div class="info-card">
                <div class="info-label">📊 总批次</div>
                <div class="info-value">{total_batches}</div>
                <div class="info-sub">批</div>
            </div>
            <div class="info-card">
                <div class="info-label">💾 JSON数据</div>
                <div class="info-value">{json_count}</div>
                <div class="info-sub">章</div>
            </div>
        </div>

        <div class="json-stats">
            <span style="display: flex; align-items: center; gap: 10px;">
                <span style="font-size: 24px;">📁</span>
                <span style="font-weight: 600;">章节JSON文件</span>
            </span>
            <span style="background: #6C5CE7; color: white; padding: 6px 16px; border-radius: 30px; font-size: 14px;">
                已处理 {json_count}/{total_chapters} 章
            </span>
        </div>

        <h2 style="margin: 30px 0 15px; color: #2D3436;">📑 批次学习系统</h2>
        <div class="batch-grid">
'''

    # 添加各个批次的链接
    for batch_num in range(1, total_batches + 1):
        start_chapter = (batch_num - 1) * batch_size + 1
        end_chapter = min(batch_num * batch_size, total_chapters)
        chapter_count = end_chapter - start_chapter + 1

        html_content += f'''
            <a href="{safe_name}_艾宾浩斯_{batch_num:02d}.html" class="batch-card">
                <div class="batch-number">{batch_num:02d}</div>
                <div class="batch-desc">第{start_chapter}-{end_chapter}章</div>
                <div class="batch-desc" style="margin-top: 8px;">共{chapter_count}章</div>
            </a>
'''

    html_content += f'''
        </div>

        <h2 style="margin: 30px 0 15px; color: #2D3436;">🎯 完整学习系统</h2>
        <a href="{safe_name}_艾宾浩斯_完整版.html" class="complete-card">
            <div style="display: flex; align-items: center; gap: 20px;">
                <div style="font-size: 40px;">📚</div>
                <div style="flex: 1;">
                    <div style="font-size: 22px; font-weight: bold; margin-bottom: 5px;">完整版 · 全{total_chapters}章</div>
                    <div style="font-size: 15px; opacity: 0.9;">包含所有章节内容，艾宾浩斯记忆曲线完整复习</div>
                </div>
            </div>
        </a>

        <div class="footer">
            <p>💡 点击上方卡片进入对应的学习系统</p>
            <p>📱 每个系统都是独立的，可单独保存到手机</p>
            <p style="margin-top: 10px; color: #999;">⏰ 生成时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}</p>
        </div>
    </div>
</body>
</html>
    '''

    index_path = os.path.join(ebbinghaus_dir, f"{safe_name}_艾宾浩斯_索引.html")
    with open(index_path, 'w', encoding='utf-8') as f:
        f.write(html_content)

    print(f"   📋 索引页面: {index_path}")


In [ ]:
# ============================================================
# 📦 在文件末尾添加辅助函数（用于从HTML加载章节数据）


In [ ]:
# ============================================================

def load_chapters_from_html(html_file):
    """
    从已生成的HTML文件加载章节数据
    这是一个简化版，你需要根据实际HTML结构调整
    """
    try:
        with open(html_file, 'r', encoding='utf-8') as f:
            content = f.read()

        # 这里需要根据你的HTML结构来解析
        # 简单起见，可以先返回空列表
        return []
    except:
        return []

def find_last_completed_chapter(html_dir, safe_name):
    """
    查找最后一个已完成的章节
    """
    if not os.path.exists(html_dir):
        return 0

    html_files = []
    for f in os.listdir(html_dir):
        if f.startswith(f"{safe_name}_") and f.endswith(".html") and "艾宾浩斯" not in f:
            try:
                batch_num = int(f.split("_")[-1].split(".")[0])
                html_files.append(batch_num)
            except:
                continue

    if not html_files:
        return 0

    max_batch = max(html_files)
    return max_batch * 20  # 假设每批20章

def test_mixed_audio_generation():
    """测试混合音频生成"""
    print("\n测试混合音频生成:")
    print("=" * 80)

    test_cases = [
        '[ch] 提示"コマーシャル"是"流れています"这个动作的主体。',
        '[ch] 由动词的"て形"加上补助动词"います"构成。',
        '[ch] 流れています → 流れてる',
        '[ch] 这是一个纯中文句子。',
        '[ch] "コマーシャル"',
    ]

    for i, test_case in enumerate(test_cases, 1):
        print(f"\n测试 {i}: {test_case}")

        # 处理文本
        audio_segments = process_mixed_language_text(test_case)

        print(f"  将生成 {len(audio_segments)} 个音频片段:")
        for j, (segment_text, voice_tag) in enumerate(audio_segments):
            print(f"    片段 {j+1}: [{voice_tag}] {segment_text}")


In [ ]:
# ============================================================
# 🎵 第二步：生成MP3和HTML文件

# ============================================================


In [ ]:
async def generate_mp3_and_html():
    """
    主入口函数：生成 MP3 和 HTML 文件。

    在此处设定：
    1. TTS_MODE — 朗读模式
    2. INPUT_PATH — 输入路径（单个文件 或 目录）
    3. FILE_EXTENSIONS — 处理的文件扩展名
    """

    # ================================================================
    # ⚙️ 在这里修改配置
    # ================================================================

    # 是否先调用大模型解析（第一步）
    #   True  = 从头开始：先调用 main() 用大模型解析句子，再生成 MP3/HTML
    #   False = 已有解析结果，只生成 MP3/HTML（跳过大模型调用）
    RUN_LLM_ANALYSIS = False

    # TTS 朗读模式（四选一）：
    #
    #   "differentiated"  = 区分中文/日文朗读
    #       - 必须有 [ch]/[jp]/[en] 语言标签才会朗读，无标签的行跳过
    #       - 不管标签是 [ch] 还是 [jp]，都按标点拆分后逐块检测语言
    #       - 适合：LLM 解析后带标签的日语学习材料
    #
    #   "tag_trust"       = 标签信任模式
    #       - 必须有语言标签才朗读，无标签跳过
    #       - [jp] 开头 → 信任标签，整行用日语语音朗读（不拆分）
    #       - [en] 开头 → 信任标签，整行用英语语音朗读（不拆分）
    #       - [ch] 开头 → 按标点拆分，逐块检测语言，分别用对应语音朗读
    #       - 适合：日语标签准确但中文标签内含日语片段的材料
    #
    #   "read_all"        = 全文朗读（不区分语言）
    #       - 不需要语言标签，不跳过任何有文字的行
    #       - 所有内容统一使用中文语音朗读（日文假名会被中文TTS乱读）
    #       - 适合：纯中文材料、不需要日文发音准确的快速通读
    #
    #   "read_all_smart"  = 智能全文朗读（推荐）
    #       - 不需要语言标签，不跳过任何有文字的行
    #       - 自动检测每段文字的语言，用对应语音朗读（中文→中文声、日文→日文声）
    #       - 兼顾"不遗漏内容"和"发音正确"
    #       - 适合：中日混合材料、无标签的通用文本
    #
    global TTS_MODE, BATCH_SIZE
    TTS_MODE = "tag_trust"

    # 每个 HTML 文件包含多少章节（每章对应输入文件的一行/一条记录）
    #   数值越大，单个 HTML 文件越长；数值越小，文件越多但加载更快
    BATCH_SIZE = 20

    # 输入路径：可以是单个文件，也可以是目录
    #   - 单个文件：只处理该文件
    #   - 目录：处理该目录根目录下所有匹配扩展名的文件（不递归子目录）
    INPUT_PATH = "/content/drive/MyDrive/ChatGPTAnkiKaraOK/Consultant02.txt"
    # INPUT_PATH = "/content/drive/MyDrive/ChatGPTAnkiKaraOK"  # 处理目录下所有文件

    # 当 INPUT_PATH 为目录时，只处理这些扩展名的文件
    FILE_EXTENSIONS = (".txt", ".csv", ".html", ".htm")

    # ================================================================

    # 第一步（可选）：调用大模型解析
    if RUN_LLM_ANALYSIS:
        print("\n" + "=" * 80)
        print("🤖 第一步：调用大模型解析句子")
        print("=" * 80)
        await main()
        # main() 生成的 anki 导出文件作为第二步的输入
        INPUT_PATH = anki_export_file
        print(f"📄 大模型解析完成，使用导出文件：{INPUT_PATH}")

    # 第二步：生成 MP3 和 HTML
    print("\n" + "=" * 80)
    print("🎵 第二步：生成MP3和HTML文件")
    print("=" * 80)
    print(f"🗣 TTS 模式：{TTS_MODE}")
    print(f"📂 输入路径：{INPUT_PATH}")

    # 收集待处理文件列表
    files_to_process = []

    if os.path.isfile(INPUT_PATH):
        files_to_process.append(INPUT_PATH)
    elif os.path.isdir(INPUT_PATH):
        for fname in sorted(os.listdir(INPUT_PATH)):
            fpath = os.path.join(INPUT_PATH, fname)
            if os.path.isfile(fpath) and fname.lower().endswith(FILE_EXTENSIONS):
                files_to_process.append(fpath)
        print(f"📄 在目录中找到 {len(files_to_process)} 个文件")
    else:
        print(f"❌ 输入路径不存在：{INPUT_PATH}")
        return

    if not files_to_process:
        print("❌ 没有找到可处理的文件")
        return

    for file_idx, file_path in enumerate(files_to_process, 1):
        print(f"\n{'=' * 60}")
        print(f"📘 [{file_idx}/{len(files_to_process)}] 处理文件：{os.path.basename(file_path)}")
        print(f"{'=' * 60}")
        await process_one_txt(file_path)

    print("\n" + "=" * 80)
    print(f"✅ 全部完成！共处理 {len(files_to_process)} 个文件")
    print("=" * 80)

# 入口


In [ ]:
if __name__ == "__main__":
    await generate_mp3_and_html()
